<a href="https://colab.research.google.com/github/nehin17/network-attack-forecasting/blob/frontend_branch/Dataset/CICIDS2017_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# =============================================================================
# CREATE NETWORK FORECASTING PROJECT STRUCTURE
# =============================================================================

import os

PROJECT_ROOT = "/content/drive/MyDrive/Network-Attack-Forecasting"

folders = [
    "notebooks",
    "src",
    "data/raw",
    "data/processed",
    "data/forecasting",
    "models",
    "results",
    "results/metrics",
    "results/figures",
]

for folder in folders:
    os.makedirs(
        os.path.join(PROJECT_ROOT, folder),
        exist_ok=True
    )

print("Project created at:")
print(PROJECT_ROOT)

print("\nStructure:")

for root, dirs, files in os.walk(PROJECT_ROOT):
    level = root.replace(PROJECT_ROOT, "").count(os.sep)
    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")

Project created at:
/content/drive/MyDrive/Network-Attack-Forecasting

Structure:
Network-Attack-Forecasting/
    results/
        metrics/
        figures/
    models/
    notebooks/
    data/
        forecasting/
        raw/
        processed/
    src/


In [6]:
# =============================================================================
# NETWORK ATTACK FORECASTING — DATASET PREPARATION
# CICIDS2017 Generated Labelled Flows / TrafficLabelling
# =============================================================================
#
# PURPOSE
# -------
# This notebook builds the canonical dataset that will later be used for:
#
#   1. Future attack prediction
#      -> Will an attack occur in the next forecast window?
#
#   2. Future attack-type prediction
#      -> What type of attack is expected?
#
#   3. Lead-time prediction
#      -> How long until the attack begins?
#
#   4. Temporal graph modelling
#      -> Which source/destination nodes or edges are likely to be involved?
#
#
# RAW DATA
# --------
# We use the 8 CICIDS2017 TrafficLabelling / GeneratedLabelledFlows CSV files.
#
# These files preserve:
#   - Timestamp
#   - Source IP
#   - Destination IP
#   - Source/Destination ports
#   - Protocol
#   - CICFlowMeter traffic features
#   - Original multiclass attack Label
#
#
# IMPORTANT
# ---------
# We will NOT immediately discard features.
# First we construct a clean canonical flow-level dataset.
#
# Forecasting windows, engineered temporal features, model-specific feature
# selection, scaling, and train/validation/test splitting will be performed
# AFTER the canonical dataset has been validated.
#
# Known raw-data issue:
# The Thursday WebAttacks CSV contains completely empty appended rows.
# These will be removed only after explicitly verifying that they are empty.
#
# =============================================================================


# =============================================================================
# CELL 1 — IMPORT LIBRARIES
# =============================================================================
#
# What we are doing:
# Importing the libraries required for dataset construction and validation.
# No dataset is loaded or modified in this cell.
#

import os
import gc
import glob
import numpy as np
import pandas as pd

from collections import Counter

print("=" * 90)
print("NETWORK FORECASTING — DATASET PREPARATION")
print("=" * 90)

print("\nPandas version :", pd.__version__)
print("NumPy version  :", np.__version__)

print("\n✓ Libraries imported successfully.")

NETWORK FORECASTING — DATASET PREPARATION

Pandas version : 2.2.3
NumPy version  : 2.1.3

✓ Libraries imported successfully.


In [7]:
# ==============================================================================================================
# RUNTIME RECOVERY — REMOUNT GOOGLE DRIVE
# ==============================================================================================================

import os

print("=" * 110)
print("CICIDS2017 — RUNTIME RECOVERY")
print("=" * 110)

# Remove stale Colab Drive mount if one exists
try:
    from google.colab import drive
    drive.flush_and_unmount()
except Exception:
    pass

# If /content/drive exists as a normal directory with stale files,
# rename it rather than deleting anything
if os.path.exists("/content/drive") and not os.path.ismount("/content/drive"):
    if os.listdir("/content/drive"):
        stale_path = "/content/drive_stale"

        if os.path.exists(stale_path):
            import shutil
            shutil.rmtree(stale_path)

        os.rename("/content/drive", stale_path)
        print("✓ Stale mountpoint moved to:", stale_path)

# Mount Drive
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

print("\n✓ Google Drive mounted.")

CICIDS2017 — RUNTIME RECOVERY
Drive not mounted, so nothing to flush and unmount.
✓ Stale mountpoint moved to: /content/drive_stale
Mounted at /content/drive

✓ Google Drive mounted.


In [8]:
# ==============================================================================================================
# VERIFY SAVED CICIDS2017 ARTIFACTS
# ==============================================================================================================

import os
import glob

PROCESSED_DIR = "/content/drive/MyDrive/Network-Attack-Forecasting/data/processed"
TEMPORAL_DIR  = "/content/drive/MyDrive/Network-Attack-Forecasting/data/temporal"

processed_files = sorted(
    glob.glob(os.path.join(PROCESSED_DIR, "*_cleaned.parquet"))
)

print("=" * 110)
print("SAVED CICIDS2017 ARTIFACT CHECK")
print("=" * 110)

print("\nProcessed Parquet files:", len(processed_files))

for i, path in enumerate(processed_files, 1):
    print(f"{i}. {os.path.basename(path)}")

print("\nTemporal artifacts:")

for filename in [
    "segment_metadata.parquet",
    "large_gap_audit.parquet",
    "flow_timeline.parquet"
]:
    path = os.path.join(TEMPORAL_DIR, filename)
    print(f"{'✓' if os.path.exists(path) else '✗'} {filename}")

print("\n" + "=" * 110)

if len(processed_files) == 8:
    print("✓ CLEANED CICIDS2017 DATA RECOVERED.")
    print("✓ NO NEED TO RE-RUN RAW CLEANING.")
else:
    print("❌ Expected 8 cleaned Parquet files but found:", len(processed_files))

print("=" * 110)

SAVED CICIDS2017 ARTIFACT CHECK

Processed Parquet files: 8
1. Friday-WorkingHours-Afternoon-DDos.pcap_ISCX_cleaned.parquet
2. Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX_cleaned.parquet
3. Friday-WorkingHours-Morning.pcap_ISCX_cleaned.parquet
4. Monday-WorkingHours.pcap_ISCX_cleaned.parquet
5. Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX_cleaned.parquet
6. Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX_cleaned.parquet
7. Tuesday-WorkingHours.pcap_ISCX_cleaned.parquet
8. Wednesday-workingHours.pcap_ISCX_cleaned.parquet

Temporal artifacts:
✓ segment_metadata.parquet
✓ large_gap_audit.parquet
✓ flow_timeline.parquet

✓ CLEANED CICIDS2017 DATA RECOVERED.
✓ NO NEED TO RE-RUN RAW CLEANING.


In [9]:
# =============================================================================
# CELL 3 — LOCATE RAW CICIDS2017 TRAFFICLABELLING FILES
# =============================================================================
#
# What we are doing:
# Finding the 8 original CICIDS2017 TrafficLabelling / GeneratedLabelledFlows
# CSV files stored in Google Drive.
#
# WHY:
# These files will be our canonical RAW source because they preserve:
#
#   - timestamps
#   - source/destination IP identities
#   - ports
#   - protocol
#   - original multiclass attack labels
#   - full CICFlowMeter feature information
#
# At this stage we are ONLY locating the files.
# We are NOT combining, cleaning, sorting, or modifying anything.
# =============================================================================


RAW_DIR = "/content/drive/MyDrive/TrafficLabelling "

print("=" * 100)
print("LOCATING CICIDS2017 TRAFFICLABELLING FILES")
print("=" * 100)

print("\nRaw data directory:")
print(repr(RAW_DIR))


# -----------------------------------------------------------------------------
# Check folder exists
# -----------------------------------------------------------------------------

assert os.path.isdir(RAW_DIR), (
    f"\nFolder not found:\n{RAW_DIR}\n\n"
    "Check the exact Google Drive folder name."
)

print("\n✓ Raw data directory exists.")


# -----------------------------------------------------------------------------
# Find CSV files
# -----------------------------------------------------------------------------

raw_files = sorted(
    glob.glob(os.path.join(RAW_DIR, "*.csv"))
)

print(f"\nCSV files found: {len(raw_files)}")


# -----------------------------------------------------------------------------
# Display files
# -----------------------------------------------------------------------------

for i, path in enumerate(raw_files, 1):

    size_mb = os.path.getsize(path) / (1024 ** 2)

    print(
        f"\n{i}. {os.path.basename(path)}"
        f"\n   Size : {size_mb:.2f} MB"
        f"\n   Path : {path}"
    )


# -----------------------------------------------------------------------------
# We expect exactly 8 CICIDS2017 TrafficLabelling files
# -----------------------------------------------------------------------------

print("\n" + "=" * 100)
print("FILE COUNT CHECK")
print("=" * 100)

if len(raw_files) == 8:

    print("\n✓ Exactly 8 CSV files found.")

else:

    print(
        f"\n❌ Expected 8 CSV files but found {len(raw_files)}."
    )

    print(
        "\nDo NOT continue until we understand why."
    )


# -----------------------------------------------------------------------------
# Save aliases used later in the notebook
# -----------------------------------------------------------------------------

glf_files = raw_files

print("\n✓ File paths stored in variable: raw_files")
print("✓ Compatibility alias created: glf_files")

LOCATING CICIDS2017 TRAFFICLABELLING FILES

Raw data directory:
'/content/drive/MyDrive/TrafficLabelling '

✓ Raw data directory exists.

CSV files found: 8

1. Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
   Size : 91.65 MB
   Path : /content/drive/MyDrive/TrafficLabelling /Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv

2. Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
   Size : 97.16 MB
   Path : /content/drive/MyDrive/TrafficLabelling /Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv

3. Friday-WorkingHours-Morning.pcap_ISCX.csv
   Size : 71.89 MB
   Path : /content/drive/MyDrive/TrafficLabelling /Friday-WorkingHours-Morning.pcap_ISCX.csv

4. Monday-WorkingHours.pcap_ISCX.csv
   Size : 256.20 MB
   Path : /content/drive/MyDrive/TrafficLabelling /Monday-WorkingHours.pcap_ISCX.csv

5. Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
   Size : 103.69 MB
   Path : /content/drive/MyDrive/TrafficLabelling /Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX

In [10]:
# =============================================================================
# CELL 4 — VALIDATE RAW DATASET SCHEMA
# =============================================================================
#
# What we are doing:
# Inspecting ONLY the column headers of all 8 CICIDS2017 TrafficLabelling files.
#
# WHY:
# Before combining millions of rows, we need to confirm that:
#
#   1. All 8 files have the same number of columns.
#   2. All 8 files use the same schema.
#   3. The columns required for our forecasting pipeline are present.
#   4. The columns required for future graph modelling are present.
#
# IMPORTANT:
# We are NOT loading the full datasets in this cell.
# nrows=0 reads only the CSV headers, so this is fast and memory-safe.
#
# We also use latin1 because the WebAttacks file contains a known
# non-UTF-8 character in some attack labels.
# =============================================================================


print("=" * 110)
print("CICIDS2017 — RAW SCHEMA VALIDATION")
print("=" * 110)


# -----------------------------------------------------------------------------
# Columns that are absolutely necessary for our final system
# -----------------------------------------------------------------------------

critical_columns = [
    "Flow ID",
    "Source IP",
    "Source Port",
    "Destination IP",
    "Destination Port",
    "Protocol",
    "Timestamp",
    "Label"
]


# -----------------------------------------------------------------------------
# Store schema information for every file
# -----------------------------------------------------------------------------

schemas = {}

for i, path in enumerate(raw_files, 1):

    filename = os.path.basename(path)

    print("\n" + "#" * 110)
    print(f"FILE {i}/8 — {filename}")
    print("#" * 110)

    # Read header only
    header = pd.read_csv(
        path,
        nrows=0,
        encoding="latin1"
    )

    # CICIDS2017 contains leading/trailing spaces in some column names.
    # We normalize ONLY the in-memory header names here.
    cleaned_columns = [
        str(col).strip()
        for col in header.columns
    ]

    schemas[filename] = cleaned_columns

    print(f"\nNumber of columns: {len(cleaned_columns)}")

    print("\nCRITICAL COLUMN CHECK:")

    for col in critical_columns:

        exists = col in cleaned_columns

        print(
            f"{'✓' if exists else '❌'} "
            f"{col:<22} : {exists}"
        )


# =============================================================================
# CHECK WHETHER ALL FILES HAVE THE SAME SCHEMA
# =============================================================================

print("\n\n" + "=" * 110)
print("CROSS-FILE SCHEMA CONSISTENCY")
print("=" * 110)

reference_filename = list(schemas.keys())[0]
reference_schema = schemas[reference_filename]

all_same_schema = all(
    schema == reference_schema
    for schema in schemas.values()
)

print("\nReference file:")
print(reference_filename)

print("\nAll files have identical schema:", all_same_schema)


# =============================================================================
# CHECK COLUMN COUNTS
# =============================================================================

column_counts = {
    filename: len(schema)
    for filename, schema in schemas.items()
}

print("\nColumn counts:")

for filename, count in column_counts.items():

    print(
        f"{filename:<65} {count}"
    )

all_85_columns = all(
    count == 85
    for count in column_counts.values()
)

print("\nAll files contain exactly 85 columns:", all_85_columns)


# =============================================================================
# CHECK CRITICAL COLUMNS ACROSS EVERY FILE
# =============================================================================

all_critical_present = all(

    all(
        col in schema
        for col in critical_columns
    )

    for schema in schemas.values()
)

print(
    "All files contain every critical column:",
    all_critical_present
)


# =============================================================================
# DISPLAY COMPLETE COLUMN LIST ONCE
# =============================================================================

print("\n" + "=" * 110)
print("COMPLETE CICIDS2017 COLUMN LIST")
print("=" * 110)

for i, col in enumerate(reference_schema, 1):

    print(
        f"{i:>2}. {col}"
    )


# =============================================================================
# FINAL RESULT
# =============================================================================

print("\n" + "=" * 110)
print("SCHEMA VALIDATION RESULT")
print("=" * 110)

checks = {

    "8 schemas inspected":
        len(schemas) == 8,

    "Identical schema across all files":
        all_same_schema,

    "Exactly 85 columns per file":
        all_85_columns,

    "All critical columns present":
        all_critical_present
}

for name, passed in checks.items():

    print(
        f"{'✓' if passed else '❌'} "
        f"{name}: {passed}"
    )


if all(checks.values()):

    print(
        "\n✓ RAW SCHEMA VALIDATION PASSED."
    )

    print(
        "The 8 files are structurally compatible for later combination."
    )

else:

    print(
        "\n❌ RAW SCHEMA VALIDATION FAILED."
    )

    print(
        "Do NOT continue to dataset construction until the issue is inspected."
    )

CICIDS2017 — RAW SCHEMA VALIDATION

##############################################################################################################
FILE 1/8 — Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
##############################################################################################################

Number of columns: 85

CRITICAL COLUMN CHECK:
✓ Flow ID                : True
✓ Source IP              : True
✓ Source Port            : True
✓ Destination IP         : True
✓ Destination Port       : True
✓ Protocol               : True
✓ Timestamp              : True
✓ Label                  : True

##############################################################################################################
FILE 2/8 — Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
##############################################################################################################

Number of columns: 85

CRITICAL COLUMN CHECK:
✓ Flow ID                : True
✓ Source IP  

In [11]:
# =============================================================================
# CELL 5 — RAW DATA INTEGRITY AND ATTACK-LABEL AUDIT
# =============================================================================
#
# What we are doing:
# Inspecting the CONTENT of all 8 raw CICIDS2017 TrafficLabelling files before
# performing any cleaning.
#
# We want to determine:
#
#   1. How many raw rows each file contains.
#   2. How many rows are completely empty.
#   3. Whether any NON-EMPTY row has a missing attack label.
#   4. The attack-label distribution in each file.
#   5. The global attack-label distribution across CICIDS2017.
#   6. Whether the known Thursday WebAttacks blank-row issue is exactly what
#      we previously observed.
#
# IMPORTANT:
#   - Nothing is modified.
#   - Nothing is saved.
#   - Empty rows are counted, NOT removed yet.
#   - Labels are inspected in their raw form.
#
# We use encoding="latin1" because the WebAttacks file contains a non-UTF-8
# dash character in its raw attack labels.
# =============================================================================


from collections import Counter

print("=" * 110)
print("CICIDS2017 — RAW DATA INTEGRITY & LABEL AUDIT")
print("=" * 110)


# =============================================================================
# STORAGE FOR RESULTS
# =============================================================================

file_audit_results = []

global_raw_labels = Counter()

global_raw_rows = 0
global_empty_rows = 0
global_real_rows = 0
global_nonempty_missing_labels = 0


# =============================================================================
# AUDIT EACH RAW FILE
# =============================================================================

for i, path in enumerate(raw_files, 1):

    filename = os.path.basename(path)

    print("\n" + "#" * 110)
    print(f"FILE {i}/8 — {filename}")
    print("#" * 110)

    # -------------------------------------------------------------------------
    # Load one raw file
    # -------------------------------------------------------------------------

    df_temp = pd.read_csv(
        path,
        encoding="latin1",
        low_memory=False
    )

    # Normalize column NAMES in memory only.
    # This does NOT modify the original CSV.
    df_temp.columns = (
        df_temp.columns
        .astype(str)
        .str.strip()
    )

    raw_rows = len(df_temp)

    # -------------------------------------------------------------------------
    # Identify completely empty rows
    # -------------------------------------------------------------------------

    empty_mask = df_temp.isna().all(axis=1)

    empty_rows = int(empty_mask.sum())

    nonempty_mask = ~empty_mask

    real_rows = int(nonempty_mask.sum())

    # -------------------------------------------------------------------------
    # Check labels ONLY among non-empty rows
    # -------------------------------------------------------------------------

    missing_label_nonempty = int(
        df_temp.loc[nonempty_mask, "Label"]
        .isna()
        .sum()
    )

    # -------------------------------------------------------------------------
    # Raw label distribution
    #
    # IMPORTANT:
    # We exclude completely empty rows from this distribution.
    # We do NOT normalize the attack names yet.
    # -------------------------------------------------------------------------

    raw_labels = (
        df_temp.loc[
            nonempty_mask & df_temp["Label"].notna(),
            "Label"
        ]
        .astype(str)
        .str.strip()
    )

    label_counts = Counter(
        raw_labels.value_counts().to_dict()
    )

    # -------------------------------------------------------------------------
    # Update global counters
    # -------------------------------------------------------------------------

    global_raw_rows += raw_rows
    global_empty_rows += empty_rows
    global_real_rows += real_rows
    global_nonempty_missing_labels += missing_label_nonempty

    global_raw_labels.update(label_counts)

    # -------------------------------------------------------------------------
    # Print file-level results
    # -------------------------------------------------------------------------

    print(f"\nRaw rows                     : {raw_rows:,}")
    print(f"Completely empty rows        : {empty_rows:,}")
    print(f"Non-empty rows               : {real_rows:,}")
    print(f"Non-empty rows missing Label : {missing_label_nonempty:,}")

    print(f"\nUnique raw labels            : {len(label_counts)}")

    print("\nRAW LABEL DISTRIBUTION:")

    for label, count in label_counts.most_common():

        pct = (
            count / real_rows * 100
            if real_rows > 0
            else 0
        )

        print(
            f"{repr(label):<48}"
            f"{count:>12,}"
            f" ({pct:7.3f}%)"
        )

    # -------------------------------------------------------------------------
    # Store summary
    # -------------------------------------------------------------------------

    file_audit_results.append({

        "File": filename,

        "Raw_Rows": raw_rows,

        "Empty_Rows": empty_rows,

        "Nonempty_Rows": real_rows,

        "Nonempty_Missing_Label":
            missing_label_nonempty,

        "Unique_Raw_Labels":
            len(label_counts)
    })

    # Free memory before loading next large CSV
    del df_temp
    gc.collect()


# =============================================================================
# GLOBAL DATASET SUMMARY
# =============================================================================

print("\n\n" + "=" * 110)
print("GLOBAL RAW DATASET SUMMARY")
print("=" * 110)

print(f"\nFiles                         : {len(raw_files)}")
print(f"Raw rows across files         : {global_raw_rows:,}")
print(f"Completely empty rows         : {global_empty_rows:,}")
print(f"Real / non-empty rows         : {global_real_rows:,}")
print(
    f"Non-empty rows missing Label  : "
    f"{global_nonempty_missing_labels:,}"
)


# =============================================================================
# GLOBAL RAW LABEL DISTRIBUTION
# =============================================================================

print("\n" + "=" * 110)
print("GLOBAL RAW LABEL DISTRIBUTION")
print("=" * 110)

print(
    f"\nUnique raw labels: "
    f"{len(global_raw_labels)}\n"
)

for label, count in global_raw_labels.most_common():

    pct = count / global_real_rows * 100

    print(
        f"{repr(label):<48}"
        f"{count:>12,}"
        f" ({pct:7.3f}%)"
    )


# =============================================================================
# FILE SUMMARY TABLE
# =============================================================================

file_audit_df = pd.DataFrame(
    file_audit_results
)

print("\n" + "=" * 110)
print("FILE-LEVEL AUDIT SUMMARY")
print("=" * 110)

display(file_audit_df)


# =============================================================================
# EMPTY-ROW LOCATION CHECK
# =============================================================================
#
# We expect the known malformed/blank-row issue to occur only in the
# Thursday WebAttacks file.
# =============================================================================

files_with_empty_rows = (
    file_audit_df.loc[
        file_audit_df["Empty_Rows"] > 0,
        ["File", "Empty_Rows"]
    ]
)

print("\n" + "=" * 110)
print("FILES CONTAINING COMPLETELY EMPTY ROWS")
print("=" * 110)

if len(files_with_empty_rows) == 0:

    print("\nNo completely empty rows found.")

else:

    display(files_with_empty_rows)


# =============================================================================
# IMPORTANT EXPECTED COUNTS
# =============================================================================

print("\n" + "=" * 110)
print("EXPECTED DATASET CHECKS")
print("=" * 110)

checks = {

    "Exactly 8 raw files":
        len(raw_files) == 8,

    "Exactly 2,830,743 real flows":
        global_real_rows == 2_830_743,

    "No non-empty row has missing Label":
        global_nonempty_missing_labels == 0,

    "Exactly 15 raw attack/benign labels":
        len(global_raw_labels) == 15,

    "BENIGN label exists":
        "BENIGN" in global_raw_labels,

    "DDoS label exists":
        "DDoS" in global_raw_labels,

    "PortScan label exists":
        "PortScan" in global_raw_labels,

    "Heartbleed label exists":
        "Heartbleed" in global_raw_labels
}


for name, passed in checks.items():

    print(
        f"{'✓' if passed else '❌'} "
        f"{name}: {passed}"
    )


# =============================================================================
# KNOWN WEBATTACKS EMPTY-ROW CHECK
# =============================================================================

webattack_rows = file_audit_df[
    file_audit_df["File"]
    .str.contains(
        "WebAttacks",
        case=False,
        na=False
    )
]

print("\n" + "=" * 110)
print("WEBATTACKS RAW-FILE CHECK")
print("=" * 110)

if len(webattack_rows) == 1:

    web_empty = int(
        webattack_rows.iloc[0]["Empty_Rows"]
    )

    web_real = int(
        webattack_rows.iloc[0]["Nonempty_Rows"]
    )

    print(
        f"\nWebAttacks completely empty rows : "
        f"{web_empty:,}"
    )

    print(
        f"WebAttacks real rows             : "
        f"{web_real:,}"
    )

    print(
        "\nExpected from our previous investigation:"
    )

    print(
        "  Empty rows : 288,602"
    )

    print(
        "  Real rows  : 170,366"
    )

    print(
        "\nMatches previous empty-row finding:",
        web_empty == 288_602
    )

    print(
        "Matches expected real-row count:",
        web_real == 170_366
    )

else:

    print(
        "\n❌ Could not uniquely identify "
        "the WebAttacks file."
    )


# =============================================================================
# FINAL STATUS
# =============================================================================

print("\n" + "=" * 110)
print("RAW INTEGRITY AUDIT STATUS")
print("=" * 110)

if all(checks.values()):

    print(
        "\n✓ CORE RAW-DATA INTEGRITY CHECKS PASSED."
    )

    print(
        "The dataset is ready for controlled cleaning."
    )

else:

    print(
        "\n❌ One or more raw-data checks failed."
    )

    print(
        "Do NOT clean or combine the files yet."
    )

CICIDS2017 — RAW DATA INTEGRITY & LABEL AUDIT

##############################################################################################################
FILE 1/8 — Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
##############################################################################################################

Raw rows                     : 225,745
Completely empty rows        : 0
Non-empty rows               : 225,745
Non-empty rows missing Label : 0

Unique raw labels            : 2

RAW LABEL DISTRIBUTION:
'DDoS'                                               128,027 ( 56.713%)
'BENIGN'                                              97,718 ( 43.287%)

##############################################################################################################
FILE 2/8 — Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
##############################################################################################################

Raw rows                     : 286,

,File,Raw_Rows,Empty_Rows,Nonempty_Rows,Nonempty_Missing_Label,Unique_Raw_Labels
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,225745,0,225745,0,2
1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,286467,0,286467,0,2
2,Friday-WorkingHours-Morning.pcap_ISCX.csv,191033,0,191033,0,2
3,Monday-WorkingHours.pcap_ISCX.csv,529918,0,529918,0,1
4,Thursday-WorkingHours-Afternoon-Infilteration....,288602,0,288602,0,2
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,458968,288602,170366,0,4
6,Tuesday-WorkingHours.pcap_ISCX.csv,445909,0,445909,0,3
7,Wednesday-workingHours.pcap_ISCX.csv,692703,0,692703,0,6



FILES CONTAINING COMPLETELY EMPTY ROWS


,File,Empty_Rows
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,288602



EXPECTED DATASET CHECKS
✓ Exactly 8 raw files: True
✓ Exactly 2,830,743 real flows: True
✓ No non-empty row has missing Label: True
✓ Exactly 15 raw attack/benign labels: True
✓ BENIGN label exists: True
✓ DDoS label exists: True
✓ PortScan label exists: True
✓ Heartbleed label exists: True

WEBATTACKS RAW-FILE CHECK

WebAttacks completely empty rows : 288,602
WebAttacks real rows             : 170,366

Expected from our previous investigation:
  Empty rows : 288,602
  Real rows  : 170,366

Matches previous empty-row finding: True
Matches expected real-row count: True

RAW INTEGRITY AUDIT STATUS

✓ CORE RAW-DATA INTEGRITY CHECKS PASSED.
The dataset is ready for controlled cleaning.


In [12]:
# =============================================================================
# CELL 6 — CLEAN CANONICAL FLOW-LEVEL FILES
# =============================================================================
#
# For EACH of the 8 raw CSV files independently, we:
#
#   1. Load ONE raw CSV file.
#   2. Strip whitespace from column names.
#   3. Remove ONLY rows that are completely empty.
#   4. Normalize malformed Web Attack label names.
#   5. Preserve the multiclass attack label as "Attack_Type".
#   6. Create a binary attack indicator:
#
#          Attack_Label = 0  -> BENIGN
#          Attack_Label = 1  -> any attack
#
#   7. Preserve the source filename for traceability.
#   8. Preserve the original timestamp string.
#   9. Parse timestamps using the mixed CICIDS2017 timestamp formats.
#  10. Verify that every timestamp parsed successfully.
#  11. Save the cleaned file to Google Drive.
#  12. Delete the dataframe from RAM before processing the next file.
#
#
# IMPORTANT DIFFERENCE FROM OLD CELL 6:
#
#
# We DO NOT create canonical_df in this cell.
#
# All original CICFlowMeter features are preserved.
# No model features are selected yet.
# No chronological sorting is performed yet.
# No temporal segments are created yet.
# No forecasting samples are created yet.
#
# =============================================================================


import os
import gc
import pandas as pd
import numpy as np


# =============================================================================
# 1. VERIFY INPUT FILES
# =============================================================================

assert "raw_files" in globals(), (
    "raw_files does not exist. Run the earlier file-location cell first."
)

assert len(raw_files) == 8, (
    f"Expected 8 raw CICIDS2017 files, found {len(raw_files)}."
)


# =============================================================================
# 2. CREATE OUTPUT DIRECTORY
# =============================================================================

PROJECT_ROOT = "/content/drive/MyDrive/Network-Attack-Forecasting"

PROCESSED_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed"
)

os.makedirs(
    PROCESSED_DIR,
    exist_ok=True
)


print("=" * 110)
print("CICIDS2017 — RAM-SAFE CANONICAL FLOW CONSTRUCTION")
print("=" * 110)

print("\nRaw files:")
print(len(raw_files))

print("\nProcessed files will be saved to:")
print(PROCESSED_DIR)


# =============================================================================
# 3. LABEL NORMALIZATION FUNCTION
# =============================================================================
#
# CICIDS2017 contains malformed/non-standard text in some Web Attack labels.
#
# Instead of relying on the exact corrupted characters, we normalize labels
# using their meaningful attack-name components.
#
# Examples ultimately become:
#
#   Web Attack - Brute Force
#   Web Attack - XSS
#   Web Attack - Sql Injection
#
# All other labels are preserved after whitespace cleaning.
#
# =============================================================================

def normalize_attack_label(value):

    if pd.isna(value):
        return value

    label = str(value).strip()

    lower = label.lower()

    # ---------------------------------------------------------
    # Web Attack labels
    # ---------------------------------------------------------

    if "web attack" in lower:

        if "brute force" in lower:
            return "Web Attack - Brute Force"

        if "xss" in lower:
            return "Web Attack - XSS"

        if "sql injection" in lower:
            return "Web Attack - Sql Injection"

    # ---------------------------------------------------------
    # Everything else
    # ---------------------------------------------------------

    return label


# =============================================================================
# 4. ROBUST CSV READER
# =============================================================================
#
# One CICIDS2017 file previously caused UTF-8 decoding problems.
#
# We therefore try UTF-8 first and fall back to Windows-1252.
#
# =============================================================================

def read_cicids_file(path):

    try:

        df = pd.read_csv(
            path,
            encoding="utf-8",
            low_memory=False
        )

        encoding_used = "utf-8"

    except UnicodeDecodeError:

        df = pd.read_csv(
            path,
            encoding="cp1252",
            low_memory=False
        )

        encoding_used = "cp1252"

    return df, encoding_used


# =============================================================================
# 5. STORAGE FOR SMALL AUDIT INFORMATION ONLY
# =============================================================================
#
# IMPORTANT:
#
# We store only tiny summary dictionaries here.
# We NEVER store the full dataframes.
#
# =============================================================================

processing_summary = []

processed_files = []


# =============================================================================
# 6. PROCESS ONE FILE AT A TIME
# =============================================================================

for file_number, path in enumerate(raw_files, start=1):

    filename = os.path.basename(path)

    print("\n\n")
    print("#" * 110)
    print(
        f"PROCESSING FILE {file_number}/{len(raw_files)}"
    )
    print(filename)
    print("#" * 110)


    # =========================================================================
    # 6A. LOAD ONLY THIS FILE
    # =========================================================================

    df, encoding_used = read_cicids_file(path)

    raw_rows = len(df)
    raw_columns = len(df.columns)

    print(f"\nEncoding used : {encoding_used}")
    print(f"Raw rows      : {raw_rows:,}")
    print(f"Raw columns   : {raw_columns}")


    # =========================================================================
    # 6B. CLEAN COLUMN NAMES
    # =========================================================================

    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
    )


    # =========================================================================
    # 6C. VERIFY REQUIRED COLUMNS
    # =========================================================================

    required_columns = [
        "Flow ID",
        "Source IP",
        "Source Port",
        "Destination IP",
        "Destination Port",
        "Protocol",
        "Timestamp",
        "Label"
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in df.columns
    ]

    assert len(missing_columns) == 0, (
        f"{filename} is missing required columns: "
        f"{missing_columns}"
    )


    # =========================================================================
    # 6D. REMOVE ONLY COMPLETELY EMPTY ROWS
    # =========================================================================
    #
    # We DO NOT drop rows merely because one feature is missing.
    #
    # Only rows where EVERY column is NaN are removed.
    #
    # =========================================================================

    rows_before_empty_removal = len(df)

    df = (
        df
        .dropna(
            how="all"
        )
        .reset_index(drop=True)
    )

    empty_rows_removed = (
        rows_before_empty_removal
        - len(df)
    )

    print(
        f"Completely empty rows removed : "
        f"{empty_rows_removed:,}"
    )


    # =========================================================================
    # 6E. NORMALIZE ATTACK LABEL
    # =========================================================================

    df["Attack_Type"] = (
        df["Label"]
        .apply(normalize_attack_label)
    )


    # =========================================================================
    # 6F. CREATE BINARY ATTACK LABEL
    # =========================================================================
    #
    # BENIGN -> 0
    # ANY ATTACK -> 1
    #
    # =========================================================================

    df["Attack_Label"] = (
        df["Attack_Type"]
        .str.upper()
        .ne("BENIGN")
        .astype("int8")
    )


    # =========================================================================
    # 6G. PRESERVE SOURCE FILE
    # =========================================================================

    df["Source_File"] = filename


    # =========================================================================
    # 6H. PRESERVE ORIGINAL TIMESTAMP
    # =========================================================================
    #
    # This lets us audit or reconstruct timestamp parsing later if necessary.
    #
    # =========================================================================

    df["Original_Timestamp"] = (
        df["Timestamp"]
        .astype("string")
        .str.strip()
    )


    # =========================================================================
    # 6I. ROBUST TIMESTAMP PARSING
    # =========================================================================
    #
    # CICIDS2017 contains mixed timestamp representations.
    #
    # Examples include:
    #
    #       7/7/2017 3:30
    #
    # and:
    #
    #       03/07/2017 11:04:03
    #
    # format="mixed" allows pandas to parse both.
    #
    # dayfirst=True is required for this dataset.
    #
    # =========================================================================

    df["Timestamp"] = pd.to_datetime(
        df["Original_Timestamp"],
        format="mixed",
        dayfirst=True,
        errors="coerce"
    )


    # =========================================================================
    # 6J. TIMESTAMP VALIDATION
    # =========================================================================

    failed_timestamps = (
        df["Timestamp"]
        .isna()
        .sum()
    )

    valid_timestamps = (
        df["Timestamp"]
        .notna()
        .sum()
    )

    print(
        f"Valid timestamps              : "
        f"{valid_timestamps:,}"
    )

    print(
        f"Failed timestamps             : "
        f"{failed_timestamps:,}"
    )


    assert failed_timestamps == 0, (
        f"Timestamp parsing failed for "
        f"{failed_timestamps:,} rows in {filename}."
    )


    # =========================================================================
    # 6K. CREATE DATE
    # =========================================================================

    df["Date"] = (
        df["Timestamp"]
        .dt.date
    )


    # =========================================================================
    # 6L. FILE-LEVEL LABEL AUDIT
    # =========================================================================

    attack_count = int(
        df["Attack_Label"]
        .sum()
    )

    benign_count = (
        len(df)
        - attack_count
    )

    unique_attack_types = (
        df["Attack_Type"]
        .nunique()
    )

    start_time = (
        df["Timestamp"]
        .min()
    )

    end_time = (
        df["Timestamp"]
        .max()
    )


    print(f"\nClean rows      : {len(df):,}")
    print(f"Benign flows    : {benign_count:,}")
    print(f"Attack flows    : {attack_count:,}")
    print(f"Unique labels   : {unique_attack_types}")

    print("\nTemporal range:")
    print("Start :", start_time)
    print("End   :", end_time)


    print("\nLabel distribution:")

    print(
        df["Attack_Type"]
        .value_counts()
    )


    # =========================================================================
    # 6M. SAVE CLEANED FILE
    # =========================================================================
    #
    # Parquet is preferable here because:
    #
    #   - it is substantially smaller than CSV
    #   - datetime types are preserved
    #   - numeric dtypes are preserved
    #   - later we can load only selected columns
    #
    # =========================================================================

    safe_filename = (
        os.path.splitext(filename)[0]
        + "_cleaned.parquet"
    )

    output_path = os.path.join(
        PROCESSED_DIR,
        safe_filename
    )


    df.to_parquet(
        output_path,
        index=False,
        compression="snappy"
    )


    processed_files.append(
        output_path
    )


    output_size_mb = (
        os.path.getsize(output_path)
        / 1024**2
    )


    print("\n✓ Saved:")
    print(output_path)

    print(
        f"Saved file size : "
        f"{output_size_mb:.2f} MB"
    )


    # =========================================================================
    # 6N. STORE ONLY SMALL SUMMARY INFORMATION
    # =========================================================================

    processing_summary.append({

        "Source_File":
            filename,

        "Encoding":
            encoding_used,

        "Raw_Rows":
            raw_rows,

        "Clean_Rows":
            len(df),

        "Empty_Rows_Removed":
            empty_rows_removed,

        "Benign_Flows":
            benign_count,

        "Attack_Flows":
            attack_count,

        "Unique_Attack_Types":
            unique_attack_types,

        "Failed_Timestamps":
            failed_timestamps,

        "Start_Time":
            start_time,

        "End_Time":
            end_time,

        "Saved_Size_MB":
            round(
                output_size_mb,
                2
            )
    })


    # =========================================================================
    # 6O. DELETE DATAFRAME FROM RAM
    # =========================================================================
    #
    # THIS IS THE CRITICAL RAM-SAVING STEP.
    #
    # Once the cleaned file has been written to Drive, we do not need the
    # dataframe anymore.
    #
    # =========================================================================

    del df

    gc.collect()

    print(
        "\n✓ Dataframe deleted from RAM "
        "before processing next file."
    )


# =============================================================================
# 7. CREATE SMALL PROCESSING SUMMARY
# =============================================================================

processing_summary_df = pd.DataFrame(
    processing_summary
)


print("\n\n")
print("=" * 110)
print("RAM-SAFE PROCESSING SUMMARY")
print("=" * 110)

display(
    processing_summary_df
)


# =============================================================================
# 8. GLOBAL COUNTS FROM SUMMARIES
# =============================================================================

total_raw_rows = (
    processing_summary_df["Raw_Rows"]
    .sum()
)

total_clean_rows = (
    processing_summary_df["Clean_Rows"]
    .sum()
)

total_empty_removed = (
    processing_summary_df["Empty_Rows_Removed"]
    .sum()
)

total_benign = (
    processing_summary_df["Benign_Flows"]
    .sum()
)

total_attacks = (
    processing_summary_df["Attack_Flows"]
    .sum()
)

total_failed_timestamps = (
    processing_summary_df["Failed_Timestamps"]
    .sum()
)


print("\n" + "=" * 110)
print("GLOBAL CLEANING TOTALS")
print("=" * 110)

print(
    f"\nRaw rows             : "
    f"{total_raw_rows:,}"
)

print(
    f"Clean rows           : "
    f"{total_clean_rows:,}"
)

print(
    f"Empty rows removed   : "
    f"{total_empty_removed:,}"
)

print(
    f"Benign flows         : "
    f"{total_benign:,}"
)

print(
    f"Attack flows         : "
    f"{total_attacks:,}"
)

print(
    f"Failed timestamps    : "
    f"{total_failed_timestamps:,}"
)

print(
    f"Processed files saved: "
    f"{len(processed_files)}"
)


# =============================================================================
# 9. FINAL VALIDATION
# =============================================================================
#
# These values were established by our previous complete audit.
#
# =============================================================================

checks = {

    "Exactly 8 processed files":
        len(processed_files) == 8,

    "Expected total flow count":
        total_clean_rows == 2_830_743,

    "Expected attack flow count":
        total_attacks == 557_646,

    "Expected benign flow count":
        total_benign == 2_273_097,

    "All timestamps parsed":
        total_failed_timestamps == 0,

    "Every processed file exists":
        all(
            os.path.exists(path)
            for path in processed_files
        )
}


print("\n" + "=" * 110)
print("FINAL CELL 6 VALIDATION")
print("=" * 110)

for name, result in checks.items():

    print(
        f"{'✓' if result else '❌'} "
        f"{name}: {result}"
    )


print("\n" + "=" * 110)

if all(checks.values()):

    print("✓ CELL 6 COMPLETED SUCCESSFULLY.")
    print()
    print(
        "All 8 CICIDS2017 files were cleaned "
        "and saved independently."
    )

    print(
        "The full 2.83-million-row dataset was "
        "NEVER held in RAM at once."
    )

    print()
    print(
        "IMPORTANT: canonical_df has intentionally "
        "NOT been created."
    )

    print()
    print(
        "NEXT: Cell 7 will validate the processed "
        "files directly from disk."
    )

else:

    print("❌ CELL 6 VALIDATION FAILED.")
    print("Do NOT continue to Cell 7.")

print("=" * 110)
# We are NOT yet:
#   - selecting model features
#   - scaling features
#   - creating minute-level aggregation
#   - creating forecasting windows
#   - creating lead-time targets
#   - splitting train/validation/test
#
# Those steps come AFTER the canonical flow dataset is validated.
#
# Expected final number of rows:
#
#       2,830,743
#
# =============================================================================


print("=" * 110)
print("BUILDING CLEAN CANONICAL CICIDS2017 FLOW DATASET")
print("=" * 110)


# =============================================================================
# STORAGE FOR CLEANED FILES
# =============================================================================

cleaned_parts = []


# =============================================================================
# PROCESS EACH RAW FILE
# =============================================================================

for i, path in enumerate(raw_files, 1):

    filename = os.path.basename(path)

    print("\n" + "#" * 110)
    print(f"PROCESSING FILE {i}/8")
    print(filename)
    print("#" * 110)


    # -------------------------------------------------------------------------
    # Load raw file
    # -------------------------------------------------------------------------

    df_part = pd.read_csv(
        path,
        encoding="latin1",
        low_memory=False
    )

    rows_before = len(df_part)


    # -------------------------------------------------------------------------
    # Normalize COLUMN NAMES
    # -------------------------------------------------------------------------

    df_part.columns = (
        df_part.columns
        .astype(str)
        .str.strip()
    )


    # -------------------------------------------------------------------------
    # Remove completely empty rows
    #
    # IMPORTANT:
    # We already proved in Cell 5 that the 288,602 problematic WebAttack rows
    # are completely empty.
    #
    # We are NOT deleting rows merely because one feature is missing.
    # -------------------------------------------------------------------------

    empty_mask = df_part.isna().all(axis=1)

    empty_removed = int(empty_mask.sum())

    df_part = (
        df_part.loc[~empty_mask]
        .copy()
    )

    rows_after = len(df_part)


    # -------------------------------------------------------------------------
    # Clean raw attack label strings
    # -------------------------------------------------------------------------

    df_part["Label"] = (
        df_part["Label"]
        .astype(str)
        .str.strip()
    )


    # -------------------------------------------------------------------------
    # Normalize malformed Web Attack labels
    #
    # Raw labels contain the Windows-1252/Latin-1 character \x96.
    #
    # Example:
    #
    #   Web Attack \x96 Brute Force
    #
    # becomes:
    #
    #   Web Attack - Brute Force
    # -------------------------------------------------------------------------

    df_part["Label"] = (
        df_part["Label"]
        .str.replace(
            "\x96",
            "-",
            regex=False
        )
        .str.replace(
            "  ",
            " ",
            regex=False
        )
        .str.strip()
    )


    # -------------------------------------------------------------------------
    # Rename multiclass target
    #
    # Attack_Type will contain:
    #
    # BENIGN
    # DDoS
    # PortScan
    # DoS Hulk
    # Bot
    # etc.
    # -------------------------------------------------------------------------

    df_part = df_part.rename(
        columns={
            "Label": "Attack_Type"
        }
    )


    # -------------------------------------------------------------------------
    # Create binary attack target
    #
    # BENIGN -> 0
    # ATTACK -> 1
    # -------------------------------------------------------------------------

    df_part["Attack_Label"] = (
        df_part["Attack_Type"]
        .ne("BENIGN")
        .astype(np.int8)
    )


    # -------------------------------------------------------------------------
    # Preserve source file identity
    #
    # This is useful for:
    #   - debugging
    #   - auditing
    #   - tracing flows back to original CICIDS2017 file
    # -------------------------------------------------------------------------

    df_part["Source_File"] = filename


    # -------------------------------------------------------------------------
    # File-level validation
    # -------------------------------------------------------------------------

    assert df_part["Attack_Type"].notna().all(), (
        f"Missing Attack_Type found in {filename}"
    )

    assert df_part["Attack_Label"].isin([0, 1]).all(), (
        f"Invalid Attack_Label found in {filename}"
    )


    print(f"\nRows before cleaning : {rows_before:,}")
    print(f"Empty rows removed   : {empty_removed:,}")
    print(f"Rows after cleaning  : {rows_after:,}")

    print(
        f"Unique attack types  : "
        f"{df_part['Attack_Type'].nunique()}"
    )

    print(
        f"Attack flows         : "
        f"{int(df_part['Attack_Label'].sum()):,}"
    )

    print(
        f"Benign flows         : "
        f"{int((df_part['Attack_Label'] == 0).sum()):,}"
    )


    # -------------------------------------------------------------------------
    # Store cleaned part
    # -------------------------------------------------------------------------

    cleaned_parts.append(df_part)


# =============================================================================
# COMBINE ALL 8 CLEANED FILES
# =============================================================================

print("\n\n" + "=" * 110)
print("COMBINING CLEANED FILES")
print("=" * 110)

canonical_df = pd.concat(
    cleaned_parts,
    ignore_index=True
)

print(
    f"\nCombined dataframe shape: "
    f"{canonical_df.shape}"
)


# Free list references after concatenation
del cleaned_parts
gc.collect()


# =============================================================================
# BASIC VALIDATION
# =============================================================================

print("\n" + "=" * 110)
print("CANONICAL DATASET — BASIC VALIDATION")
print("=" * 110)

print(
    f"\nTotal flows       : "
    f"{len(canonical_df):,}"
)

print(
    f"Total columns     : "
    f"{canonical_df.shape[1]}"
)

print(
    f"Unique attack types: "
    f"{canonical_df['Attack_Type'].nunique()}"
)

print(
    f"Attack flows      : "
    f"{int(canonical_df['Attack_Label'].sum()):,}"
)

print(
    f"Benign flows      : "
    f"{int((canonical_df['Attack_Label'] == 0).sum()):,}"
)


# =============================================================================
# ATTACK TYPE DISTRIBUTION
# =============================================================================

print("\n" + "=" * 110)
print("NORMALIZED ATTACK TYPE DISTRIBUTION")
print("=" * 110)

attack_distribution = (
    canonical_df["Attack_Type"]
    .value_counts()
)

for label, count in attack_distribution.items():

    pct = (
        count /
        len(canonical_df) *
        100
    )

    print(
        f"{label:<40}"
        f"{count:>12,}"
        f" ({pct:7.3f}%)"
    )


# =============================================================================
# BINARY TARGET DISTRIBUTION
# =============================================================================

print("\n" + "=" * 110)
print("BINARY ATTACK LABEL DISTRIBUTION")
print("=" * 110)

binary_distribution = (
    canonical_df["Attack_Label"]
    .value_counts()
    .sort_index()
)

for label, count in binary_distribution.items():

    name = (
        "BENIGN"
        if label == 0
        else "ATTACK"
    )

    pct = (
        count /
        len(canonical_df) *
        100
    )

    print(
        f"{label} ({name:<6}) "
        f"{count:>12,}"
        f" ({pct:7.3f}%)"
    )


# =============================================================================
# FINAL SANITY CHECKS
# =============================================================================

print("\n" + "=" * 110)
print("CANONICAL DATASET SANITY CHECKS")
print("=" * 110)

checks = {

    "Exactly 2,830,743 flows":
        len(canonical_df) == 2_830_743,

    "Exactly 15 attack categories":
        canonical_df["Attack_Type"].nunique() == 15,

    "No missing Attack_Type":
        canonical_df["Attack_Type"].isna().sum() == 0,

    "Binary target contains only 0/1":
        canonical_df["Attack_Label"]
        .isin([0, 1])
        .all(),

    "BENIGN maps only to 0":
        (
            canonical_df.loc[
                canonical_df["Attack_Type"] == "BENIGN",
                "Attack_Label"
            ] == 0
        ).all(),

    "Every attack maps to 1":
        (
            canonical_df.loc[
                canonical_df["Attack_Type"] != "BENIGN",
                "Attack_Label"
            ] == 1
        ).all(),

    "All 8 source files represented":
        canonical_df["Source_File"].nunique() == 8
}


for name, passed in checks.items():

    print(
        f"{'✓' if passed else '❌'} "
        f"{name}: {passed}"
    )


# =============================================================================
# PREVIEW IMPORTANT COLUMNS
# =============================================================================

print("\n" + "=" * 110)
print("CANONICAL DATASET PREVIEW")
print("=" * 110)

preview_columns = [
    "Timestamp",
    "Source IP",
    "Source Port",
    "Destination IP",
    "Destination Port",
    "Protocol",
    "Attack_Type",
    "Attack_Label",
    "Source_File"
]

display(
    canonical_df[
        preview_columns
    ].head(10)
)


# =============================================================================
# FINAL STATUS
# =============================================================================

print("\n" + "=" * 110)

if all(checks.values()):

    print(
        "✓ CANONICAL FLOW DATASET SUCCESSFULLY CONSTRUCTED"
    )

    print(
        "✓ Raw multiclass attack information preserved."
    )

    print(
        "✓ Binary attack indicator created."
    )

    print(
        "✓ Full CICFlowMeter feature space preserved."
    )

    print(
        "✓ Source/destination identity preserved for future graph modelling."
    )

else:

    print(
        "❌ CANONICAL DATASET VALIDATION FAILED"
    )

    print(
        "Do NOT continue until the failed checks are investigated."
    )

print("=" * 110)

CICIDS2017 — RAM-SAFE CANONICAL FLOW CONSTRUCTION

Raw files:
8

Processed files will be saved to:
/content/drive/MyDrive/Network-Attack-Forecasting/data/processed



##############################################################################################################
PROCESSING FILE 1/8
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
##############################################################################################################

Encoding used : utf-8
Raw rows      : 225,745
Raw columns   : 85
Completely empty rows removed : 0
Valid timestamps              : 225,745
Failed timestamps             : 0

Clean rows      : 225,745
Benign flows    : 97,718
Attack flows    : 128,027
Unique labels   : 2

Temporal range:
Start : 2017-07-07 03:30:00
End   : 2017-07-07 05:02:00

Label distribution:
Attack_Type
DDoS      128027
BENIGN     97718
Name: count, dtype: int64

✓ Saved:
/content/drive/MyDrive/Network-Attack-Forecasting/data/processed/Friday-WorkingHours-Afternoon

,Source_File,Encoding,Raw_Rows,Clean_Rows,Empty_Rows_Removed,Benign_Flows,Attack_Flows,Unique_Attack_Types,Failed_Timestamps,Start_Time,End_Time,Saved_Size_MB
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,utf-8,225745,225745,0,97718,128027,2,0,2017-07-07 03:30:00,2017-07-07 05:02:00,30.24
1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,utf-8,286467,286467,0,127537,158930,2,0,2017-07-07 01:00:00,2017-07-07 03:29:00,21.86
2,Friday-WorkingHours-Morning.pcap_ISCX.csv,utf-8,191033,191033,0,189067,1966,2,0,2017-07-07 08:59:00,2017-07-07 12:59:00,25.59
3,Monday-WorkingHours.pcap_ISCX.csv,utf-8,529918,529918,0,529918,0,1,0,2017-07-03 01:00:01,2017-07-03 12:59:58,80.33
4,Thursday-WorkingHours-Afternoon-Infilteration....,utf-8,288602,288602,0,288566,36,2,0,2017-07-06 01:00:00,2017-07-06 05:04:00,32.12
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,cp1252,458968,170366,288602,168186,2180,4,0,2017-07-06 08:59:00,2017-07-06 12:59:00,23.00
6,Tuesday-WorkingHours.pcap_ISCX.csv,utf-8,445909,445909,0,432074,13835,3,0,2017-07-04 01:00:00,2017-07-04 12:59:00,63.39
7,Wednesday-workingHours.pcap_ISCX.csv,utf-8,692703,692703,0,440031,252672,6,0,2017-07-05 01:00:00,2017-07-05 12:59:00,90.07



GLOBAL CLEANING TOTALS

Raw rows             : 3,119,345
Clean rows           : 2,830,743
Empty rows removed   : 288,602
Benign flows         : 2,273,097
Attack flows         : 557,646
Failed timestamps    : 0
Processed files saved: 8

FINAL CELL 6 VALIDATION
✓ Exactly 8 processed files: True
✓ Expected total flow count: True
✓ Expected attack flow count: True
✓ Expected benign flow count: True
✓ All timestamps parsed: True
✓ Every processed file exists: True

✓ CELL 6 COMPLETED SUCCESSFULLY.

All 8 CICIDS2017 files were cleaned and saved independently.
The full 2.83-million-row dataset was NEVER held in RAM at once.

IMPORTANT: canonical_df has intentionally NOT been created.

NEXT: Cell 7 will validate the processed files directly from disk.
BUILDING CLEAN CANONICAL CICIDS2017 FLOW DATASET

##############################################################################################################
PROCESSING FILE 1/8
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
###############

,Timestamp,Source IP,Source Port,Destination IP,Destination Port,Protocol,Attack_Type,Attack_Label,Source_File
0,7/7/2017 3:30,104.16.207.165,443.0,192.168.10.5,54865.0,6.0,BENIGN,0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
1,7/7/2017 3:30,104.16.28.216,80.0,192.168.10.5,55054.0,6.0,BENIGN,0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
2,7/7/2017 3:30,104.16.28.216,80.0,192.168.10.5,55055.0,6.0,BENIGN,0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
3,7/7/2017 3:30,104.17.241.25,443.0,192.168.10.16,46236.0,6.0,BENIGN,0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
4,7/7/2017 3:30,104.19.196.102,443.0,192.168.10.5,54863.0,6.0,BENIGN,0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
5,7/7/2017 3:30,104.20.10.120,443.0,192.168.10.5,54871.0,6.0,BENIGN,0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
6,7/7/2017 3:30,104.20.10.120,443.0,192.168.10.5,54925.0,6.0,BENIGN,0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
7,7/7/2017 3:30,104.20.10.120,443.0,192.168.10.5,54925.0,6.0,BENIGN,0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
8,7/7/2017 3:30,104.28.13.116,443.0,192.168.10.8,9282.0,6.0,BENIGN,0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
9,7/7/2017 3:30,104.97.123.193,443.0,192.168.10.5,55153.0,6.0,BENIGN,0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv



✓ CANONICAL FLOW DATASET SUCCESSFULLY CONSTRUCTED
✓ Raw multiclass attack information preserved.
✓ Binary attack indicator created.
✓ Full CICFlowMeter feature space preserved.
✓ Source/destination identity preserved for future graph modelling.


In [13]:
# =============================================================================
# CELL 7 — VALIDATE RAM-SAFE PROCESSED FLOW FILES
# =============================================================================
#
# What we are doing:
#
# Cell 6 cleaned the 8 raw CICIDS2017 TrafficLabelling CSV files and saved
# them independently as Parquet files.
#
# We intentionally did NOT create one giant canonical_df because doing so
# caused Google Colab to run out of RAM.
#
# This cell verifies that the files written by Cell 6 are correct BEFORE
# we begin temporal segmentation and forecasting-dataset construction.
#
#
# THIS CELL WILL:
#
#   1. Locate the 8 cleaned Parquet files created by Cell 6.
#
#   2. Read ONLY lightweight columns needed for validation:
#
#          Timestamp
#          Attack_Type
#          Attack_Label
#          Source_File
#
#      We DO NOT load all ~90 columns into memory.
#
#   3. Validate:
#
#          - total number of flows
#          - attack/benign counts
#          - multiclass attack labels
#          - timestamp validity
#          - temporal range of each file
#          - source-file traceability
#
#   4. Build a small global attack-label distribution.
#
#   5. Verify the expected CICIDS2017 totals established by our previous
#      audit.
#
#
# IMPORTANT:
#
# This is still FLOW-LEVEL validation.
#
# We are NOT yet:
#
#   - creating temporal segments
#   - aggregating flows by minute
#   - creating forecasting windows
#   - creating lead-time targets
#   - training any model
#
#
# After this cell passes:
#
#                 CLEAN FLOW FILES
#                         ↓
#                 TEMPORAL SEGMENTATION
#                         ↓
#                 MINUTE-LEVEL FEATURES
#                         ↓
#                 FORECASTING WINDOWS
#                         ↓
#                 TARGET CONSTRUCTION
#
# =============================================================================


import os
import glob
import gc

import pandas as pd
import numpy as np


# =============================================================================
# 1. DEFINE PROJECT DIRECTORIES
# =============================================================================

PROJECT_ROOT = (
    "/content/drive/MyDrive/"
    "Network-Attack-Forecasting"
)

PROCESSED_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed"
)


# =============================================================================
# 2. LOCATE PROCESSED FILES
# =============================================================================

processed_files = sorted(
    glob.glob(
        os.path.join(
            PROCESSED_DIR,
            "*_cleaned.parquet"
        )
    )
)


print("=" * 110)
print("CELL 7 — PROCESSED FLOW DATASET VALIDATION")
print("=" * 110)

print("\nProcessed directory:")
print(PROCESSED_DIR)

print(
    f"\nProcessed Parquet files found: "
    f"{len(processed_files)}"
)


for i, path in enumerate(
    processed_files,
    start=1
):

    size_mb = (
        os.path.getsize(path)
        / 1024**2
    )

    print(
        f"\n{i}. {os.path.basename(path)}"
    )

    print(
        f"   Size: {size_mb:.2f} MB"
    )


assert len(processed_files) == 8, (
    f"Expected 8 processed Parquet files, "
    f"but found {len(processed_files)}."
)


# =============================================================================
# 3. COLUMNS REQUIRED FOR THIS VALIDATION
# =============================================================================
#
# We deliberately load ONLY these columns.
#
# There is no reason to load the full CICFlowMeter feature space simply to
# validate chronology and labels.
#
# =============================================================================

validation_columns = [

    "Timestamp",
    "Attack_Type",
    "Attack_Label",
    "Source_File"

]


# =============================================================================
# 4. SMALL STORAGE OBJECTS
# =============================================================================
#
# These contain only summaries, not millions of flow rows.
#
# =============================================================================

file_summaries = []

global_label_counts = {}

global_binary_counts = {
    0: 0,
    1: 0
}


# =============================================================================
# 5. VALIDATE EACH PROCESSED FILE INDEPENDENTLY
# =============================================================================

for file_number, path in enumerate(
    processed_files,
    start=1
):

    filename = os.path.basename(path)

    print("\n\n")
    print("#" * 110)

    print(
        f"VALIDATING FILE "
        f"{file_number}/{len(processed_files)}"
    )

    print(filename)

    print("#" * 110)


    # =========================================================================
    # 5A. READ ONLY VALIDATION COLUMNS
    # =========================================================================

    df = pd.read_parquet(
        path,
        columns=validation_columns
    )


    # =========================================================================
    # 5B. BASIC COUNTS
    # =========================================================================

    total_rows = len(df)

    benign_flows = int(
        (df["Attack_Label"] == 0)
        .sum()
    )

    attack_flows = int(
        (df["Attack_Label"] == 1)
        .sum()
    )


    # =========================================================================
    # 5C. TIMESTAMP VALIDATION
    # =========================================================================

    missing_timestamps = int(
        df["Timestamp"]
        .isna()
        .sum()
    )

    start_time = (
        df["Timestamp"]
        .min()
    )

    end_time = (
        df["Timestamp"]
        .max()
    )


    # =========================================================================
    # 5D. ATTACK-TYPE VALIDATION
    # =========================================================================

    missing_attack_types = int(
        df["Attack_Type"]
        .isna()
        .sum()
    )

    label_counts = (
        df["Attack_Type"]
        .value_counts()
    )


    # =========================================================================
    # 5E. BINARY LABEL VALIDATION
    # =========================================================================

    unique_binary_labels = set(
        df["Attack_Label"]
        .dropna()
        .unique()
        .tolist()
    )

    binary_valid = (
        unique_binary_labels
        .issubset({0, 1})
    )


    # =========================================================================
    # 5F. VERIFY BENIGN ↔ BINARY MAPPING
    # =========================================================================

    benign_mapping_valid = bool(

        (
            df.loc[
                df["Attack_Type"]
                .str.upper()
                .eq("BENIGN"),
                "Attack_Label"
            ]
            == 0
        )
        .all()

    )


    attack_mapping_valid = bool(

        (
            df.loc[
                ~df["Attack_Type"]
                .str.upper()
                .eq("BENIGN"),
                "Attack_Label"
            ]
            == 1
        )
        .all()

    )


    # =========================================================================
    # 5G. SOURCE FILE TRACEABILITY
    # =========================================================================

    source_file_count = (
        df["Source_File"]
        .nunique()
    )

    source_file_valid = (
        source_file_count == 1
    )


    # =========================================================================
    # 5H. ADD LABEL COUNTS TO GLOBAL TOTAL
    # =========================================================================

    for label, count in label_counts.items():

        global_label_counts[label] = (
            global_label_counts.get(
                label,
                0
            )
            + int(count)
        )


    global_binary_counts[0] += (
        benign_flows
    )

    global_binary_counts[1] += (
        attack_flows
    )


    # =========================================================================
    # 5I. STORE SMALL FILE SUMMARY
    # =========================================================================

    file_summaries.append({

        "Processed_File":
            filename,

        "Rows":
            total_rows,

        "Benign_Flows":
            benign_flows,

        "Attack_Flows":
            attack_flows,

        "Unique_Attack_Types":
            df["Attack_Type"].nunique(),

        "Missing_Timestamps":
            missing_timestamps,

        "Missing_Attack_Types":
            missing_attack_types,

        "Binary_Label_Valid":
            binary_valid,

        "Benign_Mapping_Valid":
            benign_mapping_valid,

        "Attack_Mapping_Valid":
            attack_mapping_valid,

        "Source_File_Valid":
            source_file_valid,

        "Start_Time":
            start_time,

        "End_Time":
            end_time

    })


    # =========================================================================
    # 5J. DISPLAY FILE RESULTS
    # =========================================================================

    print(
        f"\nRows               : "
        f"{total_rows:,}"
    )

    print(
        f"Benign flows       : "
        f"{benign_flows:,}"
    )

    print(
        f"Attack flows       : "
        f"{attack_flows:,}"
    )

    print(
        f"Unique attack types: "
        f"{df['Attack_Type'].nunique()}"
    )

    print(
        f"Missing timestamps : "
        f"{missing_timestamps:,}"
    )

    print(
        f"Missing attack type: "
        f"{missing_attack_types:,}"
    )

    print(
        f"Temporal start     : "
        f"{start_time}"
    )

    print(
        f"Temporal end       : "
        f"{end_time}"
    )


    print("\nLabels:")

    for label, count in label_counts.items():

        print(
            f"  {label:<35} "
            f"{int(count):>10,}"
        )


    print("\nChecks:")

    print(
        "✓" if binary_valid else "❌",
        "Binary label contains only 0/1"
    )

    print(
        "✓" if benign_mapping_valid else "❌",
        "BENIGN maps to 0"
    )

    print(
        "✓" if attack_mapping_valid else "❌",
        "Attacks map to 1"
    )

    print(
        "✓" if source_file_valid else "❌",
        "Source-file traceability preserved"
    )


    # =========================================================================
    # 5K. DELETE THIS FILE FROM RAM
    # =========================================================================

    del df

    gc.collect()

    print(
        "\n✓ Validation dataframe "
        "deleted from RAM."
    )


# =============================================================================
# 6. BUILD FILE-LEVEL SUMMARY
# =============================================================================

validation_summary_df = pd.DataFrame(
    file_summaries
)


print("\n\n")
print("=" * 110)
print("FILE-LEVEL VALIDATION SUMMARY")
print("=" * 110)

display(
    validation_summary_df
)


# =============================================================================
# 7. GLOBAL DATASET COUNTS
# =============================================================================

total_flows = int(
    validation_summary_df["Rows"]
    .sum()
)

total_benign = int(
    validation_summary_df["Benign_Flows"]
    .sum()
)

total_attacks = int(
    validation_summary_df["Attack_Flows"]
    .sum()
)

total_missing_timestamps = int(
    validation_summary_df["Missing_Timestamps"]
    .sum()
)

total_missing_attack_types = int(
    validation_summary_df["Missing_Attack_Types"]
    .sum()
)


print("\n" + "=" * 110)
print("GLOBAL FLOW-LEVEL TOTALS")
print("=" * 110)

print(
    f"\nTotal flows       : "
    f"{total_flows:,}"
)

print(
    f"Benign flows      : "
    f"{total_benign:,}"
)

print(
    f"Attack flows      : "
    f"{total_attacks:,}"
)

print(
    f"Missing timestamps: "
    f"{total_missing_timestamps:,}"
)

print(
    f"Missing labels    : "
    f"{total_missing_attack_types:,}"
)


# =============================================================================
# 8. GLOBAL MULTICLASS ATTACK DISTRIBUTION
# =============================================================================

global_label_series = pd.Series(
    global_label_counts
).sort_values(
    ascending=False
)


print("\n" + "=" * 110)
print("GLOBAL ATTACK-TYPE DISTRIBUTION")
print("=" * 110)


for label, count in global_label_series.items():

    percentage = (
        count
        / total_flows
        * 100
    )

    print(
        f"{label:<40}"
        f"{count:>12,}"
        f"   ({percentage:>7.3f}%)"
    )


# =============================================================================
# 9. GLOBAL BINARY DISTRIBUTION
# =============================================================================

print("\n" + "=" * 110)
print("GLOBAL BINARY ATTACK DISTRIBUTION")
print("=" * 110)


benign_percentage = (
    total_benign
    / total_flows
    * 100
)

attack_percentage = (
    total_attacks
    / total_flows
    * 100
)


print(
    f"\n0 (BENIGN) : "
    f"{total_benign:,} "
    f"({benign_percentage:.3f}%)"
)

print(
    f"1 (ATTACK) : "
    f"{total_attacks:,} "
    f"({attack_percentage:.3f}%)"
)


# =============================================================================
# 10. GLOBAL TEMPORAL RANGE
# =============================================================================

global_start = (
    validation_summary_df["Start_Time"]
    .min()
)

global_end = (
    validation_summary_df["End_Time"]
    .max()
)


print("\n" + "=" * 110)
print("GLOBAL TEMPORAL RANGE")
print("=" * 110)

print(
    "\nStart:",
    global_start
)

print(
    "End  :",
    global_end
)


# =============================================================================
# 11. EXPECTED ATTACK TYPES
# =============================================================================

expected_attack_types = {

    "BENIGN",

    "DDoS",

    "PortScan",

    "Bot",

    "Infiltration",

    "Web Attack - Brute Force",

    "Web Attack - XSS",

    "Web Attack - Sql Injection",

    "FTP-Patator",

    "SSH-Patator",

    "DoS Hulk",

    "DoS GoldenEye",

    "DoS slowloris",

    "DoS Slowhttptest",

    "Heartbleed"

}


observed_attack_types = set(
    global_label_series.index
)


missing_expected_labels = (
    expected_attack_types
    - observed_attack_types
)

unexpected_labels = (
    observed_attack_types
    - expected_attack_types
)


# =============================================================================
# 12. FINAL VALIDATION CHECKS
# =============================================================================

checks = {

    "Exactly 8 processed files":
        len(processed_files) == 8,

    "Exactly 2,830,743 flows":
        total_flows == 2_830_743,

    "Exactly 557,646 attack flows":
        total_attacks == 557_646,

    "Exactly 2,273,097 benign flows":
        total_benign == 2_273_097,

    "Exactly 15 attack categories":
        len(observed_attack_types) == 15,

    "All expected attack categories present":
        len(missing_expected_labels) == 0,

    "No unexpected attack categories":
        len(unexpected_labels) == 0,

    "No missing timestamps":
        total_missing_timestamps == 0,

    "No missing attack types":
        total_missing_attack_types == 0,

    "Binary labels valid in every file":
        validation_summary_df[
            "Binary_Label_Valid"
        ].all(),

    "BENIGN mapping valid in every file":
        validation_summary_df[
            "Benign_Mapping_Valid"
        ].all(),

    "Attack mapping valid in every file":
        validation_summary_df[
            "Attack_Mapping_Valid"
        ].all(),

    "Source traceability valid":
        validation_summary_df[
            "Source_File_Valid"
        ].all()

}


print("\n" + "=" * 110)
print("FINAL CELL 7 VALIDATION")
print("=" * 110)


for check_name, result in checks.items():

    print(
        f"{'✓' if result else '❌'} "
        f"{check_name}: {bool(result)}"
    )


# =============================================================================
# 13. REPORT LABEL DIFFERENCES IF ANY
# =============================================================================

if missing_expected_labels:

    print(
        "\n❌ Missing expected labels:"
    )

    for label in sorted(
        missing_expected_labels
    ):

        print(
            "  -",
            label
        )


if unexpected_labels:

    print(
        "\n❌ Unexpected labels:"
    )

    for label in sorted(
        unexpected_labels
    ):

        print(
            "  -",
            label
        )


# =============================================================================
# 14. FINAL RESULT
# =============================================================================

print("\n" + "=" * 110)


if all(
    bool(value)
    for value in checks.values()
):

    print(
        "✓ CELL 7 COMPLETED SUCCESSFULLY."
    )

    print()

    print(
        "The 8 processed CICIDS2017 flow files "
        "have been validated directly from disk."
    )

    print()

    print(
        "No giant canonical_df was created."
    )

    print()

    print(
        "The cleaned flow-level dataset is ready "
        "for temporal segmentation."
    )

    print()

    print(
        "NEXT: Cell 8 will construct the temporal "
        "timeline/segments in a RAM-safe way."
    )

else:

    print(
        "❌ CELL 7 VALIDATION FAILED."
    )

    print(
        "Do NOT continue to temporal segmentation "
        "until the failed checks are investigated."
    )


print("=" * 110)


# =============================================================================
# 15. CLEAN UP
# =============================================================================

gc.collect()

CELL 7 — PROCESSED FLOW DATASET VALIDATION

Processed directory:
/content/drive/MyDrive/Network-Attack-Forecasting/data/processed

Processed Parquet files found: 8

1. Friday-WorkingHours-Afternoon-DDos.pcap_ISCX_cleaned.parquet
   Size: 30.24 MB

2. Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX_cleaned.parquet
   Size: 21.86 MB

3. Friday-WorkingHours-Morning.pcap_ISCX_cleaned.parquet
   Size: 25.59 MB

4. Monday-WorkingHours.pcap_ISCX_cleaned.parquet
   Size: 80.33 MB

5. Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX_cleaned.parquet
   Size: 32.12 MB

6. Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX_cleaned.parquet
   Size: 23.00 MB

7. Tuesday-WorkingHours.pcap_ISCX_cleaned.parquet
   Size: 63.39 MB

8. Wednesday-workingHours.pcap_ISCX_cleaned.parquet
   Size: 90.07 MB



##############################################################################################################
VALIDATING FILE 1/8
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX_cleaned.parquet
####

,Processed_File,Rows,Benign_Flows,Attack_Flows,Unique_Attack_Types,Missing_Timestamps,Missing_Attack_Types,Binary_Label_Valid,Benign_Mapping_Valid,Attack_Mapping_Valid,Source_File_Valid,Start_Time,End_Time
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX_c...,225745,97718,128027,2,0,0,True,True,True,True,2017-07-07 03:30:00,2017-07-07 05:02:00
1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,286467,127537,158930,2,0,0,True,True,True,True,2017-07-07 01:00:00,2017-07-07 03:29:00
2,Friday-WorkingHours-Morning.pcap_ISCX_cleaned....,191033,189067,1966,2,0,0,True,True,True,True,2017-07-07 08:59:00,2017-07-07 12:59:00
3,Monday-WorkingHours.pcap_ISCX_cleaned.parquet,529918,529918,0,1,0,0,True,True,True,True,2017-07-03 01:00:01,2017-07-03 12:59:58
4,Thursday-WorkingHours-Afternoon-Infilteration....,288602,288566,36,2,0,0,True,True,True,True,2017-07-06 01:00:00,2017-07-06 05:04:00
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,170366,168186,2180,4,0,0,True,True,True,True,2017-07-06 08:59:00,2017-07-06 12:59:00
6,Tuesday-WorkingHours.pcap_ISCX_cleaned.parquet,445909,432074,13835,3,0,0,True,True,True,True,2017-07-04 01:00:00,2017-07-04 12:59:00
7,Wednesday-workingHours.pcap_ISCX_cleaned.parquet,692703,440031,252672,6,0,0,True,True,True,True,2017-07-05 01:00:00,2017-07-05 12:59:00



GLOBAL FLOW-LEVEL TOTALS

Total flows       : 2,830,743
Benign flows      : 2,273,097
Attack flows      : 557,646
Missing timestamps: 0
Missing labels    : 0

GLOBAL ATTACK-TYPE DISTRIBUTION
BENIGN                                     2,273,097   ( 80.300%)
DoS Hulk                                     231,073   (  8.163%)
PortScan                                     158,930   (  5.614%)
DDoS                                         128,027   (  4.523%)
DoS GoldenEye                                 10,293   (  0.364%)
FTP-Patator                                    7,938   (  0.280%)
SSH-Patator                                    5,897   (  0.208%)
DoS slowloris                                  5,796   (  0.205%)
DoS Slowhttptest                               5,499   (  0.194%)
Bot                                            1,966   (  0.069%)
Web Attack - Brute Force                       1,507   (  0.053%)
Web Attack - XSS                                 652   (  0.023%)
Infiltration    

0

In [14]:
# =============================================================================
# CELL 8 — RAM-SAFE TEMPORAL GAP ANALYSIS AND SEGMENT CONSTRUCTION
# =============================================================================
#
# What we are doing:
#
# CICIDS2017 consists of multiple network-capture periods spread across
# several days. These capture periods must NOT automatically be treated as
# one continuous time series.
#
# For forecasting, a history window and its future prediction window must
# belong to the SAME continuous observation period.
#
# Example of what we must prevent:
#
#       Friday capture ends
#              ↓
#        several-hour gap
#              ↓
#       next capture begins
#
# A forecasting window must never jump across such a gap.
#
#
# THIS CELL WILL:
#
#   1. Read ONLY the Timestamp column from each processed Parquet file.
#
#   2. Build a lightweight global timestamp table.
#
#   3. Sort timestamps chronologically.
#
#   4. Calculate the time difference between consecutive flows.
#
#   5. Inspect the gap distribution.
#
#   6. Display all meaningful large gaps.
#
#   7. Automatically identify capture discontinuities using a conservative
#      threshold.
#
#   8. Assign Segment_ID values.
#
#   9. Create and save a small segment metadata table.
#
#
# IMPORTANT:
#
# We are NOT loading the ~85 CICFlowMeter feature columns.
#
# We are NOT constructing minute-level features yet.
#
# We are NOT creating forecasting windows yet.
#
# We are NOT training models.
#
# =============================================================================


import os
import glob
import gc

import numpy as np
import pandas as pd


# =============================================================================
# 1. PROJECT PATHS
# =============================================================================

PROJECT_ROOT = (
    "/content/drive/MyDrive/"
    "Network-Attack-Forecasting"
)

PROCESSED_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed"
)

TEMPORAL_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "temporal"
)

os.makedirs(
    TEMPORAL_DIR,
    exist_ok=True
)


# =============================================================================
# 2. LOCATE PROCESSED FILES
# =============================================================================

processed_files = sorted(
    glob.glob(
        os.path.join(
            PROCESSED_DIR,
            "*_cleaned.parquet"
        )
    )
)

assert len(processed_files) == 8, (
    f"Expected 8 processed files, "
    f"found {len(processed_files)}."
)


print("=" * 110)
print("CELL 8 — TEMPORAL GAP ANALYSIS")
print("=" * 110)

print(
    f"\nProcessed files found: "
    f"{len(processed_files)}"
)


# =============================================================================
# 3. LOAD ONLY TIMESTAMP + SOURCE FILE
# =============================================================================
#
# This is intentionally tiny compared with loading the complete feature
# dataset.
#
# =============================================================================

timeline_parts = []


for i, path in enumerate(
    processed_files,
    start=1
):

    filename = os.path.basename(path)

    print(
        f"\nReading timestamps "
        f"{i}/8: {filename}"
    )

    temp = pd.read_parquet(
        path,
        columns=[
            "Timestamp",
            "Source_File"
        ]
    )

    timeline_parts.append(
        temp
    )


# =============================================================================
# 4. COMBINE LIGHTWEIGHT TIMELINE
# =============================================================================

timeline_df = pd.concat(
    timeline_parts,
    ignore_index=True
)

del timeline_parts
del temp

gc.collect()


print("\n" + "=" * 110)
print("LIGHTWEIGHT GLOBAL TIMELINE")
print("=" * 110)

print(
    f"\nRows: "
    f"{len(timeline_df):,}"
)

print(
    "Columns:",
    timeline_df.columns.tolist()
)


assert len(timeline_df) == 2_830_743


# =============================================================================
# 5. SORT CHRONOLOGICALLY
# =============================================================================
#
# Unlike the previous failed attempt, this dataframe contains only TWO
# columns rather than ~90 columns.
#
# =============================================================================

timeline_df = (
    timeline_df
    .sort_values(
        "Timestamp",
        kind="mergesort"
    )
    .reset_index(drop=True)
)


print("\n✓ Global timestamp timeline sorted.")


# =============================================================================
# 6. CALCULATE INTER-FLOW GAPS
# =============================================================================

timeline_df["Gap_Seconds"] = (
    timeline_df["Timestamp"]
    .diff()
    .dt.total_seconds()
)


# =============================================================================
# 7. BASIC GAP STATISTICS
# =============================================================================

valid_gaps = (
    timeline_df["Gap_Seconds"]
    .dropna()
)


print("\n" + "=" * 110)
print("INTER-FLOW GAP STATISTICS")
print("=" * 110)

print(
    valid_gaps.describe(
        percentiles=[
            0.50,
            0.90,
            0.95,
            0.99,
            0.999,
            0.9999
        ]
    )
)


# =============================================================================
# 8. GAP COUNTS AT IMPORTANT THRESHOLDS
# =============================================================================

gap_thresholds = [
    1,
    5,
    10,
    30,
    60,
    120,
    300,
    600,
    1800,
    3600
]


print("\n" + "=" * 110)
print("NUMBER OF GAPS ABOVE SELECTED THRESHOLDS")
print("=" * 110)


for threshold in gap_thresholds:

    count = int(
        (
            valid_gaps
            > threshold
        )
        .sum()
    )

    print(
        f"> {threshold:>5} seconds : "
        f"{count:>10,}"
    )


# =============================================================================
# 9. DISPLAY LARGE GAPS
# =============================================================================
#
# We inspect gaps greater than 5 minutes.
#
# These are candidates for true capture discontinuities rather than normal
# packet/flow inactivity.
#
# =============================================================================

large_gap_mask = (
    timeline_df["Gap_Seconds"]
    > 300
)


large_gaps = (
    timeline_df.loc[
        large_gap_mask,
        [
            "Timestamp",
            "Gap_Seconds",
            "Source_File"
        ]
    ]
    .copy()
)


# Previous timestamp before each large gap

large_gap_indices = (
    timeline_df.index[
        large_gap_mask
    ]
)


previous_timestamps = (
    timeline_df["Timestamp"]
    .shift(1)
    .loc[large_gap_indices]
    .values
)


previous_sources = (
    timeline_df["Source_File"]
    .shift(1)
    .loc[large_gap_indices]
    .values
)


large_gaps.insert(
    0,
    "Previous_Timestamp",
    previous_timestamps
)

large_gaps.insert(
    1,
    "Previous_Source_File",
    previous_sources
)


large_gaps = large_gaps.rename(
    columns={
        "Timestamp":
            "Next_Timestamp",

        "Source_File":
            "Next_Source_File"
    }
)


large_gaps["Gap_Minutes"] = (
    large_gaps["Gap_Seconds"]
    / 60
)


print("\n" + "=" * 110)
print("GAPS GREATER THAN 5 MINUTES")
print("=" * 110)

print(
    f"\nNumber of >5-minute gaps: "
    f"{len(large_gaps)}"
)

if len(large_gaps) > 0:

    display(
        large_gaps[
            [
                "Previous_Timestamp",
                "Next_Timestamp",
                "Gap_Minutes",
                "Previous_Source_File",
                "Next_Source_File"
            ]
        ]
    )


# =============================================================================
# 10. DEFINE TEMPORAL DISCONTINUITY THRESHOLD
# =============================================================================
#
# For our minute-level forecasting task, a gap greater than 5 minutes means
# there is not enough continuous observation to safely bridge that period.
#
# Therefore:
#
#       Gap <= 5 minutes
#           same temporal segment
#
#       Gap > 5 minutes
#           new temporal segment
#
# This threshold is explicitly stored so it can later be changed and tested
# in sensitivity analysis.
#
# =============================================================================

SEGMENT_GAP_THRESHOLD_SECONDS = 300


timeline_df["New_Segment"] = (

    timeline_df["Gap_Seconds"]
    .gt(
        SEGMENT_GAP_THRESHOLD_SECONDS
    )
    .fillna(False)

)


# =============================================================================
# 11. ASSIGN SEGMENT IDs
# =============================================================================

timeline_df["Segment_ID"] = (
    timeline_df["New_Segment"]
    .cumsum()
    .astype("int16")
)


segment_count = (
    timeline_df["Segment_ID"]
    .nunique()
)


print("\n" + "=" * 110)
print("TEMPORAL SEGMENT CONSTRUCTION")
print("=" * 110)

print(
    f"\nGap threshold : "
    f"{SEGMENT_GAP_THRESHOLD_SECONDS} seconds"
)

print(
    f"Segments found: "
    f"{segment_count}"
)


# =============================================================================
# 12. CREATE SEGMENT METADATA
# =============================================================================

segment_metadata = (

    timeline_df

    .groupby(
        "Segment_ID",
        as_index=False
    )

    .agg(

        Start_Time=(
            "Timestamp",
            "min"
        ),

        End_Time=(
            "Timestamp",
            "max"
        ),

        Flow_Count=(
            "Timestamp",
            "size"
        )

    )

)


segment_metadata["Duration_Minutes"] = (

    (
        segment_metadata["End_Time"]
        -
        segment_metadata["Start_Time"]
    )

    .dt.total_seconds()

    / 60

)


print("\n" + "=" * 110)
print("SEGMENT METADATA")
print("=" * 110)

display(
    segment_metadata
)


# =============================================================================
# 13. VERIFY SEGMENTS DO NOT CONTAIN LARGE INTERNAL GAPS
# =============================================================================

segment_gap_check = (

    timeline_df

    .groupby(
        "Segment_ID"
    )["Gap_Seconds"]

    .max()

)


# Segment 0 begins with NaN Gap_Seconds.
# Ignore NaN values for validation.

segment_gap_valid = bool(

    (
        segment_gap_check
        .dropna()
        <= SEGMENT_GAP_THRESHOLD_SECONDS
    )
    .all()

)


print("\n" + "=" * 110)
print("SEGMENT CONTINUITY CHECK")
print("=" * 110)

print(
    "\nAll internal segment gaps "
    "<= threshold:",
    segment_gap_valid
)


# =============================================================================
# 14. SAVE SEGMENT METADATA
# =============================================================================

metadata_path = os.path.join(
    TEMPORAL_DIR,
    "segment_metadata.parquet"
)


segment_metadata.to_parquet(
    metadata_path,
    index=False
)


print("\n✓ Segment metadata saved:")
print(metadata_path)


# =============================================================================
# 15. SAVE LARGE-GAP AUDIT
# =============================================================================

gap_audit_path = os.path.join(
    TEMPORAL_DIR,
    "large_gap_audit.parquet"
)


large_gaps.to_parquet(
    gap_audit_path,
    index=False
)


print("\n✓ Large-gap audit saved:")
print(gap_audit_path)


# =============================================================================
# 16. SAVE LIGHTWEIGHT TIMELINE
# =============================================================================
#
# We save only the columns required to reconstruct temporal segment
# membership later.
#
# =============================================================================

timeline_output = (

    timeline_df[
        [
            "Timestamp",
            "Source_File",
            "Segment_ID"
        ]
    ]

    .copy()

)


timeline_path = os.path.join(
    TEMPORAL_DIR,
    "flow_timeline.parquet"
)


timeline_output.to_parquet(
    timeline_path,
    index=False,
    compression="snappy"
)


print("\n✓ Lightweight flow timeline saved:")
print(timeline_path)


# =============================================================================
# 17. FINAL VALIDATION
# =============================================================================

checks = {

    "Expected flow count":
        len(timeline_df) == 2_830_743,

    "No missing timestamps":
        timeline_df["Timestamp"]
        .isna()
        .sum() == 0,

    "Chronological order valid":
        timeline_df["Timestamp"]
        .is_monotonic_increasing,

    "At least one temporal segment":
        segment_count >= 1,

    "Internal segment gaps valid":
        segment_gap_valid,

    "Metadata saved":
        os.path.exists(
            metadata_path
        ),

    "Gap audit saved":
        os.path.exists(
            gap_audit_path
        ),

    "Timeline saved":
        os.path.exists(
            timeline_path
        )

}


print("\n" + "=" * 110)
print("FINAL CELL 8 VALIDATION")
print("=" * 110)


for name, result in checks.items():

    print(
        f"{'✓' if bool(result) else '❌'} "
        f"{name}: {bool(result)}"
    )


print("\n" + "=" * 110)


if all(
    bool(value)
    for value in checks.values()
):

    print(
        "✓ CELL 8 COMPLETED SUCCESSFULLY."
    )

    print()

    print(
        "The CICIDS2017 capture timeline has been "
        "constructed without loading the full feature "
        "dataset into RAM."
    )

    print()

    print(
        f"Temporal segments found: "
        f"{segment_count}"
    )

    print()

    print(
        "NEXT: inspect the gap and segment output "
        "before constructing minute-level features."
    )

else:

    print(
        "❌ CELL 8 VALIDATION FAILED."
    )

    print(
        "Do NOT continue to minute-level aggregation."
    )


print("=" * 110)


# =============================================================================
# 18. RAM CLEANUP
# =============================================================================

del timeline_output

gc.collect()

CELL 8 — TEMPORAL GAP ANALYSIS

Processed files found: 8

Reading timestamps 1/8: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX_cleaned.parquet

Reading timestamps 2/8: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX_cleaned.parquet

Reading timestamps 3/8: Friday-WorkingHours-Morning.pcap_ISCX_cleaned.parquet

Reading timestamps 4/8: Monday-WorkingHours.pcap_ISCX_cleaned.parquet

Reading timestamps 5/8: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX_cleaned.parquet

Reading timestamps 6/8: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX_cleaned.parquet

Reading timestamps 7/8: Tuesday-WorkingHours.pcap_ISCX_cleaned.parquet

Reading timestamps 8/8: Wednesday-workingHours.pcap_ISCX_cleaned.parquet

LIGHTWEIGHT GLOBAL TIMELINE

Rows: 2,830,743
Columns: ['Timestamp', 'Source_File']

✓ Global timestamp timeline sorted.

INTER-FLOW GAP STATISTICS
count     2.830742e+06
mean      1.373276e-01
std       5.461584e+01
min       0.000000e+00
50%       0.000000e+00
90%       0.000000e

,Previous_Timestamp,Next_Timestamp,Gap_Minutes,Previous_Source_File,Next_Source_File
222799,2017-07-03 05:01:34,2017-07-03 08:55:58,234.400000,Monday-WorkingHours.pcap_ISCX.csv,Monday-WorkingHours.pcap_ISCX.csv
529918,2017-07-03 12:59:58,2017-07-04 01:00:00,720.033333,Monday-WorkingHours.pcap_ISCX.csv,Tuesday-WorkingHours.pcap_ISCX.csv
750684,2017-07-04 05:00:00,2017-07-04 08:53:00,233.000000,Tuesday-WorkingHours.pcap_ISCX.csv,Tuesday-WorkingHours.pcap_ISCX.csv
975827,2017-07-04 12:59:00,2017-07-05 01:00:00,721.000000,Tuesday-WorkingHours.pcap_ISCX.csv,Wednesday-workingHours.pcap_ISCX.csv
1203164,2017-07-05 05:10:00,2017-07-05 08:42:00,212.000000,Wednesday-workingHours.pcap_ISCX.csv,Wednesday-workingHours.pcap_ISCX.csv
1668530,2017-07-05 12:59:00,2017-07-06 01:00:00,721.000000,Wednesday-workingHours.pcap_ISCX.csv,Thursday-WorkingHours-Afternoon-Infilteration....
1957132,2017-07-06 05:04:00,2017-07-06 08:59:00,235.000000,Thursday-WorkingHours-Afternoon-Infilteration....,Thursday-WorkingHours-Morning-WebAttacks.pcap_...
2127498,2017-07-06 12:59:00,2017-07-07 01:00:00,721.000000,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...
2639710,2017-07-07 05:02:00,2017-07-07 08:59:00,237.000000,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Friday-WorkingHours-Morning.pcap_ISCX.csv



TEMPORAL SEGMENT CONSTRUCTION

Gap threshold : 300 seconds
Segments found: 10

SEGMENT METADATA


,Segment_ID,Start_Time,End_Time,Flow_Count,Duration_Minutes
0,0,2017-07-03 01:00:01,2017-07-03 05:01:34,222799,241.55
1,1,2017-07-03 08:55:58,2017-07-03 12:59:58,307119,244.00
2,2,2017-07-04 01:00:00,2017-07-04 05:00:00,220766,240.00
3,3,2017-07-04 08:53:00,2017-07-04 12:59:00,225143,246.00
4,4,2017-07-05 01:00:00,2017-07-05 05:10:00,227337,250.00
5,5,2017-07-05 08:42:00,2017-07-05 12:59:00,465366,257.00
6,6,2017-07-06 01:00:00,2017-07-06 05:04:00,288602,244.00
7,7,2017-07-06 08:59:00,2017-07-06 12:59:00,170366,240.00
8,8,2017-07-07 01:00:00,2017-07-07 05:02:00,512212,242.00
9,9,2017-07-07 08:59:00,2017-07-07 12:59:00,191033,240.00



SEGMENT CONTINUITY CHECK

All internal segment gaps <= threshold: False

✓ Segment metadata saved:
/content/drive/MyDrive/Network-Attack-Forecasting/data/temporal/segment_metadata.parquet

✓ Large-gap audit saved:
/content/drive/MyDrive/Network-Attack-Forecasting/data/temporal/large_gap_audit.parquet

✓ Lightweight flow timeline saved:
/content/drive/MyDrive/Network-Attack-Forecasting/data/temporal/flow_timeline.parquet

FINAL CELL 8 VALIDATION
✓ Expected flow count: True
✓ No missing timestamps: True
✓ Chronological order valid: True
✓ At least one temporal segment: True
❌ Internal segment gaps valid: False
✓ Metadata saved: True
✓ Gap audit saved: True
✓ Timeline saved: True

❌ CELL 8 VALIDATION FAILED.
Do NOT continue to minute-level aggregation.


46

In [16]:
# =============================================================================
# CELL 8B — CORRECT TEMPORAL SEGMENT CONTINUITY VALIDATION
# =============================================================================
#
# Why this cell exists:
#
# The original Cell 8 continuity check incorrectly counted the gap BEFORE
# the first row of each new segment as if that gap occurred INSIDE the
# segment.
#
# Example:
#
#   Segment 0 ends
#          |
#          | 234-minute capture gap
#          |
#   Segment 1 begins
#
# The first row of Segment 1 naturally stores that 234-minute Gap_Seconds
# value because Gap_Seconds was calculated before segmentation.
#
# That gap is the BOUNDARY BETWEEN segments, not an internal segment gap.
#
# Therefore, this validation recalculates time differences independently
# WITHIN each Segment_ID.
#
# =============================================================================


print("=" * 110)
print("CELL 8B — CORRECTED SEGMENT CONTINUITY VALIDATION")
print("=" * 110)


# =============================================================================
# 1. RECALCULATE GAPS WITHIN EACH SEGMENT
# =============================================================================

timeline_df["Internal_Gap_Seconds"] = (

    timeline_df
    .groupby("Segment_ID")["Timestamp"]
    .diff()
    .dt.total_seconds()

)


# =============================================================================
# 2. FIND MAXIMUM TRUE INTERNAL GAP PER SEGMENT
# =============================================================================

internal_gap_summary = (

    timeline_df

    .groupby("Segment_ID")

    .agg(

        Start_Time=(
            "Timestamp",
            "min"
        ),

        End_Time=(
            "Timestamp",
            "max"
        ),

        Flow_Count=(
            "Timestamp",
            "size"
        ),

        Max_Internal_Gap_Seconds=(
            "Internal_Gap_Seconds",
            "max"
        )

    )

    .reset_index()

)


internal_gap_summary["Max_Internal_Gap_Minutes"] = (

    internal_gap_summary[
        "Max_Internal_Gap_Seconds"
    ]

    / 60

)


# =============================================================================
# 3. DISPLAY RESULT
# =============================================================================

print("\nTRUE INTERNAL GAP SUMMARY")
print("-" * 110)

display(
    internal_gap_summary
)


# =============================================================================
# 4. VALIDATE
# =============================================================================

SEGMENT_GAP_THRESHOLD_SECONDS = 300


segment_gap_valid = bool(

    (
        internal_gap_summary[
            "Max_Internal_Gap_Seconds"
        ]
        <= SEGMENT_GAP_THRESHOLD_SECONDS
    )
    .all()

)


print("\n" + "=" * 110)
print("CORRECTED CONTINUITY CHECK")
print("=" * 110)

print(
    "\nAll TRUE internal segment gaps "
    "<= 300 seconds:",
    segment_gap_valid
)


# =============================================================================
# 5. ADDITIONAL SANITY CHECK
# =============================================================================

bad_segments = (

    internal_gap_summary[

        internal_gap_summary[
            "Max_Internal_Gap_Seconds"
        ]

        > SEGMENT_GAP_THRESHOLD_SECONDS

    ]

)


print(
    "\nSegments containing an invalid "
    "internal gap:",
    len(bad_segments)
)


if len(bad_segments) > 0:

    print(
        "\n❌ Unexpected large internal gaps found:"
    )

    display(
        bad_segments
    )

else:

    print(
        "✓ No segment contains a "
        ">5-minute internal discontinuity."
    )


# =============================================================================
# 6. VERIFY EXPECTED SEGMENT STRUCTURE
# =============================================================================

segment_count = (
    timeline_df["Segment_ID"]
    .nunique()
)


boundary_count = (

    timeline_df["Segment_ID"]
    .diff()
    .fillna(0)
    .ne(0)
    .sum()

)


print("\n" + "=" * 110)
print("SEGMENT STRUCTURE")
print("=" * 110)

print(
    f"\nSegments       : {segment_count}"
)

print(
    f"Boundaries     : {boundary_count}"
)

print(
    f"Expected gaps  : {segment_count - 1}"
)


structure_valid = (

    segment_count == 10
    and
    boundary_count == 9

)


# =============================================================================
# 7. FINAL RESULT
# =============================================================================

print("\n" + "=" * 110)
print("FINAL TEMPORAL SEGMENTATION VALIDATION")
print("=" * 110)


checks = {

    "Exactly 10 continuous capture segments":
        segment_count == 10,

    "Exactly 9 capture discontinuities":
        boundary_count == 9,

    "All internal gaps <= 5 minutes":
        segment_gap_valid,

    "Segment structure valid":
        structure_valid

}


for name, result in checks.items():

    print(
        f"{'✓' if result else '❌'} "
        f"{name}: {result}"
    )


print("\n" + "=" * 110)


if all(checks.values()):

    print(
        "✓ TEMPORAL SEGMENTATION FULLY VALIDATED."
    )

    print()

    print(
        "The CICIDS2017 dataset contains "
        "10 continuous capture segments separated "
        "by 9 large observation gaps."
    )

    print()

    print(
        "Forecasting windows can now be constructed "
        "WITHOUT crossing capture discontinuities."
    )

    print()

    print(
        "NEXT: construct the minute-level temporal "
        "feature dataset."
    )

else:

    print(
        "❌ Temporal segmentation still requires investigation."
    )


print("=" * 110)

CELL 8B — CORRECTED SEGMENT CONTINUITY VALIDATION

TRUE INTERNAL GAP SUMMARY
--------------------------------------------------------------------------------------------------------------


,Segment_ID,Start_Time,End_Time,Flow_Count,Max_Internal_Gap_Seconds,Max_Internal_Gap_Minutes
0,0,2017-07-03 01:00:01,2017-07-03 05:01:34,222799,13.0,0.216667
1,1,2017-07-03 08:55:58,2017-07-03 12:59:58,307119,24.0,0.400000
2,2,2017-07-04 01:00:00,2017-07-04 05:00:00,220766,60.0,1.000000
3,3,2017-07-04 08:53:00,2017-07-04 12:59:00,225143,60.0,1.000000
4,4,2017-07-05 01:00:00,2017-07-05 05:10:00,227337,60.0,1.000000
5,5,2017-07-05 08:42:00,2017-07-05 12:59:00,465366,60.0,1.000000
6,6,2017-07-06 01:00:00,2017-07-06 05:04:00,288602,60.0,1.000000
7,7,2017-07-06 08:59:00,2017-07-06 12:59:00,170366,60.0,1.000000
8,8,2017-07-07 01:00:00,2017-07-07 05:02:00,512212,60.0,1.000000
9,9,2017-07-07 08:59:00,2017-07-07 12:59:00,191033,60.0,1.000000



CORRECTED CONTINUITY CHECK

All TRUE internal segment gaps <= 300 seconds: True

Segments containing an invalid internal gap: 0
✓ No segment contains a >5-minute internal discontinuity.

SEGMENT STRUCTURE

Segments       : 10
Boundaries     : 9
Expected gaps  : 9

FINAL TEMPORAL SEGMENTATION VALIDATION
✓ Exactly 10 continuous capture segments: True
✓ Exactly 9 capture discontinuities: True
✓ All internal gaps <= 5 minutes: True
✓ Segment structure valid: True

✓ TEMPORAL SEGMENTATION FULLY VALIDATED.

The CICIDS2017 dataset contains 10 continuous capture segments separated by 9 large observation gaps.

Forecasting windows can now be constructed WITHOUT crossing capture discontinuities.

NEXT: construct the minute-level temporal feature dataset.


In [17]:
# ================================================================================================================
# CELL 9 — CANONICAL MINUTE-LEVEL TEMPORAL DATASET
#          + ATTACK TYPE + ENDPOINT + GRAPH GROUND TRUTH
# ================================================================================================================
#
# FINAL PURPOSE
# -------------
# Construct ONE canonical 1-minute CICIDS2017 representation containing:
#
#   MODEL-SIDE INFORMATION
#   ----------------------
#   Aggregated behavioural/network statistics.
#
#   GROUND-TRUTH INFORMATION
#   ------------------------
#   Attack_State
#   Attack_Flow_Count
#   Attack_Ratio
#   Attack_Types
#   Attack_Source_IPs
#   Attack_Destination_IPs
#   Attack_Edges
#   Typed_Attack_Edges
#
# IMPORTANT:
#
#   Source/Destination identities and attack labels are NOT model predictors.
#
#   They are retained only so later forecasting cells can construct:
#
#       Future attack state
#       Future attack type(s)
#       Future source endpoint(s)
#       Future destination/victim-candidate endpoint(s)
#       Future source -> destination edges
#       Future typed edges
#
# NO SPLIT
# NO SCALING
# NO RESAMPLING
# NO MODEL TRAINING
#
# ================================================================================================================


import os
import gc
import json
import numpy as np
import pandas as pd


# ================================================================================================================
# 1. CONFIGURATION
# ================================================================================================================

PROJECT_ROOT = "/content/drive/MyDrive/Network-Attack-Forecasting"

PROCESSED_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "processed"
)

FORECASTING_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "forecasting"
)

os.makedirs(
    FORECASTING_DIR,
    exist_ok=True
)


MINUTE_DATASET_PATH = os.path.join(
    FORECASTING_DIR,
    "minute_level_dataset.parquet"
)

GRAPH_METADATA_PATH = os.path.join(
    FORECASTING_DIR,
    "minute_attack_graph_metadata.parquet"
)

MANIFEST_PATH = os.path.join(
    FORECASTING_DIR,
    "minute_dataset_manifest.json"
)


print("=" * 150)
print(
    "CELL 9 — CANONICAL MINUTE-LEVEL TEMPORAL DATASET "
    "+ ATTACK GRAPH GROUND TRUTH"
)
print("=" * 150)

print("\nConfiguration")
print("-" * 150)

print(f"Project root          : {PROJECT_ROOT}")
print(f"Processed directory   : {PROCESSED_DIR}")
print(f"Forecasting directory : {FORECASTING_DIR}")
print("Temporal resolution   : 1 minute")
print("Split created         : NO")
print("Scaling performed     : NO")
print("Resampling performed  : NO")
print("Model trained         : NO")


# ================================================================================================================
# 2. DISCOVER THE 8 CLEANED CICIDS2017 FILES
# ================================================================================================================

if not os.path.isdir(PROCESSED_DIR):

    raise RuntimeError(
        "Processed directory does not exist:\n"
        f"{PROCESSED_DIR}"
    )


processed_files = sorted(
    [
        os.path.join(PROCESSED_DIR, filename)

        for filename in os.listdir(PROCESSED_DIR)

        if filename.endswith(
            "_cleaned.parquet"
        )
    ]
)


print("\n")
print("=" * 150)
print("CLEANED FLOW FILE DISCOVERY")
print("=" * 150)

print(
    f"\nProcessed Parquet files found: "
    f"{len(processed_files)}"
)

for i, path in enumerate(
    processed_files,
    start=1
):

    print(
        f"{i}. {os.path.basename(path)}"
    )


if len(processed_files) != 8:

    raise RuntimeError(
        "Expected exactly 8 cleaned CICIDS2017 parquet files, "
        f"but found {len(processed_files)}."
    )


print(
    "\n✓ Exactly 8 cleaned CICIDS2017 flow files found."
)


# ================================================================================================================
# 3. HELPERS
# ================================================================================================================

def normalize_columns(frame):

    frame = frame.copy()

    new_columns = [
        str(column).strip()
        for column in frame.columns
    ]

    if len(new_columns) != len(
        set(new_columns)
    ):

        duplicate_columns = (
            pd.Series(new_columns)
            .value_counts()
        )

        duplicate_columns = (
            duplicate_columns[
                duplicate_columns > 1
            ]
            .index
            .tolist()
        )

        raise RuntimeError(
            "Column normalization created duplicate columns:\n"
            f"{duplicate_columns}"
        )

    frame.columns = new_columns

    return frame


def clean_string_series(series):

    return (
        series
        .astype("string")
        .fillna("")
        .str.strip()
    )


def parse_timestamp_series(
    series,
    filename
):
    """
    IMPORTANT FIX.

    Cleaned parquet files already contain parsed datetime64 timestamps.

    Therefore datetime columns MUST NOT be converted to strings and
    reparsed with dayfirst=True.

    Doing that previously transformed dates such as:

        2017-07-03

    into:

        2017-03-07

    This function preserves existing datetime values.

    Only an unexpected string timestamp column uses the CICIDS2017
    day-first parser.
    """

    # ------------------------------------------------------------------------------------------------------------
    # EXPECTED CASE: CLEANED PARQUET ALREADY STORES DATETIME
    # ------------------------------------------------------------------------------------------------------------

    if pd.api.types.is_datetime64_any_dtype(
        series
    ):

        parsed = pd.to_datetime(
            series,
            errors="coerce"
        )

    # ------------------------------------------------------------------------------------------------------------
    # FALLBACK: STRING TIMESTAMP
    # ------------------------------------------------------------------------------------------------------------

    else:

        raw = clean_string_series(
            series
        )

        parsed = pd.to_datetime(
            raw,
            dayfirst=True,
            errors="coerce"
        )


    invalid = parsed.isna()

    if invalid.any():

        examples = (
            series.loc[invalid]
            .head(10)
            .tolist()
        )

        raise RuntimeError(
            f"{filename}: "
            f"{int(invalid.sum()):,} timestamps "
            f"could not be parsed.\n"
            f"Examples: {examples}"
        )

    return parsed


def json_unique_strings(values):

    cleaned = sorted(
        {
            str(value).strip()

            for value in values

            if pd.notna(value)
            and str(value).strip() != ""
        }
    )

    return json.dumps(
        cleaned,
        ensure_ascii=False
    )


def build_attack_edges(frame):

    if frame.empty:

        return "[]"

    edges = set()

    for src, dst in zip(
        frame["Source IP"],
        frame["Destination IP"]
    ):

        src = str(src).strip()
        dst = str(dst).strip()

        if src and dst:

            edges.add(
                (
                    src,
                    dst
                )
            )

    result = sorted(
        edges
    )

    return json.dumps(
        result,
        ensure_ascii=False
    )


def build_typed_attack_edges(frame):

    if frame.empty:

        return "[]"

    edges = set()

    for src, dst, attack_type in zip(
        frame["Source IP"],
        frame["Destination IP"],
        frame["Attack_Type"]
    ):

        src = str(src).strip()
        dst = str(dst).strip()
        attack_type = str(
            attack_type
        ).strip()

        if (
            src
            and dst
            and attack_type
        ):

            edges.add(
                (
                    src,
                    dst,
                    attack_type
                )
            )

    result = sorted(
        edges
    )

    return json.dumps(
        result,
        ensure_ascii=False
    )


def json_cardinality(value):

    try:

        parsed = json.loads(
            value
        )

        return len(parsed)

    except Exception:

        return 0


def merge_json_values(series):

    merged = set()

    for value in series:

        if pd.isna(value):
            continue

        text = str(value).strip()

        if not text:
            continue

        try:

            parsed = json.loads(
                text
            )

        except Exception:

            continue

        for item in parsed:

            if isinstance(
                item,
                list
            ):

                merged.add(
                    tuple(item)
                )

            else:

                merged.add(
                    str(item)
                )

    result = sorted(
        merged,
        key=lambda x: str(x)
    )

    return json.dumps(
        result,
        ensure_ascii=False
    )


# ================================================================================================================
# 4. SCHEMA INSPECTION
# ================================================================================================================

schema_probe = pd.read_parquet(
    processed_files[0]
)

schema_probe = normalize_columns(
    schema_probe
)


print("\n")
print("=" * 150)
print("CLEANED FLOW SCHEMA")
print("=" * 150)

print(
    f"\nColumns in cleaned parquet: "
    f"{len(schema_probe.columns)}"
)


REQUIRED_COLUMNS = {
    "Source IP",
    "Destination IP",
    "Timestamp",
    "Label",
}


missing_required = (
    REQUIRED_COLUMNS
    - set(schema_probe.columns)
)


if missing_required:

    raise RuntimeError(
        "Required cleaned-flow columns are missing:\n"
        f"{sorted(missing_required)}"
    )


print(
    "\n✓ Required raw identity/target columns exist:"
)

for column in sorted(
    REQUIRED_COLUMNS
):

    print(
        f"  ✓ {column}"
    )


# ================================================================================================================
# 5. DEFINE FLOW-LEVEL MODEL FEATURE CANDIDATES
# ================================================================================================================

#
# These raw columns must NEVER become numerical model predictors.
#

RAW_NON_FEATURE_COLUMNS = {
    "Flow ID",
    "Source IP",
    "Destination IP",
    "Timestamp",
    "Label",

    # possible preprocessing columns
    "Attack_Type",
    "Attack_Label",
    "Minute",
    "Segment_ID",
}


numeric_flow_features = [
    column

    for column in schema_probe.columns

    if (
        column not in RAW_NON_FEATURE_COLUMNS

        and pd.api.types.is_numeric_dtype(
            schema_probe[column]
        )
    )
]


if len(
    numeric_flow_features
) == 0:

    raise RuntimeError(
        "No numerical flow features were discovered."
    )


print("\n")
print("=" * 150)
print("FLOW-LEVEL FEATURE DISCOVERY")
print("=" * 150)

print(
    f"\nNumeric flow-level feature columns discovered: "
    f"{len(numeric_flow_features)}"
)

print(
    "\nNOTE:"
)

print(
    "These are FLOW-LEVEL inputs."
)

print(
    "They are not automatically treated as the final model "
    "predictor list."
)

print(
    "Raw Source IP / Destination IP / Timestamp / Label "
    "are excluded."
)


# ================================================================================================================
# 6. STORAGE FOR FILE-BY-FILE MINUTE AGGREGATION
# ================================================================================================================

minute_feature_parts = []
minute_metadata_parts = []


total_rows = 0
total_attack_flows = 0
total_benign_flows = 0


# ================================================================================================================
# 7. PROCESS EACH CLEANED FILE
# ================================================================================================================

print("\n")
print("=" * 150)
print("FILE-BY-FILE MINUTE AGGREGATION")
print("=" * 150)


for file_number, path in enumerate(
    processed_files,
    start=1
):

    filename = os.path.basename(
        path
    )


    print("\n")
    print("#" * 150)

    print(
        f"FILE {file_number}/"
        f"{len(processed_files)} — "
        f"{filename}"
    )

    print("#" * 150)


    df = pd.read_parquet(
        path
    )

    df = normalize_columns(
        df
    )


    # ============================================================================================================
    # REQUIRED COLUMN VALIDATION
    # ============================================================================================================

    missing = (
        REQUIRED_COLUMNS
        - set(df.columns)
    )

    if missing:

        raise RuntimeError(
            f"{filename}: missing required columns:\n"
            f"{sorted(missing)}"
        )


    # ============================================================================================================
    # TIMESTAMP — CORRECTED
    # ============================================================================================================

    df["Timestamp"] = parse_timestamp_series(
        df["Timestamp"],
        filename
    )


    df["Minute"] = (
        df["Timestamp"]
        .dt.floor("min")
    )


    # ============================================================================================================
    # LABELS
    # ============================================================================================================

    df["Attack_Type"] = (
        clean_string_series(
            df["Label"]
        )
    )


    df["Attack_Label"] = (
        df["Attack_Type"]
        .str.upper()
        .ne("BENIGN")
        .astype(np.int8)
    )


    # ============================================================================================================
    # ENDPOINTS
    # ============================================================================================================

    df["Source IP"] = (
        clean_string_series(
            df["Source IP"]
        )
    )


    df["Destination IP"] = (
        clean_string_series(
            df["Destination IP"]
        )
    )


    # ============================================================================================================
    # FILE SUMMARY
    # ============================================================================================================

    rows = len(df)

    attack_flows = int(
        df["Attack_Label"].sum()
    )

    benign_flows = (
        rows
        - attack_flows
    )


    total_rows += rows

    total_attack_flows += (
        attack_flows
    )

    total_benign_flows += (
        benign_flows
    )


    print(
        f"Rows              : {rows:,}"
    )

    print(
        f"Start timestamp   : "
        f"{df['Timestamp'].min()}"
    )

    print(
        f"End timestamp     : "
        f"{df['Timestamp'].max()}"
    )

    print(
        f"Attack flows      : "
        f"{attack_flows:,}"
    )

    print(
        f"Benign flows      : "
        f"{benign_flows:,}"
    )


    # ============================================================================================================
    # 7A. NUMERICAL CLEANUP
    # ============================================================================================================

    for column in numeric_flow_features:

        if column not in df.columns:

            raise RuntimeError(
                f"{filename}: expected numerical column "
                f"{column!r} is missing."
            )

        df[column] = pd.to_numeric(
            df[column],
            errors="coerce"
        )

        df[column] = (
            df[column]
            .replace(
                [
                    np.inf,
                    -np.inf
                ],
                np.nan
            )
        )


    # ============================================================================================================
    # 7B. MINUTE-LEVEL BEHAVIOURAL FEATURES
    # ============================================================================================================

    grouped = df.groupby(
        "Minute",
        sort=True,
        observed=True
    )


    #
    # IMPORTANT:
    #
    # We intentionally use MEAN here as the canonical representation
    # for raw numerical CICFlowMeter columns rather than multiplying
    # every raw feature into mean/std/min/max.
    #
    # Additional structural minute-level features are created separately.
    #
    # This prevents this replacement cell from silently exploding the
    # model dimensionality.
    #

    feature_minute = (
        grouped[
            numeric_flow_features
        ]
        .mean()
        .reset_index()
    )


    #
    # Rename raw numerical aggregate columns explicitly so there is
    # no ambiguity that these are minute aggregates.
    #

    rename_map = {
        column:
            f"{column}__mean"

        for column in numeric_flow_features
    }


    feature_minute = (
        feature_minute
        .rename(
            columns=rename_map
        )
    )


    # ------------------------------------------------------------------------------------------------------------
    # Structural traffic features
    # ------------------------------------------------------------------------------------------------------------

    structural = (
        grouped
        .size()
        .rename("Flow_Count")
        .to_frame()
    )


    structural[
        "Unique_Source_IPs"
    ] = (
        grouped[
            "Source IP"
        ]
        .nunique()
    )


    structural[
        "Unique_Destination_IPs"
    ] = (
        grouped[
            "Destination IP"
        ]
        .nunique()
    )


    if "Source Port" in df.columns:

        structural[
            "Unique_Source_Ports"
        ] = (
            grouped[
                "Source Port"
            ]
            .nunique()
        )


    if "Destination Port" in df.columns:

        structural[
            "Unique_Destination_Ports"
        ] = (
            grouped[
                "Destination Port"
            ]
            .nunique()
        )


    if "Protocol" in df.columns:

        structural[
            "Unique_Protocols"
        ] = (
            grouped[
                "Protocol"
            ]
            .nunique()
        )


    structural = (
        structural
        .reset_index()
    )


    feature_minute = (
        feature_minute
        .merge(
            structural,
            on="Minute",
            how="left",
            validate="one_to_one"
        )
    )


    minute_feature_parts.append(
        feature_minute
    )


    # ============================================================================================================
    # 7C. MINUTE-LEVEL ATTACK STATE
    # ============================================================================================================

    state_metadata = (
        grouped
        .agg(
            Total_Flow_Count=(
                "Attack_Label",
                "size"
            ),

            Attack_Flow_Count=(
                "Attack_Label",
                "sum"
            ),
        )
        .reset_index()
    )


    state_metadata[
        "Benign_Flow_Count"
    ] = (
        state_metadata[
            "Total_Flow_Count"
        ]
        -
        state_metadata[
            "Attack_Flow_Count"
        ]
    )


    state_metadata[
        "Attack_State"
    ] = (
        state_metadata[
            "Attack_Flow_Count"
        ]
        .gt(0)
        .astype(np.int8)
    )


    state_metadata[
        "Attack_Ratio"
    ] = (
        state_metadata[
            "Attack_Flow_Count"
        ]
        /
        state_metadata[
            "Total_Flow_Count"
        ]
    )


    # ============================================================================================================
    # 7D. ATTACK TYPE + ENDPOINT + GRAPH GROUND TRUTH
    # ============================================================================================================

    attack_df = (
        df.loc[
            df["Attack_Label"].eq(1),
            [
                "Minute",
                "Source IP",
                "Destination IP",
                "Attack_Type",
            ]
        ]
        .copy()
    )


    graph_records = []


    if not attack_df.empty:

        for minute, attack_group in attack_df.groupby(
            "Minute",
            sort=True,
            observed=True
        ):

            attack_types = (
                json_unique_strings(
                    attack_group[
                        "Attack_Type"
                    ]
                )
            )


            attack_sources = (
                json_unique_strings(
                    attack_group[
                        "Source IP"
                    ]
                )
            )


            attack_destinations = (
                json_unique_strings(
                    attack_group[
                        "Destination IP"
                    ]
                )
            )


            attack_edges = (
                build_attack_edges(
                    attack_group
                )
            )


            typed_attack_edges = (
                build_typed_attack_edges(
                    attack_group
                )
            )


            graph_records.append(
                {
                    "Minute":
                        minute,

                    "Attack_Types":
                        attack_types,

                    "Attack_Source_IPs":
                        attack_sources,

                    "Attack_Destination_IPs":
                        attack_destinations,

                    "Attack_Edges":
                        attack_edges,

                    "Typed_Attack_Edges":
                        typed_attack_edges,
                }
            )


    graph_metadata = pd.DataFrame(
        graph_records,
        columns=[
            "Minute",
            "Attack_Types",
            "Attack_Source_IPs",
            "Attack_Destination_IPs",
            "Attack_Edges",
            "Typed_Attack_Edges",
        ]
    )


    minute_metadata = (
        state_metadata
        .merge(
            graph_metadata,
            on="Minute",
            how="left",
            validate="one_to_one"
        )
    )


    # ------------------------------------------------------------------------------------------------------------
    # Benign minute => empty graph metadata
    # ------------------------------------------------------------------------------------------------------------

    JSON_COLUMNS = [
        "Attack_Types",
        "Attack_Source_IPs",
        "Attack_Destination_IPs",
        "Attack_Edges",
        "Typed_Attack_Edges",
    ]


    for column in JSON_COLUMNS:

        minute_metadata[column] = (
            minute_metadata[column]
            .fillna("[]")
        )


    minute_metadata[
        "Attack_Type_Count"
    ] = (
        minute_metadata[
            "Attack_Types"
        ]
        .map(
            json_cardinality
        )
        .astype(np.int32)
    )


    minute_metadata[
        "Attack_Source_Count"
    ] = (
        minute_metadata[
            "Attack_Source_IPs"
        ]
        .map(
            json_cardinality
        )
        .astype(np.int32)
    )


    minute_metadata[
        "Attack_Destination_Count"
    ] = (
        minute_metadata[
            "Attack_Destination_IPs"
        ]
        .map(
            json_cardinality
        )
        .astype(np.int32)
    )


    minute_metadata[
        "Attack_Edge_Count"
    ] = (
        minute_metadata[
            "Attack_Edges"
        ]
        .map(
            json_cardinality
        )
        .astype(np.int32)
    )


    minute_metadata[
        "Typed_Attack_Edge_Count"
    ] = (
        minute_metadata[
            "Typed_Attack_Edges"
        ]
        .map(
            json_cardinality
        )
        .astype(np.int32)
    )


    minute_metadata_parts.append(
        minute_metadata
    )


    # ============================================================================================================
    # RAM CLEANUP
    # ============================================================================================================

    del grouped
    del feature_minute
    del structural
    del state_metadata
    del attack_df
    del graph_metadata
    del minute_metadata
    del df

    gc.collect()


# ================================================================================================================
# 8. GLOBAL COMBINATION
# ================================================================================================================

print("\n")
print("=" * 150)
print("COMBINING MINUTE-LEVEL PARTS")
print("=" * 150)


feature_all = pd.concat(
    minute_feature_parts,
    ignore_index=True
)


metadata_all = pd.concat(
    minute_metadata_parts,
    ignore_index=True
)


# ================================================================================================================
# 9. VALIDATE TIMESTAMP DATES BEFORE ANY SEGMENT ASSIGNMENT
# ================================================================================================================

observed_dates = sorted(
    pd.Series(
        metadata_all[
            "Minute"
        ].dt.normalize().unique()
    )
    .tolist()
)


expected_dates = {
    pd.Timestamp("2017-07-03"),
    pd.Timestamp("2017-07-04"),
    pd.Timestamp("2017-07-05"),
    pd.Timestamp("2017-07-06"),
    pd.Timestamp("2017-07-07"),
}


observed_date_set = set(
    pd.Timestamp(value)
    for value in observed_dates
)


print("\n")
print("=" * 150)
print("DATE INTEGRITY CHECK")
print("=" * 150)

print("\nObserved capture dates:")

for value in sorted(
    observed_date_set
):

    print(
        f"  {value.date()}"
    )


unexpected_dates = (
    observed_date_set
    - expected_dates
)


missing_dates = (
    expected_dates
    - observed_date_set
)


if unexpected_dates:

    raise RuntimeError(
        "Unexpected capture dates detected BEFORE segmentation:\n"
        f"{sorted(unexpected_dates)}"
    )


if missing_dates:

    raise RuntimeError(
        "Expected CICIDS2017 capture dates are missing:\n"
        f"{sorted(missing_dates)}"
    )


print(
    "\n✓ Capture dates correctly preserved as "
    "July 3–7, 2017."
)


# ================================================================================================================
# 10. COMBINE DUPLICATE MINUTES IF MULTIPLE FILES TOUCH THE SAME MINUTE
# ================================================================================================================

feature_columns = [
    column

    for column in feature_all.columns

    if column != "Minute"
]


#
# There should not normally be overlapping capture files for the same
# exact minute except potentially file boundaries.
#
# Mean is used for duplicate already-aggregated behavioural features.
#

feature_all = (
    feature_all
    .groupby(
        "Minute",
        as_index=False,
        sort=True
    )[feature_columns]
    .mean()
)


# ---------------------------------------------------------------------------------------------------------------
# Exact flow/attack counts
# ---------------------------------------------------------------------------------------------------------------

metadata_counts = (
    metadata_all
    .groupby(
        "Minute",
        as_index=False,
        sort=True
    )
    .agg(
        Total_Flow_Count=(
            "Total_Flow_Count",
            "sum"
        ),

        Attack_Flow_Count=(
            "Attack_Flow_Count",
            "sum"
        ),

        Benign_Flow_Count=(
            "Benign_Flow_Count",
            "sum"
        ),
    )
)


metadata_counts[
    "Attack_State"
] = (
    metadata_counts[
        "Attack_Flow_Count"
    ]
    .gt(0)
    .astype(np.int8)
)


metadata_counts[
    "Attack_Ratio"
] = (
    metadata_counts[
        "Attack_Flow_Count"
    ]
    /
    metadata_counts[
        "Total_Flow_Count"
    ]
)


# ---------------------------------------------------------------------------------------------------------------
# Merge graph metadata
# ---------------------------------------------------------------------------------------------------------------

JSON_COLUMNS = [
    "Attack_Types",
    "Attack_Source_IPs",
    "Attack_Destination_IPs",
    "Attack_Edges",
    "Typed_Attack_Edges",
]


metadata_json = (
    metadata_all
    .groupby(
        "Minute",
        as_index=False,
        sort=True
    )[JSON_COLUMNS]
    .agg(
        merge_json_values
    )
)


metadata_all = (
    metadata_counts
    .merge(
        metadata_json,
        on="Minute",
        how="left",
        validate="one_to_one"
    )
)


metadata_all[
    "Attack_Type_Count"
] = (
    metadata_all[
        "Attack_Types"
    ]
    .map(
        json_cardinality
    )
    .astype(np.int32)
)


metadata_all[
    "Attack_Source_Count"
] = (
    metadata_all[
        "Attack_Source_IPs"
    ]
    .map(
        json_cardinality
    )
    .astype(np.int32)
)


metadata_all[
    "Attack_Destination_Count"
] = (
    metadata_all[
        "Attack_Destination_IPs"
    ]
    .map(
        json_cardinality
    )
    .astype(np.int32)
)


metadata_all[
    "Attack_Edge_Count"
] = (
    metadata_all[
        "Attack_Edges"
    ]
    .map(
        json_cardinality
    )
    .astype(np.int32)
)


metadata_all[
    "Typed_Attack_Edge_Count"
] = (
    metadata_all[
        "Typed_Attack_Edges"
    ]
    .map(
        json_cardinality
    )
    .astype(np.int32)
)


# ================================================================================================================
# 11. VALIDATED TEMPORAL SEGMENTS FROM CELL 8B
# ================================================================================================================

SEGMENT_INTERVALS = [

    (
        0,
        pd.Timestamp(
            "2017-07-03 01:00:00"
        ),
        pd.Timestamp(
            "2017-07-03 05:01:00"
        )
    ),

    (
        1,
        pd.Timestamp(
            "2017-07-03 08:55:00"
        ),
        pd.Timestamp(
            "2017-07-03 12:59:00"
        )
    ),

    (
        2,
        pd.Timestamp(
            "2017-07-04 01:00:00"
        ),
        pd.Timestamp(
            "2017-07-04 05:00:00"
        )
    ),

    (
        3,
        pd.Timestamp(
            "2017-07-04 08:53:00"
        ),
        pd.Timestamp(
            "2017-07-04 12:59:00"
        )
    ),

    (
        4,
        pd.Timestamp(
            "2017-07-05 01:00:00"
        ),
        pd.Timestamp(
            "2017-07-05 05:10:00"
        )
    ),

    (
        5,
        pd.Timestamp(
            "2017-07-05 08:42:00"
        ),
        pd.Timestamp(
            "2017-07-05 12:59:00"
        )
    ),

    (
        6,
        pd.Timestamp(
            "2017-07-06 01:00:00"
        ),
        pd.Timestamp(
            "2017-07-06 05:04:00"
        )
    ),

    (
        7,
        pd.Timestamp(
            "2017-07-06 08:59:00"
        ),
        pd.Timestamp(
            "2017-07-06 12:59:00"
        )
    ),

    (
        8,
        pd.Timestamp(
            "2017-07-07 01:00:00"
        ),
        pd.Timestamp(
            "2017-07-07 05:02:00"
        )
    ),

    (
        9,
        pd.Timestamp(
            "2017-07-07 08:59:00"
        ),
        pd.Timestamp(
            "2017-07-07 12:59:00"
        )
    ),
]


# ================================================================================================================
# 12. ASSIGN SEGMENT IDs
# ================================================================================================================

def assign_segment(
    timestamp
):

    for (
        segment_id,
        start,
        end
    ) in SEGMENT_INTERVALS:

        if (
            start
            <= timestamp
            <= end
        ):

            return segment_id

    return pd.NA


metadata_all[
    "Segment_ID"
] = (
    metadata_all[
        "Minute"
    ]
    .map(
        assign_segment
    )
)


outside_segment = (
    metadata_all[
        "Segment_ID"
    ]
    .isna()
)


if outside_segment.any():

    bad_rows = (
        metadata_all.loc[
            outside_segment,
            [
                "Minute",
                "Attack_State",
            ]
        ]
        .head(20)
    )

    raise RuntimeError(
        f"{int(outside_segment.sum()):,} minute rows fall "
        "outside the validated 10 capture segments.\n\n"
        f"{bad_rows.to_string(index=False)}"
    )


metadata_all[
    "Segment_ID"
] = (
    metadata_all[
        "Segment_ID"
    ]
    .astype(np.int16)
)


print(
    "\n✓ Every observed minute belongs to one of "
    "the 10 validated capture segments."
)


# ================================================================================================================
# 13. CREATE THE EXACT VALIDATED 1-MINUTE TIMELINE
# ================================================================================================================

timeline_parts = []


for (
    segment_id,
    start,
    end
) in SEGMENT_INTERVALS:

    timeline_parts.append(

        pd.DataFrame(
            {
                "Segment_ID":
                    segment_id,

                "Minute":
                    pd.date_range(
                        start=start,
                        end=end,
                        freq="1min"
                    ),
            }
        )
    )


timeline = pd.concat(
    timeline_parts,
    ignore_index=True
)


if len(timeline) != 2454:

    raise RuntimeError(
        "Validated temporal intervals should produce exactly "
        f"2,454 minute rows, but produced {len(timeline):,}."
    )


print(
    f"Validated timeline rows: "
    f"{len(timeline):,}"
)


# ================================================================================================================
# 14. MERGE EVERYTHING INTO CANONICAL minute_df
# ================================================================================================================

minute_df = (
    timeline
    .merge(
        feature_all,
        on="Minute",
        how="left",
        validate="one_to_one"
    )
)


minute_df = (
    minute_df
    .merge(
        metadata_all,
        on=[
            "Segment_ID",
            "Minute"
        ],
        how="left",
        validate="one_to_one"
    )
)


# ================================================================================================================
# 15. OBSERVATION COMPLETENESS
# ================================================================================================================

missing_observations = (
    minute_df[
        "Total_Flow_Count"
    ]
    .isna()
)


if missing_observations.any():

    bad = (
        minute_df.loc[
            missing_observations,
            [
                "Segment_ID",
                "Minute"
            ]
        ]
        .head(30)
    )

    raise RuntimeError(
        f"{int(missing_observations.sum()):,} validated minutes "
        "contain no traffic observation.\n\n"
        f"{bad.to_string(index=False)}"
    )


print(
    "\n✓ Every validated minute contains observed traffic."
)


# ================================================================================================================
# 16. CORRECT FLOW COUNT USING EXACT RAW COUNT
# ================================================================================================================

if "Flow_Count" in minute_df.columns:

    minute_df[
        "Flow_Count"
    ] = (
        minute_df[
            "Total_Flow_Count"
        ]
        .astype(np.int64)
    )


# ================================================================================================================
# 17. SORT + UNIQUE KEY VALIDATION
# ================================================================================================================

minute_df = (
    minute_df
    .sort_values(
        [
            "Segment_ID",
            "Minute"
        ]
    )
    .reset_index(
        drop=True
    )
)


duplicate_keys = int(
    minute_df
    .duplicated(
        [
            "Segment_ID",
            "Minute"
        ]
    )
    .sum()
)


if duplicate_keys != 0:

    raise RuntimeError(
        "Duplicate (Segment_ID, Minute) keys detected: "
        f"{duplicate_keys:,}"
    )


print(
    "\n✓ (Segment_ID, Minute) is unique."
)


# ================================================================================================================
# 18. MINUTE CONTINUITY VALIDATION
# ================================================================================================================

continuity_records = []

continuity_errors = 0


for segment_id, group in minute_df.groupby(
    "Segment_ID",
    sort=True
):

    group = (
        group
        .sort_values(
            "Minute"
        )
    )


    differences = (
        group[
            "Minute"
        ]
        .diff()
        .dropna()
    )


    non_one_minute = int(
        differences
        .ne(
            pd.Timedelta(
                minutes=1
            )
        )
        .sum()
    )


    continuity_errors += (
        non_one_minute
    )


    continuity_records.append(
        {
            "Segment_ID":
                int(segment_id),

            "Rows":
                len(group),

            "Start_Minute":
                group[
                    "Minute"
                ].min(),

            "End_Minute":
                group[
                    "Minute"
                ].max(),

            "Non_1min_Gaps":
                non_one_minute,
        }
    )


continuity_df = pd.DataFrame(
    continuity_records
)


print("\n")
print("=" * 150)
print("MINUTE-LEVEL CONTINUITY CHECK")
print("=" * 150)

print(
    continuity_df.to_string(
        index=False
    )
)


if continuity_errors != 0:

    raise RuntimeError(
        "Internal 1-minute continuity validation failed. "
        f"Errors: {continuity_errors:,}"
    )


print(
    "\n✓ Every segment is exactly continuous "
    "at 1-minute resolution."
)


# ================================================================================================================
# 19. GROUND-TRUTH CONSISTENCY
# ================================================================================================================

print("\n")
print("=" * 150)
print("GROUND-TRUTH CONSISTENCY AUDIT")
print("=" * 150)


expected_attack_state = (
    minute_df[
        "Attack_Flow_Count"
    ]
    .gt(0)
    .astype(np.int8)
)


attack_state_errors = int(
    (
        minute_df[
            "Attack_State"
        ]
        .astype(np.int8)
        != expected_attack_state
    )
    .sum()
)


expected_attack_ratio = (
    minute_df[
        "Attack_Flow_Count"
    ]
    /
    minute_df[
        "Total_Flow_Count"
    ]
)


attack_ratio_errors = int(
    (
        ~np.isclose(
            minute_df[
                "Attack_Ratio"
            ].astype(float),

            expected_attack_ratio.astype(
                float
            ),

            rtol=1e-10,
            atol=1e-12,
            equal_nan=True
        )
    )
    .sum()
)


count_mapping = {

    "Attack_Type_Count":
        "Attack_Types",

    "Attack_Source_Count":
        "Attack_Source_IPs",

    "Attack_Destination_Count":
        "Attack_Destination_IPs",

    "Attack_Edge_Count":
        "Attack_Edges",

    "Typed_Attack_Edge_Count":
        "Typed_Attack_Edges",
}


cardinality_errors = {}


for count_column, json_column in count_mapping.items():

    recalculated = (
        minute_df[
            json_column
        ]
        .map(
            json_cardinality
        )
        .astype(int)
    )


    stored = (
        minute_df[
            count_column
        ]
        .astype(int)
    )


    cardinality_errors[
        count_column
    ] = int(
        (
            recalculated
            != stored
        )
        .sum()
    )


benign_mask = (
    minute_df[
        "Attack_State"
    ]
    .eq(0)
)


attack_mask = (
    minute_df[
        "Attack_State"
    ]
    .eq(1)
)


benign_metadata_errors = int(
    (
        minute_df.loc[
            benign_mask,
            list(
                count_mapping.keys()
            )
        ]
        .sum(
            axis=1
        )
        .ne(0)
    )
    .sum()
)


attack_without_type = int(
    minute_df.loc[
        attack_mask,
        "Attack_Type_Count"
    ]
    .eq(0)
    .sum()
)


attack_without_source = int(
    minute_df.loc[
        attack_mask,
        "Attack_Source_Count"
    ]
    .eq(0)
    .sum()
)


attack_without_destination = int(
    minute_df.loc[
        attack_mask,
        "Attack_Destination_Count"
    ]
    .eq(0)
    .sum()
)


attack_without_edge = int(
    minute_df.loc[
        attack_mask,
        "Attack_Edge_Count"
    ]
    .eq(0)
    .sum()
)


attack_without_typed_edge = int(
    minute_df.loc[
        attack_mask,
        "Typed_Attack_Edge_Count"
    ]
    .eq(0)
    .sum()
)


print(
    f"\nAttack-state inconsistencies          : "
    f"{attack_state_errors:,}"
)

print(
    f"Attack-ratio inconsistencies          : "
    f"{attack_ratio_errors:,}"
)

print(
    f"Benign metadata inconsistencies       : "
    f"{benign_metadata_errors:,}"
)

print(
    f"Attack minutes without type           : "
    f"{attack_without_type:,}"
)

print(
    f"Attack minutes without source         : "
    f"{attack_without_source:,}"
)

print(
    f"Attack minutes without destination    : "
    f"{attack_without_destination:,}"
)

print(
    f"Attack minutes without edge           : "
    f"{attack_without_edge:,}"
)

print(
    f"Attack minutes without typed edge     : "
    f"{attack_without_typed_edge:,}"
)


print(
    "\nJSON cardinality inconsistencies:"
)


for column, error_count in (
    cardinality_errors.items()
):

    print(
        f"  {column:<32}: "
        f"{error_count:,}"
    )


if (
    attack_state_errors != 0
    or attack_ratio_errors != 0
    or benign_metadata_errors != 0
    or attack_without_type != 0
    or attack_without_source != 0
    or attack_without_destination != 0
    or attack_without_edge != 0
    or attack_without_typed_edge != 0
    or any(
        value != 0
        for value
        in cardinality_errors.values()
    )
):

    raise RuntimeError(
        "Ground-truth consistency audit failed."
    )


print(
    "\n✓ Attack metadata and graph ground truth "
    "are internally consistent."
)


# ================================================================================================================
# 20. ATTACK-TYPE VOCABULARY
# ================================================================================================================

attack_type_set = set()


for value in minute_df.loc[
    attack_mask,
    "Attack_Types"
]:

    parsed = json.loads(
        value
    )

    for attack_type in parsed:

        attack_type_set.add(
            str(attack_type)
        )


ATTACK_TYPE_VOCABULARY = sorted(
    attack_type_set
)


print("\n")
print("=" * 150)
print("ATTACK-TYPE VOCABULARY")
print("=" * 150)

print(
    f"\nAttack classes represented: "
    f"{len(ATTACK_TYPE_VOCABULARY)}"
)


for index, attack_type in enumerate(
    ATTACK_TYPE_VOCABULARY,
    start=1
):

    print(
        f"{index:>2}. {attack_type}"
    )


# ================================================================================================================
# 21. DEFINE FINAL GROUND-TRUTH CONTRACT
# ================================================================================================================

GRAPH_GROUND_TRUTH_COLUMNS = [

    "Attack_Types",

    "Attack_Source_IPs",

    "Attack_Destination_IPs",

    "Attack_Edges",

    "Typed_Attack_Edges",

    "Attack_Type_Count",

    "Attack_Source_Count",

    "Attack_Destination_Count",

    "Attack_Edge_Count",

    "Typed_Attack_Edge_Count",
]


GROUND_TRUTH_COLUMNS = [

    "Segment_ID",

    "Minute",

    "Attack_State",

    "Attack_Flow_Count",

    "Benign_Flow_Count",

    "Attack_Ratio",

    *GRAPH_GROUND_TRUTH_COLUMNS,
]


#
# Total_Flow_Count is descriptive metadata because Flow_Count already
# exists as the behavioural representation.
#

NON_MODEL_COLUMNS = set(
    GROUND_TRUTH_COLUMNS
) | {
    "Total_Flow_Count"
}


MODEL_FEATURE_COLUMNS = [

    column

    for column in minute_df.columns

    if (
        column
        not in NON_MODEL_COLUMNS

        and pd.api.types.is_numeric_dtype(
            minute_df[column]
        )
    )
]


FORBIDDEN_MODEL_FEATURES = {

    "Segment_ID",

    "Minute",

    "Attack_State",

    "Attack_Flow_Count",

    "Benign_Flow_Count",

    "Attack_Ratio",

    "Attack_Types",

    "Attack_Source_IPs",

    "Attack_Destination_IPs",

    "Attack_Edges",

    "Typed_Attack_Edges",

    "Attack_Type_Count",

    "Attack_Source_Count",

    "Attack_Destination_Count",

    "Attack_Edge_Count",

    "Typed_Attack_Edge_Count",

    "Label",

    "Attack_Label",

    "Attack_Type",

    "Source IP",

    "Destination IP",

    "Flow ID",

    "Timestamp",
}


leakage_columns = sorted(
    set(
        MODEL_FEATURE_COLUMNS
    )
    &
    FORBIDDEN_MODEL_FEATURES
)


if leakage_columns:

    raise RuntimeError(
        "LABEL/TARGET LEAKAGE DETECTED:\n"
        f"{leakage_columns}"
    )


print("\n")
print("=" * 150)
print("MODEL / GROUND-TRUTH SEPARATION")
print("=" * 150)

print(
    f"\nCurrent numerical minute features : "
    f"{len(MODEL_FEATURE_COLUMNS):,}"
)

print(
    f"Ground-truth columns              : "
    f"{len(GROUND_TRUTH_COLUMNS):,}"
)

print(
    f"Forbidden model-feature entries   : "
    f"{len(FORBIDDEN_MODEL_FEATURES):,}"
)

print(
    "\n✓ Attack labels, attack types, endpoint identities "
    "and graph targets are excluded from model predictors."
)


# ================================================================================================================
# 22. NUMERICAL FINITENESS AUDIT
# ================================================================================================================

print("\n")
print("=" * 150)
print("NUMERICAL FEATURE FINITENESS AUDIT")
print("=" * 150)


inf_counts = {}

nan_counts = {}


for column in MODEL_FEATURE_COLUMNS:

    values = pd.to_numeric(
        minute_df[column],
        errors="coerce"
    )


    inf_count = int(
        np.isinf(
            values.to_numpy(
                dtype=float
            )
        )
        .sum()
    )


    nan_count = int(
        values
        .isna()
        .sum()
    )


    if inf_count:

        inf_counts[
            column
        ] = inf_count


    if nan_count:

        nan_counts[
            column
        ] = nan_count


print(
    f"\nPredictor columns containing infinity : "
    f"{len(inf_counts)}"
)

print(
    f"Predictor columns containing NaN      : "
    f"{len(nan_counts)}"
)


if inf_counts:

    raise RuntimeError(
        "Infinity remains in one or more model features:\n"
        f"{inf_counts}"
    )


if nan_counts:

    print(
        "\nNOTE:"
    )

    print(
        "NaN values are NOT statistically imputed in Cell 9."
    )

    print(
        "Any learned imputation must be fitted on TRAINING data only "
        "after the split."
    )

    print(
        "\nColumns containing NaN:"
    )

    for column, count in (
        nan_counts.items()
    ):

        print(
            f"  {column:<55} "
            f"{count:,}"
        )

else:

    print(
        "\n✓ All current numerical minute features are finite."
    )


# ================================================================================================================
# 23. FINAL DATASET SUMMARY
# ================================================================================================================

segment_summary = (
    minute_df
    .groupby(
        "Segment_ID",
        as_index=False
    )
    .agg(
        Rows=(
            "Minute",
            "size"
        ),

        Start_Minute=(
            "Minute",
            "min"
        ),

        End_Minute=(
            "Minute",
            "max"
        ),

        Attack_Minutes=(
            "Attack_State",
            "sum"
        ),
    )
)


print("\n")
print("=" * 150)
print("FINAL SEGMENT SUMMARY")
print("=" * 150)

print(
    segment_summary.to_string(
        index=False
    )
)


# ================================================================================================================
# 24. SAVE CANONICAL OUTPUTS
# ================================================================================================================

minute_df.to_parquet(
    MINUTE_DATASET_PATH,
    index=False
)


cell9_graph_metadata = (
    minute_df[
        [
            "Segment_ID",
            "Minute",

            "Attack_State",

            "Attack_Flow_Count",

            "Attack_Ratio",

            "Attack_Types",

            "Attack_Source_IPs",

            "Attack_Destination_IPs",

            "Attack_Edges",

            "Typed_Attack_Edges",

            "Attack_Type_Count",

            "Attack_Source_Count",

            "Attack_Destination_Count",

            "Attack_Edge_Count",

            "Typed_Attack_Edge_Count",
        ]
    ]
    .copy()
)


cell9_graph_metadata.to_parquet(
    GRAPH_METADATA_PATH,
    index=False
)


cell9_metadata = {

    "temporal_resolution":
        "1 minute",

    "minute_rows":
        int(
            len(
                minute_df
            )
        ),

    "segments":
        int(
            minute_df[
                "Segment_ID"
            ]
            .nunique()
        ),

    "flow_level_numeric_features":
        int(
            len(
                numeric_flow_features
            )
        ),

    "minute_level_numeric_features":
        int(
            len(
                MODEL_FEATURE_COLUMNS
            )
        ),

    "attack_types":
        ATTACK_TYPE_VOCABULARY,

    "split_created":
        False,

    "scaling_performed":
        False,

    "resampling_performed":
        False,

    "model_trained":
        False,
}


manifest = {

    **cell9_metadata,

    "model_feature_columns":
        MODEL_FEATURE_COLUMNS,

    "ground_truth_columns":
        GROUND_TRUTH_COLUMNS,

    "graph_ground_truth_columns":
        GRAPH_GROUND_TRUTH_COLUMNS,

    "forbidden_model_features":
        sorted(
            FORBIDDEN_MODEL_FEATURES
        ),
}


with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        manifest,
        file,
        indent=2,
        ensure_ascii=False
    )


# ================================================================================================================
# 25. FINAL VALIDATION
# ================================================================================================================

if len(
    minute_df
) != 2454:

    raise RuntimeError(
        "Final minute dataset does not contain "
        "the expected 2,454 rows."
    )


if minute_df[
    "Segment_ID"
].nunique() != 10:

    raise RuntimeError(
        "Final minute dataset does not contain "
        "exactly 10 temporal segments."
    )


if duplicate_keys != 0:

    raise RuntimeError(
        "Duplicate minute keys remain."
    )


if continuity_errors != 0:

    raise RuntimeError(
        "Temporal continuity errors remain."
    )


# ================================================================================================================
# 26. FINAL OUTPUT
# ================================================================================================================

print("\n")
print("=" * 150)
print("CELL 9 — FINAL VALIDATION SUMMARY")
print("=" * 150)


print(
    f"\nRaw flow rows processed         : "
    f"{total_rows:,}"
)

print(
    f"Raw attack flows                : "
    f"{total_attack_flows:,}"
)

print(
    f"Raw benign flows                : "
    f"{total_benign_flows:,}"
)

print(
    f"Minute rows                     : "
    f"{len(minute_df):,}"
)

print(
    f"Segments                        : "
    f"{minute_df['Segment_ID'].nunique():,}"
)

print(
    f"Attack minutes                  : "
    f"{int(minute_df['Attack_State'].sum()):,}"
)

print(
    f"Attack classes                  : "
    f"{len(ATTACK_TYPE_VOCABULARY):,}"
)

print(
    f"Current numerical predictors    : "
    f"{len(MODEL_FEATURE_COLUMNS):,}"
)

print(
    f"Duplicate minute keys           : "
    f"{duplicate_keys:,}"
)

print(
    f"Internal continuity errors      : "
    f"{continuity_errors:,}"
)

print(
    f"Predictor columns with infinity : "
    f"{len(inf_counts):,}"
)

print(
    f"Predictor columns with NaN      : "
    f"{len(nan_counts):,}"
)


print("\nGround truth preserved:")

print(
    "  ✓ Binary attack state"
)

print(
    "  ✓ Attack type(s)"
)

print(
    "  ✓ Attack Source IP(s)"
)

print(
    "  ✓ Attack Destination IP(s) / victim candidates"
)

print(
    "  ✓ Source -> Destination edges"
)

print(
    "  ✓ Source -> Destination -> Attack-Type edges"
)


print("\nLeakage protection:")

print(
    "  ✓ Endpoint identities are NOT predictors"
)

print(
    "  ✓ Attack types are NOT predictors"
)

print(
    "  ✓ Attack-state labels are NOT predictors"
)

print(
    "  ✓ Graph targets are NOT predictors"
)


print("\nSaved:")

print(
    f"  ✓ {MINUTE_DATASET_PATH}"
)

print(
    f"  ✓ {GRAPH_METADATA_PATH}"
)

print(
    f"  ✓ {MANIFEST_PATH}"
)


print("\nObjects created:")

print(
    "  minute_df"
)

print(
    "  MODEL_FEATURE_COLUMNS"
)

print(
    "  GROUND_TRUTH_COLUMNS"
)

print(
    "  GRAPH_GROUND_TRUTH_COLUMNS"
)

print(
    "  FORBIDDEN_MODEL_FEATURES"
)

print(
    "  ATTACK_TYPE_VOCABULARY"
)

print(
    "  cell9_graph_metadata"
)

print(
    "  cell9_metadata"
)

print(
    "  continuity_df"
)

print(
    "  segment_summary"
)


print("\n")
print("=" * 150)

print(
    "✓ CELL 9 CANONICAL DATASET CONSTRUCTION COMPLETE."
)

print(
    "✓ Timestamp month/day corruption is explicitly prevented."
)

print(
    "✓ Exactly 2,454 validated 1-minute observations retained."
)

print(
    "✓ Exactly 10 validated capture segments retained."
)

print(
    "✓ Attack endpoint identities are preserved before raw flows are released."
)

print(
    "✓ Typed attack graph edges are preserved."
)

print(
    "✓ Ground truth is separated from model predictors."
)

print(
    "✓ No split has been created."
)

print(
    "✓ No scaler has been fitted."
)

print(
    "✓ No resampling has been performed."
)

print(
    "✓ No model has been trained."
)

print(
    "\nNEXT:"
)

print(
    "Cell 10 will construct the final 15-minute history + "
    "[1, 5, 10, 15, 30]-minute point-target forecasting formulation "
    "directly from THIS canonical dataset."
)

print("=" * 150)

CELL 9 — CANONICAL MINUTE-LEVEL TEMPORAL DATASET + ATTACK GRAPH GROUND TRUTH

Configuration
------------------------------------------------------------------------------------------------------------------------------------------------------
Project root          : /content/drive/MyDrive/Network-Attack-Forecasting
Processed directory   : /content/drive/MyDrive/Network-Attack-Forecasting/data/processed
Forecasting directory : /content/drive/MyDrive/Network-Attack-Forecasting/data/forecasting
Temporal resolution   : 1 minute
Split created         : NO
Scaling performed     : NO
Resampling performed  : NO
Model trained         : NO


CLEANED FLOW FILE DISCOVERY

Processed Parquet files found: 8
1. Friday-WorkingHours-Afternoon-DDos.pcap_ISCX_cleaned.parquet
2. Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX_cleaned.parquet
3. Friday-WorkingHours-Morning.pcap_ISCX_cleaned.parquet
4. Monday-WorkingHours.pcap_ISCX_cleaned.parquet
5. Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX_c

In [18]:
# ================================================================================================================
# CELL 10 — FINAL MULTI-HORIZON FORECASTING TARGET DEFINITION
#           + ATTACK TYPE + ENDPOINT + GRAPH TARGET ALIGNMENT
# ================================================================================================================
#
# PURPOSE
# -------
# Construct the FINAL forecasting target table from the canonical minute_df
# produced by the corrected Cell 9.
#
#
# FINAL FORECASTING CONTRACT
# --------------------------
#
# Input history:
#
#       15 consecutive observed minutes ending at origin time t
#
# Forecast horizons:
#
#       +1, +5, +10, +15, +30 minutes
#
# Binary target for horizon H:
#
#       Attack_State at exactly t + H
#
#
# Additional future ground truth preserved at exactly t + H:
#
#       Attack_Types
#       Attack_Source_IPs
#       Attack_Destination_IPs
#       Attack_Edges
#       Typed_Attack_Edges
#
#       Attack_Type_Count
#       Attack_Source_Count
#       Attack_Destination_Count
#       Attack_Edge_Count
#       Typed_Attack_Edge_Count
#
#
# Context flags:
#
#       Current_State
#       Attack_In_History
#       Clean_History
#       New_Onset_H*
#       Continuation_H*
#
#
# IMPORTANT
# ---------
#
# This cell:
#
#   - DOES NOT split train/validation/test
#   - DOES NOT scale features
#   - DOES NOT resample classes
#   - DOES NOT create final model tensors
#   - DOES NOT fit preprocessing objects
#   - DOES NOT train a model
#
# All histories and future targets MUST remain inside the same Segment_ID.
#
# ================================================================================================================


import json
import numpy as np
import pandas as pd


# ================================================================================================================
# 1. CONFIGURATION
# ================================================================================================================

HISTORY_LENGTH = 15

FORECAST_HORIZONS = [
    1,
    5,
    10,
    15,
    30,
]

MAX_HORIZON = max(
    FORECAST_HORIZONS
)


print("=" * 150)
print(
    "CELL 10 — FINAL MULTI-HORIZON FORECASTING TARGET DEFINITION "
    "+ GRAPH TARGET ALIGNMENT"
)
print("=" * 150)

print("\nFinal forecasting configuration")
print("-" * 150)

print(
    f"History length      : "
    f"{HISTORY_LENGTH} minutes"
)

print(
    f"Forecast horizons   : "
    f"{FORECAST_HORIZONS}"
)

print(
    "Target semantics    : "
    "POINT target at exactly t + H"
)

print(
    "Graph semantics     : "
    "exact attack graph observed at t + H"
)

print(
    "Split created       : NO"
)

print(
    "Scaling performed   : NO"
)

print(
    "Resampling performed: NO"
)

print(
    "Model trained       : NO"
)


# ================================================================================================================
# 2. REQUIRED CELL 9 OBJECT CHECK
# ================================================================================================================

required_objects = [

    "minute_df",

    "MODEL_FEATURE_COLUMNS",

    "GROUND_TRUTH_COLUMNS",

    "GRAPH_GROUND_TRUTH_COLUMNS",

    "FORBIDDEN_MODEL_FEATURES",

    "ATTACK_TYPE_VOCABULARY",
]


missing_objects = [

    name

    for name in required_objects

    if name not in globals()
]


if missing_objects:

    raise RuntimeError(
        "Required Cell 9 objects are missing:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in missing_objects
        )
        +
        "\n\nRun the corrected Cell 9 first."
    )


print(
    "\n✓ Required Cell 9 objects found."
)


# ================================================================================================================
# 3. COPY CANONICAL DATA
# ================================================================================================================

df = minute_df.copy()


# ================================================================================================================
# 4. REQUIRED SCHEMA
# ================================================================================================================

REQUIRED_COLUMNS = {

    "Segment_ID",

    "Minute",

    "Attack_State",

    "Attack_Flow_Count",

    "Attack_Ratio",

    "Attack_Types",

    "Attack_Source_IPs",

    "Attack_Destination_IPs",

    "Attack_Edges",

    "Typed_Attack_Edges",

    "Attack_Type_Count",

    "Attack_Source_Count",

    "Attack_Destination_Count",

    "Attack_Edge_Count",

    "Typed_Attack_Edge_Count",
}


missing_columns = sorted(
    REQUIRED_COLUMNS
    -
    set(
        df.columns
    )
)


if missing_columns:

    raise RuntimeError(
        "minute_df is missing required corrected Cell 9 columns:\n"
        +
        "\n".join(
            f"  - {column}"
            for column in missing_columns
        )
    )


print(
    "✓ Required corrected Cell 9 schema found."
)


# ================================================================================================================
# 5. BASIC TYPE VALIDATION
# ================================================================================================================

df["Minute"] = pd.to_datetime(
    df["Minute"],
    errors="coerce"
)


if df["Minute"].isna().any():

    raise RuntimeError(
        f"Invalid Minute timestamps found: "
        f"{int(df['Minute'].isna().sum()):,}"
    )


if df["Segment_ID"].isna().any():

    raise RuntimeError(
        "Missing Segment_ID values found."
    )


if df["Attack_State"].isna().any():

    raise RuntimeError(
        "Missing Attack_State values found."
    )


df["Segment_ID"] = (
    df["Segment_ID"]
    .astype(int)
)


df["Attack_State"] = (
    df["Attack_State"]
    .astype(int)
)


attack_state_values = set(
    df[
        "Attack_State"
    ]
    .unique()
    .tolist()
)


if not attack_state_values.issubset(
    {
        0,
        1
    }
):

    raise RuntimeError(
        "Attack_State is not binary:\n"
        f"{sorted(attack_state_values)}"
    )


df = (
    df
    .sort_values(
        [
            "Segment_ID",
            "Minute"
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    f"\nminute_df rows      : "
    f"{len(df):,}"
)

print(
    f"Segments            : "
    f"{df['Segment_ID'].nunique():,}"
)

print(
    f"Model predictors    : "
    f"{len(MODEL_FEATURE_COLUMNS):,}"
)

print(
    f"Attack classes      : "
    f"{len(ATTACK_TYPE_VOCABULARY):,}"
)


# ================================================================================================================
# 6. EXACT EXPECTED CELL 9 CONTRACT
# ================================================================================================================

if len(df) != 2454:

    raise RuntimeError(
        "Expected exactly 2,454 canonical minute rows from Cell 9, "
        f"but found {len(df):,}."
    )


if df["Segment_ID"].nunique() != 10:

    raise RuntimeError(
        "Expected exactly 10 validated temporal segments, "
        f"but found {df['Segment_ID'].nunique():,}."
    )


print(
    "✓ Expected 2,454-minute / 10-segment Cell 9 contract confirmed."
)


# ================================================================================================================
# 7. UNIQUE MINUTE KEY CHECK
# ================================================================================================================

duplicate_keys = int(
    df
    .duplicated(
        subset=[
            "Segment_ID",
            "Minute"
        ]
    )
    .sum()
)


if duplicate_keys != 0:

    raise RuntimeError(
        "Duplicate (Segment_ID, Minute) rows found: "
        f"{duplicate_keys:,}"
    )


print(
    "✓ (Segment_ID, Minute) key is unique."
)


# ================================================================================================================
# 8. EXACT 1-MINUTE CONTINUITY CHECK
# ================================================================================================================

continuity_rows = []

continuity_errors = 0


for segment_id, seg in df.groupby(
    "Segment_ID",
    sort=True
):

    seg = (
        seg
        .sort_values(
            "Minute"
        )
        .reset_index(
            drop=True
        )
    )


    diffs = (
        seg[
            "Minute"
        ]
        .diff()
        .dropna()
    )


    bad = (
        diffs
        != pd.Timedelta(
            minutes=1
        )
    )


    bad_count = int(
        bad.sum()
    )


    continuity_errors += (
        bad_count
    )


    continuity_rows.append(
        {
            "Segment_ID":
                int(segment_id),

            "Rows":
                len(seg),

            "Start_Minute":
                seg[
                    "Minute"
                ].iloc[0],

            "End_Minute":
                seg[
                    "Minute"
                ].iloc[-1],

            "Non_1min_Gaps":
                bad_count,
        }
    )


continuity_df_cell10 = pd.DataFrame(
    continuity_rows
)


print("\n")
print("=" * 150)
print("MINUTE-LEVEL CONTINUITY CHECK")
print("=" * 150)

print(
    continuity_df_cell10.to_string(
        index=False
    )
)


if continuity_errors != 0:

    raise RuntimeError(
        "Minute-level timeline is not fully continuous "
        "inside one or more segments.\n"
        "Forecasting windows must NOT be constructed."
    )


print(
    "\n✓ Every segment is exactly continuous "
    "at 1-minute resolution."
)


# ================================================================================================================
# 9. JSON HELPERS
# ================================================================================================================

JSON_TARGET_COLUMNS = [

    "Attack_Types",

    "Attack_Source_IPs",

    "Attack_Destination_IPs",

    "Attack_Edges",

    "Typed_Attack_Edges",
]


COUNT_COLUMN_MAP = {

    "Attack_Types":
        "Attack_Type_Count",

    "Attack_Source_IPs":
        "Attack_Source_Count",

    "Attack_Destination_IPs":
        "Attack_Destination_Count",

    "Attack_Edges":
        "Attack_Edge_Count",

    "Typed_Attack_Edges":
        "Typed_Attack_Edge_Count",
}


def parse_json_list(
    value,
    column,
    row_description
):

    if pd.isna(value):

        raise RuntimeError(
            f"{row_description}: "
            f"{column} is missing."
        )


    try:

        parsed = json.loads(
            str(value)
        )

    except Exception as exc:

        raise RuntimeError(
            f"{row_description}: "
            f"{column} contains invalid JSON.\n"
            f"Value: {value!r}"
        ) from exc


    if not isinstance(
        parsed,
        list
    ):

        raise RuntimeError(
            f"{row_description}: "
            f"{column} must decode to a list."
        )


    return parsed


def canonical_json(
    value,
    column,
    row_description
):

    parsed = parse_json_list(
        value,
        column,
        row_description
    )

    return json.dumps(
        parsed,
        ensure_ascii=False,
        separators=(
            ",",
            ":"
        )
    )


# ================================================================================================================
# 10. VALIDATE ALL CELL 9 JSON GROUND TRUTH BEFORE FORECASTING
# ================================================================================================================

print("\n")
print("=" * 150)
print("CELL 9 GRAPH GROUND-TRUTH VALIDATION")
print("=" * 150)


json_errors = 0

cardinality_errors = 0

benign_metadata_errors = 0

attack_metadata_errors = 0


for row_idx, row in df.iterrows():

    description = (
        f"Segment {row['Segment_ID']} "
        f"Minute {row['Minute']}"
    )


    for json_column in JSON_TARGET_COLUMNS:

        parsed = parse_json_list(
            row[
                json_column
            ],
            json_column,
            description
        )


        expected_count_column = (
            COUNT_COLUMN_MAP[
                json_column
            ]
        )


        stored_count = row[
            expected_count_column
        ]


        if pd.isna(
            stored_count
        ):

            cardinality_errors += 1

        else:

            try:

                stored_count_int = int(
                    stored_count
                )

            except Exception:

                cardinality_errors += 1

            else:

                if (
                    stored_count_int
                    != len(parsed)
                ):

                    cardinality_errors += 1


    state = int(
        row[
            "Attack_State"
        ]
    )


    all_counts = [

        int(
            row[
                count_column
            ]
        )

        for count_column
        in COUNT_COLUMN_MAP.values()
    ]


    if state == 0:

        if any(
            count != 0
            for count in all_counts
        ):

            benign_metadata_errors += 1


    else:

        if any(
            count <= 0
            for count in all_counts
        ):

            attack_metadata_errors += 1


print(
    f"\nJSON parse errors                     : "
    f"{json_errors:,}"
)

print(
    f"JSON/count cardinality inconsistencies: "
    f"{cardinality_errors:,}"
)

print(
    f"Benign metadata inconsistencies       : "
    f"{benign_metadata_errors:,}"
)

print(
    f"Attack metadata incompleteness         : "
    f"{attack_metadata_errors:,}"
)


if (
    json_errors != 0
    or cardinality_errors != 0
    or benign_metadata_errors != 0
    or attack_metadata_errors != 0
):

    raise RuntimeError(
        "Cell 9 graph ground-truth validation failed."
    )


print(
    "\n✓ Cell 9 attack/graph ground truth is "
    "safe to align into future targets."
)


# ================================================================================================================
# 11. BUILD FINAL VALID FORECASTING ORIGINS
# ================================================================================================================
#
# For a segment containing N minute rows:
#
# earliest origin index:
#
#       HISTORY_LENGTH - 1
#
# because history is:
#
#       t-14 ... t
#
#
# latest origin index:
#
#       N - MAX_HORIZON - 1
#
# because t+30 must exist inside the same segment.
#
# ================================================================================================================

origin_records = []


for segment_id, seg in df.groupby(
    "Segment_ID",
    sort=True
):

    seg = (
        seg
        .sort_values(
            "Minute"
        )
        .reset_index(
            drop=True
        )
    )


    n = len(
        seg
    )


    first_origin_idx = (
        HISTORY_LENGTH
        - 1
    )


    last_origin_idx = (
        n
        - MAX_HORIZON
        - 1
    )


    if (
        last_origin_idx
        < first_origin_idx
    ):

        continue


    for origin_idx in range(
        first_origin_idx,
        last_origin_idx + 1
    ):

        history_start_idx = (
            origin_idx
            - HISTORY_LENGTH
            + 1
        )


        history = (
            seg.iloc[
                history_start_idx:
                origin_idx + 1
            ]
            .copy()
        )


        # --------------------------------------------------------------------------------------------------------
        # EXACT HISTORY LENGTH
        # --------------------------------------------------------------------------------------------------------

        if len(
            history
        ) != HISTORY_LENGTH:

            raise RuntimeError(
                "History-length error.\n"
                f"Segment      : {segment_id}\n"
                f"Origin index : {origin_idx}\n"
                f"History rows : {len(history)}"
            )


        # --------------------------------------------------------------------------------------------------------
        # SAME SEGMENT
        # --------------------------------------------------------------------------------------------------------

        if (
            history[
                "Segment_ID"
            ]
            .nunique()
            != 1
        ):

            raise RuntimeError(
                "History crossed a segment boundary."
            )


        origin_time = (
            seg.loc[
                origin_idx,
                "Minute"
            ]
        )


        history_start = (
            history[
                "Minute"
            ]
            .iloc[0]
        )


        expected_history_start = (
            origin_time
            -
            pd.Timedelta(
                minutes=HISTORY_LENGTH - 1
            )
        )


        if (
            history_start
            != expected_history_start
        ):

            raise RuntimeError(
                "Non-contiguous 15-minute history detected.\n"
                f"Segment  : {segment_id}\n"
                f"Origin   : {origin_time}\n"
                f"Expected : {expected_history_start}\n"
                f"Actual   : {history_start}"
            )


        history_states = (
            history[
                "Attack_State"
            ]
            .to_numpy(
                dtype=np.int8
            )
        )


        current_state = int(
            seg.loc[
                origin_idx,
                "Attack_State"
            ]
        )


        attack_in_history = int(
            history_states.max()
        )


        clean_history = int(
            attack_in_history == 0
        )


        attack_minutes_in_history = int(
            history_states.sum()
        )


        record = {

            "Segment_ID":
                int(
                    segment_id
                ),

            "History_Start":
                history_start,

            "History_End":
                origin_time,

            "Origin_Time":
                origin_time,

            "Current_State":
                current_state,

            "Attack_In_History":
                attack_in_history,

            "Clean_History":
                clean_history,

            "Attack_Minutes_In_History":
                attack_minutes_in_history,
        }


        # ========================================================================================================
        # HORIZON-SPECIFIC EXACT POINT TARGETS
        # ========================================================================================================

        for H in FORECAST_HORIZONS:

            target_idx = (
                origin_idx
                + H
            )


            target_row = (
                seg.iloc[
                    target_idx
                ]
            )


            expected_target_time = (
                origin_time
                +
                pd.Timedelta(
                    minutes=H
                )
            )


            actual_target_time = (
                target_row[
                    "Minute"
                ]
            )


            # ----------------------------------------------------------------------------------------------------
            # EXACT TIME ALIGNMENT
            # ----------------------------------------------------------------------------------------------------

            if (
                actual_target_time
                != expected_target_time
            ):

                raise RuntimeError(
                    "Target-time mismatch.\n"
                    f"Segment  : {segment_id}\n"
                    f"Origin   : {origin_time}\n"
                    f"Horizon  : +{H}\n"
                    f"Expected : {expected_target_time}\n"
                    f"Actual   : {actual_target_time}"
                )


            # ----------------------------------------------------------------------------------------------------
            # SEGMENT SAFETY
            # ----------------------------------------------------------------------------------------------------

            if (
                int(
                    target_row[
                        "Segment_ID"
                    ]
                )
                != int(
                    segment_id
                )
            ):

                raise RuntimeError(
                    "Future target crossed a segment boundary."
                )


            target_state = int(
                target_row[
                    "Attack_State"
                ]
            )


            record[
                f"Target_Time_H{H}"
            ] = (
                actual_target_time
            )


            record[
                f"Target_State_H{H}"
            ] = (
                target_state
            )


            # ====================================================================================================
            # FUTURE GRAPH GROUND TRUTH
            # ====================================================================================================

            record[
                f"Attack_Types_H{H}"
            ] = canonical_json(
                target_row[
                    "Attack_Types"
                ],
                "Attack_Types",
                f"Target H{H} at {actual_target_time}"
            )


            record[
                f"Attack_Sources_H{H}"
            ] = canonical_json(
                target_row[
                    "Attack_Source_IPs"
                ],
                "Attack_Source_IPs",
                f"Target H{H} at {actual_target_time}"
            )


            record[
                f"Attack_Destinations_H{H}"
            ] = canonical_json(
                target_row[
                    "Attack_Destination_IPs"
                ],
                "Attack_Destination_IPs",
                f"Target H{H} at {actual_target_time}"
            )


            record[
                f"Attack_Edges_H{H}"
            ] = canonical_json(
                target_row[
                    "Attack_Edges"
                ],
                "Attack_Edges",
                f"Target H{H} at {actual_target_time}"
            )


            record[
                f"Typed_Attack_Edges_H{H}"
            ] = canonical_json(
                target_row[
                    "Typed_Attack_Edges"
                ],
                "Typed_Attack_Edges",
                f"Target H{H} at {actual_target_time}"
            )


            # ====================================================================================================
            # FUTURE GRAPH CARDINALITIES
            # ====================================================================================================

            record[
                f"Attack_Type_Count_H{H}"
            ] = int(
                target_row[
                    "Attack_Type_Count"
                ]
            )


            record[
                f"Attack_Source_Count_H{H}"
            ] = int(
                target_row[
                    "Attack_Source_Count"
                ]
            )


            record[
                f"Attack_Destination_Count_H{H}"
            ] = int(
                target_row[
                    "Attack_Destination_Count"
                ]
            )


            record[
                f"Attack_Edge_Count_H{H}"
            ] = int(
                target_row[
                    "Attack_Edge_Count"
                ]
            )


            record[
                f"Typed_Attack_Edge_Count_H{H}"
            ] = int(
                target_row[
                    "Typed_Attack_Edge_Count"
                ]
            )


            # ====================================================================================================
            # NEW ONSET
            # ====================================================================================================
            #
            # Genuine pre-attack case:
            #
            #       all 15 history minutes benign
            #
            #                   AND
            #
            #       attack exists exactly at t + H
            #
            # ====================================================================================================

            record[
                f"New_Onset_H{H}"
            ] = int(

                clean_history == 1

                and

                target_state == 1
            )


            # ====================================================================================================
            # CONTINUATION
            # ====================================================================================================
            #
            # Some attack exists in the historical window AND
            # the future point t+H is also attack.
            #
            # ====================================================================================================

            record[
                f"Continuation_H{H}"
            ] = int(

                attack_in_history == 1

                and

                target_state == 1
            )


        origin_records.append(
            record
        )


# ================================================================================================================
# 12. CREATE FINAL TARGET TABLE
# ================================================================================================================

final_target_df = pd.DataFrame(
    origin_records
)


if len(
    final_target_df
) == 0:

    raise RuntimeError(
        "No valid forecasting origins were constructed."
    )


final_target_df = (
    final_target_df
    .sort_values(
        [
            "Segment_ID",
            "Origin_Time"
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    f"\n✓ Forecasting origins constructed: "
    f"{len(final_target_df):,}"
)


# ================================================================================================================
# 13. EXPECTED ORIGIN COUNT
# ================================================================================================================

expected_origin_count = 0


for _, seg in df.groupby(
    "Segment_ID",
    sort=True
):

    n = len(
        seg
    )

    possible = (
        n
        - HISTORY_LENGTH
        - MAX_HORIZON
        + 1
    )

    expected_origin_count += max(
        0,
        possible
    )


if (
    len(final_target_df)
    != expected_origin_count
):

    raise RuntimeError(
        "Forecast-origin count mismatch.\n"
        f"Expected : {expected_origin_count:,}\n"
        f"Actual   : {len(final_target_df):,}"
    )


if expected_origin_count != 2014:

    raise RuntimeError(
        "Corrected canonical dataset should produce exactly "
        "2,014 valid origins under the 15-minute history / "
        "30-minute maximum horizon contract.\n"
        f"Computed: {expected_origin_count:,}"
    )


print(
    "✓ Exactly 2,014 expected forecasting origins constructed."
)


# ================================================================================================================
# 14. UNIQUE ORIGIN CHECK
# ================================================================================================================

duplicate_origins = int(
    final_target_df
    .duplicated(
        subset=[
            "Segment_ID",
            "Origin_Time"
        ]
    )
    .sum()
)


if duplicate_origins != 0:

    raise RuntimeError(
        f"Duplicate forecasting origins found: "
        f"{duplicate_origins:,}"
    )


print(
    "✓ Forecast origins are unique."
)


# ================================================================================================================
# 15. HORIZON COMPLETENESS CHECK
# ================================================================================================================

missing_target_values = {}


for H in FORECAST_HORIZONS:

    required_horizon_columns = [

        f"Target_Time_H{H}",

        f"Target_State_H{H}",

        f"Attack_Types_H{H}",

        f"Attack_Sources_H{H}",

        f"Attack_Destinations_H{H}",

        f"Attack_Edges_H{H}",

        f"Typed_Attack_Edges_H{H}",

        f"Attack_Type_Count_H{H}",

        f"Attack_Source_Count_H{H}",

        f"Attack_Destination_Count_H{H}",

        f"Attack_Edge_Count_H{H}",

        f"Typed_Attack_Edge_Count_H{H}",

        f"New_Onset_H{H}",

        f"Continuation_H{H}",
    ]


    for column in required_horizon_columns:

        if (
            column
            not in final_target_df.columns
        ):

            raise RuntimeError(
                f"Missing horizon target column: "
                f"{column}"
            )


        missing_count = int(
            final_target_df[
                column
            ]
            .isna()
            .sum()
        )


        if missing_count:

            missing_target_values[
                column
            ] = (
                missing_count
            )


if missing_target_values:

    raise RuntimeError(
        "Missing aligned future target values detected:\n"
        f"{missing_target_values}"
    )


print(
    "✓ Every origin contains every required horizon target."
)


# ================================================================================================================
# 16. EXACT TIME-ALIGNMENT AUDIT
# ================================================================================================================

print("\n")
print("=" * 150)
print("EXACT TARGET-TIME ALIGNMENT")
print("=" * 150)


alignment_errors = 0


for H in FORECAST_HORIZONS:

    expected = (
        final_target_df[
            "Origin_Time"
        ]
        +
        pd.to_timedelta(
            H,
            unit="m"
        )
    )


    actual = (
        final_target_df[
            f"Target_Time_H{H}"
        ]
    )


    errors = int(
        (
            expected
            != actual
        )
        .sum()
    )


    alignment_errors += (
        errors
    )


    print(
        f"Horizon +{H:>2} min : "
        f"{len(final_target_df) - errors:,}/"
        f"{len(final_target_df):,} exactly aligned"
    )


if alignment_errors != 0:

    raise RuntimeError(
        "Target-time alignment errors detected: "
        f"{alignment_errors:,}"
    )


# ================================================================================================================
# 17. BINARY TARGET / ONSET / CONTINUATION VALIDATION
# ================================================================================================================

logic_errors = 0


for H in FORECAST_HORIZONS:

    state_col = (
        f"Target_State_H{H}"
    )

    onset_col = (
        f"New_Onset_H{H}"
    )

    continuation_col = (
        f"Continuation_H{H}"
    )


    for column in [
        state_col,
        onset_col,
        continuation_col,
    ]:

        values = set(
            final_target_df[
                column
            ]
            .unique()
            .tolist()
        )


        if not values.issubset(
            {
                0,
                1
            }
        ):

            raise RuntimeError(
                f"{column} contains non-binary values:\n"
                f"{sorted(values)}"
            )


    expected_onset = (

        (
            final_target_df[
                "Clean_History"
            ]
            == 1
        )

        &

        (
            final_target_df[
                state_col
            ]
            == 1
        )
    ).astype(int)


    expected_continuation = (

        (
            final_target_df[
                "Attack_In_History"
            ]
            == 1
        )

        &

        (
            final_target_df[
                state_col
            ]
            == 1
        )
    ).astype(int)


    logic_errors += int(
        (
            final_target_df[
                onset_col
            ]
            != expected_onset
        )
        .sum()
    )


    logic_errors += int(
        (
            final_target_df[
                continuation_col
            ]
            != expected_continuation
        )
        .sum()
    )


    overlap = int(
        (
            (
                final_target_df[
                    onset_col
                ]
                == 1
            )

            &

            (
                final_target_df[
                    continuation_col
                ]
                == 1
            )
        )
        .sum()
    )


    if overlap != 0:

        raise RuntimeError(
            f"H{H}: New_Onset and Continuation "
            f"overlap in {overlap:,} rows."
        )


if logic_errors != 0:

    raise RuntimeError(
        "New-onset / continuation logic audit failed. "
        f"Errors: {logic_errors:,}"
    )


print(
    "\n✓ New-onset and continuation logic validated."
)


# ================================================================================================================
# 18. FUTURE GRAPH TARGET CONSISTENCY
# ================================================================================================================

print("\n")
print("=" * 150)
print("FUTURE GRAPH TARGET CONSISTENCY")
print("=" * 150)


future_graph_errors = 0


HORIZON_JSON_COUNT_PAIRS = [

    (
        "Attack_Types_H{}",
        "Attack_Type_Count_H{}"
    ),

    (
        "Attack_Sources_H{}",
        "Attack_Source_Count_H{}"
    ),

    (
        "Attack_Destinations_H{}",
        "Attack_Destination_Count_H{}"
    ),

    (
        "Attack_Edges_H{}",
        "Attack_Edge_Count_H{}"
    ),

    (
        "Typed_Attack_Edges_H{}",
        "Typed_Attack_Edge_Count_H{}"
    ),
]


graph_audit_rows = []


for H in FORECAST_HORIZONS:

    state_column = (
        f"Target_State_H{H}"
    )


    horizon_errors = 0


    for row_idx, row in final_target_df.iterrows():

        target_state = int(
            row[
                state_column
            ]
        )


        counts = []


        for (
            json_template,
            count_template
        ) in HORIZON_JSON_COUNT_PAIRS:

            json_column = (
                json_template.format(
                    H
                )
            )

            count_column = (
                count_template.format(
                    H
                )
            )


            parsed = parse_json_list(

                row[
                    json_column
                ],

                json_column,

                (
                    f"Origin "
                    f"{row['Origin_Time']} "
                    f"H{H}"
                )
            )


            stored_count = int(
                row[
                    count_column
                ]
            )


            counts.append(
                stored_count
            )


            if (
                len(parsed)
                != stored_count
            ):

                horizon_errors += 1


        if target_state == 0:

            if any(
                count != 0
                for count in counts
            ):

                horizon_errors += 1


        else:

            if any(
                count <= 0
                for count in counts
            ):

                horizon_errors += 1


    future_graph_errors += (
        horizon_errors
    )


    graph_audit_rows.append(
        {
            "Horizon_Minutes":
                H,

            "Graph_Target_Errors":
                horizon_errors,
        }
    )


graph_target_audit_df = pd.DataFrame(
    graph_audit_rows
)


print(
    graph_target_audit_df.to_string(
        index=False
    )
)


if future_graph_errors != 0:

    raise RuntimeError(
        "Future graph-target consistency failed. "
        f"Errors: {future_graph_errors:,}"
    )


print(
    "\n✓ Every future binary target agrees with its "
    "attack-type/endpoint/edge metadata."
)


# ================================================================================================================
# 19. SEGMENT-BOUNDARY SAFETY AUDIT
# ================================================================================================================

segment_bounds = (
    df
    .groupby(
        "Segment_ID"
    )
    .agg(
        Segment_Start=(
            "Minute",
            "min"
        ),

        Segment_End=(
            "Minute",
            "max"
        ),
    )
)


boundary_errors = 0


for _, row in final_target_df.iterrows():

    sid = int(
        row[
            "Segment_ID"
        ]
    )


    segment_start = (
        segment_bounds.loc[
            sid,
            "Segment_Start"
        ]
    )


    segment_end = (
        segment_bounds.loc[
            sid,
            "Segment_End"
        ]
    )


    if (
        row[
            "History_Start"
        ]
        < segment_start
    ):

        boundary_errors += 1


    if (
        row[
            "History_End"
        ]
        > segment_end
    ):

        boundary_errors += 1


    for H in FORECAST_HORIZONS:

        target_time = (
            row[
                f"Target_Time_H{H}"
            ]
        )


        if (
            target_time
            < segment_start

            or

            target_time
            > segment_end
        ):

            boundary_errors += 1


if boundary_errors != 0:

    raise RuntimeError(
        "Segment-boundary violations detected: "
        f"{boundary_errors:,}"
    )


print(
    "\n✓ No history or future target crosses "
    "a capture-segment boundary."
)


# ================================================================================================================
# 20. SAMPLE COUNT BY SEGMENT
# ================================================================================================================

segment_sample_summary = (

    final_target_df

    .groupby(
        "Segment_ID"
    )

    .agg(

        Samples=(
            "Origin_Time",
            "size"
        ),

        First_Origin=(
            "Origin_Time",
            "min"
        ),

        Last_Origin=(
            "Origin_Time",
            "max"
        ),
    )

    .reset_index()
)


print("\n")
print("=" * 150)
print("VALID FORECASTING ORIGINS")
print("=" * 150)

print(
    segment_sample_summary.to_string(
        index=False
    )
)

print(
    f"\nTotal valid origins : "
    f"{len(final_target_df):,}"
)


# ================================================================================================================
# 21. MULTI-HORIZON BINARY TARGET DISTRIBUTION
# ================================================================================================================

target_summary = []


for H in FORECAST_HORIZONS:

    y = (
        final_target_df[
            f"Target_State_H{H}"
        ]
    )


    attacks = int(
        (
            y == 1
        )
        .sum()
    )


    benign = int(
        (
            y == 0
        )
        .sum()
    )


    new_onsets = int(
        final_target_df[
            f"New_Onset_H{H}"
        ]
        .sum()
    )


    continuations = int(
        final_target_df[
            f"Continuation_H{H}"
        ]
        .sum()
    )


    target_summary.append(
        {
            "Horizon_Minutes":
                H,

            "Samples":
                len(
                    final_target_df
                ),

            "Attack_Targets":
                attacks,

            "Benign_Targets":
                benign,

            "Attack_Percent":
                round(
                    100
                    * attacks
                    / len(
                        final_target_df
                    ),
                    4
                ),

            "New_Onset_Targets":
                new_onsets,

            "Continuation_Targets":
                continuations,
        }
    )


target_summary_df = pd.DataFrame(
    target_summary
)


print("\n")
print("=" * 150)
print("MULTI-HORIZON TARGET DISTRIBUTION")
print("=" * 150)

print(
    target_summary_df.to_string(
        index=False
    )
)


# ================================================================================================================
# 22. HISTORY CONTEXT DISTRIBUTION
# ================================================================================================================

history_summary_df = pd.DataFrame(
    {
        "Metric": [

            "Total forecasting origins",

            "Clean-history origins",

            "Origins containing attack in history",

            "Origins currently benign",

            "Origins currently attack",
        ],

        "Count": [

            len(
                final_target_df
            ),

            int(
                final_target_df[
                    "Clean_History"
                ]
                .sum()
            ),

            int(
                final_target_df[
                    "Attack_In_History"
                ]
                .sum()
            ),

            int(
                (
                    final_target_df[
                        "Current_State"
                    ]
                    == 0
                )
                .sum()
            ),

            int(
                (
                    final_target_df[
                        "Current_State"
                    ]
                    == 1
                )
                .sum()
            ),
        ]
    }
)


print("\n")
print("=" * 150)
print("HISTORY CONTEXT SUMMARY")
print("=" * 150)

print(
    history_summary_df.to_string(
        index=False
    )
)


# ================================================================================================================
# 23. GENUINE PRE-ATTACK SUPPORT
# ================================================================================================================

onset_rows = []


for H in FORECAST_HORIZONS:

    onset_rows.append(
        {
            "Horizon_Minutes":
                H,

            "New_Onset_Positive":
                int(
                    final_target_df[
                        f"New_Onset_H{H}"
                    ]
                    .sum()
                ),

            "All_Attack_Positive":
                int(
                    final_target_df[
                        f"Target_State_H{H}"
                    ]
                    .sum()
                ),
        }
    )


onset_support_df = pd.DataFrame(
    onset_rows
)


print("\n")
print("=" * 150)
print("GENUINE PRE-ATTACK / NEW-ONSET SUPPORT")
print("=" * 150)

print(
    onset_support_df.to_string(
        index=False
    )
)


# ================================================================================================================
# 24. ATTACK-TYPE FORECAST SUPPORT
# ================================================================================================================

attack_type_support_records = []


for H in FORECAST_HORIZONS:

    column = (
        f"Attack_Types_H{H}"
    )


    counts = {}


    for value in final_target_df[
        column
    ]:

        types = json.loads(
            value
        )


        for attack_type in types:

            attack_type = str(
                attack_type
            )


            counts[
                attack_type
            ] = (
                counts.get(
                    attack_type,
                    0
                )
                + 1
            )


    for attack_type in ATTACK_TYPE_VOCABULARY:

        attack_type_support_records.append(
            {
                "Horizon_Minutes":
                    H,

                "Attack_Type":
                    attack_type,

                "Positive_Target_Minutes":
                    counts.get(
                        attack_type,
                        0
                    ),
            }
        )


attack_type_support_df = pd.DataFrame(
    attack_type_support_records
)


print("\n")
print("=" * 150)
print("ATTACK-TYPE TARGET SUPPORT")
print("=" * 150)

print(
    attack_type_support_df.to_string(
        index=False
    )
)


# ================================================================================================================
# 25. GRAPH TARGET SUPPORT
# ================================================================================================================

graph_support_records = []


for H in FORECAST_HORIZONS:

    graph_support_records.append(
        {
            "Horizon_Minutes":
                H,

            "Attack_Target_Minutes":
                int(
                    final_target_df[
                        f"Target_State_H{H}"
                    ]
                    .sum()
                ),

            "Total_Attack_Types":
                int(
                    final_target_df[
                        f"Attack_Type_Count_H{H}"
                    ]
                    .sum()
                ),

            "Total_Attack_Sources":
                int(
                    final_target_df[
                        f"Attack_Source_Count_H{H}"
                    ]
                    .sum()
                ),

            "Total_Attack_Destinations":
                int(
                    final_target_df[
                        f"Attack_Destination_Count_H{H}"
                    ]
                    .sum()
                ),

            "Total_Attack_Edges":
                int(
                    final_target_df[
                        f"Attack_Edge_Count_H{H}"
                    ]
                    .sum()
                ),

            "Total_Typed_Attack_Edges":
                int(
                    final_target_df[
                        f"Typed_Attack_Edge_Count_H{H}"
                    ]
                    .sum()
                ),
        }
    )


graph_support_df = pd.DataFrame(
    graph_support_records
)


print("\n")
print("=" * 150)
print("GRAPH TARGET SUPPORT")
print("=" * 150)

print(
    graph_support_df.to_string(
        index=False
    )
)


# ================================================================================================================
# 26. MODEL FEATURE CONTRACT FROM CELL 9
# ================================================================================================================

missing_model_features = [

    column

    for column in MODEL_FEATURE_COLUMNS

    if column not in df.columns
]


if missing_model_features:

    raise RuntimeError(
        "Cell 9 MODEL_FEATURE_COLUMNS contains columns "
        "missing from minute_df:\n"
        f"{missing_model_features}"
    )


duplicate_model_features = (

    len(
        MODEL_FEATURE_COLUMNS
    )

    !=

    len(
        set(
            MODEL_FEATURE_COLUMNS
        )
    )
)


if duplicate_model_features:

    raise RuntimeError(
        "MODEL_FEATURE_COLUMNS contains duplicate feature names."
    )


# ================================================================================================================
# 27. LEAKAGE CONTRACT
# ================================================================================================================
#
# IMPORTANT:
#
# Cell 9 is the canonical authority for existing forbidden features.
# We EXTEND that set here with all newly created future target/audit columns.
#
# We do NOT overwrite the Cell 9 protection.
#
# ================================================================================================================

FORBIDDEN_MODEL_FEATURES = set(
    FORBIDDEN_MODEL_FEATURES
)


FORBIDDEN_MODEL_FEATURES.update(
    {
        "Segment_ID",

        "Minute",

        "History_Start",

        "History_End",

        "Origin_Time",

        "Current_State",

        "Attack_In_History",

        "Clean_History",

        "Attack_Minutes_In_History",
    }
)


for H in FORECAST_HORIZONS:

    FORBIDDEN_MODEL_FEATURES.update(
        {
            f"Target_Time_H{H}",

            f"Target_State_H{H}",

            f"Attack_Types_H{H}",

            f"Attack_Sources_H{H}",

            f"Attack_Destinations_H{H}",

            f"Attack_Edges_H{H}",

            f"Typed_Attack_Edges_H{H}",

            f"Attack_Type_Count_H{H}",

            f"Attack_Source_Count_H{H}",

            f"Attack_Destination_Count_H{H}",

            f"Attack_Edge_Count_H{H}",

            f"Typed_Attack_Edge_Count_H{H}",

            f"New_Onset_H{H}",

            f"Continuation_H{H}",
        }
    )


leakage_overlap = sorted(
    set(
        MODEL_FEATURE_COLUMNS
    )
    &
    FORBIDDEN_MODEL_FEATURES
)


if leakage_overlap:

    raise RuntimeError(
        "TARGET / LABEL LEAKAGE DETECTED.\n"
        "These forbidden columns are present in "
        "MODEL_FEATURE_COLUMNS:\n"
        f"{leakage_overlap}"
    )


print("\n")
print("=" * 150)
print("LABEL-LEAKAGE CONTRACT")
print("=" * 150)

print(
    f"\nCell 9 model predictors tracked : "
    f"{len(MODEL_FEATURE_COLUMNS):,}"
)

print(
    f"Forbidden feature names tracked : "
    f"{len(FORBIDDEN_MODEL_FEATURES):,}"
)

print(
    f"Predictor/forbidden overlap     : "
    f"{len(leakage_overlap):,}"
)

print(
    "\n✓ Future binary targets are excluded from predictors."
)

print(
    "✓ Future attack types are excluded from predictors."
)

print(
    "✓ Future source identities are excluded from predictors."
)

print(
    "✓ Future destination/victim identities are excluded from predictors."
)

print(
    "✓ Future graph edges are excluded from predictors."
)

print(
    "✓ Future typed graph edges are excluded from predictors."
)

print(
    "✓ Onset/continuation audit flags are excluded from predictors."
)


# ================================================================================================================
# 28. CREATE LONG-FORM FORECAST METADATA TABLE
# ================================================================================================================
#
# Wide table:
#
#       one row per forecasting origin
#
# Long table:
#
#       one row per forecasting origin × horizon
#
# Expected:
#
#       2,014 × 5 = 10,070 rows
#
# This table is extremely useful for:
#
#       Cell 11+ audits
#       split feasibility
#       attack-type distribution
#       victim/endpoint distribution
#       graph target analysis
#
# ================================================================================================================

long_records = []


for _, row in final_target_df.iterrows():

    for H in FORECAST_HORIZONS:

        long_records.append(
            {
                "Segment_ID":
                    int(
                        row[
                            "Segment_ID"
                        ]
                    ),

                "History_Start":
                    row[
                        "History_Start"
                    ],

                "History_End":
                    row[
                        "History_End"
                    ],

                "Origin_Time":
                    row[
                        "Origin_Time"
                    ],

                "Horizon_Minutes":
                    H,

                "Target_Time":
                    row[
                        f"Target_Time_H{H}"
                    ],

                "Current_State":
                    int(
                        row[
                            "Current_State"
                        ]
                    ),

                "Attack_In_History":
                    int(
                        row[
                            "Attack_In_History"
                        ]
                    ),

                "Clean_History":
                    int(
                        row[
                            "Clean_History"
                        ]
                    ),

                "Attack_Minutes_In_History":
                    int(
                        row[
                            "Attack_Minutes_In_History"
                        ]
                    ),

                "Future_Attack_State":
                    int(
                        row[
                            f"Target_State_H{H}"
                        ]
                    ),

                "Future_Attack_Types":
                    row[
                        f"Attack_Types_H{H}"
                    ],

                "Future_Attack_Sources":
                    row[
                        f"Attack_Sources_H{H}"
                    ],

                "Future_Attack_Destinations":
                    row[
                        f"Attack_Destinations_H{H}"
                    ],

                "Future_Attack_Edges":
                    row[
                        f"Attack_Edges_H{H}"
                    ],

                "Future_Typed_Attack_Edges":
                    row[
                        f"Typed_Attack_Edges_H{H}"
                    ],

                "Future_Attack_Type_Count":
                    int(
                        row[
                            f"Attack_Type_Count_H{H}"
                        ]
                    ),

                "Future_Attack_Source_Count":
                    int(
                        row[
                            f"Attack_Source_Count_H{H}"
                        ]
                    ),

                "Future_Attack_Destination_Count":
                    int(
                        row[
                            f"Attack_Destination_Count_H{H}"
                        ]
                    ),

                "Future_Attack_Edge_Count":
                    int(
                        row[
                            f"Attack_Edge_Count_H{H}"
                        ]
                    ),

                "Future_Typed_Attack_Edge_Count":
                    int(
                        row[
                            f"Typed_Attack_Edge_Count_H{H}"
                        ]
                    ),

                "New_Onset":
                    int(
                        row[
                            f"New_Onset_H{H}"
                        ]
                    ),

                "Continuation":
                    int(
                        row[
                            f"Continuation_H{H}"
                        ]
                    ),
            }
        )


cell10_forecast_metadata_long = pd.DataFrame(
    long_records
)


expected_long_rows = (
    len(
        final_target_df
    )
    *
    len(
        FORECAST_HORIZONS
    )
)


if (
    len(
        cell10_forecast_metadata_long
    )
    != expected_long_rows
):

    raise RuntimeError(
        "Long-form forecast metadata row-count mismatch.\n"
        f"Expected : {expected_long_rows:,}\n"
        f"Actual   : "
        f"{len(cell10_forecast_metadata_long):,}"
    )


if expected_long_rows != 10070:

    raise RuntimeError(
        "Expected exactly 10,070 origin×horizon rows, "
        f"but computed {expected_long_rows:,}."
    )


duplicate_long_keys = int(

    cell10_forecast_metadata_long

    .duplicated(
        subset=[
            "Segment_ID",
            "Origin_Time",
            "Horizon_Minutes"
        ]
    )

    .sum()
)


if duplicate_long_keys != 0:

    raise RuntimeError(
        "Duplicate long-form forecasting keys detected: "
        f"{duplicate_long_keys:,}"
    )


print("\n")
print("=" * 150)
print("LONG-FORM FORECAST METADATA")
print("=" * 150)

print(
    f"\nRows                    : "
    f"{len(cell10_forecast_metadata_long):,}"
)

print(
    f"Expected rows           : "
    f"{expected_long_rows:,}"
)

print(
    f"Duplicate forecast keys : "
    f"{duplicate_long_keys:,}"
)

print(
    "\n✓ Exactly 10,070 origin × horizon records created."
)


# ================================================================================================================
# 29. EXPORT CANONICAL CELL 10 OBJECTS
# ================================================================================================================

cell10_target_table = (
    final_target_df.copy()
)


cell10_valid_origins = (

    final_target_df[
        [
            "Segment_ID",

            "History_Start",

            "History_End",

            "Origin_Time",

            "Current_State",

            "Attack_In_History",

            "Clean_History",

            "Attack_Minutes_In_History",
        ]
    ]

    .copy()
)


cell10_metadata = {

    "history_length":
        HISTORY_LENGTH,

    "forecast_horizons":
        FORECAST_HORIZONS.copy(),

    "max_horizon":
        MAX_HORIZON,

    "target_semantics":
        "point_horizon",

    "graph_target_semantics":
        "exact_graph_at_point_horizon",

    "number_of_origins":
        int(
            len(
                final_target_df
            )
        ),

    "number_of_long_rows":
        int(
            len(
                cell10_forecast_metadata_long
            )
        ),

    "number_of_segments":
        int(
            df[
                "Segment_ID"
            ]
            .nunique()
        ),

    "number_of_model_features":
        int(
            len(
                MODEL_FEATURE_COLUMNS
            )
        ),

    "attack_type_vocabulary":
        list(
            ATTACK_TYPE_VOCABULARY
        ),

    "future_graph_targets": [

        "Attack_Types",

        "Attack_Sources",

        "Attack_Destinations",

        "Attack_Edges",

        "Typed_Attack_Edges",
    ],

    "split_created":
        False,

    "scaling_performed":
        False,

    "resampling_performed":
        False,

    "model_trained":
        False,
}


# ================================================================================================================
# 30. FINAL HARD VALIDATION
# ================================================================================================================

hard_validation = {

    "exact_2454_minute_rows":
        len(
            df
        )
        == 2454,

    "exact_10_segments":
        df[
            "Segment_ID"
        ]
        .nunique()
        == 10,

    "exact_2014_origins":
        len(
            final_target_df
        )
        == 2014,

    "exact_10070_long_rows":
        len(
            cell10_forecast_metadata_long
        )
        == 10070,

    "zero_duplicate_origins":
        duplicate_origins
        == 0,

    "zero_duplicate_long_keys":
        duplicate_long_keys
        == 0,

    "zero_alignment_errors":
        alignment_errors
        == 0,

    "zero_boundary_errors":
        boundary_errors
        == 0,

    "zero_graph_target_errors":
        future_graph_errors
        == 0,

    "zero_logic_errors":
        logic_errors
        == 0,

    "zero_leakage_overlap":
        len(
            leakage_overlap
        )
        == 0,
}


failed_checks = [

    name

    for name, passed
    in hard_validation.items()

    if not passed
]


if failed_checks:

    raise RuntimeError(
        "FINAL CELL 10 HARD VALIDATION FAILED:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in failed_checks
        )
    )


# ================================================================================================================
# 31. FINAL VALIDATION SUMMARY
# ================================================================================================================

print("\n")
print("=" * 150)
print("CELL 10 — FINAL VALIDATION SUMMARY")
print("=" * 150)


print(
    f"\nHistory length                    : "
    f"{HISTORY_LENGTH} minutes"
)

print(
    f"Forecast horizons                 : "
    f"{FORECAST_HORIZONS}"
)

print(
    f"Maximum forecast horizon          : "
    f"{MAX_HORIZON} minutes"
)

print(
    f"Valid forecasting origins         : "
    f"{len(final_target_df):,}"
)

print(
    f"Long-form origin×horizon rows     : "
    f"{len(cell10_forecast_metadata_long):,}"
)

print(
    f"Segments represented              : "
    f"{final_target_df['Segment_ID'].nunique():,}"
)

print(
    f"Cell 9 model predictors preserved : "
    f"{len(MODEL_FEATURE_COLUMNS):,}"
)

print(
    f"Attack classes preserved          : "
    f"{len(ATTACK_TYPE_VOCABULARY):,}"
)

print(
    f"Duplicate origins                 : "
    f"{duplicate_origins:,}"
)

print(
    f"Duplicate long-form keys          : "
    f"{duplicate_long_keys:,}"
)

print(
    f"Target alignment errors           : "
    f"{alignment_errors:,}"
)

print(
    f"Segment-boundary violations       : "
    f"{boundary_errors:,}"
)

print(
    f"Future graph-target errors         : "
    f"{future_graph_errors:,}"
)

print(
    f"Onset/continuation logic errors   : "
    f"{logic_errors:,}"
)

print(
    f"Predictor/forbidden overlap       : "
    f"{len(leakage_overlap):,}"
)


print("\nForecast targets available at EVERY horizon:")

print(
    "  ✓ Future binary attack state"
)

print(
    "  ✓ Future attack type set"
)

print(
    "  ✓ Future attacker/source set"
)

print(
    "  ✓ Future destination/victim-candidate set"
)

print(
    "  ✓ Future source -> destination attack edges"
)

print(
    "  ✓ Future source -> destination -> attack-type edges"
)

print(
    "  ✓ Future graph cardinalities"
)

print(
    "  ✓ Genuine new-onset indicator"
)

print(
    "  ✓ Attack-continuation indicator"
)


print("\nTemporal guarantees:")

print(
    "  ✓ Every sample uses exactly 15 historical minutes."
)

print(
    "  ✓ Every future target corresponds to exactly t + H."
)

print(
    "  ✓ Histories never cross capture discontinuities."
)

print(
    "  ✓ Future targets never cross capture discontinuities."
)

print(
    "  ✓ All five horizons exist simultaneously for every origin."
)


print("\nLeakage guarantees:")

print(
    "  ✓ Future labels are not model predictors."
)

print(
    "  ✓ Future attack types are not model predictors."
)

print(
    "  ✓ Future endpoint identities are not model predictors."
)

print(
    "  ✓ Future graph edges are not model predictors."
)

print(
    "  ✓ Cell 9 forbidden-feature protection was preserved and extended."
)


print("\nPipeline state:")

print(
    "  ✓ No TRAIN/VALIDATION/TEST split created."
)

print(
    "  ✓ No scaler fitted."
)

print(
    "  ✓ No resampling performed."
)

print(
    "  ✓ No model trained."
)


print("\nObjects created:")

print(
    "  cell10_target_table"
)

print(
    "  cell10_valid_origins"
)

print(
    "  cell10_forecast_metadata_long"
)

print(
    "  cell10_metadata"
)

print(
    "  target_summary_df"
)

print(
    "  history_summary_df"
)

print(
    "  onset_support_df"
)

print(
    "  attack_type_support_df"
)

print(
    "  graph_support_df"
)

print(
    "  graph_target_audit_df"
)

print(
    "  segment_sample_summary"
)

print(
    "  FORBIDDEN_MODEL_FEATURES"
)


print("\n")
print("=" * 150)

print(
    "✓ CELL 10 FINAL MULTI-HORIZON FORECASTING "
    "FORMULATION FULLY VALIDATED."
)

print(
    "✓ Binary + attack-type + attacker + victim/destination "
    "+ graph targets are now aligned."
)

print(
    "✓ No raw ground truth needs to be reconstructed again downstream."
)

print(
    "✓ Cell 11 can now audit attack/campaign structure and "
    "forecastability directly from these canonical targets."
)

print("=" * 150)

CELL 10 — FINAL MULTI-HORIZON FORECASTING TARGET DEFINITION + GRAPH TARGET ALIGNMENT

Final forecasting configuration
------------------------------------------------------------------------------------------------------------------------------------------------------
History length      : 15 minutes
Forecast horizons   : [1, 5, 10, 15, 30]
Target semantics    : POINT target at exactly t + H
Graph semantics     : exact attack graph observed at t + H
Split created       : NO
Scaling performed   : NO
Resampling performed: NO
Model trained       : NO

✓ Required Cell 9 objects found.
✓ Required corrected Cell 9 schema found.

minute_df rows      : 2,454
Segments            : 10
Model predictors    : 86
Attack classes      : 14
✓ Expected 2,454-minute / 10-segment Cell 9 contract confirmed.
✓ (Segment_ID, Minute) key is unique.


MINUTE-LEVEL CONTINUITY CHECK
 Segment_ID  Rows        Start_Minute          End_Minute  Non_1min_Gaps
          0   242 2017-07-03 01:00:00 2017-07-03 05:01:00  

In [19]:
# ==============================================================================================================
# CELL 11 — COMPREHENSIVE ATTACK / GRAPH / CAMPAIGN / SPLIT-FEASIBILITY AUDIT
# ==============================================================================================================
#
# PURPOSE
# -------
# Audit EVERYTHING needed before designing TRAIN / VALIDATION / TEST.
#
# This cell DOES NOT create the split.
#
# It audits:
#
#   1. Final Cell 10 forecasting contract
#   2. Attack-minute structure
#   3. Strict contiguous attack episodes
#   4. Campaign-like temporal groups
#   5. Attack-type support
#   6. Source / attacker support
#   7. Destination / victim-candidate support
#   8. Source -> destination edge support
#   9. Attack-type + source -> destination typed-edge support
#  10. Multi-horizon attack-type / graph forecastability
#  11. New-onset independence
#  12. Segment-level support
#  13. Attack-type concentration across segments
#  14. Graph-entity concentration across segments
#  15. Overlapping-history leakage risk
#  16. Campaign fragmentation risk
#  17. Whether segment-only splitting can preserve important target support
#  18. Constraints that the actual split MUST obey
#
# IMPORTANT
# ---------
# No split is created here.
# No scaler is fitted.
# No resampling is performed.
# No model is trained.
#
# Future metadata is aligned using:
#
#       (Segment_ID, Target_Time)
#
# against the canonical minute-level ground truth in minute_df.
#
# Therefore this cell does NOT guess future attack types, victims, attackers,
# edges, or typed edges.
#
# ==============================================================================================================


import ast
import itertools
import math
import re
from collections import Counter, defaultdict

import numpy as np
import pandas as pd


# ==============================================================================================================
# CONFIGURATION
# ==============================================================================================================

CAMPAIGN_GAP_THRESHOLDS = [0, 1, 2, 5, 10, 15, 30]
PRIMARY_CAMPAIGN_GAP = 5

# Minimum number of segments in which a target must occur before it is even
# theoretically possible for a pure segment-disjoint train/val/test split
# to contain that target in all three partitions.
MIN_SEGMENTS_FOR_3WAY_SEGMENT_SPLIT = 3


print("=" * 150)
print("CELL 11 — COMPREHENSIVE ATTACK / GRAPH / CAMPAIGN / SPLIT-FEASIBILITY AUDIT")
print("=" * 150)

print("\nConfiguration")
print("-" * 150)
print(f"Campaign-gap candidates                 : {CAMPAIGN_GAP_THRESHOLDS} minutes")
print(f"Primary descriptive campaign gap        : {PRIMARY_CAMPAIGN_GAP} minutes")
print("Split created                            : NO")
print("Scaling performed                        : NO")
print("Resampling performed                     : NO")
print("Model trained                            : NO")


# ==============================================================================================================
# REQUIRED OBJECT CHECK
# ==============================================================================================================

required_objects = [
    "minute_df",
    "cell10_target_table",
    "cell10_metadata",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Missing required objects:\n"
        + "\n".join(f"  - {x}" for x in missing_objects)
        + "\n\nRun the FINAL corrected Cells 9 and 10 first."
    )


minute = minute_df.copy()
targets = cell10_target_table.copy()
meta = dict(cell10_metadata)


# ==============================================================================================================
# SAFE COLUMN-NAME UTILITIES
# ==============================================================================================================

def norm_name(name):
    """
    Normalize a column name for schema discovery.
    """
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(name).strip().lower()
    )


def find_exact_normalized_column(df, candidates):
    """
    Find a column by normalized exact match.
    Returns None when no match exists.
    """
    normalized_map = {
        norm_name(c): c
        for c in df.columns
    }

    for candidate in candidates:
        key = norm_name(candidate)

        if key in normalized_map:
            return normalized_map[key]

    return None


def find_semantic_column(
    df,
    preferred_names=None,
    must_contain=None,
    must_not_contain=None
):
    """
    Robust semantic column finder.

    1. First tries preferred exact-normalized names.
    2. Then searches normalized column names using required/excluded tokens.
    """

    preferred_names = preferred_names or []
    must_contain = must_contain or []
    must_not_contain = must_not_contain or []

    exact = find_exact_normalized_column(
        df,
        preferred_names
    )

    if exact is not None:
        return exact

    matches = []

    for col in df.columns:

        n = norm_name(col)

        if all(
            norm_name(token) in n
            for token in must_contain
        ) and not any(
            norm_name(token) in n
            for token in must_not_contain
        ):
            matches.append(col)

    if len(matches) == 1:
        return matches[0]

    return None


# ==============================================================================================================
# FORECAST CONFIGURATION DISCOVERY
# ==============================================================================================================

if "forecast_horizons" in meta:
    FORECAST_HORIZONS = [
        int(x)
        for x in meta["forecast_horizons"]
    ]
else:
    horizon_candidates = []

    for col in targets.columns:
        m = re.fullmatch(
            r"Target_State_H(\d+)",
            str(col)
        )

        if m:
            horizon_candidates.append(
                int(m.group(1))
            )

    FORECAST_HORIZONS = sorted(
        set(horizon_candidates)
    )


if not FORECAST_HORIZONS:
    raise RuntimeError(
        "Could not discover forecast horizons from "
        "cell10_metadata or cell10_target_table."
    )


if "history_length" in meta:
    HISTORY_LENGTH = int(
        meta["history_length"]
    )
else:
    HISTORY_LENGTH = 15


MAX_HORIZON = max(
    FORECAST_HORIZONS
)


print("\n✓ Required Cells 9–10 objects found.")
print(f"minute_df rows       : {len(minute):,}")
print(f"forecast origins     : {len(targets):,}")
print(f"history length       : {HISTORY_LENGTH}")
print(f"forecast horizons    : {FORECAST_HORIZONS}")


# ==============================================================================================================
# CORE COLUMN DISCOVERY
# ==============================================================================================================

segment_col = find_exact_normalized_column(
    minute,
    ["Segment_ID", "Segment ID"]
)

minute_time_col = find_exact_normalized_column(
    minute,
    ["Minute"]
)

attack_state_col = find_exact_normalized_column(
    minute,
    [
        "Attack_State",
        "Attack State",
        "Is_Attack",
        "Is Attack"
    ]
)


if segment_col is None:
    raise RuntimeError(
        "Could not find Segment_ID in minute_df."
    )

if minute_time_col is None:
    raise RuntimeError(
        "Could not find Minute in minute_df."
    )

if attack_state_col is None:
    raise RuntimeError(
        "Could not find Attack_State in minute_df."
    )


# ==============================================================================================================
# OPTIONAL ATTACK / GRAPH METADATA DISCOVERY
# ==============================================================================================================
#
# We intentionally discover these instead of hard-coding one spelling.
#
# ==============================================================================================================

attack_type_col = find_semantic_column(
    minute,
    preferred_names=[
        "Attack_Types",
        "Attack_Type",
        "Attack Labels",
        "Attack_Labels",
        "Attack_Label_Set",
        "Attack_Type_Set",
    ],
    must_contain=["attack", "type"],
    must_not_contain=[
        "count",
        "state",
        "flow",
        "dominant"
    ]
)


source_col = find_semantic_column(
    minute,
    preferred_names=[
        "Attack_Source_IPs",
        "Attack_Sources",
        "Source_IPs",
        "Source IPs",
        "Attacker_IPs",
        "Attacker IP",
    ],
    must_contain=["source"],
    must_not_contain=[
        "count",
        "flow"
    ]
)


destination_col = find_semantic_column(
    minute,
    preferred_names=[
        "Attack_Destination_IPs",
        "Attack_Destinations",
        "Destination_IPs",
        "Destination IPs",
        "Victim_IPs",
        "Victim_Candidates",
        "Attack_Victim_IPs",
    ],
    must_contain=["destination"],
    must_not_contain=[
        "count",
        "flow"
    ]
)


if destination_col is None:
    destination_col = find_semantic_column(
        minute,
        preferred_names=[
            "Victim_IPs",
            "Victim_Candidates",
            "Attack_Victim_IPs",
        ],
        must_contain=["victim"],
        must_not_contain=[
            "count",
            "flow"
        ]
    )


edge_col = find_semantic_column(
    minute,
    preferred_names=[
        "Attack_Edges",
        "Attack_Edge_Set",
        "Edges",
        "Graph_Edges",
    ],
    must_contain=["edge"],
    must_not_contain=[
        "typed",
        "count"
    ]
)


typed_edge_col = find_semantic_column(
    minute,
    preferred_names=[
        "Typed_Attack_Edges",
        "Attack_Typed_Edges",
        "Typed_Edges",
        "Attack_Type_Edges",
    ],
    must_contain=[
        "typed",
        "edge"
    ],
    must_not_contain=[
        "count"
    ]
)


print("\n")
print("=" * 150)
print("DISCOVERED CANONICAL MINUTE-LEVEL SCHEMA")
print("=" * 150)

print(f"Segment column              : {segment_col}")
print(f"Minute column               : {minute_time_col}")
print(f"Attack-state column         : {attack_state_col}")
print(f"Attack-type metadata        : {attack_type_col}")
print(f"Source/attacker metadata    : {source_col}")
print(f"Destination/victim metadata : {destination_col}")
print(f"Attack-edge metadata        : {edge_col}")
print(f"Typed-edge metadata         : {typed_edge_col}")


# ==============================================================================================================
# STANDARDIZE CORE COLUMN NAMES LOCALLY
# ==============================================================================================================

rename_map = {
    segment_col: "Segment_ID",
    minute_time_col: "Minute",
    attack_state_col: "Attack_State",
}

minute = minute.rename(
    columns=rename_map
)


minute["Minute"] = pd.to_datetime(
    minute["Minute"],
    errors="raise"
)


minute["Segment_ID"] = pd.to_numeric(
    minute["Segment_ID"],
    errors="raise"
).astype(int)


minute["Attack_State"] = pd.to_numeric(
    minute["Attack_State"],
    errors="raise"
)


if minute["Attack_State"].isna().any():
    raise RuntimeError(
        "Attack_State contains missing values."
    )


minute["Attack_State"] = (
    minute["Attack_State"]
    .astype(int)
)


if not set(
    minute["Attack_State"].unique()
).issubset({0, 1}):
    raise RuntimeError(
        "Attack_State is not binary."
    )


minute = (
    minute
    .sort_values(
        ["Segment_ID", "Minute"]
    )
    .reset_index(drop=True)
)


if minute[
    ["Segment_ID", "Minute"]
].duplicated().any():

    dup = minute.loc[
        minute[
            ["Segment_ID", "Minute"]
        ].duplicated(
            keep=False
        ),
        ["Segment_ID", "Minute"]
    ].head(20)

    raise RuntimeError(
        "Duplicate (Segment_ID, Minute) keys exist.\n"
        + dup.to_string(index=False)
    )


# ==============================================================================================================
# UPDATE OPTIONAL COLUMN NAMES AFTER RENAME
# ==============================================================================================================

def remapped_optional_name(original_name):
    if original_name is None:
        return None

    return rename_map.get(
        original_name,
        original_name
    )


attack_type_col = remapped_optional_name(
    attack_type_col
)

source_col = remapped_optional_name(
    source_col
)

destination_col = remapped_optional_name(
    destination_col
)

edge_col = remapped_optional_name(
    edge_col
)

typed_edge_col = remapped_optional_name(
    typed_edge_col
)


# ==============================================================================================================
# ROBUST COLLECTION NORMALIZATION
# ==============================================================================================================

EMPTY_MARKERS = {
    "",
    "none",
    "nan",
    "nat",
    "<na>",
    "null",
    "benign",
}


def normalize_scalar_string(value):
    """
    Clean one scalar metadata value.
    """

    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    value = str(value).strip()

    if value.lower() in EMPTY_MARKERS:
        return None

    return value


def to_collection(value):
    """
    Convert list/set/tuple/string/scalar metadata into a deterministic tuple.

    Handles:
      - tuple
      - list
      - set
      - numpy array
      - pandas Series
      - string representations of Python collections
      - delimiter-separated strings
      - scalar strings
    """

    if value is None:
        return tuple()

    try:
        if pd.isna(value):
            return tuple()
    except Exception:
        pass

    if isinstance(
        value,
        (
            list,
            tuple,
            set,
            frozenset,
            np.ndarray,
            pd.Series
        )
    ):

        values = list(value)

    elif isinstance(value, str):

        s = value.strip()

        if s.lower() in EMPTY_MARKERS:
            return tuple()

        parsed = None

        if (
            (s.startswith("[") and s.endswith("]"))
            or
            (s.startswith("(") and s.endswith(")"))
            or
            (s.startswith("{") and s.endswith("}"))
        ):

            try:
                parsed = ast.literal_eval(s)
            except Exception:
                parsed = None

        if isinstance(
            parsed,
            (
                list,
                tuple,
                set,
                frozenset
            )
        ):
            values = list(parsed)

        elif parsed is not None and not isinstance(
            parsed,
            dict
        ):
            values = [parsed]

        else:
            # Only split obvious multi-value textual forms.
            # Do NOT split IP addresses.
            if "||" in s:
                values = s.split("||")

            elif ";" in s:
                values = s.split(";")

            elif "|" in s:
                values = s.split("|")

            else:
                values = [s]

    else:
        values = [value]


    cleaned = []

    for item in values:

        if item is None:
            continue

        try:
            if pd.isna(item):
                continue
        except Exception:
            pass

        if isinstance(
            item,
            (
                tuple,
                list,
                set,
                frozenset
            )
        ):
            item = tuple(
                str(x).strip()
                for x in item
            )

            cleaned.append(item)

        else:
            item = str(item).strip()

            if item.lower() in EMPTY_MARKERS:
                continue

            cleaned.append(item)


    # preserve deterministic uniqueness
    output = []

    seen = set()

    for item in cleaned:

        key = repr(item)

        if key not in seen:
            seen.add(key)
            output.append(item)

    return tuple(output)


def explode_collection_column(df, column):
    """
    Return one row per element from a collection-valued column.
    """

    records = []

    for idx, value in df[column].items():

        for item in to_collection(value):

            records.append({
                "_row_index": idx,
                "Value": item,
            })

    return pd.DataFrame(
        records,
        columns=[
            "_row_index",
            "Value"
        ]
    )


# ==============================================================================================================
# FORECAST-TABLE CONTRACT VALIDATION
# ==============================================================================================================

required_target_base = [
    "Segment_ID",
    "Origin_Time",
    "History_Start",
    "History_End",
    "Attack_In_History",
    "Clean_History",
]


missing_base = [
    c for c in required_target_base
    if c not in targets.columns
]


if missing_base:
    raise RuntimeError(
        "cell10_target_table is missing required base columns:\n"
        + "\n".join(
            f"  - {x}"
            for x in missing_base
        )
    )


for H in FORECAST_HORIZONS:

    required_h = [
        f"Target_Time_H{H}",
        f"Target_State_H{H}",
        f"New_Onset_H{H}",
        f"Continuation_H{H}",
    ]

    missing_h = [
        c for c in required_h
        if c not in targets.columns
    ]

    if missing_h:
        raise RuntimeError(
            f"H{H}: missing required Cell 10 columns:\n"
            + "\n".join(
                f"  - {x}"
                for x in missing_h
            )
        )


targets["Segment_ID"] = pd.to_numeric(
    targets["Segment_ID"],
    errors="raise"
).astype(int)


targets["Origin_Time"] = pd.to_datetime(
    targets["Origin_Time"],
    errors="raise"
)


targets["History_Start"] = pd.to_datetime(
    targets["History_Start"],
    errors="raise"
)


targets["History_End"] = pd.to_datetime(
    targets["History_End"],
    errors="raise"
)


for H in FORECAST_HORIZONS:

    targets[f"Target_Time_H{H}"] = pd.to_datetime(
        targets[f"Target_Time_H{H}"],
        errors="raise"
    )

    for c in [
        f"Target_State_H{H}",
        f"New_Onset_H{H}",
        f"Continuation_H{H}",
    ]:

        targets[c] = pd.to_numeric(
            targets[c],
            errors="raise"
        ).astype(int)


targets = (
    targets
    .sort_values(
        ["Segment_ID", "Origin_Time"]
    )
    .reset_index(drop=True)
)


if targets[
    ["Segment_ID", "Origin_Time"]
].duplicated().any():

    raise RuntimeError(
        "Duplicate forecasting origins exist in cell10_target_table."
    )


# ==============================================================================================================
# TARGET-TIME ALIGNMENT VALIDATION
# ==============================================================================================================

alignment_errors = 0
missing_target_minutes = 0
state_mismatches = 0


minute_state_lookup = (
    minute[
        [
            "Segment_ID",
            "Minute",
            "Attack_State"
        ]
    ]
    .set_index(
        ["Segment_ID", "Minute"]
    )["Attack_State"]
    .to_dict()
)


for H in FORECAST_HORIZONS:

    expected_times = (
        targets["Origin_Time"]
        + pd.to_timedelta(
            H,
            unit="m"
        )
    )

    alignment_errors += int(
        (
            expected_times
            != targets[f"Target_Time_H{H}"]
        ).sum()
    )

    for _, row in targets.iterrows():

        key = (
            int(row["Segment_ID"]),
            pd.Timestamp(
                row[f"Target_Time_H{H}"]
            )
        )

        if key not in minute_state_lookup:
            missing_target_minutes += 1
            continue

        actual_state = int(
            minute_state_lookup[key]
        )

        target_state = int(
            row[f"Target_State_H{H}"]
        )

        if actual_state != target_state:
            state_mismatches += 1


print("\n")
print("=" * 150)
print("FORECASTING CONTRACT VALIDATION")
print("=" * 150)

print(f"Forecast origins              : {len(targets):,}")
print(f"Forecast horizons             : {FORECAST_HORIZONS}")
print(f"Expected horizon rows         : {len(targets) * len(FORECAST_HORIZONS):,}")
print(f"Target-time alignment errors  : {alignment_errors:,}")
print(f"Missing canonical target mins : {missing_target_minutes:,}")
print(f"Target-state mismatches       : {state_mismatches:,}")


if alignment_errors != 0:
    raise RuntimeError(
        f"{alignment_errors:,} target-time alignment errors detected."
    )

if missing_target_minutes != 0:
    raise RuntimeError(
        f"{missing_target_minutes:,} forecast targets do not map "
        "to canonical minute_df rows."
    )

if state_mismatches != 0:
    raise RuntimeError(
        f"{state_mismatches:,} Cell 10 target states disagree "
        "with canonical minute_df."
    )


print("\n✓ Cell 10 forecasting contract matches canonical minute_df.")


# ==============================================================================================================
# MINUTE-LEVEL CONTINUITY
# ==============================================================================================================

continuity_records = []


for segment_id, seg in minute.groupby(
    "Segment_ID",
    sort=True
):

    seg = (
        seg
        .sort_values("Minute")
        .reset_index(drop=True)
    )

    diffs = (
        seg["Minute"]
        .diff()
        .dropna()
    )

    non_one = int(
        (
            diffs
            != pd.Timedelta(minutes=1)
        ).sum()
    )

    continuity_records.append({
        "Segment_ID": int(segment_id),
        "Rows": len(seg),
        "Start_Minute": seg["Minute"].min(),
        "End_Minute": seg["Minute"].max(),
        "Non_1min_Gaps": non_one,
    })


continuity_df = pd.DataFrame(
    continuity_records
)


print("\n")
print("=" * 150)
print("MINUTE-LEVEL CONTINUITY")
print("=" * 150)

print(
    continuity_df.to_string(
        index=False
    )
)


if continuity_df["Non_1min_Gaps"].sum() != 0:
    raise RuntimeError(
        "Canonical minute segments are not perfectly 1-minute continuous."
    )


print("\n✓ Every segment is exactly continuous at 1-minute resolution.")


# ==============================================================================================================
# ATTACK MINUTE INVENTORY
# ==============================================================================================================

attack_minutes = minute.loc[
    minute["Attack_State"] == 1
].copy()


benign_minutes = minute.loc[
    minute["Attack_State"] == 0
].copy()


if attack_minutes.empty:
    raise RuntimeError(
        "No attack-positive minutes exist."
    )


print("\n")
print("=" * 150)
print("ATTACK MINUTE INVENTORY")
print("=" * 150)

print(f"Total minute rows : {len(minute):,}")
print(f"Attack minutes    : {len(attack_minutes):,}")
print(f"Benign minutes    : {len(benign_minutes):,}")
print(
    f"Attack percentage : "
    f"{100 * len(attack_minutes) / len(minute):.4f}%"
)


# ==============================================================================================================
# STRICT CONTIGUOUS ATTACK EPISODES
# ==============================================================================================================

strict_records = []

episode_id = 0


for segment_id, seg in minute.groupby(
    "Segment_ID",
    sort=True
):

    seg = (
        seg
        .sort_values("Minute")
        .reset_index(drop=True)
    )

    active_start = None
    active_end = None
    duration = 0

    for _, row in seg.iterrows():

        t = row["Minute"]
        state = int(
            row["Attack_State"]
        )

        if state == 1:

            if active_start is None:

                active_start = t
                active_end = t
                duration = 1

            elif (
                t
                == active_end
                + pd.Timedelta(minutes=1)
            ):

                active_end = t
                duration += 1

            else:

                strict_records.append({
                    "Episode_ID": episode_id,
                    "Segment_ID": int(segment_id),
                    "Start_Minute": active_start,
                    "End_Minute": active_end,
                    "Duration_Minutes": duration,
                })

                episode_id += 1

                active_start = t
                active_end = t
                duration = 1

        else:

            if active_start is not None:

                strict_records.append({
                    "Episode_ID": episode_id,
                    "Segment_ID": int(segment_id),
                    "Start_Minute": active_start,
                    "End_Minute": active_end,
                    "Duration_Minutes": duration,
                })

                episode_id += 1

                active_start = None
                active_end = None
                duration = 0

    if active_start is not None:

        strict_records.append({
            "Episode_ID": episode_id,
            "Segment_ID": int(segment_id),
            "Start_Minute": active_start,
            "End_Minute": active_end,
            "Duration_Minutes": duration,
        })

        episode_id += 1


strict_episode_df = pd.DataFrame(
    strict_records
)


if strict_episode_df.empty:
    raise RuntimeError(
        "Strict attack-episode construction produced zero episodes."
    )


strict_attack_total = int(
    strict_episode_df[
        "Duration_Minutes"
    ].sum()
)


if strict_attack_total != len(
    attack_minutes
):
    raise RuntimeError(
        "Strict episode accounting mismatch.\n"
        f"Attack minutes : {len(attack_minutes):,}\n"
        f"Episode total  : {strict_attack_total:,}"
    )


print("\n")
print("=" * 150)
print("STRICT CONTIGUOUS ATTACK EPISODES")
print("=" * 150)

print(
    f"Strict episodes            : "
    f"{len(strict_episode_df):,}"
)

print(
    f"Attack minutes represented : "
    f"{strict_attack_total:,}"
)

print(
    "\nEpisode duration statistics:"
)

print(
    strict_episode_df[
        "Duration_Minutes"
    ]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
    .to_string()
)

print(
    "\n✓ Every attack minute belongs to exactly one strict episode."
)


# ==============================================================================================================
# ATTACK STRUCTURE BY SEGMENT
# ==============================================================================================================

segment_attack_summary = (
    minute
    .groupby("Segment_ID")
    .agg(
        Segment_Start=(
            "Minute",
            "min"
        ),
        Segment_End=(
            "Minute",
            "max"
        ),
        Minute_Rows=(
            "Minute",
            "size"
        ),
        Attack_Minutes=(
            "Attack_State",
            "sum"
        ),
    )
    .reset_index()
)


episode_counts = (
    strict_episode_df
    .groupby("Segment_ID")
    .size()
    .rename("Strict_Episodes")
    .reset_index()
)


segment_attack_summary = (
    segment_attack_summary
    .merge(
        episode_counts,
        on="Segment_ID",
        how="left"
    )
)


segment_attack_summary[
    "Strict_Episodes"
] = (
    segment_attack_summary[
        "Strict_Episodes"
    ]
    .fillna(0)
    .astype(int)
)


segment_attack_summary[
    "Attack_Percent"
] = (
    100
    * segment_attack_summary[
        "Attack_Minutes"
    ]
    / segment_attack_summary[
        "Minute_Rows"
    ]
)


print("\n")
print("=" * 150)
print("ATTACK STRUCTURE BY SEGMENT")
print("=" * 150)

print(
    segment_attack_summary.to_string(
        index=False,
        formatters={
            "Attack_Percent":
                lambda x: f"{x:.4f}"
        }
    )
)


# ==============================================================================================================
# BENIGN GAPS BETWEEN STRICT EPISODES
# ==============================================================================================================

gap_records = []


for segment_id, eps in strict_episode_df.groupby(
    "Segment_ID",
    sort=True
):

    eps = (
        eps
        .sort_values("Start_Minute")
        .reset_index(drop=True)
    )

    for i in range(
        len(eps) - 1
    ):

        left = eps.iloc[i]
        right = eps.iloc[i + 1]

        benign_gap = int(
            (
                right["Start_Minute"]
                - left["End_Minute"]
            ).total_seconds()
            // 60
        ) - 1

        if benign_gap < 0:
            raise RuntimeError(
                "Negative attack-episode gap detected."
            )

        gap_records.append({
            "Segment_ID": int(segment_id),
            "Left_Episode_ID":
                int(left["Episode_ID"]),
            "Right_Episode_ID":
                int(right["Episode_ID"]),
            "Benign_Gap_Minutes":
                benign_gap,
        })


episode_gap_df = pd.DataFrame(
    gap_records,
    columns=[
        "Segment_ID",
        "Left_Episode_ID",
        "Right_Episode_ID",
        "Benign_Gap_Minutes",
    ]
)


gap_threshold_rows = []


for threshold in CAMPAIGN_GAP_THRESHOLDS:

    if episode_gap_df.empty:
        count = 0
        pct = 0.0

    else:
        count = int(
            (
                episode_gap_df[
                    "Benign_Gap_Minutes"
                ]
                <= threshold
            ).sum()
        )

        pct = (
            100
            * count
            / len(episode_gap_df)
        )

    gap_threshold_rows.append({
        "Gap_Threshold_Minutes":
            threshold,
        "Episode_Gaps_At_Or_Below":
            count,
        "Percent_Of_All_Gaps":
            pct,
    })


episode_gap_summary_df = pd.DataFrame(
    gap_threshold_rows
)


print("\n")
print("=" * 150)
print("BENIGN-GAP STRUCTURE")
print("=" * 150)

print(
    episode_gap_summary_df.to_string(
        index=False,
        formatters={
            "Percent_Of_All_Gaps":
                lambda x: f"{x:.4f}"
        }
    )
)


# ==============================================================================================================
# CAMPAIGN-LIKE GROUP CONSTRUCTION
# ==============================================================================================================
#
# IMPORTANT:
#
# These are NOT claimed CICIDS2017 ground-truth campaigns.
#
# They are temporal audit groups created by merging strict episodes inside
# the SAME segment when the benign gap between them is <= threshold.
#
# ==============================================================================================================

def build_campaign_groups(
    strict_df,
    gap_threshold
):

    records = []
    campaign_id = 0

    for segment_id, eps in strict_df.groupby(
        "Segment_ID",
        sort=True
    ):

        eps = (
            eps
            .sort_values("Start_Minute")
            .reset_index(drop=True)
        )

        if eps.empty:
            continue

        current_episode_ids = [
            int(
                eps.loc[
                    0,
                    "Episode_ID"
                ]
            )
        ]

        current_start = eps.loc[
            0,
            "Start_Minute"
        ]

        current_end = eps.loc[
            0,
            "End_Minute"
        ]

        attack_minute_count = int(
            eps.loc[
                0,
                "Duration_Minutes"
            ]
        )

        for i in range(
            1,
            len(eps)
        ):

            row = eps.loc[i]

            benign_gap = int(
                (
                    row["Start_Minute"]
                    - current_end
                ).total_seconds()
                // 60
            ) - 1

            if benign_gap <= gap_threshold:

                current_episode_ids.append(
                    int(
                        row["Episode_ID"]
                    )
                )

                current_end = row[
                    "End_Minute"
                ]

                attack_minute_count += int(
                    row[
                        "Duration_Minutes"
                    ]
                )

            else:

                records.append({
                    "Campaign_ID":
                        campaign_id,
                    "Segment_ID":
                        int(segment_id),
                    "Campaign_Start":
                        current_start,
                    "Campaign_End":
                        current_end,
                    "Strict_Episode_Count":
                        len(
                            current_episode_ids
                        ),
                    "Attack_Minutes":
                        attack_minute_count,
                    "Episode_IDs":
                        tuple(
                            current_episode_ids
                        ),
                })

                campaign_id += 1

                current_episode_ids = [
                    int(
                        row[
                            "Episode_ID"
                        ]
                    )
                ]

                current_start = row[
                    "Start_Minute"
                ]

                current_end = row[
                    "End_Minute"
                ]

                attack_minute_count = int(
                    row[
                        "Duration_Minutes"
                    ]
                )

        records.append({
            "Campaign_ID":
                campaign_id,
            "Segment_ID":
                int(segment_id),
            "Campaign_Start":
                current_start,
            "Campaign_End":
                current_end,
            "Strict_Episode_Count":
                len(
                    current_episode_ids
                ),
            "Attack_Minutes":
                attack_minute_count,
            "Episode_IDs":
                tuple(
                    current_episode_ids
                ),
        })

        campaign_id += 1

    out = pd.DataFrame(
        records
    )

    if not out.empty:

        out[
            "Campaign_Span_Minutes"
        ] = (
            (
                out["Campaign_End"]
                - out["Campaign_Start"]
            ).dt.total_seconds()
            // 60
            + 1
        ).astype(int)

    return out


campaign_tables = {}
campaign_summary_rows = []


for threshold in CAMPAIGN_GAP_THRESHOLDS:

    cdf = build_campaign_groups(
        strict_episode_df,
        threshold
    )

    campaign_tables[
        threshold
    ] = cdf

    campaign_summary_rows.append({
        "Merge_Benign_Gap_Minutes":
            threshold,
        "Campaign_Like_Groups":
            len(cdf),
        "Strict_Episodes":
            len(strict_episode_df),
    })


campaign_threshold_summary_df = pd.DataFrame(
    campaign_summary_rows
)


campaign_df = campaign_tables[
    PRIMARY_CAMPAIGN_GAP
].copy()


print("\n")
print("=" * 150)
print("CAMPAIGN-LIKE GROUP SENSITIVITY")
print("=" * 150)

print(
    campaign_threshold_summary_df
    .to_string(
        index=False
    )
)


print("\n")
print("=" * 150)
print(
    f"PRIMARY CAMPAIGN-LIKE GROUPING — "
    f"MERGE BENIGN GAPS <= {PRIMARY_CAMPAIGN_GAP} MINUTES"
)
print("=" * 150)

print(
    campaign_df[
        [
            "Campaign_ID",
            "Segment_ID",
            "Campaign_Start",
            "Campaign_End",
            "Campaign_Span_Minutes",
            "Strict_Episode_Count",
            "Attack_Minutes",
        ]
    ].to_string(
        index=False
    )
)


# ==============================================================================================================
# MAP ATTACK MINUTES -> EPISODES -> CAMPAIGNS
# ==============================================================================================================

minute_episode_lookup = {}


for _, ep in strict_episode_df.iterrows():

    for t in pd.date_range(
        start=ep["Start_Minute"],
        end=ep["End_Minute"],
        freq="min"
    ):

        key = (
            int(
                ep["Segment_ID"]
            ),
            pd.Timestamp(t)
        )

        if key in minute_episode_lookup:
            raise RuntimeError(
                "An attack minute maps to multiple strict episodes."
            )

        minute_episode_lookup[
            key
        ] = int(
            ep["Episode_ID"]
        )


episode_to_campaign = {}


for _, campaign in campaign_df.iterrows():

    for eid in campaign[
        "Episode_IDs"
    ]:

        if eid in episode_to_campaign:
            raise RuntimeError(
                "An episode maps to multiple primary campaigns."
            )

        episode_to_campaign[
            int(eid)
        ] = int(
            campaign[
                "Campaign_ID"
            ]
        )


if len(
    episode_to_campaign
) != len(
    strict_episode_df
):
    raise RuntimeError(
        "Not every strict episode maps to one primary campaign."
    )


# ==============================================================================================================
# CANONICAL METADATA NORMALIZATION
# ==============================================================================================================

metadata_columns = {
    "Attack_Types": attack_type_col,
    "Attack_Sources": source_col,
    "Attack_Destinations": destination_col,
    "Attack_Edges": edge_col,
    "Typed_Attack_Edges": typed_edge_col,
}


available_metadata = {
    canonical: original
    for canonical, original
    in metadata_columns.items()
    if original is not None
}


print("\n")
print("=" * 150)
print("ATTACK / GRAPH METADATA AVAILABILITY")
print("=" * 150)


for canonical, original in metadata_columns.items():

    status = (
        f"AVAILABLE ({original})"
        if original is not None
        else "NOT DISCOVERED"
    )

    print(
        f"{canonical:<25} : {status}"
    )


for canonical, original in available_metadata.items():

    minute[
        f"__{canonical}"
    ] = minute[
        original
    ].apply(
        to_collection
    )


# ==============================================================================================================
# METADATA CONSISTENCY AGAINST ATTACK STATE
# ==============================================================================================================

metadata_consistency_rows = []


for canonical in available_metadata:

    c = f"__{canonical}"

    attack_rows_empty = int(
        (
            (
                minute[
                    "Attack_State"
                ] == 1
            )
            &
            (
                minute[c].apply(
                    len
                ) == 0
            )
        ).sum()
    )

    benign_rows_nonempty = int(
        (
            (
                minute[
                    "Attack_State"
                ] == 0
            )
            &
            (
                minute[c].apply(
                    len
                ) > 0
            )
        ).sum()
    )

    metadata_consistency_rows.append({
        "Metadata":
            canonical,
        "Attack_Minutes_With_Empty_Metadata":
            attack_rows_empty,
        "Benign_Minutes_With_Nonempty_Metadata":
            benign_rows_nonempty,
    })


metadata_consistency_df = pd.DataFrame(
    metadata_consistency_rows
)


print("\n")
print("=" * 150)
print("ATTACK / GRAPH METADATA CONSISTENCY")
print("=" * 150)


if metadata_consistency_df.empty:

    print(
        "No optional attack/graph metadata columns were discovered."
    )

else:

    print(
        metadata_consistency_df
        .to_string(
            index=False
        )
    )


# ==============================================================================================================
# GENERIC SUPPORT TABLE BUILDER
# ==============================================================================================================

def build_value_support_table(
    frame,
    collection_col,
    value_name
):

    records = []

    for _, row in frame.iterrows():

        values = row[
            collection_col
        ]

        for value in values:

            records.append({
                value_name:
                    value,
                "Segment_ID":
                    int(
                        row[
                            "Segment_ID"
                        ]
                    ),
                "Minute":
                    row[
                        "Minute"
                    ],
            })

    if not records:

        return pd.DataFrame(
            columns=[
                value_name,
                "Minute_Occurrences",
                "Segments",
                "Segment_IDs",
                "First_Minute",
                "Last_Minute",
            ]
        )

    long_df = pd.DataFrame(
        records
    )

    support = (
        long_df
        .groupby(
            value_name,
            dropna=False
        )
        .agg(
            Minute_Occurrences=(
                "Minute",
                "size"
            ),
            Segments=(
                "Segment_ID",
                "nunique"
            ),
            Segment_IDs=(
                "Segment_ID",
                lambda x:
                    tuple(
                        sorted(
                            set(
                                int(v)
                                for v in x
                            )
                        )
                    )
            ),
            First_Minute=(
                "Minute",
                "min"
            ),
            Last_Minute=(
                "Minute",
                "max"
            ),
        )
        .reset_index()
        .sort_values(
            [
                "Segments",
                "Minute_Occurrences",
                value_name,
            ],
            ascending=[
                True,
                False,
                True,
            ]
        )
        .reset_index(
            drop=True
        )
    )

    return support


# ==============================================================================================================
# MINUTE-LEVEL ATTACK-TYPE SUPPORT
# ==============================================================================================================

metadata_support_tables = {}


for canonical in available_metadata:

    support = build_value_support_table(
        minute.loc[
            minute[
                "Attack_State"
            ] == 1
        ],
        f"__{canonical}",
        canonical[:-1]
        if canonical.endswith("s")
        else canonical
    )

    metadata_support_tables[
        canonical
    ] = support


if "Attack_Types" in metadata_support_tables:

    attack_type_support_df = (
        metadata_support_tables[
            "Attack_Types"
        ].copy()
    )

    print("\n")
    print("=" * 150)
    print("ATTACK-TYPE SUPPORT ACROSS CANONICAL SEGMENTS")
    print("=" * 150)

    print(
        attack_type_support_df
        .to_string(
            index=False
        )
    )

else:

    attack_type_support_df = pd.DataFrame()

    print("\n")
    print("=" * 150)
    print("ATTACK-TYPE SUPPORT")
    print("=" * 150)
    print(
        "Attack-type metadata was not discovered in minute_df."
    )


# ==============================================================================================================
# SOURCE / DESTINATION / EDGE SUPPORT
# ==============================================================================================================

for canonical, title in [
    (
        "Attack_Sources",
        "ATTACK SOURCE / ATTACKER SUPPORT"
    ),
    (
        "Attack_Destinations",
        "ATTACK DESTINATION / VICTIM-CANDIDATE SUPPORT"
    ),
    (
        "Attack_Edges",
        "ATTACK EDGE SUPPORT"
    ),
    (
        "Typed_Attack_Edges",
        "TYPED ATTACK-EDGE SUPPORT"
    ),
]:

    print("\n")
    print("=" * 150)
    print(title)
    print("=" * 150)

    if canonical not in metadata_support_tables:

        print(
            f"{canonical} metadata was not discovered."
        )

        continue

    table = metadata_support_tables[
        canonical
    ]

    print(
        f"Unique values : {len(table):,}"
    )

    if len(table) <= 50:

        print(
            table.to_string(
                index=False
            )
        )

    else:

        print(
            "\nFirst 50 values ordered by "
            "segment support / occurrence count:"
        )

        print(
            table.head(50)
            .to_string(
                index=False
            )
        )


# ==============================================================================================================
# ATTACK-TYPE SUPPORT BY SEGMENT
# ==============================================================================================================

attack_type_segment_df = pd.DataFrame()


if "Attack_Types" in available_metadata:

    records = []

    for _, row in minute.loc[
        minute["Attack_State"] == 1
    ].iterrows():

        for attack_type in row[
            "__Attack_Types"
        ]:

            records.append({
                "Segment_ID":
                    int(
                        row[
                            "Segment_ID"
                        ]
                    ),
                "Attack_Type":
                    attack_type,
                "Minute":
                    row[
                        "Minute"
                    ],
            })


    if records:

        attack_type_long_df = pd.DataFrame(
            records
        )

        attack_type_segment_df = (
            attack_type_long_df
            .groupby(
                [
                    "Segment_ID",
                    "Attack_Type"
                ]
            )
            .size()
            .rename(
                "Attack_Minutes"
            )
            .reset_index()
        )

        attack_type_segment_pivot = (
            attack_type_segment_df
            .pivot_table(
                index="Attack_Type",
                columns="Segment_ID",
                values="Attack_Minutes",
                aggfunc="sum",
                fill_value=0
            )
        )

        attack_type_segment_pivot[
            "TOTAL"
        ] = (
            attack_type_segment_pivot
            .sum(
                axis=1
            )
        )

        print("\n")
        print("=" * 150)
        print("ATTACK-TYPE × SEGMENT SUPPORT MATRIX")
        print("=" * 150)

        print(
            attack_type_segment_pivot
            .to_string()
        )


# ==============================================================================================================
# ALIGN FUTURE TARGET TIMES TO CANONICAL MINUTE METADATA
# ==============================================================================================================
#
# This creates ONE long table:
#
#   origin
#   horizon
#   target time
#   future attack state
#   future attack types
#   future sources
#   future destinations
#   future edges
#   future typed edges
#
# This is derived from the canonical minute-level table.
#
# ==============================================================================================================

canonical_lookup_columns = [
    "Segment_ID",
    "Minute",
    "Attack_State",
]


for canonical in available_metadata:
    canonical_lookup_columns.append(
        f"__{canonical}"
    )


minute_lookup_df = (
    minute[
        canonical_lookup_columns
    ]
    .copy()
    .rename(
        columns={
            "Minute":
                "Target_Time",
            "Attack_State":
                "Canonical_Future_Attack_State",
        }
    )
)


forecast_parts = []


for H in FORECAST_HORIZONS:

    part = pd.DataFrame({
        "Segment_ID":
            targets[
                "Segment_ID"
            ].values,
        "Origin_Time":
            targets[
                "Origin_Time"
            ].values,
        "History_Start":
            targets[
                "History_Start"
            ].values,
        "History_End":
            targets[
                "History_End"
            ].values,
        "Attack_In_History":
            targets[
                "Attack_In_History"
            ].values,
        "Clean_History":
            targets[
                "Clean_History"
            ].values,
        "Horizon_Minutes":
            H,
        "Target_Time":
            targets[
                f"Target_Time_H{H}"
            ].values,
        "Future_Attack_State":
            targets[
                f"Target_State_H{H}"
            ].values,
        "New_Onset":
            targets[
                f"New_Onset_H{H}"
            ].values,
        "Continuation":
            targets[
                f"Continuation_H{H}"
            ].values,
    })

    forecast_parts.append(
        part
    )


forecast_long = pd.concat(
    forecast_parts,
    ignore_index=True
)


forecast_long[
    "Target_Time"
] = pd.to_datetime(
    forecast_long[
        "Target_Time"
    ],
    errors="raise"
)


forecast_long = (
    forecast_long
    .merge(
        minute_lookup_df,
        on=[
            "Segment_ID",
            "Target_Time"
        ],
        how="left",
        validate="many_to_one"
    )
)


if forecast_long[
    "Canonical_Future_Attack_State"
].isna().any():

    bad = forecast_long.loc[
        forecast_long[
            "Canonical_Future_Attack_State"
        ].isna(),
        [
            "Segment_ID",
            "Origin_Time",
            "Horizon_Minutes",
            "Target_Time"
        ]
    ].head(20)

    raise RuntimeError(
        "Some forecast targets failed canonical metadata alignment.\n"
        + bad.to_string(
            index=False
        )
    )


future_state_mismatch = int(
    (
        forecast_long[
            "Future_Attack_State"
        ].astype(int)
        !=
        forecast_long[
            "Canonical_Future_Attack_State"
        ].astype(int)
    ).sum()
)


if future_state_mismatch != 0:

    raise RuntimeError(
        f"{future_state_mismatch:,} future-state mismatches "
        "after canonical metadata alignment."
    )


# Rename canonical metadata fields for future use.

for canonical in available_metadata:

    forecast_long[
        f"Future_{canonical}"
    ] = forecast_long[
        f"__{canonical}"
    ]


drop_internal = [
    c
    for c in forecast_long.columns
    if c.startswith("__")
]


forecast_long = forecast_long.drop(
    columns=drop_internal
)


print("\n")
print("=" * 150)
print("MULTI-HORIZON FUTURE METADATA ALIGNMENT")
print("=" * 150)

print(
    f"Forecast rows expected : "
    f"{len(targets) * len(FORECAST_HORIZONS):,}"
)

print(
    f"Forecast rows aligned  : "
    f"{len(forecast_long):,}"
)

print(
    f"State mismatches       : "
    f"{future_state_mismatch:,}"
)

print(
    "✓ Future metadata aligned from canonical "
    "(Segment_ID, Target_Time) ground truth."
)


# ==============================================================================================================
# ATTACK TARGET -> EPISODE / CAMPAIGN ALIGNMENT
# ==============================================================================================================

episode_ids = []
campaign_ids = []


for _, row in forecast_long.iterrows():

    if int(
        row[
            "Future_Attack_State"
        ]
    ) == 0:

        episode_ids.append(
            pd.NA
        )

        campaign_ids.append(
            pd.NA
        )

        continue

    key = (
        int(
            row[
                "Segment_ID"
            ]
        ),
        pd.Timestamp(
            row[
                "Target_Time"
            ]
        )
    )

    eid = minute_episode_lookup.get(
        key
    )

    if eid is None:

        raise RuntimeError(
            "Attack target could not be mapped "
            f"to an episode: {key}"
        )

    cid = episode_to_campaign[
        eid
    ]

    episode_ids.append(
        eid
    )

    campaign_ids.append(
        cid
    )


forecast_long[
    "Episode_ID"
] = pd.array(
    episode_ids,
    dtype="Int64"
)


forecast_long[
    "Campaign_ID"
] = pd.array(
    campaign_ids,
    dtype="Int64"
)


# ==============================================================================================================
# MULTI-HORIZON BINARY / EPISODE SUPPORT
# ==============================================================================================================

horizon_summary_rows = []


for H in FORECAST_HORIZONS:

    hdf = forecast_long.loc[
        forecast_long[
            "Horizon_Minutes"
        ] == H
    ]

    attacks = hdf.loc[
        hdf[
            "Future_Attack_State"
        ] == 1
    ]

    new_onset = attacks.loc[
        attacks[
            "New_Onset"
        ] == 1
    ]

    horizon_summary_rows.append({
        "Horizon_Minutes":
            H,
        "Samples":
            len(hdf),
        "Attack_Targets":
            len(attacks),
        "Benign_Targets":
            int(
                (
                    hdf[
                        "Future_Attack_State"
                    ] == 0
                ).sum()
            ),
        "New_Onset_Targets":
            len(new_onset),
        "Continuation_Targets":
            int(
                attacks[
                    "Continuation"
                ].sum()
            ),
        "Strict_Episodes_Reached":
            attacks[
                "Episode_ID"
            ].nunique(),
        "Campaign_Groups_Reached":
            attacks[
                "Campaign_ID"
            ].nunique(),
        "New_Onset_Episodes":
            new_onset[
                "Episode_ID"
            ].nunique(),
        "New_Onset_Campaigns":
            new_onset[
                "Campaign_ID"
            ].nunique(),
        "New_Onset_Segments":
            new_onset[
                "Segment_ID"
            ].nunique(),
    })


horizon_episode_support_df = pd.DataFrame(
    horizon_summary_rows
)


print("\n")
print("=" * 150)
print("MULTI-HORIZON ATTACK / EPISODE / CAMPAIGN SUPPORT")
print("=" * 150)

print(
    horizon_episode_support_df
    .to_string(
        index=False
    )
)


# ==============================================================================================================
# FUTURE ATTACK-TYPE SUPPORT BY HORIZON
# ==============================================================================================================

future_attack_type_support_df = pd.DataFrame()


if "Attack_Types" in available_metadata:

    records = []

    for _, row in forecast_long.loc[
        forecast_long[
            "Future_Attack_State"
        ] == 1
    ].iterrows():

        for attack_type in row[
            "Future_Attack_Types"
        ]:

            records.append({
                "Horizon_Minutes":
                    int(
                        row[
                            "Horizon_Minutes"
                        ]
                    ),
                "Segment_ID":
                    int(
                        row[
                            "Segment_ID"
                        ]
                    ),
                "Origin_Time":
                    row[
                        "Origin_Time"
                    ],
                "Target_Time":
                    row[
                        "Target_Time"
                    ],
                "Attack_Type":
                    attack_type,
                "New_Onset":
                    int(
                        row[
                            "New_Onset"
                        ]
                    ),
                "Episode_ID":
                    row[
                        "Episode_ID"
                    ],
                "Campaign_ID":
                    row[
                        "Campaign_ID"
                    ],
            })


    future_attack_type_long_df = pd.DataFrame(
        records
    )


    if not future_attack_type_long_df.empty:

        future_attack_type_support_df = (
            future_attack_type_long_df
            .groupby(
                [
                    "Horizon_Minutes",
                    "Attack_Type"
                ],
                dropna=False
            )
            .agg(
                Forecast_Targets=(
                    "Target_Time",
                    "size"
                ),
                Segments=(
                    "Segment_ID",
                    "nunique"
                ),
                Episodes=(
                    "Episode_ID",
                    "nunique"
                ),
                Campaigns=(
                    "Campaign_ID",
                    "nunique"
                ),
                New_Onset_Targets=(
                    "New_Onset",
                    "sum"
                ),
            )
            .reset_index()
        )


        print("\n")
        print("=" * 150)
        print("FUTURE ATTACK-TYPE FORECASTABILITY BY HORIZON")
        print("=" * 150)

        print(
            future_attack_type_support_df
            .to_string(
                index=False
            )
        )


# ==============================================================================================================
# GENERIC FUTURE GRAPH SUPPORT
# ==============================================================================================================

future_graph_support_tables = {}


for canonical in [
    "Attack_Sources",
    "Attack_Destinations",
    "Attack_Edges",
    "Typed_Attack_Edges",
]:

    future_col = (
        f"Future_{canonical}"
    )

    if future_col not in forecast_long.columns:
        continue

    value_name = (
        canonical[:-1]
        if canonical.endswith("s")
        else canonical
    )

    records = []

    for _, row in forecast_long.loc[
        forecast_long[
            "Future_Attack_State"
        ] == 1
    ].iterrows():

        for value in row[
            future_col
        ]:

            records.append({
                "Horizon_Minutes":
                    int(
                        row[
                            "Horizon_Minutes"
                        ]
                    ),
                "Segment_ID":
                    int(
                        row[
                            "Segment_ID"
                        ]
                    ),
                value_name:
                    value,
                "Target_Time":
                    row[
                        "Target_Time"
                    ],
                "New_Onset":
                    int(
                        row[
                            "New_Onset"
                        ]
                    ),
            })


    if not records:
        continue


    long_df = pd.DataFrame(
        records
    )


    summary = (
        long_df
        .groupby(
            [
                "Horizon_Minutes",
                value_name
            ],
            dropna=False
        )
        .agg(
            Forecast_Targets=(
                "Target_Time",
                "size"
            ),
            Segments=(
                "Segment_ID",
                "nunique"
            ),
            New_Onset_Targets=(
                "New_Onset",
                "sum"
            ),
        )
        .reset_index()
    )


    future_graph_support_tables[
        canonical
    ] = summary


    print("\n")
    print("=" * 150)
    print(
        f"FUTURE {canonical.upper()} FORECASTABILITY"
    )
    print("=" * 150)

    print(
        f"Unique horizon/value combinations : "
        f"{len(summary):,}"
    )

    if len(summary) <= 75:

        print(
            summary.to_string(
                index=False
            )
        )

    else:

        print(
            summary
            .sort_values(
                [
                    "Segments",
                    "Forecast_Targets"
                ],
                ascending=[
                    True,
                    False
                ]
            )
            .head(75)
            .to_string(
                index=False
            )
        )

        print(
            "\n[Only first 75 rows printed; "
            "complete table retained in memory.]"
        )


# ==============================================================================================================
# NEW-ONSET SUPPORT BY SEGMENT AND HORIZON
# ==============================================================================================================

segment_onset_rows = []


for segment_id in sorted(
    targets[
        "Segment_ID"
    ].unique()
):

    row = {
        "Segment_ID":
            int(segment_id)
    }

    seg_targets = targets.loc[
        targets[
            "Segment_ID"
        ] == segment_id
    ]

    for H in FORECAST_HORIZONS:

        row[
            f"H{H}_NewOnset"
        ] = int(
            seg_targets[
                f"New_Onset_H{H}"
            ].sum()
        )

    segment_onset_rows.append(
        row
    )


segment_onset_df = pd.DataFrame(
    segment_onset_rows
)


print("\n")
print("=" * 150)
print("NEW-ONSET FORECASTING SUPPORT BY SEGMENT")
print("=" * 150)

print(
    segment_onset_df
    .to_string(
        index=False
    )
)


# ==============================================================================================================
# TARGET SUPPORT CONCENTRATION
# ==============================================================================================================

concentration_records = []


def add_concentration_rows(
    support_table,
    target_family,
    value_column
):

    if support_table is None:
        return

    if support_table.empty:
        return

    if "Segments" not in support_table.columns:
        return

    for _, row in support_table.iterrows():

        segments = int(
            row[
                "Segments"
            ]
        )

        concentration_records.append({
            "Target_Family":
                target_family,
            "Target_Value":
                str(
                    row[
                        value_column
                    ]
                ),
            "Segments_With_Target":
                segments,
            "Can_Appear_In_All_3_Segment_Disjoint_Splits":
                (
                    segments
                    >= MIN_SEGMENTS_FOR_3WAY_SEGMENT_SPLIT
                ),
        })


if not attack_type_support_df.empty:

    attack_type_value_col = (
        "Attack_Type"
        if "Attack_Type"
        in attack_type_support_df.columns
        else attack_type_support_df.columns[0]
    )

    add_concentration_rows(
        attack_type_support_df,
        "Attack_Type",
        attack_type_value_col
    )


for canonical in [
    "Attack_Sources",
    "Attack_Destinations",
    "Attack_Edges",
    "Typed_Attack_Edges",
]:

    table = metadata_support_tables.get(
        canonical
    )

    if table is None or table.empty:
        continue

    value_candidates = [
        c for c in table.columns
        if c not in {
            "Minute_Occurrences",
            "Segments",
            "Segment_IDs",
            "First_Minute",
            "Last_Minute",
        }
    ]

    if not value_candidates:
        continue

    add_concentration_rows(
        table,
        canonical,
        value_candidates[0]
    )


support_concentration_df = pd.DataFrame(
    concentration_records
)


print("\n")
print("=" * 150)
print("TARGET CONCENTRATION / SEGMENT-DISJOINT SPLIT FEASIBILITY")
print("=" * 150)


if support_concentration_df.empty:

    print(
        "No attack/graph metadata targets available "
        "for concentration analysis."
    )

else:

    concentration_summary_df = (
        support_concentration_df
        .groupby(
            "Target_Family"
        )
        .agg(
            Unique_Targets=(
                "Target_Value",
                "size"
            ),
            Targets_In_1_Segment=(
                "Segments_With_Target",
                lambda x:
                    int(
                        (x == 1).sum()
                    )
            ),
            Targets_In_2_Segments=(
                "Segments_With_Target",
                lambda x:
                    int(
                        (x == 2).sum()
                    )
            ),
            Targets_In_3plus_Segments=(
                "Segments_With_Target",
                lambda x:
                    int(
                        (x >= 3).sum()
                    )
            ),
        )
        .reset_index()
    )

    print(
        concentration_summary_df
        .to_string(
            index=False
        )
    )


    impossible_targets = (
        support_concentration_df
        .loc[
            ~support_concentration_df[
                "Can_Appear_In_All_3_Segment_Disjoint_Splits"
            ]
        ]
        .copy()
    )


    print("\nTargets that CANNOT occur in all three partitions")
    print("under a strictly segment-disjoint 3-way split:")
    print("-" * 150)

    if impossible_targets.empty:

        print(
            "None."
        )

    else:

        display_impossible = (
            impossible_targets
            .sort_values(
                [
                    "Target_Family",
                    "Segments_With_Target",
                    "Target_Value"
                ]
            )
        )

        if len(
            display_impossible
        ) <= 100:

            print(
                display_impossible
                .to_string(
                    index=False
                )
            )

        else:

            print(
                display_impossible
                .head(100)
                .to_string(
                    index=False
                )
            )

            print(
                f"\n[Showing first 100 of "
                f"{len(display_impossible):,} constrained targets.]"
            )


# ==============================================================================================================
# ATTACK-TYPE SEGMENT-EXCLUSIVITY AUDIT
# ==============================================================================================================

attack_type_exclusivity_df = pd.DataFrame()


if not attack_type_support_df.empty:

    value_col = (
        "Attack_Type"
        if "Attack_Type"
        in attack_type_support_df.columns
        else attack_type_support_df.columns[0]
    )

    attack_type_exclusivity_df = (
        attack_type_support_df[
            [
                value_col,
                "Minute_Occurrences",
                "Segments",
                "Segment_IDs",
                "First_Minute",
                "Last_Minute",
            ]
        ]
        .copy()
        .rename(
            columns={
                value_col:
                    "Attack_Type"
            }
        )
    )


    attack_type_exclusivity_df[
        "Segment_Disjoint_3Way_Status"
    ] = np.where(
        attack_type_exclusivity_df[
            "Segments"
        ] >= 3,
        "CAN theoretically appear in TRAIN+VAL+TEST",
        np.where(
            attack_type_exclusivity_df[
                "Segments"
            ] == 2,
            "ONLY 2 segments — impossible in all 3",
            "ONLY 1 segment — impossible in all 3"
        )
    )


    print("\n")
    print("=" * 150)
    print("ATTACK-TYPE SEGMENT EXCLUSIVITY — CRITICAL SPLIT AUDIT")
    print("=" * 150)

    print(
        attack_type_exclusivity_df
        .to_string(
            index=False
        )
    )


# ==============================================================================================================
# OVERLAPPING HISTORY WINDOW AUDIT
# ==============================================================================================================

origin_overlap_records = []


for segment_id, seg in targets.groupby(
    "Segment_ID",
    sort=True
):

    seg = (
        seg
        .sort_values(
            "Origin_Time"
        )
        .reset_index(
            drop=True
        )
    )

    diffs = (
        seg[
            "Origin_Time"
        ]
        .diff()
        .dropna()
        .dt.total_seconds()
        / 60
    )

    origin_overlap_records.append({
        "Segment_ID":
            int(segment_id),
        "Origins":
            len(seg),
        "One_Minute_Adjacent_Pairs":
            int(
                (
                    diffs == 1
                ).sum()
            ),
        "Non_One_Minute_Pairs":
            int(
                (
                    diffs != 1
                ).sum()
            ),
    })


origin_overlap_df = pd.DataFrame(
    origin_overlap_records
)


print("\n")
print("=" * 150)
print("FORECAST-ORIGIN OVERLAP AUDIT")
print("=" * 150)

print(
    origin_overlap_df
    .to_string(
        index=False
    )
)


total_adjacent_pairs = int(
    origin_overlap_df[
        "One_Minute_Adjacent_Pairs"
    ].sum()
)


print(
    f"\n1-minute adjacent origin pairs : "
    f"{total_adjacent_pairs:,}"
)

print(
    f"History overlap for adjacent origins : "
    f"{HISTORY_LENGTH - 1}/{HISTORY_LENGTH} minutes"
)

print(
    "\nIMPORTANT:"
)

print(
    "Random row-level TRAIN/VALIDATION/TEST splitting is forbidden."
)

print(
    "Adjacent origins would otherwise place almost-identical historical "
    "windows into different partitions."
)


# ==============================================================================================================
# REQUIRED PURGE DISTANCE FOR TEMPORAL SPLITS
# ==============================================================================================================
#
# If origins from the SAME continuous segment are separated into different
# partitions, we need to consider BOTH:
#
#   historical overlap
#   future-label reach
#
# The exact final purge design will be made in the split cell.
#
# ==============================================================================================================

MIN_HISTORY_ONLY_SEPARATION = (
    HISTORY_LENGTH
)

FULL_INFORMATION_SPAN = (
    HISTORY_LENGTH
    + MAX_HORIZON
)


print("\n")
print("=" * 150)
print("TEMPORAL SPLIT PURGE REQUIREMENTS")
print("=" * 150)

print(
    f"History length                    : "
    f"{HISTORY_LENGTH} minutes"
)

print(
    f"Maximum forecast horizon          : "
    f"{MAX_HORIZON} minutes"
)

print(
    f"History-only non-overlap distance : "
    f"{MIN_HISTORY_ONLY_SEPARATION} minutes"
)

print(
    f"Full history+future information span considered : "
    f"{FULL_INFORMATION_SPAN} minutes"
)

print(
    "\nNOTE:"
)

print(
    "The actual split cell must explicitly purge/embargo boundaries "
    "rather than randomly assigning origins."
)


# ==============================================================================================================
# SEGMENT-LEVEL FORECAST SUPPORT
# ==============================================================================================================

segment_forecast_records = []


for segment_id, seg in targets.groupby(
    "Segment_ID",
    sort=True
):

    row = {
        "Segment_ID":
            int(segment_id),
        "Forecast_Origins":
            len(seg),
        "Clean_History_Origins":
            int(
                seg[
                    "Clean_History"
                ].sum()
            ),
    }

    for H in FORECAST_HORIZONS:

        row[
            f"H{H}_Attack"
        ] = int(
            seg[
                f"Target_State_H{H}"
            ].sum()
        )

        row[
            f"H{H}_NewOnset"
        ] = int(
            seg[
                f"New_Onset_H{H}"
            ].sum()
        )

    segment_forecast_records.append(
        row
    )


segment_forecast_support_df = pd.DataFrame(
    segment_forecast_records
)


print("\n")
print("=" * 150)
print("SEGMENT-LEVEL FORECAST SUPPORT")
print("=" * 150)

print(
    segment_forecast_support_df
    .to_string(
        index=False
    )
)


# ==============================================================================================================
# SEGMENT SIGNATURE TABLE
# ==============================================================================================================
#
# Compact description of what each segment actually contains.
#
# This is one of the most important outputs for the later split design.
#
# ==============================================================================================================

segment_signature_records = []


for segment_id, seg in minute.groupby(
    "Segment_ID",
    sort=True
):

    record = {
        "Segment_ID":
            int(segment_id),
        "Start":
            seg[
                "Minute"
            ].min(),
        "End":
            seg[
                "Minute"
            ].max(),
        "Minutes":
            len(seg),
        "Attack_Minutes":
            int(
                seg[
                    "Attack_State"
                ].sum()
            ),
    }


    if "Attack_Types" in available_metadata:

        values = set()

        for collection in seg[
            "__Attack_Types"
        ]:

            values.update(
                collection
            )

        record[
            "Attack_Types"
        ] = tuple(
            sorted(
                str(v)
                for v in values
            )
        )


    if "Attack_Sources" in available_metadata:

        values = set()

        for collection in seg[
            "__Attack_Sources"
        ]:

            values.update(
                collection
            )

        record[
            "Unique_Attack_Sources"
        ] = len(
            values
        )


    if "Attack_Destinations" in available_metadata:

        values = set()

        for collection in seg[
            "__Attack_Destinations"
        ]:

            values.update(
                collection
            )

        record[
            "Unique_Attack_Destinations"
        ] = len(
            values
        )


    if "Attack_Edges" in available_metadata:

        values = set()

        for collection in seg[
            "__Attack_Edges"
        ]:

            values.update(
                collection
            )

        record[
            "Unique_Attack_Edges"
        ] = len(
            values
        )


    segment_signature_records.append(
        record
    )


segment_signature_df = pd.DataFrame(
    segment_signature_records
)


print("\n")
print("=" * 150)
print("SEGMENT SIGNATURE TABLE — IMPORTANT FOR SPLIT DESIGN")
print("=" * 150)

print(
    segment_signature_df
    .to_string(
        index=False
    )
)


# ==============================================================================================================
# PURE SEGMENT-DISJOINT SPLIT FEASIBILITY
# ==============================================================================================================
#
# We are NOT choosing a split here.
#
# We are only asking:
#
#   If entire segments are kept intact,
#   is it even mathematically possible to distribute attack types
#   across TRAIN / VALIDATION / TEST?
#
# This prevents us from blindly choosing segment IDs later.
#
# ==============================================================================================================

segment_split_feasibility_records = []


if not attack_type_exclusivity_df.empty:

    for _, row in attack_type_exclusivity_df.iterrows():

        n_segments = int(
            row[
                "Segments"
            ]
        )

        segment_split_feasibility_records.append({
            "Attack_Type":
                row[
                    "Attack_Type"
                ],
            "Segments_With_Type":
                n_segments,
            "Segment_IDs":
                row[
                    "Segment_IDs"
                ],
            "Possible_In_Train_Val_Test_With_Whole_Segments":
                (
                    n_segments >= 3
                ),
        })


segment_split_feasibility_df = pd.DataFrame(
    segment_split_feasibility_records
)


print("\n")
print("=" * 150)
print("PURE SEGMENT-DISJOINT 3-WAY SPLIT FEASIBILITY")
print("=" * 150)


if segment_split_feasibility_df.empty:

    print(
        "Attack-type metadata unavailable; "
        "attack-type feasibility cannot be evaluated."
    )

else:

    print(
        segment_split_feasibility_df
        .to_string(
            index=False
        )
    )


    impossible_attack_types = (
        segment_split_feasibility_df
        .loc[
            ~segment_split_feasibility_df[
                "Possible_In_Train_Val_Test_With_Whole_Segments"
            ]
        ]
    )


    if impossible_attack_types.empty:

        print(
            "\n✓ Every attack type occurs in at least 3 segments."
        )

        print(
            "A segment-disjoint 3-way split is not ruled out "
            "by attack-type segment count alone."
        )

    else:

        print(
            "\nCRITICAL FINDING:"
        )

        print(
            f"{len(impossible_attack_types):,} attack type(s) "
            "occur in fewer than 3 segments."
        )

        print(
            "Therefore it is MATHEMATICALLY IMPOSSIBLE for those types "
            "to appear in TRAIN, VALIDATION, and TEST simultaneously "
            "if entire segments are assigned intact."
        )

        print(
            "\nThis does NOT mean we should randomly split rows."
        )

        print(
            "It means the next split-design cell must evaluate a "
            "leakage-safe temporal-within-segment strategy with purging/"
            "embargo, or explicitly define a different generalization task."
        )


# ==============================================================================================================
# CAMPAIGN FRAGMENTATION RISK
# ==============================================================================================================

campaign_fragmentation_summary = (
    campaign_df
    .groupby(
        "Segment_ID"
    )
    .agg(
        Campaign_Groups=(
            "Campaign_ID",
            "nunique"
        ),
        Strict_Episodes=(
            "Strict_Episode_Count",
            "sum"
        ),
        Attack_Minutes=(
            "Attack_Minutes",
            "sum"
        ),
    )
    .reset_index()
)


all_segments_df = pd.DataFrame({
    "Segment_ID":
        sorted(
            minute[
                "Segment_ID"
            ].unique()
        )
})


campaign_fragmentation_summary = (
    all_segments_df
    .merge(
        campaign_fragmentation_summary,
        on="Segment_ID",
        how="left"
    )
)


for c in [
    "Campaign_Groups",
    "Strict_Episodes",
    "Attack_Minutes",
]:

    campaign_fragmentation_summary[
        c
    ] = (
        campaign_fragmentation_summary[
            c
        ]
        .fillna(0)
        .astype(int)
    )


print("\n")
print("=" * 150)
print("CAMPAIGN SUPPORT BY SEGMENT")
print("=" * 150)

print(
    campaign_fragmentation_summary
    .to_string(
        index=False
    )
)


# ==============================================================================================================
# FINAL SPLIT-SAFETY CONTRACT
# ==============================================================================================================

SPLIT_SAFETY_RULES = {

    "random_row_split_forbidden":
        True,

    "random_origin_split_forbidden":
        True,

    "overlapping_history_cross_split_forbidden":
        True,

    "future_target_overlap_cross_split_must_be_prevented":
        True,

    "segment_boundary_crossing_forbidden":
        True,

    "campaign_fragmentation_must_be_audited":
        True,

    "attack_type_support_must_be_checked":
        True,

    "new_onset_support_must_be_checked":
        True,

    "victim_destination_support_must_be_checked":
        (
            "Attack_Destinations"
            in available_metadata
        ),

    "attacker_source_support_must_be_checked":
        (
            "Attack_Sources"
            in available_metadata
        ),

    "edge_support_must_be_checked":
        (
            "Attack_Edges"
            in available_metadata
        ),

    "typed_edge_support_must_be_checked":
        (
            "Typed_Attack_Edges"
            in available_metadata
        ),

    "fit_scaler_before_split_forbidden":
        True,

    "fit_scaler_on_validation_forbidden":
        True,

    "fit_scaler_on_test_forbidden":
        True,

    "resample_before_split_forbidden":
        True,

    "test_used_for_model_selection_forbidden":
        True,

    "split_must_be_locked_before_model_training":
        True,
}


print("\n")
print("=" * 150)
print("SPLIT SAFETY CONTRACT")
print("=" * 150)


for rule, value in SPLIT_SAFETY_RULES.items():

    print(
        f"{rule:<70} : {value}"
    )


# ==============================================================================================================
# SPLIT DESIGN WARNINGS
# ==============================================================================================================

split_warnings = []


if total_adjacent_pairs > 0:

    split_warnings.append(
        "Forecast origins are heavily overlapping; "
        "random row/origin splitting would leak history."
    )


if not segment_split_feasibility_df.empty:

    impossible_count = int(
        (
            ~segment_split_feasibility_df[
                "Possible_In_Train_Val_Test_With_Whole_Segments"
            ]
        ).sum()
    )

    if impossible_count > 0:

        split_warnings.append(
            f"{impossible_count} attack type(s) occur in fewer than "
            "3 segments, so a whole-segment 3-way split cannot place "
            "every such type in TRAIN/VAL/TEST."
        )


if not support_concentration_df.empty:

    graph_constrained = support_concentration_df.loc[
        (
            support_concentration_df[
                "Target_Family"
            ] != "Attack_Type"
        )
        &
        (
            support_concentration_df[
                "Segments_With_Target"
            ] < 3
        )
    ]

    if len(
        graph_constrained
    ) > 0:

        split_warnings.append(
            f"{len(graph_constrained):,} graph/entity target value(s) "
            "occur in fewer than 3 segments; exact graph-label overlap "
            "across all three whole-segment partitions is impossible "
            "for those values."
        )


print("\n")
print("=" * 150)
print("PRE-SPLIT WARNINGS")
print("=" * 150)


if not split_warnings:

    print(
        "No structural split warning was triggered by the current audits."
    )

else:

    for i, warning in enumerate(
        split_warnings,
        start=1
    ):

        print(
            f"{i}. {warning}"
        )


# ==============================================================================================================
# EXPORT OBJECTS
# ==============================================================================================================

cell11_strict_episodes = (
    strict_episode_df.copy()
)

cell11_episode_gaps = (
    episode_gap_df.copy()
)

cell11_campaigns = (
    campaign_df.copy()
)

cell11_campaign_sensitivity = (
    campaign_threshold_summary_df.copy()
)

cell11_segment_attack_summary = (
    segment_attack_summary.copy()
)

cell11_segment_onset_support = (
    segment_onset_df.copy()
)

cell11_origin_overlap = (
    origin_overlap_df.copy()
)

cell11_forecast_long = (
    forecast_long.copy()
)

cell11_horizon_episode_support = (
    horizon_episode_support_df.copy()
)

cell11_attack_type_support = (
    attack_type_support_df.copy()
)

cell11_attack_type_segment_support = (
    attack_type_segment_df.copy()
)

cell11_attack_type_exclusivity = (
    attack_type_exclusivity_df.copy()
)

cell11_metadata_support_tables = {
    key: value.copy()
    for key, value
    in metadata_support_tables.items()
}

cell11_future_graph_support_tables = {
    key: value.copy()
    for key, value
    in future_graph_support_tables.items()
}

cell11_future_attack_type_support = (
    future_attack_type_support_df.copy()
)

cell11_support_concentration = (
    support_concentration_df.copy()
)

cell11_segment_signature = (
    segment_signature_df.copy()
)

cell11_segment_split_feasibility = (
    segment_split_feasibility_df.copy()
)

cell11_campaign_segment_summary = (
    campaign_fragmentation_summary.copy()
)

cell11_metadata_consistency = (
    metadata_consistency_df.copy()
)

cell11_split_safety_rules = (
    SPLIT_SAFETY_RULES.copy()
)

cell11_split_warnings = list(
    split_warnings
)


cell11_metadata = {

    "history_length":
        HISTORY_LENGTH,

    "forecast_horizons":
        list(
            FORECAST_HORIZONS
        ),

    "max_horizon":
        MAX_HORIZON,

    "primary_campaign_gap":
        PRIMARY_CAMPAIGN_GAP,

    "minute_rows":
        len(minute),

    "forecast_origins":
        len(targets),

    "forecast_rows":
        len(forecast_long),

    "strict_episodes":
        len(strict_episode_df),

    "primary_campaign_groups":
        len(campaign_df),

    "attack_type_column":
        attack_type_col,

    "source_column":
        source_col,

    "destination_column":
        destination_col,

    "edge_column":
        edge_col,

    "typed_edge_column":
        typed_edge_col,

    "split_created":
        False,

    "scaling_performed":
        False,

    "resampling_performed":
        False,

    "model_trained":
        False,
}


# ==============================================================================================================
# FINAL VALIDATION
# ==============================================================================================================

if strict_episode_df[
    "Episode_ID"
].duplicated().any():

    raise RuntimeError(
        "Duplicate Episode_ID values detected."
    )


if campaign_df[
    "Campaign_ID"
].duplicated().any():

    raise RuntimeError(
        "Duplicate Campaign_ID values detected."
    )


if int(
    strict_episode_df[
        "Duration_Minutes"
    ].sum()
) != int(
    minute[
        "Attack_State"
    ].sum()
):

    raise RuntimeError(
        "Final attack-minute accounting failed."
    )


expected_forecast_rows = (
    len(targets)
    * len(
        FORECAST_HORIZONS
    )
)


if len(
    forecast_long
) != expected_forecast_rows:

    raise RuntimeError(
        "Final multi-horizon forecast-row count mismatch.\n"
        f"Expected : {expected_forecast_rows:,}\n"
        f"Actual   : {len(forecast_long):,}"
    )


if forecast_long[
    [
        "Segment_ID",
        "Origin_Time",
        "Horizon_Minutes"
    ]
].duplicated().any():

    raise RuntimeError(
        "Duplicate "
        "(Segment_ID, Origin_Time, Horizon_Minutes) "
        "forecast keys detected."
    )


attack_forecast_rows = forecast_long.loc[
    forecast_long[
        "Future_Attack_State"
    ] == 1
]


if attack_forecast_rows[
    "Episode_ID"
].isna().any():

    raise RuntimeError(
        "Some attack-positive forecast targets lack Episode_ID."
    )


if attack_forecast_rows[
    "Campaign_ID"
].isna().any():

    raise RuntimeError(
        "Some attack-positive forecast targets lack Campaign_ID."
    )


benign_forecast_rows = forecast_long.loc[
    forecast_long[
        "Future_Attack_State"
    ] == 0
]


if benign_forecast_rows[
    "Episode_ID"
].notna().any():

    raise RuntimeError(
        "Some benign forecast targets incorrectly received Episode_ID."
    )


# ==============================================================================================================
# FINAL SUMMARY
# ==============================================================================================================

print("\n")
print("=" * 150)
print("CELL 11 — FINAL COMPREHENSIVE AUDIT SUMMARY")
print("=" * 150)

print(
    f"Canonical minute rows               : "
    f"{len(minute):,}"
)

print(
    f"Forecast origins                    : "
    f"{len(targets):,}"
)

print(
    f"Multi-horizon forecast rows         : "
    f"{len(forecast_long):,}"
)

print(
    f"Forecast horizons                   : "
    f"{FORECAST_HORIZONS}"
)

print(
    f"Attack minutes                      : "
    f"{len(attack_minutes):,}"
)

print(
    f"Strict contiguous attack episodes   : "
    f"{len(strict_episode_df):,}"
)

print(
    f"Primary campaign-like groups        : "
    f"{len(campaign_df):,} "
    f"(gap <= {PRIMARY_CAMPAIGN_GAP} min)"
)

print(
    f"Attack-type metadata available      : "
    f"{'YES' if attack_type_col is not None else 'NO'}"
)

print(
    f"Source/attacker metadata available  : "
    f"{'YES' if source_col is not None else 'NO'}"
)

print(
    f"Destination/victim metadata avail.  : "
    f"{'YES' if destination_col is not None else 'NO'}"
)

print(
    f"Attack-edge metadata available      : "
    f"{'YES' if edge_col is not None else 'NO'}"
)

print(
    f"Typed-edge metadata available       : "
    f"{'YES' if typed_edge_col is not None else 'NO'}"
)


print("\nVALIDATIONS")

print(
    "✓ Cell 10 target times match exact canonical future minutes."
)

print(
    "✓ Cell 10 binary future states match canonical minute ground truth."
)

print(
    "✓ Every attack minute belongs to exactly one strict episode."
)

print(
    "✓ Every strict episode belongs to exactly one primary campaign-like group."
)

print(
    "✓ Every attack-positive future target maps to a real attack episode."
)

print(
    "✓ Future attack/graph metadata aligned using exact target timestamps."
)

print(
    "✓ Attack-type concentration across segments audited."
)

print(
    "✓ Source/destination/edge concentration audited when metadata exists."
)

print(
    "✓ New-onset independence audited across episodes/campaigns."
)

print(
    "✓ Forecast-history overlap audited."
)

print(
    "✓ Whole-segment split feasibility audited."
)

print(
    "✓ No TRAIN / VALIDATION / TEST split created."
)

print(
    "✓ No scaler fitted."
)

print(
    "✓ No resampling performed."
)

print(
    "✓ No model trained."
)


print("\nIMPORTANT INTERPRETATION")

print(
    "The campaign grouping is an AUDIT heuristic, not a claimed "
    "CICIDS2017 ground-truth campaign label."
)

print(
    "A target occurring many times does NOT imply many independent attacks; "
    "episode/campaign counts must be considered."
)

print(
    "A target occurring in fewer than 3 segments cannot appear in all "
    "TRAIN/VALIDATION/TEST partitions under a strictly whole-segment "
    "3-way split."
)

print(
    "That situation must NOT be 'fixed' with random row splitting because "
    "the 15-minute historical windows overlap heavily."
)

print(
    "The next cell must DESIGN candidate leakage-safe split strategies "
    "using the support information produced here before any split is locked."
)


print("\nObjects created:")

created_objects = [
    "cell11_strict_episodes",
    "cell11_episode_gaps",
    "cell11_campaigns",
    "cell11_campaign_sensitivity",
    "cell11_segment_attack_summary",
    "cell11_segment_onset_support",
    "cell11_origin_overlap",
    "cell11_forecast_long",
    "cell11_horizon_episode_support",
    "cell11_attack_type_support",
    "cell11_attack_type_segment_support",
    "cell11_attack_type_exclusivity",
    "cell11_metadata_support_tables",
    "cell11_future_graph_support_tables",
    "cell11_future_attack_type_support",
    "cell11_support_concentration",
    "cell11_segment_signature",
    "cell11_segment_split_feasibility",
    "cell11_campaign_segment_summary",
    "cell11_metadata_consistency",
    "cell11_split_safety_rules",
    "cell11_split_warnings",
    "cell11_metadata",
]


for obj in created_objects:
    print(
        f"  {obj}"
    )


print("\nNEXT:")

print(
    "Cell 12 should DESIGN and compare candidate leakage-safe "
    "TRAIN / VALIDATION / TEST strategies."
)

print(
    "Do NOT create the final split blindly."
)

print(
    "We will use Cell 11's attack-type, new-onset, campaign, victim, "
    "source, edge, and segment-support results to choose the split."
)

print("=" * 150)

CELL 11 — COMPREHENSIVE ATTACK / GRAPH / CAMPAIGN / SPLIT-FEASIBILITY AUDIT

Configuration
------------------------------------------------------------------------------------------------------------------------------------------------------
Campaign-gap candidates                 : [0, 1, 2, 5, 10, 15, 30] minutes
Primary descriptive campaign gap        : 5 minutes
Split created                            : NO
Scaling performed                        : NO
Resampling performed                     : NO
Model trained                            : NO

✓ Required Cells 9–10 objects found.
minute_df rows       : 2,454
forecast origins     : 2,014
history length       : 15
forecast horizons    : [1, 5, 10, 15, 30]


DISCOVERED CANONICAL MINUTE-LEVEL SCHEMA
Segment column              : Segment_ID
Minute column               : Minute
Attack-state column         : Attack_State
Attack-type metadata        : Attack_Types
Source/attacker metadata    : Attack_Source_IPs
Destination/victim metadata 

In [20]:
# ==================================================================================================
# CELL 12 — CANONICAL RICH-TARGET VALIDATION + EPISODE/CAMPAIGN ALIGNMENT
# ==================================================================================================
#
# PURPOSE
# -------
# Validate and finalize the rich multi-horizon forecasting metadata produced by Cell 10.
#
# IMPORTANT DESIGN DECISION
# -------------------------
# Cell 10 is the canonical source of the forecasting contract.
#
# cell10_target_table:
#     wide representation — one row per forecasting origin
#
# cell10_forecast_metadata_long:
#     long representation — one row per
#     (Segment_ID, Origin_Time, Horizon_Minutes)
#
# Cell 12 DOES NOT reconstruct future labels independently.
# It validates Cell 10 against canonical minute-level ground truth from Cell 9
# and attaches Cell 11 episode/campaign metadata when available.
#
# NO SPLIT
# NO SCALING
# NO RESAMPLING
# NO MODEL TRAINING
# ==================================================================================================

import ast
import json
import numpy as np
import pandas as pd


# ==================================================================================================
# CONFIGURATION
# ==================================================================================================

CELL12_NAME = (
    "CELL 12 — CANONICAL RICH-TARGET VALIDATION + "
    "EPISODE/CAMPAIGN ALIGNMENT"
)

EXPECTED_HORIZONS = [1, 5, 10, 15, 30]
EXPECTED_HISTORY_LENGTH = 15

SEP = "=" * 162
SUBSEP = "-" * 162


def section(title):
    print("\n" + SEP)
    print(title)
    print(SEP)


def subsection(title):
    print("\n" + SUBSEP)
    print(title)
    print(SUBSEP)


section(CELL12_NAME)

print("\nConfiguration")
print(SUBSEP)

print("Canonical minute source          : Cell 9 minute_df")
print("Forecasting contract source      : Cell 10")
print("Forecast representation used     : cell10_forecast_metadata_long")
print("Wide validation source           : cell10_target_table")
print("Episode/campaign source          : Cell 11 if available")
print()
print("Temporal resolution              : 1 minute")
print(f"History length                   : {EXPECTED_HISTORY_LENGTH} minutes")
print(f"Forecast horizons                : {EXPECTED_HORIZONS}")
print("Forecast semantics               : POINT target at exactly t + H")
print()
print("Split created                    : NO")
print("Scaling performed                : NO")
print("Resampling performed             : NO")
print("Model trained                    : NO")

print(
    "\nIMPORTANT:\n"
    "Destination IP is retained as an observed attack destination / "
    "victim candidate.\n"
    "It is NOT blindly asserted to be a confirmed semantic victim.\n\n"
    "Source IP is retained as an observed attack source endpoint.\n"
    "It is NOT blindly asserted to identify a real-world attacker.\n\n"
    "Future graph targets are inherited from the canonical Cell 10 "
    "point-horizon contract and validated here against Cell 9."
)


# ==================================================================================================
# 1. REQUIRED OBJECT VALIDATION
# ==================================================================================================

section("REQUIRED OBJECT VALIDATION")

required_objects = [
    "minute_df",
    "cell10_target_table",
    "cell10_valid_origins",
    "cell10_forecast_metadata_long",
    "cell10_metadata",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Missing required objects:\n  - "
        + "\n  - ".join(missing_objects)
    )

if not isinstance(minute_df, pd.DataFrame):
    raise TypeError("minute_df must be a pandas DataFrame.")

if not isinstance(cell10_target_table, pd.DataFrame):
    raise TypeError("cell10_target_table must be a pandas DataFrame.")

if not isinstance(cell10_valid_origins, pd.DataFrame):
    raise TypeError("cell10_valid_origins must be a pandas DataFrame.")

if not isinstance(cell10_forecast_metadata_long, pd.DataFrame):
    raise TypeError(
        "cell10_forecast_metadata_long must be a pandas DataFrame."
    )

if not isinstance(cell10_metadata, dict):
    raise TypeError("cell10_metadata must be a dictionary.")

print("✓ Required Cell 9 / Cell 10 objects found.")
print()
print(f"minute_df rows                     : {len(minute_df):,}")
print(f"cell10_valid_origins rows          : {len(cell10_valid_origins):,}")
print(f"cell10_target_table rows           : {len(cell10_target_table):,}")
print(
    f"cell10_forecast_metadata_long rows : "
    f"{len(cell10_forecast_metadata_long):,}"
)


# ==================================================================================================
# 2. SAFE COLLECTION DESERIALIZATION
# ==================================================================================================

section("COLLECTION DESERIALIZATION HELPERS")


def parse_collection(value):
    """
    Convert serialized list-like metadata into Python collections.

    Handles:
      - already-materialized list / tuple / set
      - JSON strings
      - Python-literal strings
      - empty strings
      - NaN / None

    Never interprets '[]' as non-empty metadata.
    """

    if value is None:
        return []

    if isinstance(value, float) and np.isnan(value):
        return []

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, set):
        return list(value)

    if isinstance(value, tuple):
        return list(value)

    if isinstance(value, list):
        return value

    if isinstance(value, str):
        s = value.strip()

        if s == "" or s.lower() in {
            "nan",
            "none",
            "null",
            "<na>",
        }:
            return []

        try:
            parsed = json.loads(s)

            if parsed is None:
                return []

            if isinstance(parsed, list):
                return parsed

            if isinstance(parsed, tuple):
                return list(parsed)

            return [parsed]

        except Exception:
            pass

        try:
            parsed = ast.literal_eval(s)

            if parsed is None:
                return []

            if isinstance(parsed, list):
                return parsed

            if isinstance(parsed, tuple):
                return list(parsed)

            if isinstance(parsed, set):
                return list(parsed)

            return [parsed]

        except Exception as exc:
            raise RuntimeError(
                "Could not deserialize collection value:\n"
                f"{repr(value)}"
            ) from exc

    raise RuntimeError(
        "Unsupported rich-metadata value type:\n"
        f"type={type(value).__name__}, value={repr(value)}"
    )


def canonical_collection(value):
    """
    Convert a collection into a stable recursively-normalized form
    suitable for equality checks.
    """

    value = parse_collection(value)

    normalized = []

    for item in value:

        if isinstance(item, np.ndarray):
            item = item.tolist()

        if isinstance(item, list):
            normalized.append(
                tuple(
                    tuple(x) if isinstance(x, list) else x
                    for x in item
                )
            )

        elif isinstance(item, tuple):
            normalized.append(tuple(item))

        else:
            normalized.append(item)

    return tuple(normalized)


print("✓ Collection parser ready.")
print("✓ String '[]' is treated as an empty collection.")


# ==================================================================================================
# 3. CANONICAL CELL 9 SCHEMA VALIDATION
# ==================================================================================================

section("CANONICAL CELL 9 MINUTE-LEVEL SCHEMA")

minute_required = [
    "Segment_ID",
    "Minute",
    "Attack_State",
    "Attack_Types",
    "Attack_Source_IPs",
    "Attack_Destination_IPs",
    "Attack_Edges",
    "Typed_Attack_Edges",
]

minute_missing = [
    col for col in minute_required
    if col not in minute_df.columns
]

if minute_missing:
    raise RuntimeError(
        "minute_df is missing required canonical columns:\n  - "
        + "\n  - ".join(minute_missing)
    )

minute_meta = minute_df.copy()

minute_meta["Minute"] = pd.to_datetime(
    minute_meta["Minute"],
    errors="raise",
)

minute_meta["Segment_ID"] = pd.to_numeric(
    minute_meta["Segment_ID"],
    errors="raise",
).astype(int)

minute_meta["Attack_State"] = pd.to_numeric(
    minute_meta["Attack_State"],
    errors="raise",
).astype(int)

if not minute_meta["Attack_State"].isin([0, 1]).all():
    raise RuntimeError(
        "minute_df Attack_State contains values outside {0,1}."
    )

duplicate_minute_keys = int(
    minute_meta.duplicated(
        ["Segment_ID", "Minute"]
    ).sum()
)

if duplicate_minute_keys != 0:
    raise RuntimeError(
        f"minute_df contains {duplicate_minute_keys:,} duplicate "
        "(Segment_ID, Minute) keys."
    )

collection_columns = [
    "Attack_Types",
    "Attack_Source_IPs",
    "Attack_Destination_IPs",
    "Attack_Edges",
    "Typed_Attack_Edges",
]

for col in collection_columns:
    minute_meta[col] = minute_meta[col].map(parse_collection)

print(f"Rows                     : {len(minute_meta):,}")
print(
    f"Segments                 : "
    f"{minute_meta['Segment_ID'].nunique():,}"
)
print(
    f"Duplicate minute keys    : "
    f"{duplicate_minute_keys:,}"
)
print("✓ Canonical Cell 9 schema validated.")
print("✓ Rich metadata deserialized.")


# ==================================================================================================
# 4. CELL 9 GROUND-TRUTH CONSISTENCY
# ==================================================================================================

section("CANONICAL CELL 9 GROUND-TRUTH CONSISTENCY")

attack_mask = minute_meta["Attack_State"].eq(1)
benign_mask = ~attack_mask

attack_minutes = int(attack_mask.sum())
benign_minutes = int(benign_mask.sum())

print(f"Attack minutes : {attack_minutes:,}")
print(f"Benign minutes : {benign_minutes:,}")

benign_metadata_violations = {}

for col in collection_columns:
    benign_metadata_violations[col] = int(
        minute_meta.loc[
            benign_mask,
            col,
        ].map(len).gt(0).sum()
    )

attack_metadata_missing = {}

for col in collection_columns:
    attack_metadata_missing[col] = int(
        minute_meta.loc[
            attack_mask,
            col,
        ].map(len).eq(0).sum()
    )

print("\nBenign minutes carrying attack-only metadata:")

for col, count in benign_metadata_violations.items():
    print(f"  {col:<30}: {count:,}")

print("\nAttack minutes missing rich metadata:")

for col, count in attack_metadata_missing.items():
    print(f"  {col:<30}: {count:,}")

if any(
    count != 0
    for count in benign_metadata_violations.values()
):
    raise RuntimeError(
        "At least one benign minute contains non-empty "
        "attack-only metadata."
    )

if any(
    count != 0
    for count in attack_metadata_missing.values()
):
    raise RuntimeError(
        "At least one attack minute is missing required "
        "rich attack metadata."
    )

print()
print("✓ Benign minutes contain empty attack metadata.")
print("✓ Attack minutes contain rich attack metadata.")


# ==================================================================================================
# 5. GRAPH-METADATA INTERNAL CONSISTENCY
# ==================================================================================================

section("CELL 9 GRAPH-METADATA CROSS-CONSISTENCY")

graph_consistency_violations = 0
graph_examples = []

for row in minute_meta.loc[attack_mask].itertuples(index=False):

    attack_types = set(row.Attack_Types)
    sources = set(row.Attack_Source_IPs)
    destinations = set(row.Attack_Destination_IPs)

    edges = {
        tuple(edge)
        for edge in row.Attack_Edges
    }

    typed_edges = {
        tuple(edge)
        for edge in row.Typed_Attack_Edges
    }

    row_bad = False

    for edge in edges:

        if len(edge) != 2:
            row_bad = True
            break

        src, dst = edge

        if src not in sources or dst not in destinations:
            row_bad = True
            break

    if not row_bad:

        for edge in typed_edges:

            if len(edge) != 3:
                row_bad = True
                break

            src, dst, attack_type = edge

            if (
                src not in sources
                or dst not in destinations
                or attack_type not in attack_types
                or (src, dst) not in edges
            ):
                row_bad = True
                break

    if row_bad:
        graph_consistency_violations += 1

        if len(graph_examples) < 10:
            graph_examples.append(
                (
                    row.Segment_ID,
                    row.Minute,
                )
            )

print(
    "Graph metadata consistency violations : "
    f"{graph_consistency_violations:,}"
)

if graph_consistency_violations:
    raise RuntimeError(
        "Cell 9 graph metadata contains internal consistency "
        "violations.\n"
        f"Examples: {graph_examples}"
    )

print(
    "✓ Attack type/source/destination/edge metadata "
    "is internally consistent."
)


# ==================================================================================================
# 6. CELL 10 LONG-FORM FORECAST CONTRACT
# ==================================================================================================

section("CELL 10 LONG-FORM FORECAST CONTRACT")

forecast_required = [
    "Segment_ID",
    "History_Start",
    "History_End",
    "Origin_Time",
    "Horizon_Minutes",
    "Target_Time",
    "Current_State",
    "Attack_In_History",
    "Clean_History",
    "Attack_Minutes_In_History",
    "Future_Attack_State",
    "Future_Attack_Types",
    "Future_Attack_Sources",
    "Future_Attack_Destinations",
    "Future_Attack_Edges",
    "Future_Typed_Attack_Edges",
    "Future_Attack_Type_Count",
    "Future_Attack_Source_Count",
    "Future_Attack_Destination_Count",
    "Future_Attack_Edge_Count",
    "Future_Typed_Attack_Edge_Count",
    "New_Onset",
    "Continuation",
]

forecast_missing = [
    col for col in forecast_required
    if col not in cell10_forecast_metadata_long.columns
]

if forecast_missing:
    raise RuntimeError(
        "cell10_forecast_metadata_long is missing required fields:\n  - "
        + "\n  - ".join(forecast_missing)
    )

forecast_meta = cell10_forecast_metadata_long.copy()

datetime_columns = [
    "History_Start",
    "History_End",
    "Origin_Time",
    "Target_Time",
]

for col in datetime_columns:
    forecast_meta[col] = pd.to_datetime(
        forecast_meta[col],
        errors="raise",
    )

integer_columns = [
    "Segment_ID",
    "Horizon_Minutes",
    "Current_State",
    "Attack_In_History",
    "Clean_History",
    "Attack_Minutes_In_History",
    "Future_Attack_State",
    "Future_Attack_Type_Count",
    "Future_Attack_Source_Count",
    "Future_Attack_Destination_Count",
    "Future_Attack_Edge_Count",
    "Future_Typed_Attack_Edge_Count",
    "New_Onset",
    "Continuation",
]

for col in integer_columns:

    numeric = pd.to_numeric(
        forecast_meta[col],
        errors="coerce",
    )

    if numeric.isna().any():
        bad = forecast_meta.loc[
            numeric.isna(),
            col,
        ].head(10).tolist()

        raise RuntimeError(
            f"{col} contains non-numeric or missing values.\n"
            f"Examples: {bad}"
        )

    forecast_meta[col] = numeric.astype(int)

future_collection_columns = [
    "Future_Attack_Types",
    "Future_Attack_Sources",
    "Future_Attack_Destinations",
    "Future_Attack_Edges",
    "Future_Typed_Attack_Edges",
]

for col in future_collection_columns:
    forecast_meta[col] = forecast_meta[col].map(
        parse_collection
    )

actual_horizons = sorted(
    forecast_meta["Horizon_Minutes"]
    .unique()
    .tolist()
)

if actual_horizons != EXPECTED_HORIZONS:
    raise RuntimeError(
        "Unexpected forecast horizons.\n"
        f"Expected: {EXPECTED_HORIZONS}\n"
        f"Actual  : {actual_horizons}"
    )

print(
    f"Rows                       : "
    f"{len(forecast_meta):,}"
)
print(
    f"Columns                    : "
    f"{forecast_meta.shape[1]:,}"
)
print(
    f"Forecast origins           : "
    f"{forecast_meta['Origin_Time'].nunique():,}"
)
print(
    f"Forecast horizons          : "
    f"{actual_horizons}"
)

print("✓ Exact Cell 10 long-form schema validated.")


# ==================================================================================================
# 7. FORECAST CONTRACT STRUCTURAL AUDIT
# ==================================================================================================

section("FORECAST CONTRACT STRUCTURAL AUDIT")

origin_key = [
    "Segment_ID",
    "Origin_Time",
]

forecast_key = [
    "Segment_ID",
    "Origin_Time",
    "Horizon_Minutes",
]

duplicate_forecast_keys = int(
    forecast_meta.duplicated(
        forecast_key
    ).sum()
)

origin_count = (
    forecast_meta[
        origin_key
    ]
    .drop_duplicates()
    .shape[0]
)

expected_rows = (
    origin_count
    * len(EXPECTED_HORIZONS)
)

wrong_horizon_counts = (
    forecast_meta
    .groupby(origin_key)["Horizon_Minutes"]
    .nunique()
)

origins_missing_horizons = int(
    (
        wrong_horizon_counts
        != len(EXPECTED_HORIZONS)
    ).sum()
)

target_time_expected = (
    forecast_meta["Origin_Time"]
    + pd.to_timedelta(
        forecast_meta["Horizon_Minutes"],
        unit="m",
    )
)

target_time_errors = int(
    (
        forecast_meta["Target_Time"]
        != target_time_expected
    ).sum()
)

history_duration = (
    forecast_meta["History_End"]
    - forecast_meta["History_Start"]
)

expected_history_duration = pd.Timedelta(
    minutes=EXPECTED_HISTORY_LENGTH - 1
)

history_duration_errors = int(
    (
        history_duration
        != expected_history_duration
    ).sum()
)

history_end_origin_errors = int(
    (
        forecast_meta["History_End"]
        != forecast_meta["Origin_Time"]
    ).sum()
)

print(f"Forecast origins              : {origin_count:,}")
print(f"Expected forecast rows        : {expected_rows:,}")
print(
    f"Actual forecast rows          : "
    f"{len(forecast_meta):,}"
)
print(
    f"Duplicate forecast keys       : "
    f"{duplicate_forecast_keys:,}"
)
print(
    f"Origins missing horizons      : "
    f"{origins_missing_horizons:,}"
)
print(
    f"Target-time alignment errors  : "
    f"{target_time_errors:,}"
)
print(
    f"History-duration errors       : "
    f"{history_duration_errors:,}"
)
print(
    f"History-end/origin errors     : "
    f"{history_end_origin_errors:,}"
)

if len(forecast_meta) != expected_rows:
    raise RuntimeError(
        "Forecast row count does not equal "
        "origins × horizons."
    )

if duplicate_forecast_keys:
    raise RuntimeError(
        "Duplicate long-form forecast keys detected."
    )

if origins_missing_horizons:
    raise RuntimeError(
        "At least one forecasting origin does not contain "
        "all required horizons."
    )

if target_time_errors:
    raise RuntimeError(
        "Target_Time != Origin_Time + Horizon_Minutes "
        "for at least one row."
    )

if history_duration_errors:
    raise RuntimeError(
        "At least one history window does not contain the "
        "expected 15-minute inclusive history."
    )

if history_end_origin_errors:
    raise RuntimeError(
        "History_End does not equal Origin_Time "
        "for at least one forecast row."
    )

print()
print("✓ Forecast structural contract validated.")


# ==================================================================================================
# 8. PER-HORIZON ROW COUNT
# ==================================================================================================

section("PER-HORIZON FORECAST SUPPORT")

horizon_counts = (
    forecast_meta
    .groupby("Horizon_Minutes")
    .size()
    .rename("Rows")
    .reset_index()
)

horizon_attack_counts = (
    forecast_meta
    .groupby("Horizon_Minutes")["Future_Attack_State"]
    .agg(
        Attack_Targets="sum",
        Samples="size",
    )
    .reset_index()
)

horizon_attack_counts["Benign_Targets"] = (
    horizon_attack_counts["Samples"]
    - horizon_attack_counts["Attack_Targets"]
)

horizon_attack_counts["Attack_Percent"] = (
    100
    * horizon_attack_counts["Attack_Targets"]
    / horizon_attack_counts["Samples"]
)

horizon_attack_counts["New_Onset_Targets"] = (
    forecast_meta
    .groupby("Horizon_Minutes")["New_Onset"]
    .sum()
    .values
)

horizon_attack_counts["Continuation_Targets"] = (
    forecast_meta
    .groupby("Horizon_Minutes")["Continuation"]
    .sum()
    .values
)

print(
    horizon_attack_counts.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)


# ==================================================================================================
# 9. TARGET COLLECTION COUNT CONSISTENCY
# ==================================================================================================

section("RICH TARGET COUNT CONSISTENCY")

count_contracts = {
    "Future_Attack_Types":
        "Future_Attack_Type_Count",

    "Future_Attack_Sources":
        "Future_Attack_Source_Count",

    "Future_Attack_Destinations":
        "Future_Attack_Destination_Count",

    "Future_Attack_Edges":
        "Future_Attack_Edge_Count",

    "Future_Typed_Attack_Edges":
        "Future_Typed_Attack_Edge_Count",
}

count_mismatches = {}

for collection_col, count_col in count_contracts.items():

    actual_count = forecast_meta[
        collection_col
    ].map(len)

    expected_count = forecast_meta[
        count_col
    ]

    mismatch = actual_count.ne(expected_count)

    count_mismatches[count_col] = int(
        mismatch.sum()
    )

for col, count in count_mismatches.items():
    print(f"{col:<40}: {count:,}")

if any(
    value != 0
    for value in count_mismatches.values()
):
    raise RuntimeError(
        "At least one rich-target count does not match "
        "the corresponding collection length."
    )

print()
print(
    "✓ Every rich-target count matches its "
    "corresponding collection."
)


# ==================================================================================================
# 10. BENIGN / ATTACK TARGET SEMANTICS
# ==================================================================================================

section("RICH TARGET STATE CONSISTENCY")

future_attack_mask = (
    forecast_meta["Future_Attack_State"].eq(1)
)

future_benign_mask = ~future_attack_mask

benign_future_metadata_violations = {}

for col in future_collection_columns:

    benign_future_metadata_violations[col] = int(
        forecast_meta.loc[
            future_benign_mask,
            col,
        ].map(len).gt(0).sum()
    )

attack_future_metadata_missing = {}

for col in future_collection_columns:

    attack_future_metadata_missing[col] = int(
        forecast_meta.loc[
            future_attack_mask,
            col,
        ].map(len).eq(0).sum()
    )

print("Benign future targets carrying attack metadata:")

for col, count in benign_future_metadata_violations.items():
    print(f"  {col:<35}: {count:,}")

print("\nAttack future targets missing attack metadata:")

for col, count in attack_future_metadata_missing.items():
    print(f"  {col:<35}: {count:,}")

if any(
    count != 0
    for count in benign_future_metadata_violations.values()
):
    raise RuntimeError(
        "A benign future target contains non-empty "
        "attack metadata."
    )

if any(
    count != 0
    for count in attack_future_metadata_missing.values()
):
    raise RuntimeError(
        "An attack future target is missing required "
        "rich attack metadata."
    )

print()
print("✓ Future binary state and rich metadata agree.")


# ==================================================================================================
# 11. NEW-ONSET / CONTINUATION CONSISTENCY
# ==================================================================================================

section("NEW-ONSET / CONTINUATION CONSISTENCY")

# IMPORTANT:
# Cell 10 defines onset/continuation relative to the FULL 15-minute
# historical forecasting window, not merely the state at Origin_Time.
#
# New_Onset:
#     future target is attack
#     AND there was NO attack anywhere in the historical window
#
# Continuation:
#     future target is attack
#     AND the historical window already contains attack activity
#
# Therefore Attack_In_History / Clean_History are the correct
# context variables for this audit.

# ----------------------------------------------------------------------------------
# Validate history-context fields first
# ----------------------------------------------------------------------------------

history_complement_errors = int(
    (
        (
            forecast_meta["Attack_In_History"]
            + forecast_meta["Clean_History"]
        )
        != 1
    ).sum()
)

attack_history_count_errors = int(
    (
        forecast_meta["Attack_In_History"]
        != forecast_meta["Attack_Minutes_In_History"].gt(0).astype(int)
    ).sum()
)

clean_history_count_errors = int(
    (
        forecast_meta["Clean_History"]
        != forecast_meta["Attack_Minutes_In_History"].eq(0).astype(int)
    ).sum()
)

print(
    f"Attack/Clean history complement errors : "
    f"{history_complement_errors:,}"
)

print(
    f"Attack-history count errors             : "
    f"{attack_history_count_errors:,}"
)

print(
    f"Clean-history count errors              : "
    f"{clean_history_count_errors:,}"
)

if (
    history_complement_errors
    or attack_history_count_errors
    or clean_history_count_errors
):
    raise RuntimeError(
        "Cell 10 history-context metadata is internally inconsistent."
    )


# ----------------------------------------------------------------------------------
# Correct Cell 10 onset semantics
# ----------------------------------------------------------------------------------

expected_new_onset = (
    forecast_meta["Future_Attack_State"].eq(1)
    & forecast_meta["Attack_In_History"].eq(0)
).astype(int)

expected_continuation = (
    forecast_meta["Future_Attack_State"].eq(1)
    & forecast_meta["Attack_In_History"].eq(1)
).astype(int)


# ----------------------------------------------------------------------------------
# Compare stored Cell 10 labels against expected semantics
# ----------------------------------------------------------------------------------

new_onset_errors = int(
    (
        forecast_meta["New_Onset"]
        != expected_new_onset
    ).sum()
)

continuation_errors = int(
    (
        forecast_meta["Continuation"]
        != expected_continuation
    ).sum()
)


# ----------------------------------------------------------------------------------
# Every future attack must belong to exactly one of the two classes
# ----------------------------------------------------------------------------------

mutual_exclusivity_errors = int(
    (
        (
            forecast_meta["New_Onset"]
            + forecast_meta["Continuation"]
        )
        > 1
    ).sum()
)

attack_partition_errors = int(
    (
        (
            forecast_meta["New_Onset"]
            + forecast_meta["Continuation"]
        )
        != forecast_meta["Future_Attack_State"]
    ).sum()
)


# ----------------------------------------------------------------------------------
# Additional logical checks
# ----------------------------------------------------------------------------------

new_onset_with_dirty_history = int(
    (
        forecast_meta["New_Onset"].eq(1)
        & forecast_meta["Attack_In_History"].eq(1)
    ).sum()
)

continuation_with_clean_history = int(
    (
        forecast_meta["Continuation"].eq(1)
        & forecast_meta["Clean_History"].eq(1)
    ).sum()
)

benign_marked_as_onset = int(
    (
        forecast_meta["New_Onset"].eq(1)
        & forecast_meta["Future_Attack_State"].eq(0)
    ).sum()
)

benign_marked_as_continuation = int(
    (
        forecast_meta["Continuation"].eq(1)
        & forecast_meta["Future_Attack_State"].eq(0)
    ).sum()
)


# ----------------------------------------------------------------------------------
# Report
# ----------------------------------------------------------------------------------

print()
print(
    f"New-onset semantic errors             : "
    f"{new_onset_errors:,}"
)

print(
    f"Continuation semantic errors          : "
    f"{continuation_errors:,}"
)

print(
    f"Mutual-exclusivity errors             : "
    f"{mutual_exclusivity_errors:,}"
)

print(
    f"Attack partition errors               : "
    f"{attack_partition_errors:,}"
)

print()
print(
    f"New onset with attack in history      : "
    f"{new_onset_with_dirty_history:,}"
)

print(
    f"Continuation with clean history       : "
    f"{continuation_with_clean_history:,}"
)

print(
    f"Benign target marked new onset        : "
    f"{benign_marked_as_onset:,}"
)

print(
    f"Benign target marked continuation     : "
    f"{benign_marked_as_continuation:,}"
)


# ----------------------------------------------------------------------------------
# Final validation
# ----------------------------------------------------------------------------------

onset_continuation_failures = {
    "history_complement_errors":
        history_complement_errors,

    "attack_history_count_errors":
        attack_history_count_errors,

    "clean_history_count_errors":
        clean_history_count_errors,

    "new_onset_errors":
        new_onset_errors,

    "continuation_errors":
        continuation_errors,

    "mutual_exclusivity_errors":
        mutual_exclusivity_errors,

    "attack_partition_errors":
        attack_partition_errors,

    "new_onset_with_dirty_history":
        new_onset_with_dirty_history,

    "continuation_with_clean_history":
        continuation_with_clean_history,

    "benign_marked_as_onset":
        benign_marked_as_onset,

    "benign_marked_as_continuation":
        benign_marked_as_continuation,
}

if any(
    value != 0
    for value in onset_continuation_failures.values()
):
    raise RuntimeError(
        "New_Onset / Continuation / history-context "
        "semantics are inconsistent.\n\n"
        + "\n".join(
            f"{name}: {value:,}"
            for name, value
            in onset_continuation_failures.items()
            if value != 0
        )
    )

print()
print(
    "✓ History-context metadata is internally consistent."
)

print(
    "✓ New_Onset means future attack after a completely "
    "clean 15-minute history."
)

print(
    "✓ Continuation means future attack when attack activity "
    "already exists somewhere in the 15-minute history."
)

print(
    "✓ New-onset and continuation targets are mutually exclusive."
)

print(
    "✓ Together they exactly partition all future attack targets."
)

# ==================================================================================================
# 12. EXACT TARGET ALIGNMENT AGAINST CELL 9
# ==================================================================================================

section("EXACT TARGET ALIGNMENT AGAINST CELL 9")

minute_lookup_columns = [
    "Segment_ID",
    "Minute",
    "Attack_State",
    "Attack_Types",
    "Attack_Source_IPs",
    "Attack_Destination_IPs",
    "Attack_Edges",
    "Typed_Attack_Edges",
]

target_truth = minute_meta[
    minute_lookup_columns
].copy()

target_truth = target_truth.rename(
    columns={
        "Minute":
            "_Truth_Target_Time",

        "Attack_State":
            "_Truth_Attack_State",

        "Attack_Types":
            "_Truth_Attack_Types",

        "Attack_Source_IPs":
            "_Truth_Attack_Sources",

        "Attack_Destination_IPs":
            "_Truth_Attack_Destinations",

        "Attack_Edges":
            "_Truth_Attack_Edges",

        "Typed_Attack_Edges":
            "_Truth_Typed_Attack_Edges",
    }
)

aligned = forecast_meta.merge(
    target_truth,
    left_on=[
        "Segment_ID",
        "Target_Time",
    ],
    right_on=[
        "Segment_ID",
        "_Truth_Target_Time",
    ],
    how="left",
    validate="many_to_one",
    indicator=True,
)

missing_target_minutes = int(
    aligned["_merge"].ne("both").sum()
)

print(
    f"Forecast rows without canonical target minute : "
    f"{missing_target_minutes:,}"
)

if missing_target_minutes:

    examples = (
        aligned.loc[
            aligned["_merge"].ne("both"),
            [
                "Segment_ID",
                "Origin_Time",
                "Horizon_Minutes",
                "Target_Time",
            ],
        ]
        .head(10)
        .to_dict("records")
    )

    raise RuntimeError(
        "At least one Cell 10 target does not map to a "
        "canonical Cell 9 minute.\n"
        f"Examples: {examples}"
    )

state_alignment_errors = int(
    (
        aligned["Future_Attack_State"]
        != aligned["_Truth_Attack_State"]
    ).sum()
)

collection_alignment_pairs = [
    (
        "Future_Attack_Types",
        "_Truth_Attack_Types",
    ),
    (
        "Future_Attack_Sources",
        "_Truth_Attack_Sources",
    ),
    (
        "Future_Attack_Destinations",
        "_Truth_Attack_Destinations",
    ),
    (
        "Future_Attack_Edges",
        "_Truth_Attack_Edges",
    ),
    (
        "Future_Typed_Attack_Edges",
        "_Truth_Typed_Attack_Edges",
    ),
]

collection_alignment_errors = {}

for future_col, truth_col in collection_alignment_pairs:

    mismatch = aligned.apply(
        lambda row: (
            canonical_collection(row[future_col])
            != canonical_collection(row[truth_col])
        ),
        axis=1,
    )

    collection_alignment_errors[future_col] = int(
        mismatch.sum()
    )

print(
    f"Binary-state alignment errors       : "
    f"{state_alignment_errors:,}"
)

for col, count in collection_alignment_errors.items():
    print(
        f"{col + ' alignment errors':<40}: "
        f"{count:,}"
    )

if state_alignment_errors:
    raise RuntimeError(
        "Future_Attack_State does not exactly match "
        "Cell 9 target-minute ground truth."
    )

if any(
    count != 0
    for count in collection_alignment_errors.values()
):
    raise RuntimeError(
        "At least one rich future target does not exactly "
        "match Cell 9 target-minute ground truth."
    )

print()
print(
    "✓ All 10,070 point-horizon targets exactly match "
    "canonical Cell 9 minute-level ground truth."
)


# ==================================================================================================
# 13. SEGMENT-BOUNDARY VALIDATION
# ==================================================================================================

section("SEGMENT-BOUNDARY VALIDATION")

segment_bounds = (
    minute_meta
    .groupby("Segment_ID")["Minute"]
    .agg(
        Segment_Start="min",
        Segment_End="max",
    )
    .reset_index()
)

boundary_check = forecast_meta.merge(
    segment_bounds,
    on="Segment_ID",
    how="left",
    validate="many_to_one",
)

history_before_segment = (
    boundary_check["History_Start"]
    < boundary_check["Segment_Start"]
)

target_after_segment = (
    boundary_check["Target_Time"]
    > boundary_check["Segment_End"]
)

origin_outside_segment = (
    (
        boundary_check["Origin_Time"]
        < boundary_check["Segment_Start"]
    )
    |
    (
        boundary_check["Origin_Time"]
        > boundary_check["Segment_End"]
    )
)

segment_boundary_violations = int(
    (
        history_before_segment
        | target_after_segment
        | origin_outside_segment
    ).sum()
)

print(
    f"Segment-boundary violations : "
    f"{segment_boundary_violations:,}"
)

if segment_boundary_violations:
    raise RuntimeError(
        "At least one forecasting sample crosses a "
        "capture-segment boundary."
    )

print(
    "✓ No history, origin, or future target crosses "
    "a segment boundary."
)


# ==================================================================================================
# 14. WIDE-vs-LONG CELL 10 CROSS-CHECK
# ==================================================================================================

section("CELL 10 WIDE-vs-LONG CROSS-CHECK")

wide = cell10_target_table.copy()

wide["Origin_Time"] = pd.to_datetime(
    wide["Origin_Time"],
    errors="raise",
)

wide["Segment_ID"] = pd.to_numeric(
    wide["Segment_ID"],
    errors="raise",
).astype(int)

wide_key_duplicates = int(
    wide.duplicated(
        ["Segment_ID", "Origin_Time"]
    ).sum()
)

if wide_key_duplicates:
    raise RuntimeError(
        "cell10_target_table contains duplicate "
        "(Segment_ID, Origin_Time) rows."
    )

wide_long_errors = {}

for horizon in EXPECTED_HORIZONS:

    required_wide_columns = [
        f"Target_Time_H{horizon}",
        f"Target_State_H{horizon}",
        f"Attack_Types_H{horizon}",
        f"Attack_Sources_H{horizon}",
        f"Attack_Destinations_H{horizon}",
        f"Attack_Edges_H{horizon}",
        f"Typed_Attack_Edges_H{horizon}",
        f"Attack_Type_Count_H{horizon}",
        f"Attack_Source_Count_H{horizon}",
        f"Attack_Destination_Count_H{horizon}",
        f"Attack_Edge_Count_H{horizon}",
        f"Typed_Attack_Edge_Count_H{horizon}",
        f"New_Onset_H{horizon}",
        f"Continuation_H{horizon}",
    ]

    missing_wide = [
        col for col in required_wide_columns
        if col not in wide.columns
    ]

    if missing_wide:
        raise RuntimeError(
            f"Wide Cell 10 table is missing H={horizon} columns:\n  - "
            + "\n  - ".join(missing_wide)
        )

    wide_h = wide[
        [
            "Segment_ID",
            "Origin_Time",
        ]
        + required_wide_columns
    ].copy()

    rename_map = {
        f"Target_Time_H{horizon}":
            "_Wide_Target_Time",

        f"Target_State_H{horizon}":
            "_Wide_Target_State",

        f"Attack_Types_H{horizon}":
            "_Wide_Attack_Types",

        f"Attack_Sources_H{horizon}":
            "_Wide_Attack_Sources",

        f"Attack_Destinations_H{horizon}":
            "_Wide_Attack_Destinations",

        f"Attack_Edges_H{horizon}":
            "_Wide_Attack_Edges",

        f"Typed_Attack_Edges_H{horizon}":
            "_Wide_Typed_Attack_Edges",

        f"Attack_Type_Count_H{horizon}":
            "_Wide_Attack_Type_Count",

        f"Attack_Source_Count_H{horizon}":
            "_Wide_Attack_Source_Count",

        f"Attack_Destination_Count_H{horizon}":
            "_Wide_Attack_Destination_Count",

        f"Attack_Edge_Count_H{horizon}":
            "_Wide_Attack_Edge_Count",

        f"Typed_Attack_Edge_Count_H{horizon}":
            "_Wide_Typed_Attack_Edge_Count",

        f"New_Onset_H{horizon}":
            "_Wide_New_Onset",

        f"Continuation_H{horizon}":
            "_Wide_Continuation",
    }

    wide_h = wide_h.rename(
        columns=rename_map
    )

    wide_h["_Wide_Target_Time"] = pd.to_datetime(
        wide_h["_Wide_Target_Time"],
        errors="raise",
    )

    long_h = forecast_meta.loc[
        forecast_meta["Horizon_Minutes"].eq(horizon)
    ].copy()

    check = long_h.merge(
        wide_h,
        on=[
            "Segment_ID",
            "Origin_Time",
        ],
        how="left",
        validate="one_to_one",
        indicator=True,
    )

    errors = 0

    errors += int(
        check["_merge"].ne("both").sum()
    )

    errors += int(
        (
            check["Target_Time"]
            != check["_Wide_Target_Time"]
        ).sum()
    )

    errors += int(
        (
            check["Future_Attack_State"]
            != pd.to_numeric(
                check["_Wide_Target_State"],
                errors="coerce",
            )
        ).sum()
    )

    collection_pairs = [
        (
            "Future_Attack_Types",
            "_Wide_Attack_Types",
        ),
        (
            "Future_Attack_Sources",
            "_Wide_Attack_Sources",
        ),
        (
            "Future_Attack_Destinations",
            "_Wide_Attack_Destinations",
        ),
        (
            "Future_Attack_Edges",
            "_Wide_Attack_Edges",
        ),
        (
            "Future_Typed_Attack_Edges",
            "_Wide_Typed_Attack_Edges",
        ),
    ]

    for left_col, right_col in collection_pairs:

        errors += int(
            check.apply(
                lambda row: (
                    canonical_collection(row[left_col])
                    != canonical_collection(row[right_col])
                ),
                axis=1,
            ).sum()
        )

    numeric_pairs = [
        (
            "Future_Attack_Type_Count",
            "_Wide_Attack_Type_Count",
        ),
        (
            "Future_Attack_Source_Count",
            "_Wide_Attack_Source_Count",
        ),
        (
            "Future_Attack_Destination_Count",
            "_Wide_Attack_Destination_Count",
        ),
        (
            "Future_Attack_Edge_Count",
            "_Wide_Attack_Edge_Count",
        ),
        (
            "Future_Typed_Attack_Edge_Count",
            "_Wide_Typed_Attack_Edge_Count",
        ),
        (
            "New_Onset",
            "_Wide_New_Onset",
        ),
        (
            "Continuation",
            "_Wide_Continuation",
        ),
    ]

    for left_col, right_col in numeric_pairs:

        right_numeric = pd.to_numeric(
            check[right_col],
            errors="coerce",
        )

        errors += int(
            (
                check[left_col]
                != right_numeric
            ).sum()
        )

    wide_long_errors[horizon] = errors

for horizon in EXPECTED_HORIZONS:
    print(
        f"Horizon +{horizon:>2} min : "
        f"{wide_long_errors[horizon]:,} mismatches"
    )

if any(
    count != 0
    for count in wide_long_errors.values()
):
    raise RuntimeError(
        "Cell 10 wide and long representations are inconsistent."
    )

print()
print(
    "✓ Cell 10 wide and long forecasting representations "
    "are exactly consistent."
)


# ==================================================================================================
# 15. ATTACK-TYPE VOCABULARY VALIDATION
# ==================================================================================================

section("ATTACK-TYPE VOCABULARY")

observed_attack_types = sorted(
    {
        attack_type
        for values in forecast_meta[
            "Future_Attack_Types"
        ]
        for attack_type in values
    }
)

metadata_attack_types = sorted(
    cell10_metadata.get(
        "attack_type_vocabulary",
        []
    )
)

print(
    f"Observed future attack types : "
    f"{len(observed_attack_types):,}"
)

for attack_type in observed_attack_types:
    print(f"  - {attack_type}")

if metadata_attack_types:

    missing_from_metadata = sorted(
        set(observed_attack_types)
        - set(metadata_attack_types)
    )

    metadata_only = sorted(
        set(metadata_attack_types)
        - set(observed_attack_types)
    )

    print(
        f"\nObserved types absent from metadata vocabulary : "
        f"{len(missing_from_metadata):,}"
    )

    print(
        f"Metadata vocabulary types not observed here    : "
        f"{len(metadata_only):,}"
    )

    if missing_from_metadata:
        raise RuntimeError(
            "Observed attack types are missing from "
            "cell10_metadata attack_type_vocabulary:\n  - "
            + "\n  - ".join(missing_from_metadata)
        )

print("✓ Attack-type vocabulary validated.")


# ==================================================================================================
# 16. GRAPH TARGET SUPPORT SUMMARY
# ==================================================================================================

section("GRAPH TARGET SUPPORT SUMMARY")

graph_support_rows = []

for horizon in EXPECTED_HORIZONS:

    hdf = forecast_meta.loc[
        forecast_meta["Horizon_Minutes"].eq(horizon)
    ]

    attack_h = hdf.loc[
        hdf["Future_Attack_State"].eq(1)
    ]

    graph_support_rows.append(
        {
            "Horizon_Minutes":
                horizon,

            "Samples":
                len(hdf),

            "Attack_Targets":
                len(attack_h),

            "With_Attack_Type":
                int(
                    attack_h[
                        "Future_Attack_Types"
                    ].map(len).gt(0).sum()
                ),

            "With_Source":
                int(
                    attack_h[
                        "Future_Attack_Sources"
                    ].map(len).gt(0).sum()
                ),

            "With_Destination":
                int(
                    attack_h[
                        "Future_Attack_Destinations"
                    ].map(len).gt(0).sum()
                ),

            "With_Edge":
                int(
                    attack_h[
                        "Future_Attack_Edges"
                    ].map(len).gt(0).sum()
                ),

            "With_Typed_Edge":
                int(
                    attack_h[
                        "Future_Typed_Attack_Edges"
                    ].map(len).gt(0).sum()
                ),
        }
    )

graph_support_df = pd.DataFrame(
    graph_support_rows
)

print(
    graph_support_df.to_string(
        index=False
    )
)

print(
    "\nNOTE:\n"
    "Future_Attack_Destinations is the future observed attack "
    "destination set and therefore the available victim-candidate "
    "target for later graph forecasting.\n\n"
    "Future_Attack_Edges preserves source → destination structure.\n"
    "Future_Typed_Attack_Edges preserves "
    "source → destination → attack-type structure."
)


# ==================================================================================================
# 17. DESTINATION / VICTIM-CANDIDATE SUPPORT
# ==================================================================================================

section("FUTURE DESTINATION / VICTIM-CANDIDATE SUPPORT")

destination_support = []

for horizon in EXPECTED_HORIZONS:

    hdf = forecast_meta.loc[
        (
            forecast_meta["Horizon_Minutes"].eq(horizon)
            & forecast_meta["Future_Attack_State"].eq(1)
        )
    ]

    single_destination = int(
        hdf[
            "Future_Attack_Destinations"
        ].map(len).eq(1).sum()
    )

    multiple_destination = int(
        hdf[
            "Future_Attack_Destinations"
        ].map(len).gt(1).sum()
    )

    no_destination = int(
        hdf[
            "Future_Attack_Destinations"
        ].map(len).eq(0).sum()
    )

    destination_support.append(
        {
            "Horizon_Minutes":
                horizon,

            "Attack_Targets":
                len(hdf),

            "Exactly_One_Destination":
                single_destination,

            "Multiple_Destinations":
                multiple_destination,

            "No_Destination":
                no_destination,
        }
    )

destination_support_df = pd.DataFrame(
    destination_support
)

print(
    destination_support_df.to_string(
        index=False
    )
)

print(
    "\nIMPORTANT:\n"
    "Multiple destinations are NOT collapsed into an arbitrary "
    "single victim label.\n"
    "The collection is retained for graph/set prediction."
)


# ==================================================================================================
# 18. CELL 11 EPISODE / CAMPAIGN METADATA DISCOVERY
# ==================================================================================================

section("CELL 11 EPISODE / CAMPAIGN ALIGNMENT")

episode_col = None
campaign_col = None
episode_source_name = None

episode_candidates = [
    "Episode_ID",
    "Attack_Episode_ID",
    "Strict_Episode_ID",
]

campaign_candidates = [
    "Campaign_ID",
    "Attack_Campaign_ID",
    "Campaign_Group_ID",
]

candidate_cell11_objects = []

for name, obj in list(globals().items()):

    if (
        name.startswith("cell11")
        and isinstance(obj, pd.DataFrame)
    ):
        candidate_cell11_objects.append(
            (name, obj)
        )

for name, obj in candidate_cell11_objects:

    if not {
        "Segment_ID",
        "Minute",
    }.issubset(obj.columns):
        continue

    found_episode = next(
        (
            col
            for col in episode_candidates
            if col in obj.columns
        ),
        None,
    )

    found_campaign = next(
        (
            col
            for col in campaign_candidates
            if col in obj.columns
        ),
        None,
    )

    if (
        found_episode is not None
        or found_campaign is not None
    ):
        episode_source_name = name
        episode_col = found_episode
        campaign_col = found_campaign
        break

print(
    f"Minute-aligned Cell 11 source : "
    f"{episode_source_name or 'NOT FOUND'}"
)
print(
    f"Episode column                : "
    f"{episode_col or 'NOT FOUND'}"
)
print(
    f"Campaign column               : "
    f"{campaign_col or 'NOT FOUND'}"
)

forecast_meta["Future_Episode_ID"] = pd.NA
forecast_meta["Future_Campaign_ID"] = pd.NA

if episode_source_name is not None:

    cell11_alignment = globals()[
        episode_source_name
    ].copy()

    cell11_alignment["Minute"] = pd.to_datetime(
        cell11_alignment["Minute"],
        errors="raise",
    )

    cell11_alignment["Segment_ID"] = pd.to_numeric(
        cell11_alignment["Segment_ID"],
        errors="raise",
    ).astype(int)

    keep_cols = [
        "Segment_ID",
        "Minute",
    ]

    rename_map = {
        "Minute":
            "Target_Time",
    }

    if episode_col is not None:
        keep_cols.append(episode_col)
        rename_map[episode_col] = (
            "Future_Episode_ID"
        )

    if campaign_col is not None:
        keep_cols.append(campaign_col)
        rename_map[campaign_col] = (
            "Future_Campaign_ID"
        )

    cell11_alignment = (
        cell11_alignment[
            keep_cols
        ]
        .rename(columns=rename_map)
        .drop_duplicates(
            ["Segment_ID", "Target_Time"]
        )
    )

    # Remove placeholder columns before merge.
    forecast_meta = forecast_meta.drop(
        columns=[
            "Future_Episode_ID",
            "Future_Campaign_ID",
        ]
    )

    forecast_meta = forecast_meta.merge(
        cell11_alignment,
        on=[
            "Segment_ID",
            "Target_Time",
        ],
        how="left",
        validate="many_to_one",
    )

    if "Future_Episode_ID" not in forecast_meta.columns:
        forecast_meta["Future_Episode_ID"] = pd.NA

    if "Future_Campaign_ID" not in forecast_meta.columns:
        forecast_meta["Future_Campaign_ID"] = pd.NA

    print()
    print("✓ Available Cell 11 metadata aligned to target time.")

else:

    print(
        "\nNOTE:\n"
        "No minute-aligned Cell 11 episode/campaign table was "
        "discovered automatically.\n"
        "This does NOT invalidate the forecasting targets.\n"
        "Future_Episode_ID and Future_Campaign_ID remain nullable "
        "audit fields."
    )


# ==================================================================================================
# 19. FINAL CANONICAL OUTPUT TABLE
# ==================================================================================================

section("FINAL CELL 12 FORECAST METADATA")

# Stable deterministic ordering.
forecast_meta = (
    forecast_meta
    .sort_values(
        [
            "Segment_ID",
            "Origin_Time",
            "Horizon_Minutes",
        ]
    )
    .reset_index(drop=True)
)

# Rename only for clarity in downstream cells.
# We KEEP the canonical Cell 10 names rather than inventing
# alternative names that future cells would have to rediscover.

cell12_forecast_metadata = forecast_meta.copy()

cell12_target_table = (
    cell12_forecast_metadata.copy()
)

# Explicit aliases useful for later graph code.
cell12_forecast_metadata[
    "Future_Victim_Candidates"
] = cell12_forecast_metadata[
    "Future_Attack_Destinations"
].map(lambda x: list(x))

cell12_forecast_metadata[
    "Future_Source_Candidates"
] = cell12_forecast_metadata[
    "Future_Attack_Sources"
].map(lambda x: list(x))

cell12_forecast_metadata[
    "Future_Graph_Edges"
] = cell12_forecast_metadata[
    "Future_Attack_Edges"
].map(lambda x: list(x))

cell12_forecast_metadata[
    "Future_Typed_Graph_Edges"
] = cell12_forecast_metadata[
    "Future_Typed_Attack_Edges"
].map(lambda x: list(x))

# Keep target-table alias synchronized after graph aliases.
cell12_target_table = (
    cell12_forecast_metadata.copy()
)

print(
    f"Rows                       : "
    f"{len(cell12_forecast_metadata):,}"
)
print(
    f"Columns                    : "
    f"{cell12_forecast_metadata.shape[1]:,}"
)
print(
    f"Forecast origins           : "
    f"{cell12_forecast_metadata[['Segment_ID', 'Origin_Time']].drop_duplicates().shape[0]:,}"
)
print(
    f"Horizons                   : "
    f"{sorted(cell12_forecast_metadata['Horizon_Minutes'].unique().tolist())}"
)
print(
    f"Attack target rows         : "
    f"{int(cell12_forecast_metadata['Future_Attack_State'].sum()):,}"
)

print(
    "\nGraph-target aliases created:"
)
print(
    "  Future_Victim_Candidates     <- "
    "Future_Attack_Destinations"
)
print(
    "  Future_Source_Candidates     <- "
    "Future_Attack_Sources"
)
print(
    "  Future_Graph_Edges           <- "
    "Future_Attack_Edges"
)
print(
    "  Future_Typed_Graph_Edges     <- "
    "Future_Typed_Attack_Edges"
)


# ==================================================================================================
# 20. TARGET AVAILABILITY MATRIX
# ==================================================================================================

section("FINAL TARGET AVAILABILITY MATRIX")

target_availability = pd.DataFrame(
    [
        {
            "Output":
                "Binary attack forecast",

            "Canonical_Target":
                "Future_Attack_State",

            "Available":
                True,

            "Semantics":
                "Attack state exactly at t + H",
        },
        {
            "Output":
                "Attack-type forecast",

            "Canonical_Target":
                "Future_Attack_Types",

            "Available":
                True,

            "Semantics":
                "Attack type set exactly at t + H",
        },
        {
            "Output":
                "Attack source forecast",

            "Canonical_Target":
                "Future_Attack_Sources",

            "Available":
                True,

            "Semantics":
                "Observed attack source endpoint set at t + H",
        },
        {
            "Output":
                "Victim-candidate forecast",

            "Canonical_Target":
                "Future_Attack_Destinations",

            "Available":
                True,

            "Semantics":
                "Observed attack destination endpoint set at t + H",
        },
        {
            "Output":
                "Graph edge forecast",

            "Canonical_Target":
                "Future_Attack_Edges",

            "Available":
                True,

            "Semantics":
                "Observed source → destination attack edges at t + H",
        },
        {
            "Output":
                "Typed graph edge forecast",

            "Canonical_Target":
                "Future_Typed_Attack_Edges",

            "Available":
                True,

            "Semantics":
                "Observed source → destination → attack-type edges at t + H",
        },
        {
            "Output":
                "New attack onset",

            "Canonical_Target":
                "New_Onset",

            "Available":
                True,

            "Semantics":
                "Future attack with currently benign origin",
        },
        {
            "Output":
                "Attack continuation",

            "Canonical_Target":
                "Continuation",

            "Available":
                True,

            "Semantics":
                "Future attack with currently attacking origin",
        },
    ]
)

print(
    target_availability.to_string(
        index=False
    )
)


# ==================================================================================================
# 21. LEAKAGE-SAFETY REGISTRY
# ==================================================================================================

section("LABEL-LEAKAGE SAFETY REGISTRY")

cell12_forbidden_model_features = {
    "Segment_ID",
    "History_Start",
    "History_End",
    "Origin_Time",
    "Horizon_Minutes",
    "Target_Time",
    "Current_State",
    "Attack_In_History",
    "Clean_History",
    "Attack_Minutes_In_History",

    "Future_Attack_State",
    "Future_Attack_Types",
    "Future_Attack_Sources",
    "Future_Attack_Destinations",
    "Future_Attack_Edges",
    "Future_Typed_Attack_Edges",

    "Future_Attack_Type_Count",
    "Future_Attack_Source_Count",
    "Future_Attack_Destination_Count",
    "Future_Attack_Edge_Count",
    "Future_Typed_Attack_Edge_Count",

    "New_Onset",
    "Continuation",

    "Future_Episode_ID",
    "Future_Campaign_ID",

    "Future_Victim_Candidates",
    "Future_Source_Candidates",
    "Future_Graph_Edges",
    "Future_Typed_Graph_Edges",
}

if "FORBIDDEN_MODEL_FEATURES" in globals():

    FORBIDDEN_MODEL_FEATURES = set(
        FORBIDDEN_MODEL_FEATURES
    ).union(
        cell12_forbidden_model_features
    )

else:

    FORBIDDEN_MODEL_FEATURES = set(
        cell12_forbidden_model_features
    )

print(
    f"Forbidden downstream feature names tracked : "
    f"{len(FORBIDDEN_MODEL_FEATURES):,}"
)

print(
    "✓ Future targets and audit metadata are explicitly "
    "blocked from predictor use."
)


# ==================================================================================================
# 22. CELL 12 METADATA OBJECT
# ==================================================================================================

cell12_metadata = {
    "history_length":
        EXPECTED_HISTORY_LENGTH,

    "forecast_horizons":
        EXPECTED_HORIZONS.copy(),

    "forecast_semantics":
        "point_horizon",

    "graph_target_semantics":
        "exact_graph_at_point_horizon",

    "number_of_forecast_rows":
        len(cell12_forecast_metadata),

    "number_of_origins":
        origin_count,

    "number_of_segments":
        int(
            cell12_forecast_metadata[
                "Segment_ID"
            ].nunique()
        ),

    "canonical_binary_target":
        "Future_Attack_State",

    "canonical_attack_type_target":
        "Future_Attack_Types",

    "canonical_source_target":
        "Future_Attack_Sources",

    "canonical_destination_target":
        "Future_Attack_Destinations",

    "canonical_edge_target":
        "Future_Attack_Edges",

    "canonical_typed_edge_target":
        "Future_Typed_Attack_Edges",

    "victim_candidate_alias":
        "Future_Victim_Candidates",

    "episode_metadata_available":
        bool(episode_col is not None),

    "campaign_metadata_available":
        bool(campaign_col is not None),

    "split_created":
        False,

    "scaling_performed":
        False,

    "resampling_performed":
        False,

    "model_trained":
        False,
}


# ==================================================================================================
# 23. FINAL VALIDATION SUMMARY
# ==================================================================================================

section("CELL 12 — FINAL VALIDATION SUMMARY")

print(
    f"Canonical minute rows              : "
    f"{len(minute_meta):,}"
)

print(
    f"Forecast origins                   : "
    f"{origin_count:,}"
)

print(
    f"Forecast rows                      : "
    f"{len(cell12_forecast_metadata):,}"
)

print(
    f"Forecast horizons                  : "
    f"{EXPECTED_HORIZONS}"
)

print(
    f"Duplicate forecast keys            : "
    f"{duplicate_forecast_keys:,}"
)

print(
    f"Target-time alignment errors       : "
    f"{target_time_errors:,}"
)

print(
    f"Segment-boundary violations        : "
    f"{segment_boundary_violations:,}"
)

print(
    f"Cell 9 binary alignment errors     : "
    f"{state_alignment_errors:,}"
)

print(
    "Cell 9 rich-target alignment errors: "
    f"{sum(collection_alignment_errors.values()):,}"
)

print(
    "Cell 10 wide/long mismatches       : "
    f"{sum(wide_long_errors.values()):,}"
)

print(
    "Rich-target count mismatches       : "
    f"{sum(count_mismatches.values()):,}"
)

print()
print("✓ Cell 9 rich ground truth validated.")
print("✓ Cell 10 long-form forecasting contract validated.")
print("✓ Cell 10 wide and long representations cross-validated.")
print("✓ Exact point-horizon timestamps validated.")
print("✓ Binary future attack targets validated.")
print("✓ Future attack-type targets validated.")
print("✓ Future source endpoint targets validated.")
print("✓ Future destination/victim-candidate targets validated.")
print("✓ Future source→destination graph edges validated.")
print("✓ Future typed graph edges validated.")
print("✓ New-onset / continuation semantics validated.")
print("✓ Multiple destinations remain multi-target collections.")
print("✓ No arbitrary victim selection performed.")
print("✓ No split created.")
print("✓ No scaling performed.")
print("✓ No resampling performed.")
print("✓ No model trained.")

print("\nObjects created:")
print("  cell12_forecast_metadata")
print("  cell12_target_table")
print("  cell12_metadata")
print("  graph_support_df")
print("  destination_support_df")
print("  target_availability")
print("  cell12_forbidden_model_features")
print("  FORBIDDEN_MODEL_FEATURES")

print(
    "\nNEXT:\n"
    "Cell 13 should perform the comprehensive forecastability / "
    "class-support / graph-target / episode / campaign audit using "
    "cell12_forecast_metadata BEFORE any train/validation/test split "
    "is designed."
)

print(SEP)


CELL 12 — CANONICAL RICH-TARGET VALIDATION + EPISODE/CAMPAIGN ALIGNMENT

Configuration
------------------------------------------------------------------------------------------------------------------------------------------------------------------
Canonical minute source          : Cell 9 minute_df
Forecasting contract source      : Cell 10
Forecast representation used     : cell10_forecast_metadata_long
Wide validation source           : cell10_target_table
Episode/campaign source          : Cell 11 if available

Temporal resolution              : 1 minute
History length                   : 15 minutes
Forecast horizons                : [1, 5, 10, 15, 30]
Forecast semantics               : POINT target at exactly t + H

Split created                    : NO
Scaling performed                : NO
Resampling performed             : NO
Model trained                    : NO

IMPORTANT:
Destination IP is retained as an observed attack destination / victim candidate.
It is NOT blindly asse

In [22]:
# ==================================================================================================
# CELL 13 — FINAL PRE-SPLIT FORECASTABILITY + TEMPORAL STRUCTURE + SPLIT-FEASIBILITY AUDIT
# ==================================================================================================
#
# PURPOSE
# -------
# Final diagnostic cell before train/validation/test construction.
#
# This cell:
#   1. Validates Cell 12's canonical forecasting table.
#   2. Verifies exact point-target alignment.
#   3. Rejoins Future_Attack_State to minute-level ground truth.
#   4. Audits rich attack-type/source/destination/edge metadata.
#   5. Audits New_Onset / Continuation ONLY according to the canonical
#      partition contract already established upstream.
#   6. Measures per-horizon target support.
#   7. Measures attack-type forecastability.
#   8. Measures source/destination/edge forecastability.
#   9. Measures temporal concentration by capture segment.
#  10. Reconstructs descriptive attack episodes for diagnostics only.
#  11. Examines every contiguous chronological segment-level
#      TRAIN / VALIDATION / TEST candidate.
#  12. Reports unseen attack types and unseen graph entities.
#  13. Reports whether each partition has attack support at every horizon.
#  14. Exports diagnostics for Cell 14.
#
# THIS CELL DOES NOT:
#   - create the final split
#   - scale
#   - resample
#   - train
#   - redefine Cell 12 targets
#
# ==================================================================================================

import ast
import json
import numpy as np
import pandas as pd
from collections import Counter

pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 300)
pd.set_option("display.max_colwidth", 180)

LINE = "=" * 160
SUBLINE = "-" * 160

print(LINE)
print("CELL 13 — FINAL PRE-SPLIT FORECASTABILITY + TEMPORAL STRUCTURE + SPLIT-FEASIBILITY AUDIT")
print(LINE)


# ==================================================================================================
# 1. CONFIGURATION
# ==================================================================================================

EPISODE_GAP_CANDIDATES = [0, 1, 2, 5]

print("\nConfiguration")
print(SUBLINE)
print("Episode-gap candidates        :", EPISODE_GAP_CANDIDATES, "minutes")
print("Split created                 : NO")
print("Scaling performed             : NO")
print("Resampling performed          : NO")
print("Model trained                 : NO")
print()
print("This cell is diagnostic only.")


# ==================================================================================================
# 2. REQUIRED OBJECTS
# ==================================================================================================

required_objects = [
    "minute_df",
    "cell12_forecast_metadata",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Required upstream objects are missing:\n"
        + "\n".join(f"  - {x}" for x in missing_objects)
    )

print("\n✓ Required canonical objects found.")


# ==================================================================================================
# 3. SAFE WORKING COPIES
# ==================================================================================================

minute_meta = minute_df.copy(deep=True)
forecast_meta = cell12_forecast_metadata.copy(deep=True)

print(f"minute_df rows                : {len(minute_meta):,}")
print(f"cell12_forecast_metadata rows : {len(forecast_meta):,}")


# ==================================================================================================
# 4. REQUIRED SCHEMA
# ==================================================================================================

required_minute_columns = [
    "Segment_ID",
    "Minute",
    "Attack_State",
]

required_forecast_columns = [
    "Segment_ID",
    "Origin_Time",
    "Horizon_Minutes",
    "Target_Time",
    "Future_Attack_State",

    "Future_Attack_Types",
    "Future_Attack_Sources",
    "Future_Attack_Destinations",
    "Future_Attack_Edges",
    "Future_Typed_Attack_Edges",

    "Future_Attack_Type_Count",
    "Future_Attack_Source_Count",
    "Future_Attack_Destination_Count",
    "Future_Attack_Edge_Count",
    "Future_Typed_Attack_Edge_Count",

    "New_Onset",
    "Continuation",
]

missing_minute_cols = [
    c for c in required_minute_columns
    if c not in minute_meta.columns
]

missing_forecast_cols = [
    c for c in required_forecast_columns
    if c not in forecast_meta.columns
]

if missing_minute_cols:
    raise RuntimeError(
        "minute_df is missing required columns:\n"
        + "\n".join(f"  - {x}" for x in missing_minute_cols)
    )

if missing_forecast_cols:
    raise RuntimeError(
        "cell12_forecast_metadata is missing required columns:\n"
        + "\n".join(f"  - {x}" for x in missing_forecast_cols)
    )

print("✓ Canonical Cell 9 / Cell 12 schema validated.")


# ==================================================================================================
# 5. NORMALIZE CORE TYPES
# ==================================================================================================

minute_meta["Minute"] = pd.to_datetime(
    minute_meta["Minute"],
    errors="raise"
)

forecast_meta["Origin_Time"] = pd.to_datetime(
    forecast_meta["Origin_Time"],
    errors="raise"
)

forecast_meta["Target_Time"] = pd.to_datetime(
    forecast_meta["Target_Time"],
    errors="raise"
)

minute_meta["Segment_ID"] = pd.to_numeric(
    minute_meta["Segment_ID"],
    errors="raise"
).astype(int)

minute_meta["Attack_State"] = pd.to_numeric(
    minute_meta["Attack_State"],
    errors="raise"
).astype(int)

forecast_meta["Segment_ID"] = pd.to_numeric(
    forecast_meta["Segment_ID"],
    errors="raise"
).astype(int)

forecast_meta["Horizon_Minutes"] = pd.to_numeric(
    forecast_meta["Horizon_Minutes"],
    errors="raise"
).astype(int)

for col in [
    "Future_Attack_State",
    "New_Onset",
    "Continuation",
]:
    forecast_meta[col] = pd.to_numeric(
        forecast_meta[col],
        errors="raise"
    ).astype(int)


# ==================================================================================================
# 6. SAFE COLLECTION HELPERS
# ==================================================================================================

def normalize_collection(value):
    """
    Convert Cell 12 rich metadata to a tuple without changing its meaning.
    Handles lists, tuples, sets, arrays, JSON strings, literal strings,
    empty strings and nulls.
    """

    if value is None:
        return tuple()

    try:
        if pd.isna(value):
            return tuple()
    except (TypeError, ValueError):
        pass

    if isinstance(value, np.ndarray):
        value = value.tolist()

    if isinstance(value, set):
        return tuple(sorted(value, key=str))

    if isinstance(value, list):
        return tuple(value)

    if isinstance(value, tuple):
        return value

    if isinstance(value, str):

        text = value.strip()

        if text == "" or text.lower() in {
            "nan", "none", "null", "[]", "()", "{}"
        }:
            return tuple()

        try:
            parsed = json.loads(text)

            if parsed is None:
                return tuple()

            if isinstance(parsed, list):
                return tuple(parsed)

            if isinstance(parsed, tuple):
                return parsed

            return (parsed,)

        except Exception:
            pass

        try:
            parsed = ast.literal_eval(text)

            if parsed is None:
                return tuple()

            if isinstance(parsed, list):
                return tuple(parsed)

            if isinstance(parsed, tuple):
                return parsed

            if isinstance(parsed, set):
                return tuple(sorted(parsed, key=str))

            return (parsed,)

        except Exception:
            return (text,)

    return (value,)


def hashable_item(value):

    if isinstance(value, np.ndarray):
        value = value.tolist()

    if isinstance(value, list):
        return tuple(hashable_item(x) for x in value)

    if isinstance(value, tuple):
        return tuple(hashable_item(x) for x in value)

    if isinstance(value, set):
        return tuple(
            sorted(
                (hashable_item(x) for x in value),
                key=str
            )
        )

    return value


def collection_count(value):
    return len(normalize_collection(value))


def metadata_set(df, column):

    output = set()

    if len(df) == 0:
        return output

    for value in df[column]:
        for item in normalize_collection(value):
            output.add(hashable_item(item))

    return output


# ==================================================================================================
# 7. BINARY DOMAIN VALIDATION
# ==================================================================================================

print("\n" + LINE)
print("BINARY TARGET VALIDATION")
print(LINE)

binary_columns = {
    "Attack_State": minute_meta["Attack_State"],
    "Future_Attack_State": forecast_meta["Future_Attack_State"],
    "New_Onset": forecast_meta["New_Onset"],
    "Continuation": forecast_meta["Continuation"],
}

for name, series in binary_columns.items():

    values = set(
        series.dropna().unique().tolist()
    )

    invalid = values - {0, 1}

    if invalid:
        raise RuntimeError(
            f"{name} contains invalid binary values: {sorted(invalid)}"
        )

print("✓ All binary state columns contain only 0/1.")


# ==================================================================================================
# 8. FINAL INPUT SCHEMA
# ==================================================================================================

HORIZONS = sorted(
    forecast_meta["Horizon_Minutes"]
    .unique()
    .tolist()
)

segments = sorted(
    minute_meta["Segment_ID"]
    .unique()
    .tolist()
)

print("\n" + LINE)
print("FINAL INPUT SCHEMA")
print(LINE)

print(f"Minute rows                : {len(minute_meta):,}")
print(f"Minute columns             : {minute_meta.shape[1]:,}")
print(f"Forecast rows              : {len(forecast_meta):,}")
print(f"Forecast columns           : {forecast_meta.shape[1]:,}")
print(f"Segments                   : {len(segments):,}")
print(f"Forecast horizons          : {HORIZONS}")


# ==================================================================================================
# 9. FORECASTING CONTRACT
# ==================================================================================================

print("\n" + LINE)
print("FORECASTING CONTRACT VALIDATION")
print(LINE)

origin_keys = (
    forecast_meta[
        ["Segment_ID", "Origin_Time"]
    ]
    .drop_duplicates()
)

n_origins = len(origin_keys)
n_horizons = len(HORIZONS)

expected_rows = n_origins * n_horizons
actual_rows = len(forecast_meta)

duplicate_forecast_keys = int(
    forecast_meta.duplicated(
        [
            "Segment_ID",
            "Origin_Time",
            "Horizon_Minutes",
        ]
    ).sum()
)

expected_target_time = (
    forecast_meta["Origin_Time"]
    + pd.to_timedelta(
        forecast_meta["Horizon_Minutes"],
        unit="m"
    )
)

target_alignment_errors = int(
    (
        forecast_meta["Target_Time"]
        != expected_target_time
    ).sum()
)

origin_horizon_counts = (
    forecast_meta
    .groupby(
        ["Segment_ID", "Origin_Time"]
    )["Horizon_Minutes"]
    .agg(["count", "nunique"])
)

origins_missing_horizons = int(
    (
        (origin_horizon_counts["count"] != n_horizons)
        |
        (origin_horizon_counts["nunique"] != n_horizons)
    ).sum()
)

per_horizon_counts = (
    forecast_meta["Horizon_Minutes"]
    .value_counts()
    .sort_index()
)

wrong_per_horizon_counts = {
    int(h): int(n)
    for h, n in per_horizon_counts.items()
    if int(n) != n_origins
}

segment_bounds = (
    minute_meta
    .groupby("Segment_ID")["Minute"]
    .agg(["min", "max"])
)

segment_boundary_violations = 0

for seg_id, grp in forecast_meta.groupby("Segment_ID"):

    if seg_id not in segment_bounds.index:
        segment_boundary_violations += len(grp)
        continue

    seg_min = segment_bounds.loc[seg_id, "min"]
    seg_max = segment_bounds.loc[seg_id, "max"]

    bad = (
        (grp["Origin_Time"] < seg_min)
        | (grp["Origin_Time"] > seg_max)
        | (grp["Target_Time"] < seg_min)
        | (grp["Target_Time"] > seg_max)
    )

    segment_boundary_violations += int(bad.sum())

print(f"Forecast origins              : {n_origins:,}")
print(f"Forecast horizons             : {HORIZONS}")
print(f"Expected forecast rows        : {expected_rows:,}")
print(f"Actual forecast rows          : {actual_rows:,}")
print(f"Duplicate forecast keys       : {duplicate_forecast_keys:,}")
print(f"Target-time alignment errors  : {target_alignment_errors:,}")
print(f"Origins missing horizons      : {origins_missing_horizons:,}")
print(f"Wrong per-horizon row counts  : {wrong_per_horizon_counts}")
print(f"Segment-boundary violations   : {segment_boundary_violations:,}")

contract_errors = (
    int(actual_rows != expected_rows)
    + duplicate_forecast_keys
    + target_alignment_errors
    + origins_missing_horizons
    + len(wrong_per_horizon_counts)
    + segment_boundary_violations
)

if contract_errors:
    raise RuntimeError(
        "Canonical forecasting contract is structurally inconsistent."
    )

print("\n✓ Forecasting contract validated.")


# ==================================================================================================
# 10. TARGET GROUND-TRUTH REJOIN
# ==================================================================================================

print("\n" + LINE)
print("TARGET GROUND-TRUTH REJOIN AUDIT")
print(LINE)

minute_target_lookup = (
    minute_meta[
        [
            "Segment_ID",
            "Minute",
            "Attack_State",
        ]
    ]
    .rename(
        columns={
            "Minute": "_Audit_Target_Time",
            "Attack_State": "_Audit_Attack_State",
        }
    )
)

target_audit = forecast_meta.merge(
    minute_target_lookup,
    left_on=["Segment_ID", "Target_Time"],
    right_on=["Segment_ID", "_Audit_Target_Time"],
    how="left",
    validate="many_to_one",
)

missing_target_minutes = int(
    target_audit["_Audit_Target_Time"]
    .isna()
    .sum()
)

valid_target_mask = (
    target_audit["_Audit_Target_Time"]
    .notna()
)

binary_target_mismatches = int(
    (
        target_audit.loc[
            valid_target_mask,
            "Future_Attack_State"
        ]
        !=
        target_audit.loc[
            valid_target_mask,
            "_Audit_Attack_State"
        ]
    ).sum()
)

print(
    f"Forecast targets missing minute ground truth : "
    f"{missing_target_minutes:,}"
)

print(
    f"Binary target mismatches                     : "
    f"{binary_target_mismatches:,}"
)

if missing_target_minutes or binary_target_mismatches:
    raise RuntimeError(
        "Forecast target ground truth is inconsistent with minute_df."
    )

print("\n✓ Future binary targets exactly match minute-level ground truth.")


# ==================================================================================================
# 11. RICH TARGET INTERNAL CONSISTENCY
# ==================================================================================================

print("\n" + LINE)
print("RICH TARGET INTERNAL-CONSISTENCY AUDIT")
print(LINE)

attack_mask = (
    forecast_meta["Future_Attack_State"] == 1
)

benign_mask = ~attack_mask

collection_to_count = {
    "Future_Attack_Types":
        "Future_Attack_Type_Count",

    "Future_Attack_Sources":
        "Future_Attack_Source_Count",

    "Future_Attack_Destinations":
        "Future_Attack_Destination_Count",

    "Future_Attack_Edges":
        "Future_Attack_Edge_Count",

    "Future_Typed_Attack_Edges":
        "Future_Typed_Attack_Edge_Count",
}

consistency_rows = []

for collection_col, count_col in collection_to_count.items():

    actual_counts = (
        forecast_meta[collection_col]
        .apply(collection_count)
        .astype(int)
    )

    stored_counts = pd.to_numeric(
        forecast_meta[count_col],
        errors="coerce"
    )

    missing_count_on_attack = int(
        (
            attack_mask
            & stored_counts.isna()
        ).sum()
    )

    attack_count_mismatches = int(
        (
            attack_mask
            & stored_counts.notna()
            & (
                actual_counts
                != stored_counts.fillna(-1).astype(int)
            )
        ).sum()
    )

    benign_nonempty = int(
        (
            benign_mask
            & (actual_counts != 0)
        ).sum()
    )

    attack_empty = int(
        (
            attack_mask
            & (actual_counts == 0)
        ).sum()
    )

    consistency_rows.append({
        "Collection": collection_col,
        "Count_Column": count_col,
        "Missing_Count_On_Attack": missing_count_on_attack,
        "Attack_Count_Mismatches": attack_count_mismatches,
        "Benign_Nonempty_Collections": benign_nonempty,
        "Attack_Empty_Collections": attack_empty,
    })

cell13_target_consistency = pd.DataFrame(
    consistency_rows
)

print(
    cell13_target_consistency.to_string(
        index=False
    )
)

critical_metadata_errors = int(
    cell13_target_consistency[
        [
            "Missing_Count_On_Attack",
            "Attack_Count_Mismatches",
            "Benign_Nonempty_Collections",
            "Attack_Empty_Collections",
        ]
    ].sum().sum()
)

if critical_metadata_errors:
    raise RuntimeError(
        "Canonical rich target metadata is internally inconsistent."
    )

print("\n✓ Rich attack metadata is internally consistent.")


# ==================================================================================================
# 12. GRAPH STRUCTURE CONSISTENCY
# ==================================================================================================

print("\n" + LINE)
print("GRAPH STRUCTURE CONSISTENCY")
print(LINE)

graph_structure_errors = 0

for _, row in forecast_meta.loc[attack_mask].iterrows():

    sources = {
        str(x)
        for x in normalize_collection(
            row["Future_Attack_Sources"]
        )
    }

    destinations = {
        str(x)
        for x in normalize_collection(
            row["Future_Attack_Destinations"]
        )
    }

    attack_types = {
        str(x)
        for x in normalize_collection(
            row["Future_Attack_Types"]
        )
    }

    edges = [
        hashable_item(x)
        for x in normalize_collection(
            row["Future_Attack_Edges"]
        )
    ]

    typed_edges = [
        hashable_item(x)
        for x in normalize_collection(
            row["Future_Typed_Attack_Edges"]
        )
    ]

    for edge in edges:

        if (
            not isinstance(edge, tuple)
            or len(edge) != 2
        ):
            graph_structure_errors += 1
            continue

        src, dst = edge

        if (
            str(src) not in sources
            or str(dst) not in destinations
        ):
            graph_structure_errors += 1

    for edge in typed_edges:

        if (
            not isinstance(edge, tuple)
            or len(edge) != 3
        ):
            graph_structure_errors += 1
            continue

        src, dst, attack_type = edge

        if (
            str(src) not in sources
            or str(dst) not in destinations
            or str(attack_type) not in attack_types
        ):
            graph_structure_errors += 1

print(
    "Graph metadata consistency violations :",
    graph_structure_errors
)

if graph_structure_errors:
    raise RuntimeError(
        "Canonical graph metadata is internally inconsistent."
    )

print(
    "✓ Graph source/destination/type relationships are internally consistent."
)


# ==================================================================================================
# 13. NEW-ONSET / CONTINUATION — CANONICAL CONTRACT ONLY
# ==================================================================================================
#
# CRITICAL:
#
# Cell 13 DOES NOT redefine these labels.
#
# It does NOT compare them with Current_State.
# It does NOT infer them from history.
# It does NOT infer them from intermediate states.
#
# Cell 12 is the canonical source.
#
# The only invariants checked here are:
#
#   1. both are binary
#   2. they are mutually exclusive
#   3. benign future target -> both are 0
#   4. attack future target -> exactly one is 1
#   5. New_Onset + Continuation == Future_Attack_State
#
# ==================================================================================================

print("\n" + LINE)
print("NEW-ONSET / CONTINUATION CONSISTENCY")
print(LINE)

new_onset = forecast_meta["New_Onset"].astype(int)
continuation = forecast_meta["Continuation"].astype(int)
future_attack = forecast_meta["Future_Attack_State"].astype(int)

invalid_new_onset_values = int(
    (~new_onset.isin([0, 1])).sum()
)

invalid_continuation_values = int(
    (~continuation.isin([0, 1])).sum()
)

mutual_exclusivity_errors = int(
    (
        (new_onset == 1)
        & (continuation == 1)
    ).sum()
)

attack_partition_errors = int(
    (
        (new_onset + continuation)
        != future_attack
    ).sum()
)

benign_target_errors = int(
    (
        (future_attack == 0)
        &
        (
            (new_onset != 0)
            | (continuation != 0)
        )
    ).sum()
)

attack_target_errors = int(
    (
        (future_attack == 1)
        &
        (
            (new_onset + continuation)
            != 1
        )
    ).sum()
)

canonical_onset_errors = (
    invalid_new_onset_values
    + invalid_continuation_values
    + mutual_exclusivity_errors
    + attack_partition_errors
    + benign_target_errors
    + attack_target_errors
)

print(
    f"Invalid New_Onset values          : "
    f"{invalid_new_onset_values:,}"
)

print(
    f"Invalid Continuation values       : "
    f"{invalid_continuation_values:,}"
)

print(
    f"Mutual-exclusivity errors         : "
    f"{mutual_exclusivity_errors:,}"
)

print(
    f"Attack partition errors           : "
    f"{attack_partition_errors:,}"
)

print(
    f"Benign-target semantic errors     : "
    f"{benign_target_errors:,}"
)

print(
    f"Attack-target semantic errors     : "
    f"{attack_target_errors:,}"
)

if canonical_onset_errors:
    raise RuntimeError(
        "Cell 12 New_Onset / Continuation canonical partition "
        "contract is internally inconsistent."
    )

print()
print("✓ Canonical New_Onset / Continuation partition contract validated.")
print("✓ No independent reinterpretation from Current_State was performed.")


# ==================================================================================================
# 14. TARGET DISTRIBUTION BY HORIZON
# ==================================================================================================

print("\n" + LINE)
print("FORECAST TARGET DISTRIBUTION BY HORIZON")
print(LINE)

horizon_rows = []

for horizon in HORIZONS:

    h = forecast_meta.loc[
        forecast_meta["Horizon_Minutes"] == horizon
    ]

    attack_n = int(
        h["Future_Attack_State"].sum()
    )

    benign_n = int(
        len(h) - attack_n
    )

    onset_n = int(
        h["New_Onset"].sum()
    )

    continuation_n = int(
        h["Continuation"].sum()
    )

    horizon_rows.append({
        "Horizon_Minutes": int(horizon),
        "Samples": int(len(h)),
        "Attack_Targets": attack_n,
        "Benign_Targets": benign_n,
        "Attack_Percent":
            (100.0 * attack_n / len(h))
            if len(h) else np.nan,
        "New_Onset_Targets": onset_n,
        "Continuation_Targets": continuation_n,
    })

cell13_horizon_distribution = pd.DataFrame(
    horizon_rows
)

print(
    cell13_horizon_distribution.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ==================================================================================================
# 15. ATTACK-TYPE SUPPORT BY HORIZON
# ==================================================================================================

print("\n" + LINE)
print("ATTACK-TYPE FORECASTABILITY BY HORIZON")
print(LINE)

attack_forecasts = forecast_meta.loc[
    attack_mask
].copy()

attack_type_rows = []

for horizon in HORIZONS:

    h = attack_forecasts.loc[
        attack_forecasts["Horizon_Minutes"]
        == horizon
    ]

    counter = Counter()

    for value in h["Future_Attack_Types"]:
        for attack_type in normalize_collection(value):
            counter[str(attack_type)] += 1

    for attack_type, count in sorted(
        counter.items(),
        key=lambda x: (-x[1], x[0])
    ):

        attack_type_rows.append({
            "Horizon_Minutes": int(horizon),
            "Attack_Type": attack_type,
            "Target_Rows": int(count),
        })

cell13_attack_type_by_horizon = pd.DataFrame(
    attack_type_rows
)

if len(cell13_attack_type_by_horizon):
    print(
        cell13_attack_type_by_horizon.to_string(
            index=False
        )
    )
else:
    print("No attack-type targets found.")


# ==================================================================================================
# 16. GLOBAL ATTACK-TYPE SUPPORT
# ==================================================================================================

print("\n" + LINE)
print("GLOBAL ATTACK-TYPE SUPPORT")
print(LINE)

global_type_counter = Counter()

for value in attack_forecasts["Future_Attack_Types"]:
    for attack_type in normalize_collection(value):
        global_type_counter[str(attack_type)] += 1

cell13_attack_type_support = pd.DataFrame(
    [
        {
            "Attack_Type": attack_type,
            "Forecast_Target_Rows": int(count),
        }
        for attack_type, count in sorted(
            global_type_counter.items(),
            key=lambda x: (-x[1], x[0])
        )
    ]
)

if len(cell13_attack_type_support):
    print(
        cell13_attack_type_support.to_string(
            index=False
        )
    )
else:
    print("No attack types found.")


# ==================================================================================================
# 17. GRAPH TARGET SUPPORT BY HORIZON
# ==================================================================================================

print("\n" + LINE)
print("SOURCE / DESTINATION / GRAPH-EDGE FORECASTABILITY")
print(LINE)

graph_rows = []

for horizon in HORIZONS:

    h = attack_forecasts.loc[
        attack_forecasts["Horizon_Minutes"]
        == horizon
    ]

    source_counts = (
        h["Future_Attack_Sources"]
        .apply(collection_count)
    )

    destination_counts = (
        h["Future_Attack_Destinations"]
        .apply(collection_count)
    )

    edge_counts = (
        h["Future_Attack_Edges"]
        .apply(collection_count)
    )

    typed_edge_counts = (
        h["Future_Typed_Attack_Edges"]
        .apply(collection_count)
    )

    graph_rows.append({
        "Horizon_Minutes": int(horizon),
        "Attack_Targets": int(len(h)),

        "Mean_Source_Count":
            float(source_counts.mean())
            if len(h) else np.nan,

        "Max_Source_Count":
            int(source_counts.max())
            if len(h) else 0,

        "Mean_Destination_Count":
            float(destination_counts.mean())
            if len(h) else np.nan,

        "Max_Destination_Count":
            int(destination_counts.max())
            if len(h) else 0,

        "Mean_Edge_Count":
            float(edge_counts.mean())
            if len(h) else np.nan,

        "Max_Edge_Count":
            int(edge_counts.max())
            if len(h) else 0,

        "Mean_Typed_Edge_Count":
            float(typed_edge_counts.mean())
            if len(h) else np.nan,

        "Max_Typed_Edge_Count":
            int(typed_edge_counts.max())
            if len(h) else 0,
    })

cell13_graph_support = pd.DataFrame(
    graph_rows
)

print(
    cell13_graph_support.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ==================================================================================================
# 18. GLOBAL GRAPH ENTITY SUPPORT
# ==================================================================================================

print("\n" + LINE)
print("GLOBAL GRAPH ENTITY SUPPORT")
print(LINE)

all_sources = metadata_set(
    attack_forecasts,
    "Future_Attack_Sources"
)

all_destinations = metadata_set(
    attack_forecasts,
    "Future_Attack_Destinations"
)

all_edges = metadata_set(
    attack_forecasts,
    "Future_Attack_Edges"
)

all_typed_edges = metadata_set(
    attack_forecasts,
    "Future_Typed_Attack_Edges"
)

print(f"Unique observed attack sources      : {len(all_sources):,}")
print(f"Unique observed attack destinations : {len(all_destinations):,}")
print(f"Unique attack edges                 : {len(all_edges):,}")
print(f"Unique typed attack edges           : {len(all_typed_edges):,}")


# ==================================================================================================
# 19. MINUTE-LEVEL ATTACK DISTRIBUTION BY SEGMENT
# ==================================================================================================

print("\n" + LINE)
print("MINUTE-LEVEL ATTACK DISTRIBUTION BY SEGMENT")
print(LINE)

segment_attack_summary = (
    minute_meta
    .groupby("Segment_ID")
    .agg(
        Minutes=("Minute", "size"),
        Attack_Minutes=("Attack_State", "sum"),
        Start_Minute=("Minute", "min"),
        End_Minute=("Minute", "max"),
    )
    .reset_index()
)

segment_attack_summary["Benign_Minutes"] = (
    segment_attack_summary["Minutes"]
    - segment_attack_summary["Attack_Minutes"]
)

segment_attack_summary["Attack_Percent"] = (
    100.0
    * segment_attack_summary["Attack_Minutes"]
    / segment_attack_summary["Minutes"]
)

print(
    segment_attack_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ==================================================================================================
# 20. FORECAST SUPPORT BY SEGMENT + HORIZON
# ==================================================================================================

print("\n" + LINE)
print("FORECAST SUPPORT BY SEGMENT AND HORIZON")
print(LINE)

cell13_segment_horizon_support = (
    forecast_meta
    .groupby(
        ["Segment_ID", "Horizon_Minutes"]
    )
    .agg(
        Samples=("Future_Attack_State", "size"),
        Attack_Targets=("Future_Attack_State", "sum"),
        New_Onset_Targets=("New_Onset", "sum"),
        Continuation_Targets=("Continuation", "sum"),
    )
    .reset_index()
)

cell13_segment_horizon_support["Benign_Targets"] = (
    cell13_segment_horizon_support["Samples"]
    - cell13_segment_horizon_support["Attack_Targets"]
)

cell13_segment_horizon_support["Attack_Percent"] = (
    100.0
    * cell13_segment_horizon_support["Attack_Targets"]
    / cell13_segment_horizon_support["Samples"]
)

print(
    cell13_segment_horizon_support.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ==================================================================================================
# 21. ATTACK-TYPE SUPPORT BY SEGMENT
# ==================================================================================================

print("\n" + LINE)
print("ATTACK-TYPE SUPPORT BY SEGMENT")
print(LINE)

segment_type_rows = []

for seg_id in segments:

    grp = attack_forecasts.loc[
        attack_forecasts["Segment_ID"] == seg_id
    ]

    counter = Counter()

    for value in grp["Future_Attack_Types"]:
        for attack_type in normalize_collection(value):
            counter[str(attack_type)] += 1

    segment_type_rows.append({
        "Segment_ID": int(seg_id),
        "Attack_Forecast_Rows": int(len(grp)),
        "Attack_Type_Count": int(len(counter)),
        "Attack_Type_List": ", ".join(sorted(counter.keys())),
    })

cell13_segment_type_support = pd.DataFrame(
    segment_type_rows
)

print(
    cell13_segment_type_support.to_string(
        index=False
    )
)


# ==================================================================================================
# 22. ATTACK-TYPE TEMPORAL CONCENTRATION
# ==================================================================================================

print("\n" + LINE)
print("ATTACK-TYPE TEMPORAL CONCENTRATION")
print(LINE)

type_segment_map = {}

for attack_type in global_type_counter.keys():

    containing_segments = set()

    for seg_id in segments:

        grp = attack_forecasts.loc[
            attack_forecasts["Segment_ID"] == seg_id
        ]

        found = False

        for value in grp["Future_Attack_Types"]:

            current_types = {
                str(x)
                for x in normalize_collection(value)
            }

            if attack_type in current_types:
                found = True
                break

        if found:
            containing_segments.add(int(seg_id))

    type_segment_map[attack_type] = containing_segments

type_concentration_rows = []

for attack_type, segs in sorted(
    type_segment_map.items()
):

    type_concentration_rows.append({
        "Attack_Type": attack_type,
        "Segment_Count": int(len(segs)),
        "Segments": ",".join(map(str, sorted(segs))),
        "Single_Segment_Only": bool(len(segs) == 1),
    })

cell13_attack_type_concentration = pd.DataFrame(
    type_concentration_rows
)

if len(cell13_attack_type_concentration):
    print(
        cell13_attack_type_concentration.to_string(
            index=False
        )
    )
else:
    print("No attack types found.")


# ==================================================================================================
# 23. DESCRIPTIVE ATTACK EPISODES
# ==================================================================================================
#
# Diagnostic only.
#
# These episode definitions DO NOT become training labels and DO NOT
# automatically determine the final split.
#
# ==================================================================================================

print("\n" + LINE)
print("DESCRIPTIVE ATTACK EPISODE RECONSTRUCTION")
print(LINE)


def build_attack_episodes(
    minute_dataframe,
    allowed_benign_gap_minutes
):

    episodes = []
    episode_id = 0

    ordered = minute_dataframe.sort_values(
        ["Segment_ID", "Minute"]
    )

    for seg_id, seg_df in ordered.groupby("Segment_ID"):

        attack_times = (
            seg_df.loc[
                seg_df["Attack_State"] == 1,
                "Minute"
            ]
            .sort_values()
            .tolist()
        )

        if len(attack_times) == 0:
            continue

        current_start = attack_times[0]
        current_end = attack_times[0]
        attack_minute_count = 1

        for timestamp in attack_times[1:]:

            minute_difference = int(
                (timestamp - current_end)
                / pd.Timedelta(minutes=1)
            )

            benign_gap = minute_difference - 1

            if benign_gap <= allowed_benign_gap_minutes:

                current_end = timestamp
                attack_minute_count += 1

            else:

                episodes.append({
                    "Episode_ID": episode_id,
                    "Segment_ID": int(seg_id),
                    "Start_Time": current_start,
                    "End_Time": current_end,
                    "Attack_Minutes": int(attack_minute_count),
                    "Span_Minutes": int(
                        (current_end - current_start)
                        / pd.Timedelta(minutes=1)
                    ) + 1,
                })

                episode_id += 1

                current_start = timestamp
                current_end = timestamp
                attack_minute_count = 1

        episodes.append({
            "Episode_ID": episode_id,
            "Segment_ID": int(seg_id),
            "Start_Time": current_start,
            "End_Time": current_end,
            "Attack_Minutes": int(attack_minute_count),
            "Span_Minutes": int(
                (current_end - current_start)
                / pd.Timedelta(minutes=1)
            ) + 1,
        })

        episode_id += 1

    return pd.DataFrame(episodes)


episode_tables = {}
episode_summary_rows = []

for gap in EPISODE_GAP_CANDIDATES:

    episodes = build_attack_episodes(
        minute_meta,
        gap
    )

    episode_tables[gap] = episodes

    if len(episodes):

        episode_summary_rows.append({
            "Allowed_Benign_Gap_Minutes": int(gap),
            "Episodes": int(len(episodes)),
            "Median_Attack_Minutes":
                float(episodes["Attack_Minutes"].median()),
            "Max_Attack_Minutes":
                int(episodes["Attack_Minutes"].max()),
            "Median_Span_Minutes":
                float(episodes["Span_Minutes"].median()),
            "Max_Span_Minutes":
                int(episodes["Span_Minutes"].max()),
        })

    else:

        episode_summary_rows.append({
            "Allowed_Benign_Gap_Minutes": int(gap),
            "Episodes": 0,
            "Median_Attack_Minutes": np.nan,
            "Max_Attack_Minutes": np.nan,
            "Median_Span_Minutes": np.nan,
            "Max_Span_Minutes": np.nan,
        })

cell13_episode_gap_summary = pd.DataFrame(
    episode_summary_rows
)

print(
    cell13_episode_gap_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.2f}"
    )
)


# ==================================================================================================
# 24. CHRONOLOGICAL SEGMENT ORDER
# ==================================================================================================

print("\n" + LINE)
print("CHRONOLOGICAL SEGMENT ORDER")
print(LINE)

segment_time_table = (
    minute_meta
    .groupby("Segment_ID")
    .agg(
        Start_Time=("Minute", "min"),
        End_Time=("Minute", "max"),
    )
    .reset_index()
    .sort_values(
        ["Start_Time", "Segment_ID"]
    )
    .reset_index(drop=True)
)

ordered_segments = (
    segment_time_table["Segment_ID"]
    .astype(int)
    .tolist()
)

print(
    segment_time_table.to_string(
        index=False
    )
)

print(
    "\nChronological segment order:",
    ordered_segments
)


# ==================================================================================================
# 25. PARTITION SUMMARY HELPER
# ==================================================================================================

def summarize_partition(df, name):

    attack = df.loc[
        df["Future_Attack_State"] == 1
    ]

    per_horizon_attack = {
        int(k): int(v)
        for k, v in (
            df.groupby("Horizon_Minutes")[
                "Future_Attack_State"
            ]
            .sum()
            .to_dict()
        ).items()
    }

    per_horizon_rows = {
        int(k): int(v)
        for k, v in (
            df.groupby("Horizon_Minutes")
            .size()
            .to_dict()
        ).items()
    }

    return {
        "name": name,

        "rows": int(len(df)),

        "origins": int(
            df[
                ["Segment_ID", "Origin_Time"]
            ]
            .drop_duplicates()
            .shape[0]
        ),

        "attack_rows": int(len(attack)),

        "benign_rows": int(
            len(df) - len(attack)
        ),

        "attack_types":
            metadata_set(
                attack,
                "Future_Attack_Types"
            ),

        "sources":
            metadata_set(
                attack,
                "Future_Attack_Sources"
            ),

        "destinations":
            metadata_set(
                attack,
                "Future_Attack_Destinations"
            ),

        "edges":
            metadata_set(
                attack,
                "Future_Attack_Edges"
            ),

        "typed_edges":
            metadata_set(
                attack,
                "Future_Typed_Attack_Edges"
            ),

        "per_horizon_attack":
            per_horizon_attack,

        "per_horizon_rows":
            per_horizon_rows,
    }


# ==================================================================================================
# 26. EXHAUSTIVE CONTIGUOUS CHRONOLOGICAL 3-WAY SPLIT AUDIT
# ==================================================================================================
#
# Every candidate is:
#
#   TRAIN = chronological prefix of whole segments
#   VAL   = chronological middle block of whole segments
#   TEST  = chronological suffix of whole segments
#
# No final split is selected here.
#
# ==================================================================================================

print("\n" + LINE)
print("EXHAUSTIVE CHRONOLOGICAL TRAIN / VALIDATION / TEST FEASIBILITY")
print(LINE)

if len(ordered_segments) < 3:
    raise RuntimeError(
        "At least 3 temporal segments are required."
    )

global_attack_types = set(
    global_type_counter.keys()
)

split_candidate_rows = []

n_segments = len(ordered_segments)

for train_end_idx in range(
    1,
    n_segments - 1
):

    for val_end_idx in range(
        train_end_idx + 1,
        n_segments
    ):

        train_segments = (
            ordered_segments[:train_end_idx]
        )

        val_segments = (
            ordered_segments[
                train_end_idx:val_end_idx
            ]
        )

        test_segments = (
            ordered_segments[val_end_idx:]
        )

        train_df = forecast_meta.loc[
            forecast_meta["Segment_ID"]
            .isin(train_segments)
        ]

        val_df = forecast_meta.loc[
            forecast_meta["Segment_ID"]
            .isin(val_segments)
        ]

        test_df = forecast_meta.loc[
            forecast_meta["Segment_ID"]
            .isin(test_segments)
        ]

        train_summary = summarize_partition(
            train_df,
            "TRAIN"
        )

        val_summary = summarize_partition(
            val_df,
            "VALIDATION"
        )

        test_summary = summarize_partition(
            test_df,
            "TEST"
        )

        # ------------------------------------------------------------------------------------------
        # ATTACK TYPE COVERAGE
        # ------------------------------------------------------------------------------------------

        unseen_val_types = (
            val_summary["attack_types"]
            - train_summary["attack_types"]
        )

        unseen_test_types = (
            test_summary["attack_types"]
            - train_summary["attack_types"]
        )

        unseen_test_types_vs_trainval = (
            test_summary["attack_types"]
            - (
                train_summary["attack_types"]
                | val_summary["attack_types"]
            )
        )

        missing_global_types_from_train = (
            global_attack_types
            - train_summary["attack_types"]
        )

        # ------------------------------------------------------------------------------------------
        # GRAPH ENTITY COVERAGE
        # ------------------------------------------------------------------------------------------

        unseen_val_sources = (
            val_summary["sources"]
            - train_summary["sources"]
        )

        unseen_test_sources = (
            test_summary["sources"]
            - train_summary["sources"]
        )

        unseen_val_destinations = (
            val_summary["destinations"]
            - train_summary["destinations"]
        )

        unseen_test_destinations = (
            test_summary["destinations"]
            - train_summary["destinations"]
        )

        unseen_val_edges = (
            val_summary["edges"]
            - train_summary["edges"]
        )

        unseen_test_edges = (
            test_summary["edges"]
            - train_summary["edges"]
        )

        unseen_val_typed_edges = (
            val_summary["typed_edges"]
            - train_summary["typed_edges"]
        )

        unseen_test_typed_edges = (
            test_summary["typed_edges"]
            - train_summary["typed_edges"]
        )

        # ------------------------------------------------------------------------------------------
        # PER-HORIZON ATTACK SUPPORT
        # ------------------------------------------------------------------------------------------

        train_missing_attack_horizons = [
            int(h)
            for h in HORIZONS
            if train_summary[
                "per_horizon_attack"
            ].get(int(h), 0) == 0
        ]

        val_missing_attack_horizons = [
            int(h)
            for h in HORIZONS
            if val_summary[
                "per_horizon_attack"
            ].get(int(h), 0) == 0
        ]

        test_missing_attack_horizons = [
            int(h)
            for h in HORIZONS
            if test_summary[
                "per_horizon_attack"
            ].get(int(h), 0) == 0
        ]

        # ------------------------------------------------------------------------------------------
        # BASIC FEASIBILITY
        # ------------------------------------------------------------------------------------------

        nonempty_partitions = (
            len(train_df) > 0
            and len(val_df) > 0
            and len(test_df) > 0
        )

        all_partitions_have_attack = (
            train_summary["attack_rows"] > 0
            and val_summary["attack_rows"] > 0
            and test_summary["attack_rows"] > 0
        )

        type_coverage_safe = (
            len(unseen_val_types) == 0
            and len(unseen_test_types) == 0
        )

        all_horizons_have_attack = (
            len(train_missing_attack_horizons) == 0
            and len(val_missing_attack_horizons) == 0
            and len(test_missing_attack_horizons) == 0
        )

        strict_type_safe = (
            nonempty_partitions
            and all_partitions_have_attack
            and type_coverage_safe
            and all_horizons_have_attack
        )

        # ------------------------------------------------------------------------------------------
        # ROW
        # ------------------------------------------------------------------------------------------

        split_candidate_rows.append({

            "Train_End_After_Segment":
                int(train_segments[-1]),

            "Validation_End_After_Segment":
                int(val_segments[-1]),

            "Train_Segments":
                ",".join(map(str, train_segments)),

            "Validation_Segments":
                ",".join(map(str, val_segments)),

            "Test_Segments":
                ",".join(map(str, test_segments)),

            "Train_Origins":
                train_summary["origins"],

            "Validation_Origins":
                val_summary["origins"],

            "Test_Origins":
                test_summary["origins"],

            "Train_Attack_Rows":
                train_summary["attack_rows"],

            "Validation_Attack_Rows":
                val_summary["attack_rows"],

            "Test_Attack_Rows":
                test_summary["attack_rows"],

            "Train_Attack_Types":
                len(train_summary["attack_types"]),

            "Validation_Attack_Types":
                len(val_summary["attack_types"]),

            "Test_Attack_Types":
                len(test_summary["attack_types"]),

            "Missing_Global_Types_From_Train":
                len(missing_global_types_from_train),

            "Missing_Global_Type_List":
                ", ".join(
                    sorted(
                        map(
                            str,
                            missing_global_types_from_train
                        )
                    )
                ),

            "Unseen_Validation_Types":
                len(unseen_val_types),

            "Unseen_Validation_Type_List":
                ", ".join(
                    sorted(
                        map(str, unseen_val_types)
                    )
                ),

            "Unseen_Test_Types":
                len(unseen_test_types),

            "Unseen_Test_Type_List":
                ", ".join(
                    sorted(
                        map(str, unseen_test_types)
                    )
                ),

            "Unseen_Test_Types_vs_TrainVal":
                len(unseen_test_types_vs_trainval),

            "Unseen_Validation_Sources":
                len(unseen_val_sources),

            "Unseen_Test_Sources":
                len(unseen_test_sources),

            "Unseen_Validation_Destinations":
                len(unseen_val_destinations),

            "Unseen_Test_Destinations":
                len(unseen_test_destinations),

            "Unseen_Validation_Edges":
                len(unseen_val_edges),

            "Unseen_Test_Edges":
                len(unseen_test_edges),

            "Unseen_Validation_Typed_Edges":
                len(unseen_val_typed_edges),

            "Unseen_Test_Typed_Edges":
                len(unseen_test_typed_edges),

            "Train_Missing_Attack_Horizons":
                ",".join(
                    map(
                        str,
                        train_missing_attack_horizons
                    )
                ),

            "Validation_Missing_Attack_Horizons":
                ",".join(
                    map(
                        str,
                        val_missing_attack_horizons
                    )
                ),

            "Test_Missing_Attack_Horizons":
                ",".join(
                    map(
                        str,
                        test_missing_attack_horizons
                    )
                ),

            "All_Partitions_Have_Attack":
                bool(all_partitions_have_attack),

            "Type_Coverage_Safe":
                bool(type_coverage_safe),

            "All_Horizons_Have_Attack":
                bool(all_horizons_have_attack),

            "Strict_Type_Safe":
                bool(strict_type_safe),
        })


cell13_split_feasibility = pd.DataFrame(
    split_candidate_rows
)

print(
    f"Candidate chronological 3-way splits examined: "
    f"{len(cell13_split_feasibility):,}"
)


# ==================================================================================================
# 27. STRICT TYPE-SAFE CANDIDATES
# ==================================================================================================

print("\n" + LINE)
print("STRICT TYPE-SAFE CHRONOLOGICAL SPLIT CANDIDATES")
print(LINE)

type_safe_candidates = (
    cell13_split_feasibility.loc[
        cell13_split_feasibility[
            "Strict_Type_Safe"
        ]
    ]
    .copy()
)

candidate_display_columns = [
    "Train_Segments",
    "Validation_Segments",
    "Test_Segments",

    "Train_Origins",
    "Validation_Origins",
    "Test_Origins",

    "Train_Attack_Rows",
    "Validation_Attack_Rows",
    "Test_Attack_Rows",

    "Train_Attack_Types",
    "Validation_Attack_Types",
    "Test_Attack_Types",

    "Unseen_Validation_Types",
    "Unseen_Test_Types",

    "Unseen_Validation_Destinations",
    "Unseen_Test_Destinations",

    "Unseen_Validation_Edges",
    "Unseen_Test_Edges",
]

if len(type_safe_candidates):

    print(
        type_safe_candidates[
            candidate_display_columns
        ].to_string(
            index=False
        )
    )

else:

    print(
        "No strict type-safe contiguous chronological "
        "segment-only 3-way split exists."
    )

    print()
    print(
        "This is a DATASET STRUCTURE RESULT, not a code failure."
    )

    print(
        "Cell 14 must account for this explicitly rather "
        "than silently using a random row split."
    )


# ==================================================================================================
# 28. COVERAGE-RISK DIAGNOSTIC
# ==================================================================================================
#
# NOT a final ranking or selected split.
#
# It simply exposes candidates with fewer coverage problems so Cell 14
# can inspect the actual trade-offs.
#
# ==================================================================================================

print("\n" + LINE)
print("COVERAGE-RISK DIAGNOSTIC")
print(LINE)

coverage_diagnostic = (
    cell13_split_feasibility.copy()
)

coverage_diagnostic["_Type_Risk"] = (
    coverage_diagnostic["Unseen_Validation_Types"]
    + coverage_diagnostic["Unseen_Test_Types"]
)

coverage_diagnostic["_Destination_Risk"] = (
    coverage_diagnostic[
        "Unseen_Validation_Destinations"
    ]
    + coverage_diagnostic[
        "Unseen_Test_Destinations"
    ]
)

coverage_diagnostic["_Edge_Risk"] = (
    coverage_diagnostic[
        "Unseen_Validation_Edges"
    ]
    + coverage_diagnostic[
        "Unseen_Test_Edges"
    ]
)

coverage_diagnostic["_Typed_Edge_Risk"] = (
    coverage_diagnostic[
        "Unseen_Validation_Typed_Edges"
    ]
    + coverage_diagnostic[
        "Unseen_Test_Typed_Edges"
    ]
)

coverage_diagnostic = (
    coverage_diagnostic
    .sort_values(
        [
            "_Type_Risk",
            "_Destination_Risk",
            "_Edge_Risk",
            "_Typed_Edge_Risk",
        ],
        ascending=True
    )
    .reset_index(drop=True)
)

diagnostic_columns = [
    "Train_Segments",
    "Validation_Segments",
    "Test_Segments",

    "Train_Origins",
    "Validation_Origins",
    "Test_Origins",

    "Train_Attack_Rows",
    "Validation_Attack_Rows",
    "Test_Attack_Rows",

    "Missing_Global_Types_From_Train",

    "Unseen_Validation_Types",
    "Unseen_Test_Types",

    "Unseen_Validation_Destinations",
    "Unseen_Test_Destinations",

    "Unseen_Validation_Edges",
    "Unseen_Test_Edges",

    "Unseen_Validation_Typed_Edges",
    "Unseen_Test_Typed_Edges",

    "Strict_Type_Safe",
]

print(
    coverage_diagnostic[
        diagnostic_columns
    ]
    .head(15)
    .to_string(
        index=False
    )
)


# ==================================================================================================
# 29. ORIGIN-LEVEL MULTI-HORIZON INTEGRITY
# ==================================================================================================

print("\n" + LINE)
print("ORIGIN-LEVEL MULTI-HORIZON INTEGRITY")
print(LINE)

bad_origin_count = int(
    (
        (origin_horizon_counts["count"] != len(HORIZONS))
        |
        (origin_horizon_counts["nunique"] != len(HORIZONS))
    ).sum()
)

print(
    f"Origins with incomplete/duplicate horizon sets : "
    f"{bad_origin_count:,}"
)

if bad_origin_count:
    raise RuntimeError(
        "Origin-level multi-horizon integrity failed."
    )

print(
    "✓ Every forecasting origin remains one indivisible "
    "multi-horizon unit."
)


# ==================================================================================================
# 30. PRE-SPLIT STRUCTURAL RISK FLAGS
# ==================================================================================================

print("\n" + LINE)
print("PRE-SPLIT STRUCTURAL RISK FLAGS")
print(LINE)

risk_flags = []

zero_attack_segments = (
    segment_attack_summary.loc[
        segment_attack_summary["Attack_Minutes"] == 0,
        "Segment_ID"
    ]
    .astype(int)
    .tolist()
)

single_segment_types = {
    attack_type: sorted(segs)
    for attack_type, segs
    in type_segment_map.items()
    if len(segs) == 1
}

if zero_attack_segments:
    risk_flags.append(
        "SEGMENTS_WITH_NO_ATTACK_MINUTES"
    )

if single_segment_types:
    risk_flags.append(
        "ATTACK_TYPES_CONFINED_TO_SINGLE_SEGMENT"
    )

if len(type_safe_candidates) == 0:
    risk_flags.append(
        "NO_STRICT_TYPE_SAFE_CONTIGUOUS_CHRONOLOGICAL_3WAY_SPLIT"
    )

print(
    "Segments with zero attack minutes :",
    zero_attack_segments
)

print(
    "Attack types confined to one segment:",
    single_segment_types
)

print()

if risk_flags:

    print("Structural risk flags:")

    for flag in risk_flags:
        print("  -", flag)

else:

    print(
        "No major structural pre-split risk flags detected."
    )


# ==================================================================================================
# 31. CELL 14 SPLIT CONTRACT
# ==================================================================================================

print("\n" + LINE)
print("SPLIT-DESIGN CONTRACT FOR CELL 14")
print(LINE)

print(
"""
Cell 14 must obey these rules:

1. The atomic split unit is a forecasting ORIGIN.

2. All horizon rows for one origin:
      +1
      +5
      +10
      +15
      +30
   must remain in exactly the same partition.

3. Individual horizon rows must never be randomly split.

4. Capture-segment boundaries must remain respected.

5. No scaler may be fitted before the training partition exists.

6. No resampling may occur before the training partition exists.

7. Validation/test attack types absent from training must be explicitly
   identified rather than silently treated as normal closed-set cases.

8. Source, destination, edge and typed-edge novelty must be reported
   separately from attack-type novelty.

9. The final split must be checked for overlapping temporal information.

10. If Cell 14 ever introduces a boundary inside one continuous capture
    segment, it must use an appropriate purge/embargo based on the
    15-minute history and maximum 30-minute forecast horizon.

11. Cell 14 must audit the final selected split after constructing it.

12. Cell 13 does NOT select the final split.
"""
)


# ==================================================================================================
# 32. EXPORT METADATA
# ==================================================================================================

cell13_metadata = {

    "forecast_horizons":
        list(map(int, HORIZONS)),

    "forecast_origins":
        int(n_origins),

    "forecast_rows":
        int(actual_rows),

    "segments":
        list(map(int, ordered_segments)),

    "duplicate_forecast_keys":
        int(duplicate_forecast_keys),

    "target_alignment_errors":
        int(target_alignment_errors),

    "segment_boundary_violations":
        int(segment_boundary_violations),

    "missing_target_minutes":
        int(missing_target_minutes),

    "binary_target_mismatches":
        int(binary_target_mismatches),

    "critical_metadata_errors":
        int(critical_metadata_errors),

    "graph_structure_errors":
        int(graph_structure_errors),

    "canonical_onset_errors":
        int(canonical_onset_errors),

    "attack_targets":
        int(attack_mask.sum()),

    "benign_targets":
        int(benign_mask.sum()),

    "unique_attack_types":
        int(len(global_type_counter)),

    "unique_attack_sources":
        int(len(all_sources)),

    "unique_attack_destinations":
        int(len(all_destinations)),

    "unique_attack_edges":
        int(len(all_edges)),

    "unique_typed_attack_edges":
        int(len(all_typed_edges)),

    "chronological_split_candidates":
        int(len(cell13_split_feasibility)),

    "strict_type_safe_candidates":
        int(len(type_safe_candidates)),

    "zero_attack_segments":
        zero_attack_segments,

    "single_segment_attack_types":
        single_segment_types,

    "risk_flags":
        risk_flags,

    "split_created":
        False,

    "scaling_performed":
        False,

    "resampling_performed":
        False,

    "model_trained":
        False,
}


# ==================================================================================================
# 33. FINAL SUMMARY
# ==================================================================================================

print("\n" + LINE)
print("CELL 13 — FINAL VALIDATION SUMMARY")
print(LINE)

print(f"Forecast origins                  : {n_origins:,}")
print(f"Forecast rows                     : {actual_rows:,}")
print(f"Forecast horizons                 : {HORIZONS}")

print(
    f"Attack forecast rows              : "
    f"{int(attack_mask.sum()):,}"
)

print(
    f"Benign forecast rows              : "
    f"{int(benign_mask.sum()):,}"
)

print(
    f"Unique attack types               : "
    f"{len(global_type_counter):,}"
)

print(
    f"Unique attack sources             : "
    f"{len(all_sources):,}"
)

print(
    f"Unique attack destinations        : "
    f"{len(all_destinations):,}"
)

print(
    f"Unique attack edges               : "
    f"{len(all_edges):,}"
)

print(
    f"Unique typed attack edges         : "
    f"{len(all_typed_edges):,}"
)

print(
    f"Forecast-contract errors          : "
    f"{contract_errors:,}"
)

print(
    f"Binary target mismatches          : "
    f"{binary_target_mismatches:,}"
)

print(
    f"Rich metadata errors              : "
    f"{critical_metadata_errors:,}"
)

print(
    f"Graph consistency errors          : "
    f"{graph_structure_errors:,}"
)

print(
    f"Canonical onset partition errors  : "
    f"{canonical_onset_errors:,}"
)

print(
    f"Chronological split candidates    : "
    f"{len(cell13_split_feasibility):,}"
)

print(
    f"Strict type-safe candidates       : "
    f"{len(type_safe_candidates):,}"
)

print(
    f"Structural risk flags             : "
    f"{risk_flags}"
)

print()
print("✓ Forecasting contract audited.")
print("✓ Exact target timestamps audited.")
print("✓ Binary ground truth audited.")
print("✓ Rich attack targets audited.")
print("✓ Graph ground truth audited.")
print("✓ Canonical onset/continuation partition audited.")
print("✓ No unsupported Current_State reinterpretation performed.")
print("✓ Attack-type support audited.")
print("✓ Source/destination support audited.")
print("✓ Edge and typed-edge support audited.")
print("✓ Segment concentration audited.")
print("✓ Descriptive episode structure audited.")
print("✓ Every contiguous chronological 3-way segment split examined.")
print("✓ Unseen validation/test attack types reported.")
print("✓ Unseen validation/test graph entities reported.")
print("✓ Per-horizon attack support reported.")
print("✓ Multi-horizon origins remain indivisible.")
print("✓ NO final split created.")
print("✓ NO scaler fitted.")
print("✓ NO resampling performed.")
print("✓ NO model trained.")

print("\nObjects created:")
print("  cell13_target_consistency")
print("  cell13_horizon_distribution")
print("  cell13_attack_type_by_horizon")
print("  cell13_attack_type_support")
print("  cell13_graph_support")
print("  segment_attack_summary")
print("  cell13_segment_horizon_support")
print("  cell13_segment_type_support")
print("  cell13_attack_type_concentration")
print("  cell13_episode_gap_summary")
print("  episode_tables")
print("  segment_time_table")
print("  cell13_split_feasibility")
print("  type_safe_candidates")
print("  coverage_diagnostic")
print("  cell13_metadata")

print()
print("NEXT:")
print(
    "Use Cell 13's observed split-feasibility results to design "
    "the final leakage-safe split in Cell 14."
)

print(LINE)

CELL 13 — FINAL PRE-SPLIT FORECASTABILITY + TEMPORAL STRUCTURE + SPLIT-FEASIBILITY AUDIT

Configuration
----------------------------------------------------------------------------------------------------------------------------------------------------------------
Episode-gap candidates        : [0, 1, 2, 5] minutes
Split created                 : NO
Scaling performed             : NO
Resampling performed          : NO
Model trained                 : NO

This cell is diagnostic only.

✓ Required canonical objects found.
minute_df rows                : 2,454
cell12_forecast_metadata rows : 10,070
✓ Canonical Cell 9 / Cell 12 schema validated.

BINARY TARGET VALIDATION
✓ All binary state columns contain only 0/1.

FINAL INPUT SCHEMA
Minute rows                : 2,454
Minute columns             : 103
Forecast rows              : 10,070
Forecast columns           : 29
Segments                   : 10
Forecast horizons          : [1, 5, 10, 15, 30]

FORECASTING CONTRACT VALIDATION
Forecast o

/tmp/ipykernel_396/1775437421.py:228: DeprecationWarning: The truth value of an empty array is ambiguous. Returning False, but in future this will result in an error. Use `array.size > 0` to check that an array is not empty.
  if pd.isna(value):
/tmp/ipykernel_396/1775437421.py:228: DeprecationWarning: The truth value of an empty array is ambiguous. Returning False, but in future this will result in an error. Use `array.size > 0` to check that an array is not empty.
  if pd.isna(value):
/tmp/ipykernel_396/1775437421.py:228: DeprecationWarning: The truth value of an empty array is ambiguous. Returning False, but in future this will result in an error. Use `array.size > 0` to check that an array is not empty.
  if pd.isna(value):
/tmp/ipykernel_396/1775437421.py:228: DeprecationWarning: The truth value of an empty array is ambiguous. Returning False, but in future this will result in an error. Use `array.size > 0` to check that an array is not empty.
  if pd.isna(value):
/tmp/ipykernel_3

                Collection                    Count_Column  Missing_Count_On_Attack  Attack_Count_Mismatches  Benign_Nonempty_Collections  Attack_Empty_Collections
       Future_Attack_Types        Future_Attack_Type_Count                        0                        0                            0                         0
     Future_Attack_Sources      Future_Attack_Source_Count                        0                        0                            0                         0
Future_Attack_Destinations Future_Attack_Destination_Count                        0                        0                            0                         0
       Future_Attack_Edges        Future_Attack_Edge_Count                        0                        0                            0                         0
 Future_Typed_Attack_Edges  Future_Typed_Attack_Edge_Count                        0                        0                            0                         0

✓ Rich attack m

In [24]:
# ==================================================================================================
# CELL 14 — FINAL LEAKAGE-SAFE DEVELOPMENT SPLIT + STRICT CHRONOLOGICAL HOLDOUT
# ==================================================================================================
#
# PURPOSE
# -------
# Construct the modelling partitions AFTER the complete pre-split audit in Cell 13.
#
# TWO DISTINCT EVALUATION PROTOCOLS ARE CREATED:
#
#   PROTOCOL A — DEVELOPMENT / CLOSED-SET-ORIENTED SPLIT
#   ---------------------------------------------------
#   • Each capture segment is divided chronologically into:
#         TRAIN -> PURGE -> VALIDATION -> PURGE -> TEST
#
#   • Forecast ORIGINS are the atomic unit.
#   • All five horizons belonging to one origin remain together.
#   • No row-level random split is used.
#   • No origin is duplicated between partitions.
#   • A full history + maximum-horizon temporal footprint is respected.
#
#   IMPORTANT:
#   This split does NOT artificially move attack rows or attack types between
#   partitions. Therefore some attack types may still be absent from validation
#   or test. That limitation is measured and reported rather than hidden.
#
#
#   PROTOCOL B — STRICT CHRONOLOGICAL FUTURE HOLDOUT
#   ------------------------------------------------
#   • Whole capture segments only.
#   • Earliest segments = train.
#   • Next segment(s) = validation.
#   • Latest segment(s) = test.
#   • No within-segment mixing.
#
#   This protocol is retained for genuine future/generalization evaluation,
#   including unseen attack types / sources / destinations / edges.
#
#
# THIS CELL DOES NOT:
#   • scale features
#   • fit a scaler
#   • resample
#   • SMOTE
#   • train a model
#
# ==================================================================================================

import ast
import json
import numpy as np
import pandas as pd
from collections import Counter, defaultdict

pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 260)
pd.set_option("display.max_colwidth", 140)

SEP = "=" * 170
SUB = "-" * 170

print(SEP)
print("CELL 14 — FINAL LEAKAGE-SAFE DEVELOPMENT SPLIT + STRICT CHRONOLOGICAL HOLDOUT")
print(SEP)


# ==================================================================================================
# 1. CONFIGURATION
# ==================================================================================================

HISTORY_MINUTES = 15
EXPECTED_HORIZONS = [1, 5, 10, 15, 30]
MAX_HORIZON = max(EXPECTED_HORIZONS)

# Historical footprint:
#
# history starts at origin - (HISTORY_MINUTES - 1)
# future target extends to origin + MAX_HORIZON
#
# Therefore two neighbouring partitions must not use overlapping raw minutes.
LEFT_HISTORY_SPAN = HISTORY_MINUTES - 1
RIGHT_FUTURE_SPAN = MAX_HORIZON

# Development split proportions INSIDE each segment.
#
# These proportions are applied to the AVAILABLE ORIGIN TIMELINE,
# not directly to individual forecast rows.
DEV_TRAIN_FRACTION = 0.60
DEV_VAL_FRACTION = 0.20
DEV_TEST_FRACTION = 0.20

# Purge between development partitions.
#
# If train's final origin is T,
# validation's first origin must be sufficiently later that:
#
#   validation_history_start > train_max_target_time
#
# Minimum origin-to-origin separation:
#     (HISTORY_MINUTES - 1) + MAX_HORIZON + 1
#
# = 14 + 30 + 1 = 45 minutes.
MIN_ORIGIN_BOUNDARY_SEPARATION = (
    LEFT_HISTORY_SPAN
    + RIGHT_FUTURE_SPAN
    + 1
)

# Strict chronological protocol.
#
# We DO NOT claim this is closed-set.
#
# With 10 segments:
#     first 60% -> train
#     next 20%  -> validation
#     final 20% -> test
#
STRICT_TRAIN_FRACTION = 0.60
STRICT_VAL_FRACTION = 0.20

print("\nConfiguration")
print(SUB)
print(f"History length                         : {HISTORY_MINUTES} minutes")
print(f"Forecast horizons                      : {EXPECTED_HORIZONS}")
print(f"Maximum forecast horizon               : {MAX_HORIZON} minutes")
print(f"History span before origin             : {LEFT_HISTORY_SPAN} minutes")
print(f"Minimum inter-partition origin gap     : {MIN_ORIGIN_BOUNDARY_SEPARATION} minutes")
print()
print("Development protocol")
print(f"  Train fraction                       : {DEV_TRAIN_FRACTION:.2f}")
print(f"  Validation fraction                  : {DEV_VAL_FRACTION:.2f}")
print(f"  Test fraction                        : {DEV_TEST_FRACTION:.2f}")
print()
print("Strict chronological protocol")
print(f"  Train fraction                       : {STRICT_TRAIN_FRACTION:.2f}")
print(f"  Validation fraction                  : {STRICT_VAL_FRACTION:.2f}")
print(f"  Test fraction                        : {1 - STRICT_TRAIN_FRACTION - STRICT_VAL_FRACTION:.2f}")
print()
print("Scaling performed                      : NO")
print("Resampling performed                   : NO")
print("Model trained                          : NO")


# ==================================================================================================
# 2. REQUIRED UPSTREAM OBJECTS
# ==================================================================================================

required_objects = [
    "minute_df",
    "cell12_forecast_metadata",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Cell 14 cannot run because required upstream objects are missing:\n"
        + "\n".join(f"  - {x}" for x in missing_objects)
    )

minute_meta = minute_df.copy()
forecast_meta = cell12_forecast_metadata.copy()

print("\n" + SEP)
print("UPSTREAM OBJECT VALIDATION")
print(SEP)

print(f"minute_df rows                  : {len(minute_meta):,}")
print(f"cell12_forecast_metadata rows   : {len(forecast_meta):,}")


# ==================================================================================================
# 3. CANONICAL COLUMN CONTRACT
# ==================================================================================================

required_forecast_columns = [
    "Segment_ID",
    "History_Start",
    "History_End",
    "Origin_Time",
    "Horizon_Minutes",
    "Target_Time",
    "Current_State",
    "Attack_In_History",
    "Clean_History",
    "Attack_Minutes_In_History",
    "Future_Attack_State",
    "Future_Attack_Types",
    "Future_Attack_Sources",
    "Future_Attack_Destinations",
    "Future_Attack_Edges",
    "Future_Typed_Attack_Edges",
    "Future_Attack_Type_Count",
    "Future_Attack_Source_Count",
    "Future_Attack_Destination_Count",
    "Future_Attack_Edge_Count",
    "Future_Typed_Attack_Edge_Count",
    "New_Onset",
    "Continuation",
]

missing_forecast_columns = [
    c
    for c in required_forecast_columns
    if c not in forecast_meta.columns
]

if missing_forecast_columns:
    raise RuntimeError(
        "Cell 12 forecast metadata is missing canonical columns:\n"
        + "\n".join(f"  - {x}" for x in missing_forecast_columns)
    )

required_minute_columns = [
    "Segment_ID",
    "Minute",
    "Attack_State",
]

missing_minute_columns = [
    c
    for c in required_minute_columns
    if c not in minute_meta.columns
]

if missing_minute_columns:
    raise RuntimeError(
        "minute_df is missing required columns:\n"
        + "\n".join(f"  - {x}" for x in missing_minute_columns)
    )

print("✓ Canonical Cell 12 forecasting schema found.")
print("✓ Canonical Cell 9 minute-level schema found.")


# ==================================================================================================
# 4. BASIC NORMALIZATION
# ==================================================================================================

datetime_columns = [
    "History_Start",
    "History_End",
    "Origin_Time",
    "Target_Time",
]

for col in datetime_columns:
    forecast_meta[col] = pd.to_datetime(
        forecast_meta[col],
        errors="raise",
    )

minute_meta["Minute"] = pd.to_datetime(
    minute_meta["Minute"],
    errors="raise",
)

forecast_meta["Segment_ID"] = pd.to_numeric(
    forecast_meta["Segment_ID"],
    errors="raise",
).astype(int)

forecast_meta["Horizon_Minutes"] = pd.to_numeric(
    forecast_meta["Horizon_Minutes"],
    errors="raise",
).astype(int)

binary_columns = [
    "Current_State",
    "Attack_In_History",
    "Clean_History",
    "Future_Attack_State",
    "New_Onset",
    "Continuation",
]

for col in binary_columns:
    forecast_meta[col] = pd.to_numeric(
        forecast_meta[col],
        errors="raise",
    ).astype(int)

minute_meta["Segment_ID"] = pd.to_numeric(
    minute_meta["Segment_ID"],
    errors="raise",
).astype(int)

minute_meta["Attack_State"] = pd.to_numeric(
    minute_meta["Attack_State"],
    errors="raise",
).astype(int)


# ==================================================================================================
# 5. COLLECTION NORMALIZATION
# ==================================================================================================

def normalize_collection(value):
    """
    Convert canonical collection metadata into a Python list.

    Cell 12 should already contain proper collections, but this function
    safely handles JSON/string representations if they are encountered.
    """

    if value is None:
        return []

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, (list, tuple, set)):
        return list(value)

    try:
        if pd.isna(value):
            return []
    except Exception:
        pass

    if isinstance(value, str):
        s = value.strip()

        if s == "" or s.lower() in {
            "none",
            "nan",
            "null",
            "<na>",
        }:
            return []

        try:
            parsed = json.loads(s)

            if isinstance(parsed, list):
                return parsed

        except Exception:
            pass

        try:
            parsed = ast.literal_eval(s)

            if isinstance(parsed, (list, tuple, set)):
                return list(parsed)

        except Exception:
            pass

        return [s]

    return [value]


collection_columns = [
    "Future_Attack_Types",
    "Future_Attack_Sources",
    "Future_Attack_Destinations",
    "Future_Attack_Edges",
    "Future_Typed_Attack_Edges",
]

for col in collection_columns:
    forecast_meta[col] = (
        forecast_meta[col]
        .apply(normalize_collection)
    )

print("\n✓ Forecast collection metadata normalized.")


# ==================================================================================================
# 6. FORECASTING CONTRACT REVALIDATION
# ==================================================================================================

print("\n" + SEP)
print("FORECASTING CONTRACT REVALIDATION")
print(SEP)

actual_horizons = sorted(
    forecast_meta["Horizon_Minutes"]
    .unique()
    .tolist()
)

if actual_horizons != EXPECTED_HORIZONS:
    raise RuntimeError(
        f"Unexpected forecast horizons.\n"
        f"Expected: {EXPECTED_HORIZONS}\n"
        f"Found   : {actual_horizons}"
    )

duplicate_keys = int(
    forecast_meta.duplicated(
        subset=[
            "Segment_ID",
            "Origin_Time",
            "Horizon_Minutes",
        ]
    ).sum()
)

expected_target_time = (
    forecast_meta["Origin_Time"]
    + pd.to_timedelta(
        forecast_meta["Horizon_Minutes"],
        unit="m",
    )
)

target_alignment_errors = int(
    (
        forecast_meta["Target_Time"]
        != expected_target_time
    ).sum()
)

origin_horizon_counts = (
    forecast_meta
    .groupby(
        ["Segment_ID", "Origin_Time"]
    )["Horizon_Minutes"]
    .nunique()
)

bad_origin_horizon_counts = int(
    (
        origin_horizon_counts
        != len(EXPECTED_HORIZONS)
    ).sum()
)

print(f"Duplicate forecast keys              : {duplicate_keys:,}")
print(f"Target-time alignment errors         : {target_alignment_errors:,}")
print(f"Origins missing one/more horizons    : {bad_origin_horizon_counts:,}")

if duplicate_keys:
    raise RuntimeError(
        "Duplicate forecast keys exist."
    )

if target_alignment_errors:
    raise RuntimeError(
        "Target timestamps are misaligned."
    )

if bad_origin_horizon_counts:
    raise RuntimeError(
        "At least one origin does not contain all five horizons."
    )

print("✓ Forecasting contract remains valid.")


# ==================================================================================================
# 7. BUILD ONE ROW PER FORECAST ORIGIN
# ==================================================================================================

print("\n" + SEP)
print("BUILDING ATOMIC FORECAST-ORIGIN TABLE")
print(SEP)


def union_nested(series):
    """
    Union arbitrary scalar/list metadata into a tuple of unique values.
    """

    result = set()

    for value in series:

        values = normalize_collection(value)

        for item in values:

            if isinstance(item, list):
                item = tuple(item)

            if isinstance(item, tuple):
                result.add(tuple(item))
            else:
                result.add(item)

    return tuple(
        sorted(
            result,
            key=lambda x: str(x),
        )
    )


origin_base = (
    forecast_meta
    .sort_values(
        [
            "Segment_ID",
            "Origin_Time",
            "Horizon_Minutes",
        ]
    )
    .groupby(
        ["Segment_ID", "Origin_Time"],
        as_index=False,
    )
    .agg(
        History_Start=("History_Start", "first"),
        History_End=("History_End", "first"),
        Current_State=("Current_State", "first"),
        Attack_In_History=("Attack_In_History", "first"),
        Clean_History=("Clean_History", "first"),
        Attack_Minutes_In_History=("Attack_Minutes_In_History", "first"),
    )
)

# Aggregate future information ACROSS horizons only for split auditing.
#
# IMPORTANT:
# These are NOT model targets.
# They simply describe what appears anywhere in the five target horizons
# belonging to an origin.

origin_future_state = (
    forecast_meta
    .groupby(
        ["Segment_ID", "Origin_Time"],
        as_index=False,
    )
    .agg(
        Any_Future_Attack=("Future_Attack_State", "max"),
        Any_New_Onset=("New_Onset", "max"),
        Any_Continuation=("Continuation", "max"),
        Max_Target_Time=("Target_Time", "max"),
        Min_Target_Time=("Target_Time", "min"),
    )
)

origin_types = (
    forecast_meta
    .groupby(
        ["Segment_ID", "Origin_Time"]
    )["Future_Attack_Types"]
    .apply(union_nested)
    .rename("Future_Attack_Types_AnyHorizon")
    .reset_index()
)

origin_sources = (
    forecast_meta
    .groupby(
        ["Segment_ID", "Origin_Time"]
    )["Future_Attack_Sources"]
    .apply(union_nested)
    .rename("Future_Attack_Sources_AnyHorizon")
    .reset_index()
)

origin_destinations = (
    forecast_meta
    .groupby(
        ["Segment_ID", "Origin_Time"]
    )["Future_Attack_Destinations"]
    .apply(union_nested)
    .rename("Future_Attack_Destinations_AnyHorizon")
    .reset_index()
)

origin_edges = (
    forecast_meta
    .groupby(
        ["Segment_ID", "Origin_Time"]
    )["Future_Attack_Edges"]
    .apply(union_nested)
    .rename("Future_Attack_Edges_AnyHorizon")
    .reset_index()
)

origin_typed_edges = (
    forecast_meta
    .groupby(
        ["Segment_ID", "Origin_Time"]
    )["Future_Typed_Attack_Edges"]
    .apply(union_nested)
    .rename("Future_Typed_Attack_Edges_AnyHorizon")
    .reset_index()
)

origin_table = (
    origin_base
    .merge(
        origin_future_state,
        on=["Segment_ID", "Origin_Time"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        origin_types,
        on=["Segment_ID", "Origin_Time"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        origin_sources,
        on=["Segment_ID", "Origin_Time"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        origin_destinations,
        on=["Segment_ID", "Origin_Time"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        origin_edges,
        on=["Segment_ID", "Origin_Time"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        origin_typed_edges,
        on=["Segment_ID", "Origin_Time"],
        how="left",
        validate="one_to_one",
    )
)

origin_table["Footprint_Start"] = (
    origin_table["Origin_Time"]
    - pd.to_timedelta(
        LEFT_HISTORY_SPAN,
        unit="m",
    )
)

origin_table["Footprint_End"] = (
    origin_table["Origin_Time"]
    + pd.to_timedelta(
        RIGHT_FUTURE_SPAN,
        unit="m",
    )
)

origin_table = (
    origin_table
    .sort_values(
        ["Segment_ID", "Origin_Time"]
    )
    .reset_index(drop=True)
)

print(f"Atomic forecast origins : {len(origin_table):,}")

expected_origins = (
    len(forecast_meta)
    // len(EXPECTED_HORIZONS)
)

if len(origin_table) != expected_origins:
    raise RuntimeError(
        "Origin-table size does not match forecasting contract."
    )

print("✓ Exactly one origin-level record created per five-horizon forecasting sample.")


# ==================================================================================================
# 8. SEGMENT SUMMARY
# ==================================================================================================

print("\n" + SEP)
print("CAPTURE SEGMENT SUMMARY")
print(SEP)

segment_summary = (
    minute_meta
    .groupby(
        "Segment_ID",
        as_index=False,
    )
    .agg(
        Minute_Rows=("Minute", "size"),
        Segment_Start=("Minute", "min"),
        Segment_End=("Minute", "max"),
        Attack_Minutes=("Attack_State", "sum"),
    )
)

segment_summary["Benign_Minutes"] = (
    segment_summary["Minute_Rows"]
    - segment_summary["Attack_Minutes"]
)

origin_counts = (
    origin_table
    .groupby("Segment_ID")
    .size()
    .rename("Forecast_Origins")
    .reset_index()
)

segment_summary = (
    segment_summary
    .merge(
        origin_counts,
        on="Segment_ID",
        how="left",
    )
    .sort_values("Segment_ID")
    .reset_index(drop=True)
)

segment_summary["Forecast_Origins"] = (
    segment_summary["Forecast_Origins"]
    .fillna(0)
    .astype(int)
)

print(
    segment_summary.to_string(
        index=False
    )
)

segment_ids = (
    segment_summary["Segment_ID"]
    .astype(int)
    .tolist()
)

if len(segment_ids) < 3:
    raise RuntimeError(
        "At least three capture segments are required."
    )


# ==================================================================================================
# 9. RECONSTRUCT STRICT ATTACK EPISODES FOR BOUNDARY AUDITING
# ==================================================================================================
#
# IMPORTANT:
# This does NOT redefine forecasting labels.
#
# Episode = contiguous attack run at one-minute resolution.
#
# A new episode begins whenever:
#   • segment changes, OR
#   • there is at least one benign minute between attack minutes.
#
# ==================================================================================================

print("\n" + SEP)
print("STRICT ATTACK-EPISODE RECONSTRUCTION")
print(SEP)

attack_minutes = (
    minute_meta[
        minute_meta["Attack_State"].eq(1)
    ][
        ["Segment_ID", "Minute"]
    ]
    .drop_duplicates()
    .sort_values(
        ["Segment_ID", "Minute"]
    )
    .reset_index(drop=True)
)

episode_records = []
episode_id = 0

for seg, group in attack_minutes.groupby(
    "Segment_ID",
    sort=True,
):

    times = (
        group["Minute"]
        .sort_values()
        .tolist()
    )

    if not times:
        continue

    start = times[0]
    end = times[0]

    for ts in times[1:]:

        if ts - end == pd.Timedelta(minutes=1):
            end = ts

        else:
            episode_records.append({
                "Episode_ID": episode_id,
                "Segment_ID": int(seg),
                "Episode_Start": start,
                "Episode_End": end,
            })

            episode_id += 1
            start = ts
            end = ts

    episode_records.append({
        "Episode_ID": episode_id,
        "Segment_ID": int(seg),
        "Episode_Start": start,
        "Episode_End": end,
    })

    episode_id += 1

episode_table = pd.DataFrame(
    episode_records
)

if not episode_table.empty:

    episode_table["Duration_Minutes"] = (
        (
            episode_table["Episode_End"]
            - episode_table["Episode_Start"]
        ).dt.total_seconds()
        / 60.0
        + 1
    )

print(f"Strict attack episodes : {len(episode_table):,}")

if not episode_table.empty:

    print(
        episode_table
        .groupby("Segment_ID")
        .size()
        .rename("Episodes")
        .reset_index()
        .to_string(index=False)
    )


# ==================================================================================================
# 10. DEVELOPMENT SPLIT
# ==================================================================================================
#
# FINAL RULE:
#
# Each segment is initially divided chronologically:
#
#       TRAIN | VALIDATION | TEST
#
# Then TWO leakage barriers are enforced:
#
#   1. RAW-TEMPORAL FOOTPRINT ISOLATION
#      No 15-minute history / +30-minute future footprint may overlap.
#
#   2. ATTACK-EPISODE ISOLATION
#      A strict contiguous attack episode may not be touched by the
#      footprints of two different partitions.
#
# IMPORTANT:
# We NEVER move an attack sample from one partition to another because of
# its label/type.
#
# Boundary samples are PURGED instead.
#
# ==================================================================================================

print("\n" + SEP)
print("PROTOCOL A — DEVELOPMENT SPLIT CONSTRUCTION")
print(SEP)


def choose_initial_boundaries(n):
    """
    Initial chronological 60/20/20 split by forecast origin.
    """

    if n < 3:
        return None, None

    train_n = max(
        1,
        int(np.floor(n * DEV_TRAIN_FRACTION)),
    )

    val_n = max(
        1,
        int(np.floor(n * DEV_VAL_FRACTION)),
    )

    if train_n + val_n >= n:
        val_n = 1
        train_n = n - 2

    train_end_idx = train_n - 1
    val_end_idx = train_n + val_n - 1

    return train_end_idx, val_end_idx


# ==================================================================================================
# 10A. RAW-FOOTPRINT BOUNDARY PURGE
# ==================================================================================================

def purge_raw_boundary(left_df, right_df):
    """
    Purge boundary origins until:

        max(left Footprint_End)
            <
        min(right Footprint_Start)

    Chronology is preserved.

    IMPORTANT:
    We do NOT alternate arbitrary removals.

    The left partition keeps its earlier block.
    The right partition keeps its later block.

    Boundary origins are removed only.
    """

    left = (
        left_df
        .sort_values("Origin_Time")
        .reset_index(drop=True)
        .copy()
    )

    right = (
        right_df
        .sort_values("Origin_Time")
        .reset_index(drop=True)
        .copy()
    )

    purged = []

    while (
        len(left) > 0
        and len(right) > 0
        and left["Footprint_End"].max()
            >= right["Footprint_Start"].min()
    ):

        # Compare the two boundary origins.
        #
        # We remove whichever boundary origin produces the smaller
        # distortion while preserving chronological ordering.

        left_boundary = left.iloc[-1]
        right_boundary = right.iloc[0]

        left_remaining = len(left)
        right_remaining = len(right)

        # Prefer removing from the larger side.
        if left_remaining > right_remaining:

            purged.append(
                (
                    int(left_boundary["Segment_ID"]),
                    left_boundary["Origin_Time"],
                    "RAW_FOOTPRINT",
                )
            )

            left = (
                left.iloc[:-1]
                .reset_index(drop=True)
            )

        else:

            purged.append(
                (
                    int(right_boundary["Segment_ID"]),
                    right_boundary["Origin_Time"],
                    "RAW_FOOTPRINT",
                )
            )

            right = (
                right.iloc[1:]
                .reset_index(drop=True)
            )

    return left, right, purged


# ==================================================================================================
# 10B. EPISODE-TOUCH HELPERS
# ==================================================================================================

def touched_episode_ids(partition_df, segment_id):
    """
    Return attack episode IDs whose attack interval intersects at least
    one complete temporal footprint in partition_df.
    """

    if (
        partition_df.empty
        or episode_table.empty
    ):
        return set()

    part = (
        partition_df[
            partition_df["Segment_ID"].eq(segment_id)
        ]
        .copy()
    )

    eps = (
        episode_table[
            episode_table["Segment_ID"].eq(segment_id)
        ]
        .copy()
    )

    if part.empty or eps.empty:
        return set()

    touched = set()

    for ep in eps.itertuples(index=False):

        intersects = (
            (
                part["Footprint_Start"]
                <= ep.Episode_End
            )
            &
            (
                part["Footprint_End"]
                >= ep.Episode_Start
            )
        )

        if intersects.any():
            touched.add(
                int(ep.Episode_ID)
            )

    return touched


def get_episode_row(episode_id):

    row = (
        episode_table[
            episode_table[
                "Episode_ID"
            ].eq(episode_id)
        ]
    )

    if len(row) != 1:
        raise RuntimeError(
            f"Could not uniquely resolve Episode_ID={episode_id}"
        )

    return row.iloc[0]


# ==================================================================================================
# 10C. EPISODE-AWARE BOUNDARY PURGE
# ==================================================================================================

def purge_shared_episodes(
    left_df,
    right_df,
    left_name,
    right_name,
):
    """
    Ensure no strict attack episode is touched by both partitions.

    Strategy
    --------
    If an episode is touched by both sides, the episode is treated as a
    protected boundary region.

    We purge EVERY boundary origin whose full temporal footprint touches
    that shared episode.

    This deliberately sacrifices samples instead of allowing the same
    attack event to influence two evaluation partitions.

    After episode purging, raw-footprint purging is run again because
    episode removal can change the boundary.

    The procedure repeats until BOTH conditions hold:

        1. no shared episode
        2. no raw footprint overlap
    """

    left = (
        left_df
        .sort_values("Origin_Time")
        .reset_index(drop=True)
        .copy()
    )

    right = (
        right_df
        .sort_values("Origin_Time")
        .reset_index(drop=True)
        .copy()
    )

    purged = []

    max_iterations = 1000
    iteration = 0

    while True:

        iteration += 1

        if iteration > max_iterations:
            raise RuntimeError(
                f"Episode-aware purge failed to converge for "
                f"{left_name}/{right_name}."
            )

        if left.empty or right.empty:
            break

        segment_ids_here = sorted(
            set(
                left["Segment_ID"]
                .astype(int)
                .tolist()
            )
            |
            set(
                right["Segment_ID"]
                .astype(int)
                .tolist()
            )
        )

        shared_episode_ids = set()

        for seg in segment_ids_here:

            left_eps = touched_episode_ids(
                left,
                seg,
            )

            right_eps = touched_episode_ids(
                right,
                seg,
            )

            shared_episode_ids.update(
                left_eps & right_eps
            )

        # ------------------------------------------------------------------------------------------
        # No shared episode remains.
        # ------------------------------------------------------------------------------------------

        if not shared_episode_ids:

            # Still re-run raw purge as a final safety step.

            left, right, raw_purged = (
                purge_raw_boundary(
                    left,
                    right,
                )
            )

            purged.extend(
                raw_purged
            )

            # Verify raw purge did not somehow leave/create shared episodes.

            post_shared = set()

            for seg in segment_ids_here:

                post_shared.update(
                    touched_episode_ids(
                        left,
                        seg,
                    )
                    &
                    touched_episode_ids(
                        right,
                        seg,
                    )
                )

            if not post_shared:
                break

            shared_episode_ids = post_shared

        # ------------------------------------------------------------------------------------------
        # Purge all origins touching every shared episode.
        # ------------------------------------------------------------------------------------------

        removed_this_iteration = 0

        for episode_id in sorted(
            shared_episode_ids
        ):

            ep = get_episode_row(
                episode_id
            )

            seg = int(
                ep["Segment_ID"]
            )

            ep_start = ep[
                "Episode_Start"
            ]

            ep_end = ep[
                "Episode_End"
            ]

            # Any origin footprint touching the episode is unsafe at
            # this boundary.

            left_touch = (
                left["Segment_ID"].eq(seg)
                &
                (
                    left["Footprint_Start"]
                    <= ep_end
                )
                &
                (
                    left["Footprint_End"]
                    >= ep_start
                )
            )

            right_touch = (
                right["Segment_ID"].eq(seg)
                &
                (
                    right["Footprint_Start"]
                    <= ep_end
                )
                &
                (
                    right["Footprint_End"]
                    >= ep_start
                )
            )

            left_removed = (
                left.loc[
                    left_touch,
                    [
                        "Segment_ID",
                        "Origin_Time",
                    ],
                ]
                .copy()
            )

            right_removed = (
                right.loc[
                    right_touch,
                    [
                        "Segment_ID",
                        "Origin_Time",
                    ],
                ]
                .copy()
            )

            for row in left_removed.itertuples(
                index=False
            ):

                purged.append(
                    (
                        int(row.Segment_ID),
                        row.Origin_Time,
                        f"SHARED_EPISODE_{episode_id}",
                    )
                )

            for row in right_removed.itertuples(
                index=False
            ):

                purged.append(
                    (
                        int(row.Segment_ID),
                        row.Origin_Time,
                        f"SHARED_EPISODE_{episode_id}",
                    )
                )

            removed_this_iteration += (
                int(left_touch.sum())
                +
                int(right_touch.sum())
            )

            left = (
                left.loc[
                    ~left_touch
                ]
                .sort_values(
                    "Origin_Time"
                )
                .reset_index(drop=True)
            )

            right = (
                right.loc[
                    ~right_touch
                ]
                .sort_values(
                    "Origin_Time"
                )
                .reset_index(drop=True)
            )

        if removed_this_iteration == 0:

            raise RuntimeError(
                f"Shared attack episode remained across "
                f"{left_name}/{right_name}, but no removable "
                f"origin footprint was identified."
            )

        # ------------------------------------------------------------------------------------------
        # Re-enforce raw temporal isolation after episode removal.
        # ------------------------------------------------------------------------------------------

        left, right, raw_purged = (
            purge_raw_boundary(
                left,
                right,
            )
        )

        purged.extend(
            raw_purged
        )

    return (
        left.reset_index(drop=True),
        right.reset_index(drop=True),
        purged,
    )


# ==================================================================================================
# 10D. CONSTRUCT EACH SEGMENT'S FINAL DEVELOPMENT PARTITIONS
# ==================================================================================================

development_assignments = []
development_purge_records = []
development_segment_rows = []

for seg in segment_ids:

    seg_origins = (
        origin_table[
            origin_table[
                "Segment_ID"
            ].eq(seg)
        ]
        .sort_values(
            "Origin_Time"
        )
        .reset_index(drop=True)
        .copy()
    )

    n = len(
        seg_origins
    )

    train_end_idx, val_end_idx = (
        choose_initial_boundaries(
            n
        )
    )

    # ----------------------------------------------------------------------------------------------
    # Segment too small.
    # ----------------------------------------------------------------------------------------------

    if train_end_idx is None:

        for row in seg_origins.itertuples(
            index=False
        ):

            development_assignments.append({
                "Segment_ID":
                    int(seg),

                "Origin_Time":
                    row.Origin_Time,

                "Development_Split":
                    "PURGED",
            })

            development_purge_records.append({
                "Segment_ID":
                    int(seg),

                "Origin_Time":
                    row.Origin_Time,

                "Purge_Reason":
                    "SEGMENT_TOO_SMALL",
            })

        development_segment_rows.append({
            "Segment_ID":
                int(seg),

            "Original_Origins":
                n,

            "Train_Origins":
                0,

            "Validation_Origins":
                0,

            "Test_Origins":
                0,

            "Purged_Origins":
                n,
        })

        continue

    # ----------------------------------------------------------------------------------------------
    # Initial chronological 60/20/20 partition.
    # ----------------------------------------------------------------------------------------------

    train_part = (
        seg_origins
        .iloc[
            :train_end_idx + 1
        ]
        .copy()
    )

    val_part = (
        seg_origins
        .iloc[
            train_end_idx + 1:
            val_end_idx + 1
        ]
        .copy()
    )

    test_part = (
        seg_origins
        .iloc[
            val_end_idx + 1:
        ]
        .copy()
    )

    # ----------------------------------------------------------------------------------------------
    # STEP 1:
    # Raw TRAIN / VALIDATION purge.
    # ----------------------------------------------------------------------------------------------

    train_part, val_part, purged_tv_raw = (
        purge_raw_boundary(
            train_part,
            val_part,
        )
    )

    # ----------------------------------------------------------------------------------------------
    # STEP 2:
    # Raw VALIDATION / TEST purge.
    # ----------------------------------------------------------------------------------------------

    val_part, test_part, purged_vt_raw = (
        purge_raw_boundary(
            val_part,
            test_part,
        )
    )

    # ----------------------------------------------------------------------------------------------
    # STEP 3:
    # Episode-aware TRAIN / VALIDATION purge.
    # ----------------------------------------------------------------------------------------------

    train_part, val_part, purged_tv_episode = (
        purge_shared_episodes(
            train_part,
            val_part,
            "TRAIN",
            "VALIDATION",
        )
    )

    # ----------------------------------------------------------------------------------------------
    # STEP 4:
    # Episode-aware VALIDATION / TEST purge.
    # ----------------------------------------------------------------------------------------------

    val_part, test_part, purged_vt_episode = (
        purge_shared_episodes(
            val_part,
            test_part,
            "VALIDATION",
            "TEST",
        )
    )

    # ----------------------------------------------------------------------------------------------
    # STEP 5:
    # TRAIN / VALIDATION again.
    #
    # VAL/TEST purging may alter validation's left boundary, so run the
    # TRAIN/VAL protection once more.
    # ----------------------------------------------------------------------------------------------

    train_part, val_part, purged_tv_final = (
        purge_shared_episodes(
            train_part,
            val_part,
            "TRAIN",
            "VALIDATION",
        )
    )

    # ----------------------------------------------------------------------------------------------
    # STEP 6:
    # Final raw purge both boundaries.
    # ----------------------------------------------------------------------------------------------

    train_part, val_part, purged_tv_final_raw = (
        purge_raw_boundary(
            train_part,
            val_part,
        )
    )

    val_part, test_part, purged_vt_final_raw = (
        purge_raw_boundary(
            val_part,
            test_part,
        )
    )

    all_purged = (
        purged_tv_raw
        + purged_vt_raw
        + purged_tv_episode
        + purged_vt_episode
        + purged_tv_final
        + purged_tv_final_raw
        + purged_vt_final_raw
    )

    # ----------------------------------------------------------------------------------------------
    # Deduplicate purge records.
    # ----------------------------------------------------------------------------------------------

    purge_reason_map = defaultdict(
        set
    )

    for (
        purge_seg,
        purge_origin,
        purge_reason,
    ) in all_purged:

        purge_reason_map[
            (
                int(purge_seg),
                purge_origin,
            )
        ].add(
            purge_reason
        )

    # ----------------------------------------------------------------------------------------------
    # Verify that no origin appears in two ACTIVE partitions.
    # ----------------------------------------------------------------------------------------------

    train_keys_seg = set(
        train_part[
            "Origin_Time"
        ].tolist()
    )

    val_keys_seg = set(
        val_part[
            "Origin_Time"
        ].tolist()
    )

    test_keys_seg = set(
        test_part[
            "Origin_Time"
        ].tolist()
    )

    if (
        train_keys_seg & val_keys_seg
        or
        train_keys_seg & test_keys_seg
        or
        val_keys_seg & test_keys_seg
    ):

        raise RuntimeError(
            f"Segment {seg}: active origin overlap after purging."
        )

    # ----------------------------------------------------------------------------------------------
    # All original origins must resolve to exactly one of:
    #
    # TRAIN / VALIDATION / TEST / PURGED
    # ----------------------------------------------------------------------------------------------

    active_origin_times = (
        train_keys_seg
        | val_keys_seg
        | test_keys_seg
    )

    original_origin_times = set(
        seg_origins[
            "Origin_Time"
        ].tolist()
    )

    purged_origin_times = (
        original_origin_times
        - active_origin_times
    )

    # ----------------------------------------------------------------------------------------------
    # Record ACTIVE assignments.
    # ----------------------------------------------------------------------------------------------

    for row in train_part.itertuples(
        index=False
    ):

        development_assignments.append({
            "Segment_ID":
                int(seg),

            "Origin_Time":
                row.Origin_Time,

            "Development_Split":
                "TRAIN",
        })

    for row in val_part.itertuples(
        index=False
    ):

        development_assignments.append({
            "Segment_ID":
                int(seg),

            "Origin_Time":
                row.Origin_Time,

            "Development_Split":
                "VALIDATION",
        })

    for row in test_part.itertuples(
        index=False
    ):

        development_assignments.append({
            "Segment_ID":
                int(seg),

            "Origin_Time":
                row.Origin_Time,

            "Development_Split":
                "TEST",
        })

    # ----------------------------------------------------------------------------------------------
    # Record PURGED assignments.
    # ----------------------------------------------------------------------------------------------

    for origin_time in sorted(
        purged_origin_times
    ):

        development_assignments.append({
            "Segment_ID":
                int(seg),

            "Origin_Time":
                origin_time,

            "Development_Split":
                "PURGED",
        })

        reasons = purge_reason_map.get(
            (
                int(seg),
                origin_time,
            ),
            {"BOUNDARY_SAFETY"},
        )

        development_purge_records.append({
            "Segment_ID":
                int(seg),

            "Origin_Time":
                origin_time,

            "Purge_Reason":
                " | ".join(
                    sorted(
                        reasons
                    )
                ),
        })

    # ----------------------------------------------------------------------------------------------
    # Final segment accounting.
    # ----------------------------------------------------------------------------------------------

    accounted = (
        len(train_part)
        + len(val_part)
        + len(test_part)
        + len(purged_origin_times)
    )

    if accounted != n:

        raise RuntimeError(
            f"Segment {seg}: origin accounting error. "
            f"Expected {n}, accounted for {accounted}."
        )

    development_segment_rows.append({
        "Segment_ID":
            int(seg),

        "Original_Origins":
            n,

        "Train_Origins":
            len(train_part),

        "Validation_Origins":
            len(val_part),

        "Test_Origins":
            len(test_part),

        "Purged_Origins":
            len(purged_origin_times),
    })


# ==================================================================================================
# 10E. BUILD FINAL DEVELOPMENT ASSIGNMENT TABLE
# ==================================================================================================

development_origin_split = pd.DataFrame(
    development_assignments
)

development_purge_log = pd.DataFrame(
    development_purge_records
)

development_segment_summary = pd.DataFrame(
    development_segment_rows
)

duplicate_dev_assignments = int(
    development_origin_split.duplicated(
        subset=[
            "Segment_ID",
            "Origin_Time",
        ]
    ).sum()
)

if duplicate_dev_assignments:

    duplicate_examples = (
        development_origin_split[
            development_origin_split.duplicated(
                subset=[
                    "Segment_ID",
                    "Origin_Time",
                ],
                keep=False,
            )
        ]
        .sort_values(
            [
                "Segment_ID",
                "Origin_Time",
            ]
        )
        .head(20)
    )

    raise RuntimeError(
        "Development split assigned at least one origin more than once.\n\n"
        + duplicate_examples.to_string(
            index=False
        )
    )

# Every original origin must exist exactly once.
if len(
    development_origin_split
) != len(
    origin_table
):

    raise RuntimeError(
        "Development assignment table does not contain exactly one "
        "assignment for every forecast origin."
    )

development_origin_split = (
    origin_table
    .merge(
        development_origin_split,
        on=[
            "Segment_ID",
            "Origin_Time",
        ],
        how="left",
        validate="one_to_one",
    )
)

if development_origin_split[
    "Development_Split"
].isna().any():

    raise RuntimeError(
        "At least one origin received no development split assignment."
    )

print(
    development_segment_summary
    .to_string(
        index=False
    )
)

print()
print(
    "Total original origins :",
    f"{len(origin_table):,}",
)

print(
    "Total retained origins :",
    f"{int((development_origin_split['Development_Split'] != 'PURGED').sum()):,}",
)

print(
    "Total purged origins   :",
    f"{int((development_origin_split['Development_Split'] == 'PURGED').sum()):,}",
)


# ==================================================================================================
# 11. DEVELOPMENT SPLIT — HARD FOOTPRINT LEAKAGE AUDIT
# ==================================================================================================

print("\n" + SEP)
print("DEVELOPMENT SPLIT — TEMPORAL FOOTPRINT LEAKAGE AUDIT")
print(SEP)


def footprint_overlap_count(
    left,
    right,
):
    """
    Count overlapping temporal-footprint pairs inside the same segment.
    """

    overlaps = 0

    shared_segments = sorted(
        set(
            left["Segment_ID"]
            .astype(int)
        )
        &
        set(
            right["Segment_ID"]
            .astype(int)
        )
    )

    for seg in shared_segments:

        a = (
            left[
                left[
                    "Segment_ID"
                ].eq(seg)
            ][
                [
                    "Footprint_Start",
                    "Footprint_End",
                ]
            ]
        )

        b = (
            right[
                right[
                    "Segment_ID"
                ].eq(seg)
            ][
                [
                    "Footprint_Start",
                    "Footprint_End",
                ]
            ]
        )

        for (
            a_start,
            a_end,
        ) in a.itertuples(
            index=False,
            name=None,
        ):

            overlap_mask = (
                (
                    b["Footprint_Start"]
                    <= a_end
                )
                &
                (
                    b["Footprint_End"]
                    >= a_start
                )
            )

            overlaps += int(
                overlap_mask.sum()
            )

    return overlaps


dev_train_origins = (
    development_origin_split[
        development_origin_split[
            "Development_Split"
        ].eq("TRAIN")
    ]
    .copy()
)

dev_val_origins = (
    development_origin_split[
        development_origin_split[
            "Development_Split"
        ].eq("VALIDATION")
    ]
    .copy()
)

dev_test_origins = (
    development_origin_split[
        development_origin_split[
            "Development_Split"
        ].eq("TEST")
    ]
    .copy()
)

dev_purged_origins = (
    development_origin_split[
        development_origin_split[
            "Development_Split"
        ].eq("PURGED")
    ]
    .copy()
)

train_val_overlap = (
    footprint_overlap_count(
        dev_train_origins,
        dev_val_origins,
    )
)

train_test_overlap = (
    footprint_overlap_count(
        dev_train_origins,
        dev_test_origins,
    )
)

val_test_overlap = (
    footprint_overlap_count(
        dev_val_origins,
        dev_test_origins,
    )
)

print(
    f"Train origins                     : {len(dev_train_origins):,}"
)

print(
    f"Validation origins                : {len(dev_val_origins):,}"
)

print(
    f"Test origins                      : {len(dev_test_origins):,}"
)

print(
    f"Purged origins                    : {len(dev_purged_origins):,}"
)

print()

print(
    f"TRAIN ↔ VALIDATION footprint overlap pairs : {train_val_overlap:,}"
)

print(
    f"TRAIN ↔ TEST footprint overlap pairs       : {train_test_overlap:,}"
)

print(
    f"VALIDATION ↔ TEST footprint overlap pairs  : {val_test_overlap:,}"
)

if (
    train_val_overlap
    or train_test_overlap
    or val_test_overlap
):

    raise RuntimeError(
        "Development split contains raw-time footprint leakage."
    )

print(
    "\n✓ No history/future raw-minute footprint is shared across development partitions."
)


# ==================================================================================================
# 12. DEVELOPMENT SPLIT — FINAL ATTACK-EPISODE ISOLATION AUDIT
# ==================================================================================================

print("\n" + SEP)
print("DEVELOPMENT SPLIT — ATTACK EPISODE ISOLATION")
print(SEP)


def episodes_touched_by_partition(
    partition_df,
):
    """
    Return strict attack episodes touched by any temporal footprint
    in the supplied partition.
    """

    touched = set()

    if (
        partition_df.empty
        or episode_table.empty
    ):
        return touched

    for seg in sorted(
        partition_df[
            "Segment_ID"
        ]
        .astype(int)
        .unique()
        .tolist()
    ):

        part_seg = (
            partition_df[
                partition_df[
                    "Segment_ID"
                ].eq(seg)
            ]
        )

        ep_seg = (
            episode_table[
                episode_table[
                    "Segment_ID"
                ].eq(seg)
            ]
        )

        if (
            part_seg.empty
            or ep_seg.empty
        ):
            continue

        for ep in ep_seg.itertuples(
            index=False
        ):

            intersects = (
                (
                    part_seg[
                        "Footprint_Start"
                    ]
                    <= ep.Episode_End
                )
                &
                (
                    part_seg[
                        "Footprint_End"
                    ]
                    >= ep.Episode_Start
                )
            )

            if intersects.any():

                touched.add(
                    int(
                        ep.Episode_ID
                    )
                )

    return touched


train_episodes_touched = (
    episodes_touched_by_partition(
        dev_train_origins
    )
)

val_episodes_touched = (
    episodes_touched_by_partition(
        dev_val_origins
    )
)

test_episodes_touched = (
    episodes_touched_by_partition(
        dev_test_origins
    )
)

episode_train_val_overlap = (
    train_episodes_touched
    &
    val_episodes_touched
)

episode_train_test_overlap = (
    train_episodes_touched
    &
    test_episodes_touched
)

episode_val_test_overlap = (
    val_episodes_touched
    &
    test_episodes_touched
)

print(
    f"Attack episodes touched by TRAIN      : {len(train_episodes_touched):,}"
)

print(
    f"Attack episodes touched by VALIDATION : {len(val_episodes_touched):,}"
)

print(
    f"Attack episodes touched by TEST       : {len(test_episodes_touched):,}"
)

print()

print(
    f"TRAIN ∩ VALIDATION episodes           : {len(episode_train_val_overlap):,}"
)

print(
    f"TRAIN ∩ TEST episodes                 : {len(episode_train_test_overlap):,}"
)

print(
    f"VALIDATION ∩ TEST episodes            : {len(episode_val_test_overlap):,}"
)

if (
    episode_train_val_overlap
    or episode_train_test_overlap
    or episode_val_test_overlap
):

    print("\nShared episode details:")

    shared_ids = sorted(
        episode_train_val_overlap
        | episode_train_test_overlap
        | episode_val_test_overlap
    )

    print(
        episode_table[
            episode_table[
                "Episode_ID"
            ].isin(
                shared_ids
            )
        ]
        .sort_values(
            [
                "Segment_ID",
                "Episode_Start",
            ]
        )
        .to_string(
            index=False
        )
    )

    raise RuntimeError(
        "Episode-aware purge failed: at least one strict attack episode "
        "is still visible to more than one development partition."
    )

print(
    "\n✓ No strict attack episode is visible to more than one development partition."
)


# ==================================================================================================
# 12B. PURGE LOG
# ==================================================================================================

print("\n" + SEP)
print("DEVELOPMENT SPLIT — PURGE AUDIT")
print(SEP)

if development_purge_log.empty:

    print(
        "No origins were purged."
    )

else:

    print(
        "Purged origins:",
        f"{len(development_purge_log):,}",
    )

    print()

    purge_reason_summary = (
        development_purge_log[
            "Purge_Reason"
        ]
        .value_counts()
        .rename_axis(
            "Purge_Reason"
        )
        .reset_index(
            name="Origins"
        )
    )

    print(
        purge_reason_summary
        .to_string(
            index=False
        )
    )



# ==================================================================================================
# 13. ATTACH DEVELOPMENT SPLIT TO ALL FIVE-HORIZON FORECAST ROWS
# ==================================================================================================

development_assignment_lookup = (
    development_origin_split[
        [
            "Segment_ID",
            "Origin_Time",
            "Development_Split",
        ]
    ]
    .copy()
)

cell14_development_forecast = (
    forecast_meta
    .merge(
        development_assignment_lookup,
        on=[
            "Segment_ID",
            "Origin_Time",
        ],
        how="left",
        validate="many_to_one",
    )
)

if cell14_development_forecast[
    "Development_Split"
].isna().any():

    raise RuntimeError(
        "At least one forecast row did not receive a development partition."
    )

# Every origin must have exactly one split across all horizons.
split_counts_per_origin = (
    cell14_development_forecast
    .groupby(
        ["Segment_ID", "Origin_Time"]
    )["Development_Split"]
    .nunique()
)

cross_partition_origins = int(
    (
        split_counts_per_origin != 1
    ).sum()
)

if cross_partition_origins:
    raise RuntimeError(
        "At least one origin has horizons assigned to different partitions."
    )

print("\n✓ All five horizons remain together for every forecast origin.")


# ==================================================================================================
# 14. DEVELOPMENT SPLIT — PER-HORIZON SUPPORT
# ==================================================================================================

print("\n" + SEP)
print("DEVELOPMENT SPLIT — PER-HORIZON TARGET SUPPORT")
print(SEP)

dev_active = (
    cell14_development_forecast[
        cell14_development_forecast[
            "Development_Split"
        ].isin(
            [
                "TRAIN",
                "VALIDATION",
                "TEST",
            ]
        )
    ]
    .copy()
)

development_horizon_summary = (
    dev_active
    .groupby(
        [
            "Development_Split",
            "Horizon_Minutes",
        ]
    )
    .agg(
        Rows=("Future_Attack_State", "size"),
        Attack_Targets=("Future_Attack_State", "sum"),
        New_Onset_Targets=("New_Onset", "sum"),
        Continuation_Targets=("Continuation", "sum"),
    )
    .reset_index()
)

development_horizon_summary[
    "Benign_Targets"
] = (
    development_horizon_summary["Rows"]
    - development_horizon_summary["Attack_Targets"]
)

development_horizon_summary[
    "Attack_Percent"
] = (
    100
    * development_horizon_summary["Attack_Targets"]
    / development_horizon_summary["Rows"]
)

print(
    development_horizon_summary
    .sort_values(
        [
            "Development_Split",
            "Horizon_Minutes",
        ]
    )
    .to_string(index=False)
)


# ==================================================================================================
# 15. CLEAN-HISTORY / GENUINE PRE-ATTACK SUPPORT
# ==================================================================================================

print("\n" + SEP)
print("DEVELOPMENT SPLIT — CLEAN-HISTORY PRE-ATTACK SUPPORT")
print(SEP)

clean_attack_mask = (
    dev_active["Clean_History"].eq(1)
    &
    dev_active["Future_Attack_State"].eq(1)
)

development_clean_history_summary = (
    dev_active[
        clean_attack_mask
    ]
    .groupby(
        [
            "Development_Split",
            "Horizon_Minutes",
        ]
    )
    .size()
    .rename("CleanHistory_Attack_Targets")
    .reset_index()
)

full_partition_horizon_index = pd.MultiIndex.from_product(
    [
        ["TRAIN", "VALIDATION", "TEST"],
        EXPECTED_HORIZONS,
    ],
    names=[
        "Development_Split",
        "Horizon_Minutes",
    ],
)

development_clean_history_summary = (
    development_clean_history_summary
    .set_index(
        [
            "Development_Split",
            "Horizon_Minutes",
        ]
    )
    .reindex(
        full_partition_horizon_index,
        fill_value=0,
    )
    .reset_index()
)

print(
    development_clean_history_summary
    .to_string(index=False)
)


# ==================================================================================================
# 16. HELPER FUNCTIONS FOR RICH-TARGET COVERAGE
# ==================================================================================================

def collect_values(
    df,
    column,
):
    values = set()

    for item in df[column]:

        for value in normalize_collection(item):

            if isinstance(value, list):
                value = tuple(value)

            if isinstance(value, tuple):
                values.add(tuple(value))
            else:
                values.add(value)

    return values


def attack_only(df):

    return df[
        df["Future_Attack_State"].eq(1)
    ].copy()


def partition_rich_sets(df):

    attacks = attack_only(df)

    return {
        "types":
            collect_values(
                attacks,
                "Future_Attack_Types",
            ),

        "sources":
            collect_values(
                attacks,
                "Future_Attack_Sources",
            ),

        "destinations":
            collect_values(
                attacks,
                "Future_Attack_Destinations",
            ),

        "edges":
            collect_values(
                attacks,
                "Future_Attack_Edges",
            ),

        "typed_edges":
            collect_values(
                attacks,
                "Future_Typed_Attack_Edges",
            ),
    }


dev_train_rows = (
    dev_active[
        dev_active[
            "Development_Split"
        ].eq("TRAIN")
    ]
)

dev_val_rows = (
    dev_active[
        dev_active[
            "Development_Split"
        ].eq("VALIDATION")
    ]
)

dev_test_rows = (
    dev_active[
        dev_active[
            "Development_Split"
        ].eq("TEST")
    ]
)

train_sets = partition_rich_sets(
    dev_train_rows
)

val_sets = partition_rich_sets(
    dev_val_rows
)

test_sets = partition_rich_sets(
    dev_test_rows
)


# ==================================================================================================
# 17. ATTACK-TYPE COVERAGE
# ==================================================================================================

print("\n" + SEP)
print("DEVELOPMENT SPLIT — ATTACK-TYPE COVERAGE")
print(SEP)

all_types = sorted(
    train_sets["types"]
    | val_sets["types"]
    | test_sets["types"],
    key=str,
)

type_rows = []

for attack_type in all_types:

    type_rows.append({
        "Attack_Type": attack_type,
        "Seen_In_Train":
            attack_type in train_sets["types"],
        "Present_In_Validation":
            attack_type in val_sets["types"],
        "Present_In_Test":
            attack_type in test_sets["types"],
    })

development_attack_type_coverage = pd.DataFrame(
    type_rows
)

if development_attack_type_coverage.empty:

    print("No attack types found.")

else:

    print(
        development_attack_type_coverage
        .to_string(index=False)
    )

unseen_val_types = (
    val_sets["types"]
    - train_sets["types"]
)

unseen_test_types = (
    test_sets["types"]
    - train_sets["types"]
)

print()
print(
    "Attack types represented in TRAIN      :",
    len(train_sets["types"]),
)

print(
    "Attack types represented in VALIDATION :",
    len(val_sets["types"]),
)

print(
    "Attack types represented in TEST       :",
    len(test_sets["types"]),
)

print(
    "Unseen validation attack types         :",
    len(unseen_val_types),
)

print(
    "Unseen test attack types               :",
    len(unseen_test_types),
)

if unseen_val_types:

    print(
        "Validation unseen type list            :",
        sorted(
            unseen_val_types,
            key=str,
        ),
    )

if unseen_test_types:

    print(
        "Test unseen type list                  :",
        sorted(
            unseen_test_types,
            key=str,
        ),
    )


# ==================================================================================================
# 18. PER-TYPE / PER-HORIZON SUPPORT
# ==================================================================================================

print("\n" + SEP)
print("DEVELOPMENT SPLIT — PER-TYPE / PER-HORIZON POSITIVE SUPPORT")
print(SEP)

type_horizon_rows = []

for split_name in [
    "TRAIN",
    "VALIDATION",
    "TEST",
]:

    split_df = dev_active[
        dev_active[
            "Development_Split"
        ].eq(split_name)
    ]

    for h in EXPECTED_HORIZONS:

        hdf = split_df[
            split_df[
                "Horizon_Minutes"
            ].eq(h)
            &
            split_df[
                "Future_Attack_State"
            ].eq(1)
        ]

        type_counter = Counter()

        clean_type_counter = Counter()

        for _, row in hdf.iterrows():

            types = normalize_collection(
                row["Future_Attack_Types"]
            )

            for attack_type in types:

                type_counter[
                    attack_type
                ] += 1

                if int(
                    row["Clean_History"]
                ) == 1:

                    clean_type_counter[
                        attack_type
                    ] += 1

        for attack_type in all_types:

            type_horizon_rows.append({
                "Development_Split":
                    split_name,

                "Horizon_Minutes":
                    h,

                "Attack_Type":
                    attack_type,

                "Positive_Target_Rows":
                    int(
                        type_counter.get(
                            attack_type,
                            0,
                        )
                    ),

                "CleanHistory_Positive_Rows":
                    int(
                        clean_type_counter.get(
                            attack_type,
                            0,
                        )
                    ),
            })

development_type_horizon_support = pd.DataFrame(
    type_horizon_rows
)

if development_type_horizon_support.empty:

    print("No attack-type support rows.")

else:

    print(
        development_type_horizon_support
        .sort_values(
            [
                "Attack_Type",
                "Horizon_Minutes",
                "Development_Split",
            ]
        )
        .to_string(index=False)
    )


# ==================================================================================================
# 19. GRAPH ENTITY / EDGE NOVELTY
# ==================================================================================================

print("\n" + SEP)
print("DEVELOPMENT SPLIT — SOURCE / DESTINATION / EDGE NOVELTY")
print(SEP)

novelty_rows = []

for entity_name, key in [
    ("Attack_Type", "types"),
    ("Source", "sources"),
    ("Destination", "destinations"),
    ("Source→Destination_Edge", "edges"),
    ("Typed_Source→Destination_Edge", "typed_edges"),
]:

    val_unseen = (
        val_sets[key]
        - train_sets[key]
    )

    test_unseen = (
        test_sets[key]
        - train_sets[key]
    )

    novelty_rows.append({
        "Entity": entity_name,
        "Train_Unique": len(train_sets[key]),
        "Validation_Unique": len(val_sets[key]),
        "Test_Unique": len(test_sets[key]),
        "Validation_Unseen_vs_Train": len(val_unseen),
        "Test_Unseen_vs_Train": len(test_unseen),
    })

development_graph_novelty = pd.DataFrame(
    novelty_rows
)

print(
    development_graph_novelty
    .to_string(index=False)
)


# ==================================================================================================
# 20. DEVELOPMENT PARTITION TIME RANGES
# ==================================================================================================

print("\n" + SEP)
print("DEVELOPMENT SPLIT — TIME RANGE BY SEGMENT")
print(SEP)

development_time_rows = []

for seg in segment_ids:

    for split_name in [
        "TRAIN",
        "VALIDATION",
        "TEST",
        "PURGED",
    ]:

        sdf = development_origin_split[
            development_origin_split[
                "Segment_ID"
            ].eq(seg)
            &
            development_origin_split[
                "Development_Split"
            ].eq(split_name)
        ]

        development_time_rows.append({
            "Segment_ID":
                seg,

            "Partition":
                split_name,

            "Origins":
                len(sdf),

            "First_Origin":
                (
                    sdf["Origin_Time"].min()
                    if len(sdf)
                    else pd.NaT
                ),

            "Last_Origin":
                (
                    sdf["Origin_Time"].max()
                    if len(sdf)
                    else pd.NaT
                ),
        })

development_time_ranges = pd.DataFrame(
    development_time_rows
)

print(
    development_time_ranges
    .to_string(index=False)
)


# ==================================================================================================
# 21. STRICT CHRONOLOGICAL FUTURE HOLDOUT
# ==================================================================================================
#
# This is deliberately separate from the development split.
#
# Whole capture segments are assigned chronologically.
#
# No within-segment mixing occurs.
#
# This protocol is the appropriate place to report:
#   • future unseen attack types
#   • future unseen sources
#   • future unseen destinations
#   • future unseen graph edges
#
# ==================================================================================================

print("\n" + SEP)
print("PROTOCOL B — STRICT CHRONOLOGICAL FUTURE HOLDOUT")
print(SEP)

n_segments = len(segment_ids)

strict_train_n = max(
    1,
    int(
        np.floor(
            n_segments
            * STRICT_TRAIN_FRACTION
        )
    ),
)

strict_val_n = max(
    1,
    int(
        np.floor(
            n_segments
            * STRICT_VAL_FRACTION
        )
    ),
)

if (
    strict_train_n
    + strict_val_n
    >= n_segments
):

    strict_val_n = 1
    strict_train_n = n_segments - 2

strict_train_segments = (
    segment_ids[
        :strict_train_n
    ]
)

strict_val_segments = (
    segment_ids[
        strict_train_n:
        strict_train_n + strict_val_n
    ]
)

strict_test_segments = (
    segment_ids[
        strict_train_n + strict_val_n:
    ]
)

print(
    "Strict TRAIN segments      :",
    strict_train_segments,
)

print(
    "Strict VALIDATION segments :",
    strict_val_segments,
)

print(
    "Strict TEST segments       :",
    strict_test_segments,
)

strict_assignment_map = {}

for seg in strict_train_segments:
    strict_assignment_map[int(seg)] = "TRAIN"

for seg in strict_val_segments:
    strict_assignment_map[int(seg)] = "VALIDATION"

for seg in strict_test_segments:
    strict_assignment_map[int(seg)] = "TEST"

strict_origin_split = (
    origin_table.copy()
)

strict_origin_split[
    "Strict_Chronological_Split"
] = (
    strict_origin_split[
        "Segment_ID"
    ]
    .map(
        strict_assignment_map
    )
)

if strict_origin_split[
    "Strict_Chronological_Split"
].isna().any():

    raise RuntimeError(
        "At least one segment was not assigned in strict chronological protocol."
    )

cell14_strict_chronological_forecast = (
    forecast_meta
    .merge(
        strict_origin_split[
            [
                "Segment_ID",
                "Origin_Time",
                "Strict_Chronological_Split",
            ]
        ],
        on=[
            "Segment_ID",
            "Origin_Time",
        ],
        how="left",
        validate="many_to_one",
    )
)

print("\nStrict chronological forecast rows")

print(
    cell14_strict_chronological_forecast[
        "Strict_Chronological_Split"
    ]
    .value_counts()
    .reindex(
        [
            "TRAIN",
            "VALIDATION",
            "TEST",
        ],
        fill_value=0,
    )
    .to_string()
)


# ==================================================================================================
# 22. STRICT CHRONOLOGICAL ATTACK-TYPE / GRAPH NOVELTY
# ==================================================================================================

print("\n" + SEP)
print("STRICT CHRONOLOGICAL HOLDOUT — FUTURE NOVELTY")
print(SEP)

strict_train_rows = (
    cell14_strict_chronological_forecast[
        cell14_strict_chronological_forecast[
            "Strict_Chronological_Split"
        ].eq("TRAIN")
    ]
)

strict_val_rows = (
    cell14_strict_chronological_forecast[
        cell14_strict_chronological_forecast[
            "Strict_Chronological_Split"
        ].eq("VALIDATION")
    ]
)

strict_test_rows = (
    cell14_strict_chronological_forecast[
        cell14_strict_chronological_forecast[
            "Strict_Chronological_Split"
        ].eq("TEST")
    ]
)

strict_train_sets = partition_rich_sets(
    strict_train_rows
)

strict_val_sets = partition_rich_sets(
    strict_val_rows
)

strict_test_sets = partition_rich_sets(
    strict_test_rows
)

strict_novelty_rows = []

for entity_name, key in [
    ("Attack_Type", "types"),
    ("Source", "sources"),
    ("Destination", "destinations"),
    ("Source→Destination_Edge", "edges"),
    ("Typed_Source→Destination_Edge", "typed_edges"),
]:

    val_unseen = (
        strict_val_sets[key]
        - strict_train_sets[key]
    )

    test_unseen = (
        strict_test_sets[key]
        - strict_train_sets[key]
    )

    strict_novelty_rows.append({
        "Entity":
            entity_name,

        "Train_Unique":
            len(
                strict_train_sets[key]
            ),

        "Validation_Unique":
            len(
                strict_val_sets[key]
            ),

        "Test_Unique":
            len(
                strict_test_sets[key]
            ),

        "Validation_Unseen_vs_Train":
            len(
                val_unseen
            ),

        "Test_Unseen_vs_Train":
            len(
                test_unseen
            ),
    })

strict_chronological_novelty = pd.DataFrame(
    strict_novelty_rows
)

print(
    strict_chronological_novelty
    .to_string(index=False)
)

strict_unseen_val_types = (
    strict_val_sets["types"]
    - strict_train_sets["types"]
)

strict_unseen_test_types = (
    strict_test_sets["types"]
    - strict_train_sets["types"]
)

print()

print(
    "Strict validation unseen attack types :",
    sorted(
        strict_unseen_val_types,
        key=str,
    ),
)

print(
    "Strict test unseen attack types       :",
    sorted(
        strict_unseen_test_types,
        key=str,
    ),
)


# ==================================================================================================
# 23. STRICT CHRONOLOGICAL PER-HORIZON SUPPORT
# ==================================================================================================

print("\n" + SEP)
print("STRICT CHRONOLOGICAL HOLDOUT — PER-HORIZON SUPPORT")
print(SEP)

strict_horizon_summary = (
    cell14_strict_chronological_forecast
    .groupby(
        [
            "Strict_Chronological_Split",
            "Horizon_Minutes",
        ]
    )
    .agg(
        Rows=("Future_Attack_State", "size"),
        Attack_Targets=("Future_Attack_State", "sum"),
        New_Onset_Targets=("New_Onset", "sum"),
        Continuation_Targets=("Continuation", "sum"),
    )
    .reset_index()
)

strict_horizon_summary[
    "Benign_Targets"
] = (
    strict_horizon_summary["Rows"]
    - strict_horizon_summary["Attack_Targets"]
)

strict_horizon_summary[
    "Attack_Percent"
] = (
    100
    * strict_horizon_summary["Attack_Targets"]
    / strict_horizon_summary["Rows"]
)

print(
    strict_horizon_summary
    .sort_values(
        [
            "Strict_Chronological_Split",
            "Horizon_Minutes",
        ]
    )
    .to_string(index=False)
)


# ==================================================================================================
# 24. HARD ORIGIN DISJOINTNESS ASSERTIONS
# ==================================================================================================

print("\n" + SEP)
print("HARD ORIGIN DISJOINTNESS ASSERTIONS")
print(SEP)


def origin_key_set(df):

    return set(
        zip(
            df["Segment_ID"].astype(int),
            df["Origin_Time"],
        )
    )


train_origin_keys = origin_key_set(
    dev_train_origins
)

val_origin_keys = origin_key_set(
    dev_val_origins
)

test_origin_keys = origin_key_set(
    dev_test_origins
)

purged_origin_keys = origin_key_set(
    dev_purged_origins
)

train_val_origin_overlap = (
    train_origin_keys
    & val_origin_keys
)

train_test_origin_overlap = (
    train_origin_keys
    & test_origin_keys
)

val_test_origin_overlap = (
    val_origin_keys
    & test_origin_keys
)

active_purge_overlap = (
    (
        train_origin_keys
        | val_origin_keys
        | test_origin_keys
    )
    & purged_origin_keys
)

print(
    f"TRAIN ∩ VALIDATION origins : {len(train_val_origin_overlap):,}"
)

print(
    f"TRAIN ∩ TEST origins       : {len(train_test_origin_overlap):,}"
)

print(
    f"VALIDATION ∩ TEST origins  : {len(val_test_origin_overlap):,}"
)

print(
    f"ACTIVE ∩ PURGED origins    : {len(active_purge_overlap):,}"
)

if (
    train_val_origin_overlap
    or train_test_origin_overlap
    or val_test_origin_overlap
    or active_purge_overlap
):
    raise RuntimeError(
        "Origin-level split leakage detected."
    )

print("✓ Development origin sets are perfectly disjoint.")


# ==================================================================================================
# 25. RAW MINUTE FOOTPRINT SET AUDIT
# ==================================================================================================
#
# This explicitly materializes every raw minute touched by every origin:
#
#     Origin-14 ... Origin+30
#
# and checks that no raw minute key appears in multiple development partitions.
#
# ==================================================================================================

print("\n" + SEP)
print("RAW MINUTE FOOTPRINT SET AUDIT")
print(SEP)


def raw_minute_footprint_set(df):

    footprint = set()

    for seg, origin in df[
        [
            "Segment_ID",
            "Origin_Time",
        ]
    ].itertuples(
        index=False,
        name=None,
    ):

        times = pd.date_range(
            start=(
                origin
                - pd.Timedelta(
                    minutes=LEFT_HISTORY_SPAN
                )
            ),
            end=(
                origin
                + pd.Timedelta(
                    minutes=RIGHT_FUTURE_SPAN
                )
            ),
            freq="min",
        )

        footprint.update(
            (
                int(seg),
                ts,
            )
            for ts in times
        )

    return footprint


train_raw_minutes = (
    raw_minute_footprint_set(
        dev_train_origins
    )
)

val_raw_minutes = (
    raw_minute_footprint_set(
        dev_val_origins
    )
)

test_raw_minutes = (
    raw_minute_footprint_set(
        dev_test_origins
    )
)

train_val_raw_overlap = (
    train_raw_minutes
    & val_raw_minutes
)

train_test_raw_overlap = (
    train_raw_minutes
    & test_raw_minutes
)

val_test_raw_overlap = (
    val_raw_minutes
    & test_raw_minutes
)

print(
    f"TRAIN raw-minute footprint      : {len(train_raw_minutes):,}"
)

print(
    f"VALIDATION raw-minute footprint : {len(val_raw_minutes):,}"
)

print(
    f"TEST raw-minute footprint       : {len(test_raw_minutes):,}"
)

print()

print(
    f"TRAIN ∩ VALIDATION raw minutes  : {len(train_val_raw_overlap):,}"
)

print(
    f"TRAIN ∩ TEST raw minutes        : {len(train_test_raw_overlap):,}"
)

print(
    f"VALIDATION ∩ TEST raw minutes   : {len(val_test_raw_overlap):,}"
)

if (
    train_val_raw_overlap
    or train_test_raw_overlap
    or val_test_raw_overlap
):
    raise RuntimeError(
        "Raw-minute leakage exists across development partitions."
    )

print(
    "\n✓ No raw minute contributes to more than one development partition."
)


# ==================================================================================================
# 26. FEATURE-LEAKAGE CONTRACT
# ==================================================================================================

print("\n" + SEP)
print("FEATURE-LEAKAGE CONTRACT")
print(SEP)

# Preserve upstream forbidden-feature contract if available.
if "FORBIDDEN_MODEL_FEATURES" in globals():

    cell14_forbidden_model_features = set(
        FORBIDDEN_MODEL_FEATURES
    )

else:

    cell14_forbidden_model_features = set()

# Explicitly add all known target/audit/split fields.
cell14_forbidden_model_features.update({
    "Segment_ID",
    "History_Start",
    "History_End",
    "Origin_Time",
    "Horizon_Minutes",
    "Target_Time",

    "Current_State",
    "Attack_In_History",
    "Clean_History",
    "Attack_Minutes_In_History",

    "Future_Attack_State",
    "Future_Attack_Types",
    "Future_Attack_Sources",
    "Future_Attack_Destinations",
    "Future_Attack_Edges",
    "Future_Typed_Attack_Edges",

    "Future_Attack_Type_Count",
    "Future_Attack_Source_Count",
    "Future_Attack_Destination_Count",
    "Future_Attack_Edge_Count",
    "Future_Typed_Attack_Edge_Count",

    "New_Onset",
    "Continuation",

    "Development_Split",
    "Strict_Chronological_Split",

    "Footprint_Start",
    "Footprint_End",
})

print(
    "Forbidden model-feature names tracked :",
    len(cell14_forbidden_model_features),
)

print(
    "✓ Target, future metadata, audit metadata, and split labels are explicitly forbidden as predictors."
)


# ==================================================================================================
# 27. FINAL DEVELOPMENT SPLIT SUMMARY
# ==================================================================================================

print("\n" + SEP)
print("FINAL DEVELOPMENT SPLIT SUMMARY")
print(SEP)

development_split_summary = (
    cell14_development_forecast
    .groupby(
        "Development_Split"
    )
    .agg(
        Forecast_Rows=(
            "Future_Attack_State",
            "size",
        ),
        Unique_Origins=(
            "Origin_Time",
            "nunique",
        ),
        Attack_Target_Rows=(
            "Future_Attack_State",
            "sum",
        ),
        New_Onset_Target_Rows=(
            "New_Onset",
            "sum",
        ),
        Continuation_Target_Rows=(
            "Continuation",
            "sum",
        ),
    )
    .reset_index()
)

development_split_summary[
    "Benign_Target_Rows"
] = (
    development_split_summary[
        "Forecast_Rows"
    ]
    - development_split_summary[
        "Attack_Target_Rows"
    ]
)

development_split_summary[
    "Attack_Percent"
] = (
    100
    * development_split_summary[
        "Attack_Target_Rows"
    ]
    / development_split_summary[
        "Forecast_Rows"
    ]
)

print(
    development_split_summary
    .to_string(index=False)
)


# ==================================================================================================
# 28. SCIENTIFIC INTERPRETATION FLAGS
# ==================================================================================================

print("\n" + SEP)
print("SCIENTIFIC INTERPRETATION FLAGS")
print(SEP)

dev_validation_has_attack = bool(
    dev_val_rows[
        "Future_Attack_State"
    ].eq(1).any()
)

dev_test_has_attack = bool(
    dev_test_rows[
        "Future_Attack_State"
    ].eq(1).any()
)

dev_validation_has_clean_attack = bool(
    (
        dev_val_rows[
            "Future_Attack_State"
        ].eq(1)
        &
        dev_val_rows[
            "Clean_History"
        ].eq(1)
    ).any()
)

dev_test_has_clean_attack = bool(
    (
        dev_test_rows[
            "Future_Attack_State"
        ].eq(1)
        &
        dev_test_rows[
            "Clean_History"
        ].eq(1)
    ).any()
)

dev_closed_set_val_types = (
    len(unseen_val_types) == 0
)

dev_closed_set_test_types = (
    len(unseen_test_types) == 0
)

interpretation_flags = {
    "validation_has_attack":
        dev_validation_has_attack,

    "test_has_attack":
        dev_test_has_attack,

    "validation_has_clean_history_attack":
        dev_validation_has_clean_attack,

    "test_has_clean_history_attack":
        dev_test_has_clean_attack,

    "validation_attack_types_all_seen_in_train":
        dev_closed_set_val_types,

    "test_attack_types_all_seen_in_train":
        dev_closed_set_test_types,

    "raw_time_leakage":
        False,

    "origin_leakage":
        False,

    "episode_leakage":
        False,
}

for key, value in interpretation_flags.items():

    print(
        f"{key:<48}: {value}"
    )

print()

if (
    not dev_closed_set_val_types
    or not dev_closed_set_test_types
):

    print(
        "NOTE:"
    )

    print(
        "The leakage-safe development split is NOT perfectly closed-set for attack type."
    )

    print(
        "This is reported honestly rather than moving labelled attack samples between partitions."
    )

    print(
        "Known-type and unseen-type evaluation can therefore be reported separately downstream."
    )

else:

    print(
        "✓ All validation/test attack types are represented in development training."
    )


# ==================================================================================================
# 29. HARD FINAL ASSERTIONS
# ==================================================================================================

print("\n" + SEP)
print("HARD FINAL ASSERTIONS")
print(SEP)

active_dev_splits = {
    "TRAIN",
    "VALIDATION",
    "TEST",
}

found_active_splits = set(
    cell14_development_forecast[
        "Development_Split"
    ].unique()
) & active_dev_splits

if found_active_splits != active_dev_splits:

    raise RuntimeError(
        "Development split does not contain TRAIN, VALIDATION, and TEST."
    )

for split_name in [
    "TRAIN",
    "VALIDATION",
    "TEST",
]:

    split_rows = (
        cell14_development_forecast[
            cell14_development_forecast[
                "Development_Split"
            ].eq(split_name)
        ]
    )

    horizons = sorted(
        split_rows[
            "Horizon_Minutes"
        ]
        .unique()
        .tolist()
    )

    if horizons != EXPECTED_HORIZONS:

        raise RuntimeError(
            f"{split_name} does not contain all forecast horizons.\n"
            f"Found: {horizons}"
        )

if (
    train_val_overlap
    or train_test_overlap
    or val_test_overlap
):
    raise RuntimeError(
        "Temporal footprint leakage survived final validation."
    )

if (
    train_val_raw_overlap
    or train_test_raw_overlap
    or val_test_raw_overlap
):
    raise RuntimeError(
        "Raw-minute overlap survived final validation."
    )

if (
    episode_train_val_overlap
    or episode_train_test_overlap
    or episode_val_test_overlap
):
    raise RuntimeError(
        "Attack-episode leakage survived final validation."
    )

if cross_partition_origins:
    raise RuntimeError(
        "Forecast horizons were separated across partitions."
    )

print("✓ TRAIN / VALIDATION / TEST all exist.")
print("✓ Every active partition contains all five horizons.")
print("✓ Every forecast origin belongs to exactly one development partition.")
print("✓ All five horizons belonging to an origin stay together.")
print("✓ No development origin overlap.")
print("✓ No history/future temporal-footprint overlap.")
print("✓ No raw-minute overlap.")
print("✓ No attack episode is shared across development partitions.")
print("✓ No scaling has been performed.")
print("✓ No resampling has been performed.")
print("✓ No model has been trained.")


# ==================================================================================================
# 30. EXPOSE FINAL CELL 14 OBJECTS
# ==================================================================================================

cell14_origin_table = (
    origin_table.copy()
)

cell14_development_origin_split = (
    development_origin_split.copy()
)

cell14_development_train_origins = (
    dev_train_origins.copy()
)

cell14_development_validation_origins = (
    dev_val_origins.copy()
)

cell14_development_test_origins = (
    dev_test_origins.copy()
)

cell14_development_purged_origins = (
    dev_purged_origins.copy()
)

cell14_development_train = (
    dev_train_rows.copy()
)

cell14_development_validation = (
    dev_val_rows.copy()
)

cell14_development_test = (
    dev_test_rows.copy()
)

cell14_development_horizon_summary = (
    development_horizon_summary.copy()
)

cell14_development_clean_history_summary = (
    development_clean_history_summary.copy()
)

cell14_development_attack_type_coverage = (
    development_attack_type_coverage.copy()
)

cell14_development_type_horizon_support = (
    development_type_horizon_support.copy()
)

cell14_development_graph_novelty = (
    development_graph_novelty.copy()
)

cell14_development_time_ranges = (
    development_time_ranges.copy()
)

cell14_strict_origin_split = (
    strict_origin_split.copy()
)

cell14_strict_train = (
    strict_train_rows.copy()
)

cell14_strict_validation = (
    strict_val_rows.copy()
)

cell14_strict_test = (
    strict_test_rows.copy()
)

cell14_strict_horizon_summary = (
    strict_horizon_summary.copy()
)

cell14_strict_novelty = (
    strict_chronological_novelty.copy()
)

cell14_episode_table = (
    episode_table.copy()
)

cell14_segment_summary = (
    segment_summary.copy()
)

cell14_development_split_summary = (
    development_split_summary.copy()
)

cell14_interpretation_flags = (
    interpretation_flags.copy()
)

# Convenient canonical aliases for downstream modelling.
#
# Cell 15 can use these directly.
train_forecast_df = (
    cell14_development_train.copy()
)

validation_forecast_df = (
    cell14_development_validation.copy()
)

test_forecast_df = (
    cell14_development_test.copy()
)


cell14_metadata = {

    "history_minutes":
        HISTORY_MINUTES,

    "forecast_horizons":
        EXPECTED_HORIZONS.copy(),

    "max_horizon_minutes":
        MAX_HORIZON,

    "history_left_span_minutes":
        LEFT_HISTORY_SPAN,

    "minimum_origin_boundary_separation_minutes":
        MIN_ORIGIN_BOUNDARY_SEPARATION,

    "development_train_fraction":
        DEV_TRAIN_FRACTION,

    "development_validation_fraction":
        DEV_VAL_FRACTION,

    "development_test_fraction":
        DEV_TEST_FRACTION,

    "strict_train_segments":
        strict_train_segments.copy(),

    "strict_validation_segments":
        strict_val_segments.copy(),

    "strict_test_segments":
        strict_test_segments.copy(),

    "development_train_origins":
        len(dev_train_origins),

    "development_validation_origins":
        len(dev_val_origins),

    "development_test_origins":
        len(dev_test_origins),

    "development_purged_origins":
        len(dev_purged_origins),

    "development_validation_unseen_attack_types":
        sorted(
            unseen_val_types,
            key=str,
        ),

    "development_test_unseen_attack_types":
        sorted(
            unseen_test_types,
            key=str,
        ),

    "strict_validation_unseen_attack_types":
        sorted(
            strict_unseen_val_types,
            key=str,
        ),

    "strict_test_unseen_attack_types":
        sorted(
            strict_unseen_test_types,
            key=str,
        ),

    "raw_time_leakage":
        False,

    "origin_leakage":
        False,

    "episode_leakage":
        False,

    "scaling_performed":
        False,

    "resampling_performed":
        False,

    "model_trained":
        False,
}


# ==================================================================================================
# 31. FINAL CELL 14 REPORT
# ==================================================================================================

print("\n" + SEP)
print("CELL 14 — COMPLETE")
print(SEP)

print(
f"""
DEVELOPMENT PROTOCOL
--------------------
Train origins       : {len(dev_train_origins):,}
Validation origins  : {len(dev_val_origins):,}
Test origins        : {len(dev_test_origins):,}
Purged origins      : {len(dev_purged_origins):,}

Validation unseen attack types : {len(unseen_val_types):,}
Test unseen attack types       : {len(unseen_test_types):,}

STRICT CHRONOLOGICAL PROTOCOL
-----------------------------
Train segments      : {strict_train_segments}
Validation segments : {strict_val_segments}
Test segments       : {strict_test_segments}

Strict validation unseen attack types : {len(strict_unseen_val_types):,}
Strict test unseen attack types       : {len(strict_unseen_test_types):,}

SAFETY CONTRACT
---------------
✓ Forecast origin is the atomic split unit.
✓ +1/+5/+10/+15/+30 targets remain together.
✓ Development partitions are temporally purged.
✓ 15-minute histories cannot overlap another partition.
✓ Future targets up to +30 minutes cannot overlap another partition.
✓ Raw-minute footprints are disjoint.
✓ Attack episodes are isolated.
✓ Future attack-type metadata was NOT used to move individual samples.
✓ Source/destination/edge novelty is preserved and measured.
✓ Strict chronological future holdout is retained separately.
✓ Target/audit/split columns are forbidden from model predictors.

NO SCALER HAS BEEN FIT.
NO RESAMPLING HAS BEEN PERFORMED.
NO MODEL HAS BEEN TRAINED.

Objects created:
  cell14_origin_table

  cell14_development_origin_split
  cell14_development_train_origins
  cell14_development_validation_origins
  cell14_development_test_origins
  cell14_development_purged_origins

  cell14_development_forecast
  cell14_development_train
  cell14_development_validation
  cell14_development_test

  cell14_development_horizon_summary
  cell14_development_clean_history_summary
  cell14_development_attack_type_coverage
  cell14_development_type_horizon_support
  cell14_development_graph_novelty
  cell14_development_time_ranges
  cell14_development_split_summary

  cell14_strict_origin_split
  cell14_strict_chronological_forecast
  cell14_strict_train
  cell14_strict_validation
  cell14_strict_test
  cell14_strict_horizon_summary
  cell14_strict_novelty

  cell14_episode_table
  cell14_segment_summary
  cell14_interpretation_flags
  cell14_forbidden_model_features
  cell14_metadata

Convenience aliases:
  train_forecast_df
  validation_forecast_df
  test_forecast_df


NEXT
----
DO NOT write/train the model yet.

First inspect this Cell 14 output, especially:

  1. DEVELOPMENT SPLIT — PER-HORIZON TARGET SUPPORT
  2. DEVELOPMENT SPLIT — CLEAN-HISTORY PRE-ATTACK SUPPORT
  3. DEVELOPMENT SPLIT — ATTACK-TYPE COVERAGE
  4. DEVELOPMENT SPLIT — PER-TYPE / PER-HORIZON POSITIVE SUPPORT
  5. DEVELOPMENT SPLIT — SOURCE / DESTINATION / EDGE NOVELTY
  6. SCIENTIFIC INTERPRETATION FLAGS
  7. STRICT CHRONOLOGICAL HOLDOUT — FUTURE NOVELTY

Cell 15 should be designed from THESE observed split statistics.
"""
)

print(SEP)

CELL 14 — FINAL LEAKAGE-SAFE DEVELOPMENT SPLIT + STRICT CHRONOLOGICAL HOLDOUT

Configuration
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------
History length                         : 15 minutes
Forecast horizons                      : [1, 5, 10, 15, 30]
Maximum forecast horizon               : 30 minutes
History span before origin             : 14 minutes
Minimum inter-partition origin gap     : 45 minutes

Development protocol
  Train fraction                       : 0.60
  Validation fraction                  : 0.20
  Test fraction                        : 0.20

Strict chronological protocol
  Train fraction                       : 0.60
  Validation fraction                  : 0.20
  Test fraction                        : 0.20

Scaling performed                      : NO
Resampling performed                   : NO
Model trained                          : NO

UPSTR

In [25]:
# ==================================================================================================
# CELL 14A — ATTACK-TYPE × EPISODE SPLITABILITY AUDIT
# ==================================================================================================
#
# PURPOSE
# -------
# Determine whether each attack type has genuinely independent temporal
# episodes that can support leakage-safe TRAIN / VALIDATION / TEST allocation.
#
# IMPORTANT
# ---------
# This cell is DIAGNOSTIC ONLY.
#
# It does NOT:
#   - create a train/validation/test split
#   - scale features
#   - resample data
#   - train a model
#
# WHY THIS CELL EXISTS
# --------------------
# CICIDS2017 attack types are strongly concentrated inside individual
# capture segments. Therefore "attack type occurs in one segment" does
# NOT by itself tell us whether that attack type is safely splittable.
#
# We must inspect independent attack episodes WITHIN each segment.
#
# The audit answers:
#
#   1. How many temporal episodes does each attack type have?
#   2. How many forecasting origins are associated with each episode?
#   3. How many clean-history forecasting origins exist?
#   4. Which horizons are supported?
#   5. Are there at least two/three independent usable episodes?
#   6. Could the attack type potentially support:
#         TRAIN only
#         TRAIN + EVALUATION
#         TRAIN + VALIDATION + TEST
#
# No episode is actually assigned to a partition here.
# Cell 15 will use this evidence to construct the final split.
# ==================================================================================================

import ast
import json
import numpy as np
import pandas as pd

SEP = "=" * 170
SUB = "-" * 170

# --------------------------------------------------------------------------------------------------
# CONFIGURATION
# --------------------------------------------------------------------------------------------------

EPISODE_GAPS = [0, 1, 2, 5]

# We use the 5-minute-gap definition as the conservative campaign-like
# structure for FINAL splittability decisions.
FINAL_EPISODE_GAP = 5

EXPECTED_HORIZONS = [1, 5, 10, 15, 30]

print(SEP)
print("CELL 14A — ATTACK-TYPE × EPISODE SPLITABILITY AUDIT")
print(SEP)

print("\nConfiguration")
print(SUB)
print(f"Episode gaps audited              : {EPISODE_GAPS}")
print(f"Final conservative episode gap    : {FINAL_EPISODE_GAP} minutes")
print(f"Expected forecasting horizons     : {EXPECTED_HORIZONS}")
print("Split created                     : NO")
print("Scaling performed                 : NO")
print("Resampling performed              : NO")
print("Model trained                     : NO")


# ==================================================================================================
# 1. LOAD CANONICAL INPUTS
# ==================================================================================================

print("\n" + SEP)
print("CANONICAL INPUT VALIDATION")
print(SEP)

if "minute_df" not in globals():
    raise RuntimeError(
        "minute_df not found. Run canonical Cell 9 first."
    )

forecast_candidates = [
    "cell12_forecast_metadata",
    "cell12_forecast_metadata_long",
    "forecast_meta",
]

forecast_name = None
forecast = None

required_forecast_columns = {
    "Segment_ID",
    "Origin_Time",
    "Target_Time",
    "Horizon_Minutes",
    "Future_Attack_State",
    "Future_Attack_Types",
    "Clean_History",
}

for name in forecast_candidates:

    if name not in globals():
        continue

    obj = globals()[name]

    if (
        isinstance(obj, pd.DataFrame)
        and required_forecast_columns.issubset(obj.columns)
    ):
        forecast_name = name
        forecast = obj.copy()
        break

if forecast is None:
    raise RuntimeError(
        "Canonical Cell 12 forecasting metadata could not be found."
    )

minute = minute_df.copy()

required_minute_columns = {
    "Segment_ID",
    "Minute",
    "Attack_State",
    "Attack_Types",
}

missing = required_minute_columns - set(minute.columns)

if missing:
    raise RuntimeError(
        "minute_df is missing required columns: "
        + ", ".join(sorted(missing))
    )

print(f"Minute source                     : minute_df")
print(f"Forecast source                   : {forecast_name}")
print(f"Minute rows                       : {len(minute):,}")
print(f"Forecast rows                     : {len(forecast):,}")


# ==================================================================================================
# 2. NORMALIZATION
# ==================================================================================================

def normalize_collection(value):

    if isinstance(value, list):
        return value

    if isinstance(value, tuple):
        return list(value)

    if isinstance(value, set):
        return list(value)

    if value is None:
        return []

    try:
        if pd.isna(value):
            return []
    except Exception:
        pass

    if isinstance(value, str):

        s = value.strip()

        if s in {
            "",
            "[]",
            "{}",
            "None",
            "nan",
            "NaN",
        }:
            return []

        try:
            x = json.loads(s)

            if isinstance(x, list):
                return x

            if x is None:
                return []

            return [x]

        except Exception:
            pass

        try:
            x = ast.literal_eval(s)

            if isinstance(
                x,
                (list, tuple, set),
            ):
                return list(x)

            if x is None:
                return []

            return [x]

        except Exception:
            return [s]

    return [value]


minute["Minute"] = pd.to_datetime(
    minute["Minute"]
)

minute["Segment_ID"] = (
    minute["Segment_ID"]
    .astype(int)
)

minute["Attack_State"] = (
    minute["Attack_State"]
    .astype(int)
)

minute["Attack_Types"] = (
    minute["Attack_Types"]
    .apply(normalize_collection)
)

forecast["Origin_Time"] = pd.to_datetime(
    forecast["Origin_Time"]
)

forecast["Target_Time"] = pd.to_datetime(
    forecast["Target_Time"]
)

forecast["Segment_ID"] = (
    forecast["Segment_ID"]
    .astype(int)
)

forecast["Horizon_Minutes"] = (
    forecast["Horizon_Minutes"]
    .astype(int)
)

forecast["Future_Attack_State"] = (
    forecast["Future_Attack_State"]
    .astype(int)
)

forecast["Clean_History"] = (
    forecast["Clean_History"]
    .astype(int)
)

forecast["Future_Attack_Types"] = (
    forecast["Future_Attack_Types"]
    .apply(normalize_collection)
)

minute = (
    minute
    .sort_values(
        ["Segment_ID", "Minute"]
    )
    .reset_index(drop=True)
)

forecast = (
    forecast
    .sort_values(
        [
            "Segment_ID",
            "Origin_Time",
            "Horizon_Minutes",
        ]
    )
    .reset_index(drop=True)
)

print("✓ Canonical collections normalized.")


# ==================================================================================================
# 3. BASIC CONTRACT CHECK
# ==================================================================================================

print("\n" + SEP)
print("FORECASTING CONTRACT CHECK")
print(SEP)

found_horizons = sorted(
    forecast["Horizon_Minutes"]
    .unique()
    .tolist()
)

duplicate_keys = int(
    forecast.duplicated(
        [
            "Segment_ID",
            "Origin_Time",
            "Horizon_Minutes",
        ]
    ).sum()
)

alignment_errors = int(
    (
        forecast["Target_Time"]
        != (
            forecast["Origin_Time"]
            + pd.to_timedelta(
                forecast["Horizon_Minutes"],
                unit="m",
            )
        )
    ).sum()
)

print(f"Forecast horizons                 : {found_horizons}")
print(f"Duplicate forecast keys           : {duplicate_keys}")
print(f"Target-time alignment errors      : {alignment_errors}")

if found_horizons != EXPECTED_HORIZONS:
    raise RuntimeError(
        "Unexpected forecasting horizons."
    )

if duplicate_keys != 0:
    raise RuntimeError(
        "Duplicate canonical forecast keys detected."
    )

if alignment_errors != 0:
    raise RuntimeError(
        "Canonical target-time alignment is invalid."
    )

print("✓ Forecasting contract validated.")


# ==================================================================================================
# 4. DISCOVER ATTACK TYPES
# ==================================================================================================

all_attack_types = sorted({
    str(attack_type)

    for values in minute.loc[
        minute["Attack_State"] == 1,
        "Attack_Types",
    ]

    for attack_type in values
})

print("\n" + SEP)
print("ATTACK TYPES DISCOVERED")
print(SEP)

print(f"Attack types found                : {len(all_attack_types)}")

for i, attack_type in enumerate(
    all_attack_types,
    start=1,
):
    print(f"{i:>2}. {attack_type}")


# ==================================================================================================
# 5. ATTACK-TYPE-SPECIFIC EPISODE RECONSTRUCTION
# ==================================================================================================
#
# IMPORTANT:
#
# Episodes are reconstructed PER ATTACK TYPE.
#
# This is different from simply reconstructing generic "Attack_State"
# episodes because two attack types can occur close together without
# necessarily being the same attack-type event.
#
# For each attack type:
#
#   attack minute ---- attack minute ---- benign/other minute ---- attack minute
#
# is grouped according to the selected benign-gap tolerance.
# ==================================================================================================

def reconstruct_type_episodes(
    minute_data,
    attack_type,
    gap_tolerance,
):

    rows = []
    episode_counter = 0

    for seg, g in minute_data.groupby(
        "Segment_ID",
        sort=True,
    ):

        type_minutes = []

        for row in g.itertuples():

            if int(row.Attack_State) != 1:
                continue

            values = {
                str(x)
                for x in normalize_collection(
                    row.Attack_Types
                )
            }

            if attack_type in values:
                type_minutes.append(
                    row.Minute
                )

        type_minutes = sorted(
            type_minutes
        )

        if not type_minutes:
            continue

        episode_start = type_minutes[0]
        previous = type_minutes[0]

        for current in type_minutes[1:]:

            minute_delta = int(
                (
                    current - previous
                ).total_seconds()
                / 60
            )

            # Number of minutes BETWEEN the two
            # attack-type-positive minutes.
            benign_gap = minute_delta - 1

            if benign_gap > gap_tolerance:

                rows.append({
                    "Attack_Type":
                        attack_type,

                    "Episode_Gap":
                        gap_tolerance,

                    "Type_Episode_ID":
                        episode_counter,

                    "Segment_ID":
                        int(seg),

                    "Episode_Start":
                        episode_start,

                    "Episode_End":
                        previous,
                })

                episode_counter += 1
                episode_start = current

            previous = current

        rows.append({
            "Attack_Type":
                attack_type,

            "Episode_Gap":
                gap_tolerance,

            "Type_Episode_ID":
                episode_counter,

            "Segment_ID":
                int(seg),

            "Episode_Start":
                episode_start,

            "Episode_End":
                previous,
        })

        episode_counter += 1

    result = pd.DataFrame(rows)

    if len(result):

        result["Episode_Duration_Minutes"] = (
            (
                result["Episode_End"]
                - result["Episode_Start"]
            )
            .dt.total_seconds()
            .div(60)
            .astype(int)
            + 1
        )

    return result


episode_tables = {}

episode_count_rows = []

for gap in EPISODE_GAPS:

    gap_parts = []

    for attack_type in all_attack_types:

        ep = reconstruct_type_episodes(
            minute,
            attack_type,
            gap,
        )

        if len(ep):
            gap_parts.append(ep)

        episode_count_rows.append({
            "Attack_Type":
                attack_type,

            "Episode_Gap":
                gap,

            "Episode_Count":
                len(ep),
        })

    if gap_parts:

        episode_tables[gap] = pd.concat(
            gap_parts,
            ignore_index=True,
        )

    else:

        episode_tables[gap] = pd.DataFrame()


cell14a_episode_counts = pd.DataFrame(
    episode_count_rows
)

print("\n" + SEP)
print("ATTACK-TYPE EPISODE COUNTS")
print(SEP)

episode_count_pivot = (
    cell14a_episode_counts
    .pivot(
        index="Attack_Type",
        columns="Episode_Gap",
        values="Episode_Count",
    )
    .reset_index()
)

episode_count_pivot.columns = [
    (
        "Attack_Type"
        if c == "Attack_Type"
        else f"Episodes_Gap_{c}min"
    )
    for c in episode_count_pivot.columns
]

print(
    episode_count_pivot
    .to_string(index=False)
)


# ==================================================================================================
# 6. FINAL CONSERVATIVE EPISODE TABLE
# ==================================================================================================

final_episodes = (
    episode_tables[
        FINAL_EPISODE_GAP
    ]
    .copy()
)

if final_episodes.empty:
    raise RuntimeError(
        "No conservative attack-type episodes reconstructed."
    )

# Globally unique identifier.
final_episodes[
    "Canonical_Type_Episode_ID"
] = (
    final_episodes[
        "Attack_Type"
    ].astype(str)
    + "__SEG"
    + final_episodes[
        "Segment_ID"
    ].astype(str)
    + "__EP"
    + final_episodes[
        "Type_Episode_ID"
    ].astype(str)
)

print("\n" + SEP)
print(
    f"CONSERVATIVE {FINAL_EPISODE_GAP}-MINUTE ATTACK-TYPE EPISODES"
)
print(SEP)

print(
    final_episodes[
        [
            "Attack_Type",
            "Canonical_Type_Episode_ID",
            "Segment_ID",
            "Episode_Start",
            "Episode_End",
            "Episode_Duration_Minutes",
        ]
    ]
    .sort_values(
        [
            "Attack_Type",
            "Segment_ID",
            "Episode_Start",
        ]
    )
    .to_string(index=False)
)


# ==================================================================================================
# 7. MAP FORECAST TARGETS TO ATTACK-TYPE EPISODES
# ==================================================================================================
#
# IMPORTANT:
#
# We map the TARGET TIME to an episode.
#
# We do NOT assign the forecast origin merely because it occurs near
# an attack episode.
#
# A row belongs to an attack-type episode only when:
#
#     Future_Attack_State == 1
#     attack type is present at Target_Time
#     Target_Time falls inside that attack-type episode
#
# ==================================================================================================

episode_lookup = {}

for row in final_episodes.itertuples():

    key = (
        str(row.Attack_Type),
        int(row.Segment_ID),
    )

    episode_lookup.setdefault(
        key,
        [],
    ).append(
        (
            row.Episode_Start,
            row.Episode_End,
            row.Canonical_Type_Episode_ID,
        )
    )


mapped_rows = []

attack_forecast = forecast[
    forecast["Future_Attack_State"] == 1
].copy()

for row in attack_forecast.itertuples():

    target_types = {
        str(x)
        for x in normalize_collection(
            row.Future_Attack_Types
        )
    }

    for attack_type in target_types:

        candidates = episode_lookup.get(
            (
                attack_type,
                int(row.Segment_ID),
            ),
            [],
        )

        matched_episode = None

        for (
            episode_start,
            episode_end,
            episode_id,
        ) in candidates:

            if (
                row.Target_Time
                >= episode_start
                and row.Target_Time
                <= episode_end
            ):

                matched_episode = episode_id
                break

        if matched_episode is None:

            raise RuntimeError(
                "Attack forecast target could not be mapped "
                "to its conservative attack-type episode.\n"
                f"Attack type: {attack_type}\n"
                f"Segment: {row.Segment_ID}\n"
                f"Target time: {row.Target_Time}"
            )

        mapped_rows.append({
            "Attack_Type":
                attack_type,

            "Canonical_Type_Episode_ID":
                matched_episode,

            "Segment_ID":
                int(row.Segment_ID),

            "Origin_Time":
                row.Origin_Time,

            "Target_Time":
                row.Target_Time,

            "Horizon_Minutes":
                int(row.Horizon_Minutes),

            "Clean_History":
                int(row.Clean_History),
        })


cell14a_forecast_episode_map = pd.DataFrame(
    mapped_rows
)

print("\n" + SEP)
print("FORECAST TARGET → ATTACK-TYPE EPISODE MAPPING")
print(SEP)

print(
    f"Mapped attack-type forecast rows : "
    f"{len(cell14a_forecast_episode_map):,}"
)

print(
    "Unmapped rows                    : 0"
)


# ==================================================================================================
# 8. EPISODE-LEVEL FORECAST SUPPORT
# ==================================================================================================

episode_support_rows = []

for episode in final_episodes.itertuples():

    q = cell14a_forecast_episode_map[
        cell14a_forecast_episode_map[
            "Canonical_Type_Episode_ID"
        ]
        == episode.Canonical_Type_Episode_ID
    ].copy()

    unique_origins = (
        q[
            [
                "Segment_ID",
                "Origin_Time",
            ]
        ]
        .drop_duplicates()
    )

    clean_origins = (
        q.loc[
            q["Clean_History"] == 1,
            [
                "Segment_ID",
                "Origin_Time",
            ],
        ]
        .drop_duplicates()
    )

    horizons = sorted(
        q["Horizon_Minutes"]
        .unique()
        .tolist()
    )

    episode_support_rows.append({
        "Attack_Type":
            episode.Attack_Type,

        "Canonical_Type_Episode_ID":
            episode.Canonical_Type_Episode_ID,

        "Segment_ID":
            episode.Segment_ID,

        "Episode_Start":
            episode.Episode_Start,

        "Episode_End":
            episode.Episode_End,

        "Episode_Duration_Minutes":
            episode.Episode_Duration_Minutes,

        "Forecast_Rows":
            len(q),

        "Unique_Forecast_Origins":
            len(unique_origins),

        "Clean_History_Origins":
            len(clean_origins),

        "Supported_Horizons":
            horizons,

        "All_5_Horizons_Present":
            int(
                horizons
                == EXPECTED_HORIZONS
            ),
    })


cell14a_episode_forecast_support = pd.DataFrame(
    episode_support_rows
)

print("\n" + SEP)
print("EPISODE-LEVEL FORECASTING SUPPORT")
print(SEP)

print(
    cell14a_episode_forecast_support[
        [
            "Attack_Type",
            "Canonical_Type_Episode_ID",
            "Segment_ID",
            "Episode_Start",
            "Episode_End",
            "Episode_Duration_Minutes",
            "Forecast_Rows",
            "Unique_Forecast_Origins",
            "Clean_History_Origins",
            "Supported_Horizons",
            "All_5_Horizons_Present",
        ]
    ]
    .sort_values(
        [
            "Attack_Type",
            "Episode_Start",
        ]
    )
    .to_string(index=False)
)


# ==================================================================================================
# 9. ATTACK-TYPE SPLITABILITY SUMMARY
# ==================================================================================================
#
# IMPORTANT:
#
# We intentionally distinguish several concepts:
#
# Episode_Count
#     Number of independent conservative attack-type episodes.
#
# Forecastable_Episodes
#     Episodes associated with at least one canonical future attack target.
#
# Clean_Forecastable_Episodes
#     Episodes associated with at least one clean-history forecasting origin.
#
# Potential_KnownPattern_TrainEval
#     At least TWO independent forecastable episodes.
#
#     This means one episode could potentially establish training support
#     and another could potentially be used for validation OR test.
#
# Potential_KnownPattern_TrainValTest
#     At least THREE independent forecastable episodes.
#
#     This means TRAIN / VALIDATION / TEST episode-level representation
#     may be possible.
#
# NOTE:
# These are POTENTIAL classifications.
#
# Cell 15 must still verify complete history/future footprints and prevent
# any origin from crossing selected partition boundaries.
# ==================================================================================================

type_summary_rows = []

for attack_type in all_attack_types:

    ep = cell14a_episode_forecast_support[
        cell14a_episode_forecast_support[
            "Attack_Type"
        ]
        == attack_type
    ].copy()

    episode_count = len(ep)

    forecastable = ep[
        ep["Unique_Forecast_Origins"] > 0
    ]

    clean_forecastable = ep[
        ep["Clean_History_Origins"] > 0
    ]

    all_horizon_episodes = ep[
        ep["All_5_Horizons_Present"] == 1
    ]

    segments = sorted(
        ep["Segment_ID"]
        .unique()
        .tolist()
    )

    forecastable_episode_count = len(
        forecastable
    )

    clean_episode_count = len(
        clean_forecastable
    )

    potential_train_eval = (
        forecastable_episode_count >= 2
    )

    potential_train_val_test = (
        forecastable_episode_count >= 3
    )

    potential_clean_train_eval = (
        clean_episode_count >= 2
    )

    potential_clean_train_val_test = (
        clean_episode_count >= 3
    )

    # Descriptive recommendation category.
    #
    # This is NOT yet an actual partition assignment.

    if potential_train_val_test:

        structural_status = (
            "POTENTIAL TRAIN+VAL+TEST"
        )

    elif potential_train_eval:

        structural_status = (
            "POTENTIAL TRAIN+ONE-EVAL"
        )

    elif forecastable_episode_count == 1:

        structural_status = (
            "ONE FORECASTABLE EPISODE"
        )

    else:

        structural_status = (
            "NO FORECASTABLE EPISODE"
        )

    type_summary_rows.append({
        "Attack_Type":
            attack_type,

        "Segments":
            segments,

        "Segment_Count":
            len(segments),

        "Conservative_Episodes":
            episode_count,

        "Forecastable_Episodes":
            forecastable_episode_count,

        "Clean_Forecastable_Episodes":
            clean_episode_count,

        "Episodes_With_All_5_Horizons":
            len(all_horizon_episodes),

        "Total_Unique_Forecast_Origins":
            int(
                ep[
                    "Unique_Forecast_Origins"
                ].sum()
            ),

        "Total_Clean_History_Origins":
            int(
                ep[
                    "Clean_History_Origins"
                ].sum()
            ),

        "Potential_TrainPlusEval":
            int(
                potential_train_eval
            ),

        "Potential_TrainValTest":
            int(
                potential_train_val_test
            ),

        "Potential_Clean_TrainPlusEval":
            int(
                potential_clean_train_eval
            ),

        "Potential_Clean_TrainValTest":
            int(
                potential_clean_train_val_test
            ),

        "Structural_Status":
            structural_status,
    })


cell14a_attack_type_splitability = pd.DataFrame(
    type_summary_rows
)

print("\n" + SEP)
print("FINAL ATTACK-TYPE SPLITABILITY SUMMARY")
print(SEP)

print(
    cell14a_attack_type_splitability[
        [
            "Attack_Type",
            "Segments",
            "Conservative_Episodes",
            "Forecastable_Episodes",
            "Clean_Forecastable_Episodes",
            "Episodes_With_All_5_Horizons",
            "Total_Unique_Forecast_Origins",
            "Total_Clean_History_Origins",
            "Potential_TrainPlusEval",
            "Potential_TrainValTest",
            "Potential_Clean_TrainPlusEval",
            "Potential_Clean_TrainValTest",
            "Structural_Status",
        ]
    ]
    .sort_values(
        [
            "Potential_TrainValTest",
            "Potential_TrainPlusEval",
            "Forecastable_Episodes",
            "Total_Unique_Forecast_Origins",
        ],
        ascending=[
            False,
            False,
            False,
            False,
        ],
    )
    .to_string(index=False)
)


# ==================================================================================================
# 10. CLEAN-HISTORY FORECASTABILITY WARNING TABLE
# ==================================================================================================

print("\n" + SEP)
print("CLEAN-HISTORY / PRE-ATTACK SUPPORT")
print(SEP)

clean_support = (
    cell14a_attack_type_splitability[
        [
            "Attack_Type",
            "Forecastable_Episodes",
            "Clean_Forecastable_Episodes",
            "Total_Unique_Forecast_Origins",
            "Total_Clean_History_Origins",
            "Potential_Clean_TrainPlusEval",
            "Potential_Clean_TrainValTest",
        ]
    ]
    .copy()
)

print(
    clean_support
    .sort_values(
        [
            "Total_Clean_History_Origins",
            "Attack_Type",
        ],
        ascending=[
            True,
            True,
        ],
    )
    .to_string(index=False)
)


# ==================================================================================================
# 11. EPISODES WITH ZERO CLEAN-HISTORY SUPPORT
# ==================================================================================================

print("\n" + SEP)
print("EPISODES WITH NO CLEAN-HISTORY FORECAST ORIGINS")
print(SEP)

zero_clean = (
    cell14a_episode_forecast_support[
        (
            cell14a_episode_forecast_support[
                "Unique_Forecast_Origins"
            ]
            > 0
        )
        &
        (
            cell14a_episode_forecast_support[
                "Clean_History_Origins"
            ]
            == 0
        )
    ]
    .copy()
)

if len(zero_clean):

    print(
        zero_clean[
            [
                "Attack_Type",
                "Canonical_Type_Episode_ID",
                "Segment_ID",
                "Episode_Start",
                "Episode_End",
                "Unique_Forecast_Origins",
                "Supported_Horizons",
            ]
        ]
        .to_string(index=False)
    )

else:

    print("NONE")


# ==================================================================================================
# 12. FINAL DECISION SUPPORT
# ==================================================================================================

print("\n" + SEP)
print("CELL 14A — DECISION SUPPORT")
print(SEP)

three_way_types = (
    cell14a_attack_type_splitability.loc[
        cell14a_attack_type_splitability[
            "Potential_TrainValTest"
        ]
        == 1,
        "Attack_Type",
    ]
    .tolist()
)

two_way_types = (
    cell14a_attack_type_splitability.loc[
        (
            cell14a_attack_type_splitability[
                "Potential_TrainPlusEval"
            ]
            == 1
        )
        &
        (
            cell14a_attack_type_splitability[
                "Potential_TrainValTest"
            ]
            == 0
        ),
        "Attack_Type",
    ]
    .tolist()
)

single_episode_types = (
    cell14a_attack_type_splitability.loc[
        cell14a_attack_type_splitability[
            "Forecastable_Episodes"
        ]
        <= 1,
        "Attack_Type",
    ]
    .tolist()
)

print("\nPotential TRAIN + VALIDATION + TEST attack types:")
if three_way_types:
    for x in three_way_types:
        print(f"  ✓ {x}")
else:
    print("  NONE")

print("\nPotential TRAIN + ONE evaluation partition attack types:")
if two_way_types:
    for x in two_way_types:
        print(f"  • {x}")
else:
    print("  NONE")

print("\nAttack types with <=1 forecastable conservative episode:")
if single_episode_types:
    for x in single_episode_types:
        print(f"  • {x}")
else:
    print("  NONE")


# ==================================================================================================
# 13. FINAL INTERPRETATION
# ==================================================================================================

print("\n" + SEP)
print("FINAL INTERPRETATION")
print(SEP)

print(
f"""
This cell has NOT created a split.

It has answered the structural question required before Cell 15:

    Can each attack type legitimately appear in multiple dataset
    partitions without cutting the same conservative attack episode?

Definitions:

    Potential_TrainPlusEval = 1

        At least two independent {FINAL_EPISODE_GAP}-minute-gap
        forecastable episodes exist.

        Therefore one episode may potentially establish TRAIN support
        while another may potentially support VALIDATION or TEST.

    Potential_TrainValTest = 1

        At least three independent {FINAL_EPISODE_GAP}-minute-gap
        forecastable episodes exist.

        Therefore TRAIN + VALIDATION + TEST representation may be
        structurally possible.

These flags do NOT authorize Cell 15 to split blindly.

Cell 15 must still enforce:

    1. all five horizons from one origin remain atomic;
    2. complete history-to-max-horizon footprints remain in one partition;
    3. attack episodes are never cut across partitions;
    4. validation attack types are represented in TRAIN;
    5. test attack types are represented in TRAIN;
    6. clean-history forecasting support is reported separately;
    7. types that cannot be safely evaluated outside TRAIN are not
       artificially split merely to manufacture class coverage.

This audit should be retained permanently as methodological evidence
for the final split design.
"""
)

print(SEP)
print("CELL 14A COMPLETE")
print(SEP)

CELL 14A — ATTACK-TYPE × EPISODE SPLITABILITY AUDIT

Configuration
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Episode gaps audited              : [0, 1, 2, 5]
Final conservative episode gap    : 5 minutes
Expected forecasting horizons     : [1, 5, 10, 15, 30]
Split created                     : NO
Scaling performed                 : NO
Resampling performed              : NO
Model trained                     : NO

CANONICAL INPUT VALIDATION
Minute source                     : minute_df
Forecast source                   : cell12_forecast_metadata
Minute rows                       : 2,454
Forecast rows                     : 10,070
✓ Canonical collections normalized.

FORECASTING CONTRACT CHECK
Forecast horizons                 : [1, 5, 10, 15, 30]
Duplicate forecast keys           : 0
Target-time alignment errors      : 0
✓ Forecasting contract validated.

ATTACK

In [26]:
checks = [
    "minute_df",
    "cell12_forecast_metadata",
    "cell14a_forecast_episode_map",
]

for x in checks:
    if x in globals():
        obj = globals()[x]
        shape = getattr(obj, "shape", None)
        print(f"✓ {x:<40} FOUND   shape={shape}")
    else:
        print(f"✗ {x:<40} MISSING")

✓ minute_df                                FOUND   shape=(2454, 103)
✓ cell12_forecast_metadata                 FOUND   shape=(10070, 29)
✓ cell14a_forecast_episode_map             FOUND   shape=(2439, 7)


In [28]:
# ==================================================================================================
# CELL 15 — PERMANENT LEAKAGE-SAFE TEMPORAL SPLIT + EVALUATION CONTRACT
# ==================================================================================================
#
# FINAL PURPOSE
# -------------
# Freeze the train / validation / test partition BEFORE any model-dependent
# preprocessing or training.
#
# This cell implements the final evaluation philosophy of the project:
#
#   PRIMARY EXPERIMENT:
#       Strict out-of-time forecasting.
#
#   Evaluation attack targets are subsequently labelled as:
#
#       KNOWN_TO_TRAIN = 1
#           The attack type was represented in TRAIN.
#
#       KNOWN_TO_TRAIN = 0
#           The attack type was NOT represented in TRAIN.
#
#   Therefore:
#
#       * binary attack forecasting can be evaluated on ALL retained targets;
#
#       * closed-set attack-type classification is evaluated separately on
#         attack types represented in TRAIN;
#
#       * unseen-type attacks remain valuable for binary/generalization analysis;
#
#       * clean-history attack targets are separately marked for genuine
#         pre-attack forecasting evaluation.
#
#
# HARD LEAKAGE RULES
# ------------------
#
#   1. One origin's five horizons NEVER cross partitions.
#
#   2. Complete raw temporal footprint:
#
#           [Origin_Time - 14 min, Origin_Time + 30 min]
#
#      NEVER overlaps another retained partition inside the same capture segment.
#
#   3. Capture segments are treated as atomic split units.
#
#      This is intentionally stronger than splitting individual origins inside
#      the same segment.
#
#   4. No attack episode is deliberately cut across partitions.
#
#   5. TRAIN / VALIDATION / TEST are selected without looking at model
#      performance.
#
#
# THIS CELL DOES NOT
# ------------------
#
#   * scale
#   * normalize
#   * impute using learned statistics
#   * resample
#   * perform SMOTE
#   * select features using labels
#   * train a model
#
#
# REQUIRED UPSTREAM OBJECTS
# -------------------------
#
#   minute_df
#   Cell 12 canonical forecasting metadata
#
# Optional but strongly preferred:
#
#   cell14a_forecast_episode_map
#
#
# PERMANENT DOWNSTREAM OUTPUTS
# ----------------------------
#
#   cell15_origin_table
#   cell15_forecast_metadata
#
#   cell15_train_origins
#   cell15_validation_origins
#   cell15_test_origins
#
#   cell15_train_forecast
#   cell15_validation_forecast
#   cell15_test_forecast
#
#   cell15_partition_lookup
#   cell15_attack_type_support
#   cell15_evaluation_attack_metadata
#   cell15_split_summary
#   cell15_split_metadata
#
# ==================================================================================================


import ast
import json
import itertools
from collections import Counter

import numpy as np
import pandas as pd


# ==================================================================================================
# 0. DISPLAY / CONFIGURATION
# ==================================================================================================

SEP = "=" * 180
SUB = "-" * 180

HISTORY_LENGTH = 15
HISTORY_BACK_MINUTES = HISTORY_LENGTH - 1

EXPECTED_HORIZONS = [1, 5, 10, 15, 30]
MAX_HORIZON = max(EXPECTED_HORIZONS)

EXPECTED_ORIGINS = 2014

# ----------------------------------------------------------------------------------
# Desired split proportions.
#
# IMPORTANT:
# These are TARGETS, not hard quotas.
#
# We are splitting whole temporal capture segments, so exact 60/20/20 is neither
# expected nor desirable if obtaining it would require cutting a segment.
# ----------------------------------------------------------------------------------

TARGET_TRAIN_FRAC = 0.60
TARGET_VALIDATION_FRAC = 0.20
TARGET_TEST_FRAC = 0.20

# ----------------------------------------------------------------------------------
# Minimum structural requirements.
# ----------------------------------------------------------------------------------

MIN_VALIDATION_ORIGINS = 100
MIN_TEST_ORIGINS = 100

MIN_VALIDATION_ATTACK_TARGETS = 1
MIN_TEST_ATTACK_TARGETS = 1

MIN_VALIDATION_CLEAN_HISTORY_ATTACK_TARGETS = 1
MIN_TEST_CLEAN_HISTORY_ATTACK_TARGETS = 1

# ----------------------------------------------------------------------------------
# Split search philosophy.
#
# We search WHOLE capture-segment assignments.
#
# We do NOT optimize on:
#
#   accuracy
#   F1
#   AUROC
#   model loss
#   model predictions
#
# Candidate selection uses ONLY structural support:
#
#   partition size
#   attack support
#   clean-history attack support
#   known/unseen composition
#
# ----------------------------------------------------------------------------------


print(SEP)
print("CELL 15 — PERMANENT LEAKAGE-SAFE TEMPORAL SPLIT + EVALUATION CONTRACT")
print(SEP)

print("\nConfiguration")
print(SUB)

print(f"History length                         : {HISTORY_LENGTH} minutes")
print(f"History span before origin             : {HISTORY_BACK_MINUTES} minutes")
print(f"Forecast horizons                      : {EXPECTED_HORIZONS}")
print(f"Maximum forecast horizon               : {MAX_HORIZON} minutes")

print()

print("Primary evaluation                     : strict out-of-time forecasting")
print("Split unit                             : complete capture segment")
print("Known-type requirement                 : REPORT ONLY — NOT a split constraint")
print("Unseen attack types in evaluation      : ALLOWED and explicitly marked")

print()

print(f"Target TRAIN fraction                  : {TARGET_TRAIN_FRAC:.2f}")
print(f"Target VALIDATION fraction             : {TARGET_VALIDATION_FRAC:.2f}")
print(f"Target TEST fraction                   : {TARGET_TEST_FRAC:.2f}")

print()

print("Scaling performed                      : NO")
print("Resampling performed                   : NO")
print("Feature selection performed            : NO")
print("Model trained                          : NO")


# ==================================================================================================
# 1. HELPER FUNCTIONS
# ==================================================================================================

def normalize_collection(value):
    """
    Convert JSON strings / Python collection strings / tuples / sets / scalars
    into a normal Python list.
    """

    if isinstance(value, list):
        return value

    if isinstance(value, tuple):
        return list(value)

    if isinstance(value, set):
        return list(value)

    if value is None:
        return []

    try:
        if pd.isna(value):
            return []
    except Exception:
        pass

    if isinstance(value, str):

        s = value.strip()

        if s in {
            "",
            "[]",
            "{}",
            "None",
            "nan",
            "NaN",
            "<NA>",
        }:
            return []

        try:
            parsed = json.loads(s)

            if parsed is None:
                return []

            if isinstance(parsed, list):
                return parsed

            if isinstance(parsed, (tuple, set)):
                return list(parsed)

            return [parsed]

        except Exception:
            pass

        try:
            parsed = ast.literal_eval(s)

            if parsed is None:
                return []

            if isinstance(parsed, (list, tuple, set)):
                return list(parsed)

            return [parsed]

        except Exception:
            return [s]

    return [value]


def collection_union(series):
    """
    Union collection-valued rows into a sorted string list.
    """

    result = set()

    for value in series:

        for item in normalize_collection(value):
            result.add(str(item))

    return sorted(result)


def attack_type_set(df):
    """
    Return attack types occurring in attack-positive rows.
    """

    result = set()

    if len(df) == 0:
        return result

    positive = df[
        df["Future_Attack_State"] == 1
    ]

    for values in positive["Future_Attack_Types"]:

        result.update(
            str(x)
            for x in normalize_collection(values)
        )

    return result


def safe_ratio(a, b):

    if b == 0:
        return np.nan

    return a / b


def footprint_overlap(
    start_a,
    end_a,
    start_b,
    end_b,
):

    return (
        start_a <= end_b
        and start_b <= end_a
    )


# ==================================================================================================
# 2. DISCOVER / VALIDATE UPSTREAM OBJECTS
# ==================================================================================================

print("\n" + SEP)
print("UPSTREAM OBJECT VALIDATION")
print(SEP)


if "minute_df" not in globals():

    raise RuntimeError(
        "minute_df is missing. Run canonical Cell 9 first."
    )


forecast_candidates = [
    "cell12_forecast_metadata",
    "cell12_forecast_metadata_long",
    "forecast_meta",
]


required_forecast_columns = {
    "Segment_ID",
    "Origin_Time",
    "Target_Time",
    "Horizon_Minutes",
    "Future_Attack_State",
    "Future_Attack_Types",
    "Clean_History",
}


forecast_name = None
forecast = None


for candidate_name in forecast_candidates:

    if candidate_name not in globals():
        continue

    candidate = globals()[candidate_name]

    if not isinstance(candidate, pd.DataFrame):
        continue

    if required_forecast_columns.issubset(
        candidate.columns
    ):

        forecast_name = candidate_name
        forecast = candidate.copy()
        break


if forecast is None:

    raise RuntimeError(
        "Could not discover canonical Cell 12 forecasting metadata."
    )


minute = minute_df.copy()


print(f"minute_df rows                          : {len(minute):,}")
print(f"Forecast metadata source                : {forecast_name}")
print(f"Forecast metadata rows                  : {len(forecast):,}")


episode_map_available = (
    "cell14a_forecast_episode_map"
    in globals()
    and isinstance(
        cell14a_forecast_episode_map,
        pd.DataFrame,
    )
)


print(
    "Cell 14A episode map available         : "
    f"{episode_map_available}"
)


# ==================================================================================================
# 3. NORMALIZE CANONICAL SCHEMA
# ==================================================================================================

forecast["Segment_ID"] = (
    pd.to_numeric(
        forecast["Segment_ID"],
        errors="raise",
    )
    .astype(int)
)


forecast["Origin_Time"] = pd.to_datetime(
    forecast["Origin_Time"],
    errors="raise",
)


forecast["Target_Time"] = pd.to_datetime(
    forecast["Target_Time"],
    errors="raise",
)


forecast["Horizon_Minutes"] = (
    pd.to_numeric(
        forecast["Horizon_Minutes"],
        errors="raise",
    )
    .astype(int)
)


forecast["Future_Attack_State"] = (
    pd.to_numeric(
        forecast["Future_Attack_State"],
        errors="raise",
    )
    .astype(int)
)


forecast["Clean_History"] = (
    pd.to_numeric(
        forecast["Clean_History"],
        errors="raise",
    )
    .astype(int)
)


forecast["Future_Attack_Types"] = (
    forecast["Future_Attack_Types"]
    .apply(normalize_collection)
)


minute["Segment_ID"] = (
    pd.to_numeric(
        minute["Segment_ID"],
        errors="raise",
    )
    .astype(int)
)


minute["Minute"] = pd.to_datetime(
    minute["Minute"],
    errors="raise",
)


if "Attack_State" not in minute.columns:

    raise RuntimeError(
        "minute_df does not contain Attack_State."
    )


minute["Attack_State"] = (
    pd.to_numeric(
        minute["Attack_State"],
        errors="raise",
    )
    .astype(int)
)


if not set(
    forecast["Future_Attack_State"].unique()
).issubset({0, 1}):

    raise RuntimeError(
        "Future_Attack_State contains values other than 0/1."
    )


if not set(
    forecast["Clean_History"].unique()
).issubset({0, 1}):

    raise RuntimeError(
        "Clean_History contains values other than 0/1."
    )


print("✓ Canonical schema normalized.")


# ==================================================================================================
# 4. FORECASTING CONTRACT — HARD REVALIDATION
# ==================================================================================================

print("\n" + SEP)
print("FORECASTING CONTRACT REVALIDATION")
print(SEP)


observed_horizons = sorted(
    forecast["Horizon_Minutes"]
    .unique()
    .tolist()
)


duplicate_forecast_keys = int(
    forecast.duplicated(
        [
            "Segment_ID",
            "Origin_Time",
            "Horizon_Minutes",
        ]
    ).sum()
)


expected_target_time = (
    forecast["Origin_Time"]
    + pd.to_timedelta(
        forecast["Horizon_Minutes"],
        unit="m",
    )
)


target_alignment_errors = int(
    (
        forecast["Target_Time"]
        != expected_target_time
    ).sum()
)


origin_horizon_sets = (
    forecast
    .groupby(
        [
            "Segment_ID",
            "Origin_Time",
        ]
    )["Horizon_Minutes"]
    .apply(
        lambda s: tuple(
            sorted(
                set(
                    int(x)
                    for x in s.tolist()
                )
            )
        )
    )
)

expected_horizon_tuple = tuple(EXPECTED_HORIZONS)

bad_origin_horizon_sets = int(
    origin_horizon_sets.apply(
        lambda horizons:
            horizons != expected_horizon_tuple
    ).sum()
)


origin_count = int(
    forecast[
        [
            "Segment_ID",
            "Origin_Time",
        ]
    ]
    .drop_duplicates()
    .shape[0]
)


expected_forecast_rows = (
    origin_count
    * len(EXPECTED_HORIZONS)
)


print(f"Forecast horizons                      : {observed_horizons}")
print(f"Forecast origins                       : {origin_count:,}")
print(f"Expected forecast rows                 : {expected_forecast_rows:,}")
print(f"Actual forecast rows                   : {len(forecast):,}")
print(f"Duplicate forecast keys                : {duplicate_forecast_keys}")
print(f"Target-time alignment errors           : {target_alignment_errors}")
print(f"Origins with wrong horizon set         : {bad_origin_horizon_sets}")


if observed_horizons != EXPECTED_HORIZONS:

    raise RuntimeError(
        "Unexpected forecasting horizons."
    )


if duplicate_forecast_keys != 0:

    raise RuntimeError(
        "Duplicate forecast keys detected."
    )


if target_alignment_errors != 0:

    raise RuntimeError(
        "Target-time alignment errors detected."
    )


if bad_origin_horizon_sets != 0:

    raise RuntimeError(
        "At least one origin does not contain exactly the expected five horizons."
    )


if len(forecast) != expected_forecast_rows:

    raise RuntimeError(
        "Forecast row count does not equal origins × horizons."
    )


if origin_count != EXPECTED_ORIGINS:

    print(
        f"WARNING: expected {EXPECTED_ORIGINS:,} canonical origins "
        f"from the current notebook formulation, but found {origin_count:,}."
    )


print("✓ Forecasting contract validated.")


# ==================================================================================================
# 5. BUILD ATOMIC ORIGIN TABLE
# ==================================================================================================

print("\n" + SEP)
print("BUILDING ATOMIC FIVE-HORIZON ORIGIN TABLE")
print(SEP)


origin_rows = []


for (
    segment_id,
    origin_time
), g in forecast.groupby(
    [
        "Segment_ID",
        "Origin_Time",
    ],
    sort=True,
):

    if len(g) != len(EXPECTED_HORIZONS):

        raise RuntimeError(
            "Unexpected number of horizon rows for an origin."
        )

    clean_values = sorted(
        g["Clean_History"]
        .unique()
        .tolist()
    )

    if len(clean_values) != 1:

        raise RuntimeError(
            "Clean_History is inconsistent across horizons of one origin."
        )

    future_types = collection_union(
        g.loc[
            g["Future_Attack_State"] == 1,
            "Future_Attack_Types",
        ]
    )

    attack_horizons = sorted(
        g.loc[
            g["Future_Attack_State"] == 1,
            "Horizon_Minutes",
        ]
        .astype(int)
        .tolist()
    )

    origin_rows.append({

        "Segment_ID":
            int(segment_id),

        "Origin_Time":
            origin_time,

        "Footprint_Start":
            origin_time
            - pd.Timedelta(
                minutes=HISTORY_BACK_MINUTES
            ),

        "Footprint_End":
            origin_time
            + pd.Timedelta(
                minutes=MAX_HORIZON
            ),

        "Clean_History":
            int(clean_values[0]),

        "Any_Future_Attack":
            int(
                g["Future_Attack_State"].max()
            ),

        "Attack_Horizons":
            attack_horizons,

        "Future_Attack_Types":
            future_types,

        "Future_Attack_Type_Count":
            len(future_types),
    })


cell15_origin_table = (
    pd.DataFrame(origin_rows)
    .sort_values(
        [
            "Segment_ID",
            "Origin_Time",
        ]
    )
    .reset_index(drop=True)
)


print(
    f"Atomic origins                          : "
    f"{len(cell15_origin_table):,}"
)

print(
    "✓ One atomic record represents all five horizons of an origin."
)


# ==================================================================================================
# 6. BUILD CAPTURE-SEGMENT SUMMARY
# ==================================================================================================

print("\n" + SEP)
print("CAPTURE-SEGMENT SUMMARY")
print(SEP)


segment_rows = []


for segment_id in sorted(
    minute["Segment_ID"]
    .unique()
    .tolist()
):

    m = minute[
        minute["Segment_ID"]
        == segment_id
    ].copy()

    o = cell15_origin_table[
        cell15_origin_table["Segment_ID"]
        == segment_id
    ].copy()

    f = forecast[
        forecast["Segment_ID"]
        == segment_id
    ].copy()

    attack_types = sorted(
        attack_type_set(f)
    )

    attack_target_rows = int(
        f["Future_Attack_State"].sum()
    )

    clean_attack_target_rows = int(
        (
            (
                f["Future_Attack_State"] == 1
            )
            &
            (
                f["Clean_History"] == 1
            )
        ).sum()
    )

    segment_rows.append({

        "Segment_ID":
            int(segment_id),

        "Segment_Start":
            m["Minute"].min(),

        "Segment_End":
            m["Minute"].max(),

        "Minute_Rows":
            len(m),

        "Attack_Minutes":
            int(
                m["Attack_State"].sum()
            ),

        "Benign_Minutes":
            int(
                len(m)
                - m["Attack_State"].sum()
            ),

        "Forecast_Origins":
            len(o),

        "Forecast_Rows":
            len(f),

        "Attack_Target_Rows":
            attack_target_rows,

        "Clean_History_Attack_Target_Rows":
            clean_attack_target_rows,

        "Attack_Types":
            attack_types,

        "Attack_Type_Count":
            len(attack_types),
    })


cell15_segment_summary = (
    pd.DataFrame(segment_rows)
    .sort_values("Segment_ID")
    .reset_index(drop=True)
)


print(
    cell15_segment_summary
    .to_string(index=False)
)


# ==================================================================================================
# 7. DEFINE CANONICAL SEGMENT ORDER
# ==================================================================================================
#
# IMPORTANT
# ---------
#
# We do NOT infer chronological order by naïvely sorting the ambiguous raw
# datetime representation.
#
# Segment_ID was already created upstream from the validated CICIDS capture
# chronology.
#
# Therefore the canonical temporal order used here is Segment_ID order.
#
# ==================================================================================================

segment_order = sorted(
    cell15_segment_summary[
        "Segment_ID"
    ]
    .astype(int)
    .tolist()
)


if segment_order != list(
    range(
        min(segment_order),
        max(segment_order) + 1,
    )
):

    print(
        "WARNING: Segment_ID values are not contiguous. "
        "Existing Segment_ID order will still be respected."
    )


print("\nCanonical segment order")
print(SUB)
print(segment_order)


# ==================================================================================================
# 8. GENERATE STRICT CHRONOLOGICAL BLOCK CANDIDATES
# ==================================================================================================
#
# We use two cut points:
#
#       TRAIN      = segments before cut_1
#       VALIDATION = segments [cut_1, cut_2)
#       TEST       = segments from cut_2 onward
#
# Every partition therefore contains whole capture segments.
#
# This guarantees:
#
#   TRAIN precedes VALIDATION
#   VALIDATION precedes TEST
#
# in the canonical capture order.
#
# ==================================================================================================

print("\n" + SEP)
print("SEARCHING STRICT CHRONOLOGICAL WHOLE-SEGMENT SPLITS")
print(SEP)


candidate_rows = []


n_segments = len(segment_order)


for cut_1 in range(
    1,
    n_segments - 1,
):

    for cut_2 in range(
        cut_1 + 1,
        n_segments,
    ):

        train_segments = (
            segment_order[:cut_1]
        )

        validation_segments = (
            segment_order[
                cut_1:cut_2
            ]
        )

        test_segments = (
            segment_order[cut_2:]
        )


        train_origins_df = (
            cell15_origin_table[
                cell15_origin_table[
                    "Segment_ID"
                ].isin(train_segments)
            ]
        )

        validation_origins_df = (
            cell15_origin_table[
                cell15_origin_table[
                    "Segment_ID"
                ].isin(validation_segments)
            ]
        )

        test_origins_df = (
            cell15_origin_table[
                cell15_origin_table[
                    "Segment_ID"
                ].isin(test_segments)
            ]
        )


        train_f = (
            forecast[
                forecast[
                    "Segment_ID"
                ].isin(train_segments)
            ]
        )

        validation_f = (
            forecast[
                forecast[
                    "Segment_ID"
                ].isin(validation_segments)
            ]
        )

        test_f = (
            forecast[
                forecast[
                    "Segment_ID"
                ].isin(test_segments)
            ]
        )


        n_train = len(
            train_origins_df
        )

        n_validation = len(
            validation_origins_df
        )

        n_test = len(
            test_origins_df
        )


        train_frac = (
            n_train
            / origin_count
        )

        validation_frac = (
            n_validation
            / origin_count
        )

        test_frac = (
            n_test
            / origin_count
        )


        validation_attack_targets = int(
            validation_f[
                "Future_Attack_State"
            ].sum()
        )

        test_attack_targets = int(
            test_f[
                "Future_Attack_State"
            ].sum()
        )


        validation_clean_attack_targets = int(
            (
                (
                    validation_f[
                        "Future_Attack_State"
                    ]
                    == 1
                )
                &
                (
                    validation_f[
                        "Clean_History"
                    ]
                    == 1
                )
            ).sum()
        )

        test_clean_attack_targets = int(
            (
                (
                    test_f[
                        "Future_Attack_State"
                    ]
                    == 1
                )
                &
                (
                    test_f[
                        "Clean_History"
                    ]
                    == 1
                )
            ).sum()
        )


        train_types = attack_type_set(
            train_f
        )

        validation_types = attack_type_set(
            validation_f
        )

        test_types = attack_type_set(
            test_f
        )


        validation_known = (
            validation_types
            & train_types
        )

        validation_unseen = (
            validation_types
            - train_types
        )

        test_known = (
            test_types
            & train_types
        )

        test_unseen = (
            test_types
            - train_types
        )


        # --------------------------------------------------------------------------
        # HARD structural feasibility.
        # --------------------------------------------------------------------------

        feasible = (

            n_train > 0

            and n_validation
            >= MIN_VALIDATION_ORIGINS

            and n_test
            >= MIN_TEST_ORIGINS

            and validation_attack_targets
            >= MIN_VALIDATION_ATTACK_TARGETS

            and test_attack_targets
            >= MIN_TEST_ATTACK_TARGETS

            and validation_clean_attack_targets
            >= MIN_VALIDATION_CLEAN_HISTORY_ATTACK_TARGETS

            and test_clean_attack_targets
            >= MIN_TEST_CLEAN_HISTORY_ATTACK_TARGETS
        )


        # --------------------------------------------------------------------------
        # Structural score.
        #
        # Lower = better.
        #
        # MOST IMPORTANT:
        # This score does NOT use model performance.
        #
        # Primary term:
        #   closeness to desired origin fractions.
        #
        # Secondary terms:
        #   discourage evaluation partitions with no known types at all,
        #   while still ALLOWING unseen types.
        #
        # We are NOT maximizing label overlap.
        # --------------------------------------------------------------------------

        fraction_error = (

            abs(
                train_frac
                - TARGET_TRAIN_FRAC
            )

            +

            abs(
                validation_frac
                - TARGET_VALIDATION_FRAC
            )

            +

            abs(
                test_frac
                - TARGET_TEST_FRAC
            )
        )


        no_validation_known_penalty = (
            0.10
            if (
                len(validation_types) > 0
                and len(validation_known) == 0
            )
            else 0.0
        )


        no_test_known_penalty = (
            0.10
            if (
                len(test_types) > 0
                and len(test_known) == 0
            )
            else 0.0
        )


        structural_score = (

            fraction_error

            + no_validation_known_penalty

            + no_test_known_penalty
        )


        candidate_rows.append({

            "Cut_1":
                cut_1,

            "Cut_2":
                cut_2,

            "Train_Segments":
                train_segments,

            "Validation_Segments":
                validation_segments,

            "Test_Segments":
                test_segments,

            "Train_Origins":
                n_train,

            "Validation_Origins":
                n_validation,

            "Test_Origins":
                n_test,

            "Train_Fraction":
                train_frac,

            "Validation_Fraction":
                validation_frac,

            "Test_Fraction":
                test_frac,

            "Validation_Attack_Targets":
                validation_attack_targets,

            "Test_Attack_Targets":
                test_attack_targets,

            "Validation_Clean_Attack_Targets":
                validation_clean_attack_targets,

            "Test_Clean_Attack_Targets":
                test_clean_attack_targets,

            "Train_Attack_Type_Count":
                len(train_types),

            "Validation_Attack_Type_Count":
                len(validation_types),

            "Test_Attack_Type_Count":
                len(test_types),

            "Validation_Known_Type_Count":
                len(validation_known),

            "Validation_Unseen_Type_Count":
                len(validation_unseen),

            "Test_Known_Type_Count":
                len(test_known),

            "Test_Unseen_Type_Count":
                len(test_unseen),

            "Validation_Known_Types":
                sorted(
                    validation_known
                ),

            "Validation_Unseen_Types":
                sorted(
                    validation_unseen
                ),

            "Test_Known_Types":
                sorted(
                    test_known
                ),

            "Test_Unseen_Types":
                sorted(
                    test_unseen
                ),

            "Fraction_Error":
                fraction_error,

            "Structural_Score":
                structural_score,

            "Feasible":
                int(feasible),
        })


cell15_split_candidates = pd.DataFrame(
    candidate_rows
)


feasible_candidates = (
    cell15_split_candidates[
        cell15_split_candidates[
            "Feasible"
        ]
        == 1
    ]
    .copy()
)


print(
    f"Candidate chronological splits          : "
    f"{len(cell15_split_candidates):,}"
)

print(
    f"Structurally feasible candidates        : "
    f"{len(feasible_candidates):,}"
)


if len(feasible_candidates) == 0:

    print("\nALL CANDIDATES")
    print(SUB)

    print(
        cell15_split_candidates[
            [
                "Train_Segments",
                "Validation_Segments",
                "Test_Segments",
                "Train_Origins",
                "Validation_Origins",
                "Test_Origins",
                "Validation_Attack_Targets",
                "Test_Attack_Targets",
                "Validation_Clean_Attack_Targets",
                "Test_Clean_Attack_Targets",
            ]
        ]
        .to_string(index=False)
    )

    raise RuntimeError(
        "No whole-segment chronological split satisfies the minimum "
        "forecasting-support requirements."
    )


# ==================================================================================================
# 9. SELECT FINAL STRUCTURAL SPLIT
# ==================================================================================================
#
# Deterministic tie-breaking:
#
#   1. minimum Structural_Score
#   2. minimum Fraction_Error
#   3. larger TRAIN partition
#   4. earlier cut points
#
# ==================================================================================================

feasible_candidates = (
    feasible_candidates
    .sort_values(
        [
            "Structural_Score",
            "Fraction_Error",
            "Train_Origins",
            "Cut_1",
            "Cut_2",
        ],
        ascending=[
            True,
            True,
            False,
            True,
            True,
        ],
    )
    .reset_index(drop=True)
)


selected = (
    feasible_candidates.iloc[0]
)


TRAIN_SEGMENTS = list(
    selected[
        "Train_Segments"
    ]
)

VALIDATION_SEGMENTS = list(
    selected[
        "Validation_Segments"
    ]
)

TEST_SEGMENTS = list(
    selected[
        "Test_Segments"
    ]
)


print("\n" + SEP)
print("SELECTED STRICT TEMPORAL SPLIT")
print(SEP)


print(f"TRAIN segments                          : {TRAIN_SEGMENTS}")
print(f"VALIDATION segments                     : {VALIDATION_SEGMENTS}")
print(f"TEST segments                           : {TEST_SEGMENTS}")

print()

print(
    f"TRAIN origins                           : "
    f"{int(selected['Train_Origins']):,} "
    f"({selected['Train_Fraction'] * 100:.2f}%)"
)

print(
    f"VALIDATION origins                      : "
    f"{int(selected['Validation_Origins']):,} "
    f"({selected['Validation_Fraction'] * 100:.2f}%)"
)

print(
    f"TEST origins                            : "
    f"{int(selected['Test_Origins']):,} "
    f"({selected['Test_Fraction'] * 100:.2f}%)"
)

print()

print(
    f"VALIDATION attack targets               : "
    f"{int(selected['Validation_Attack_Targets']):,}"
)

print(
    f"TEST attack targets                     : "
    f"{int(selected['Test_Attack_Targets']):,}"
)

print(
    f"VALIDATION clean-history attack targets : "
    f"{int(selected['Validation_Clean_Attack_Targets']):,}"
)

print(
    f"TEST clean-history attack targets       : "
    f"{int(selected['Test_Clean_Attack_Targets']):,}"
)


# ==================================================================================================
# 10. ASSIGN FINAL PARTITIONS
# ==================================================================================================

segment_partition_map = {}


for segment_id in TRAIN_SEGMENTS:
    segment_partition_map[
        int(segment_id)
    ] = "TRAIN"


for segment_id in VALIDATION_SEGMENTS:
    segment_partition_map[
        int(segment_id)
    ] = "VALIDATION"


for segment_id in TEST_SEGMENTS:
    segment_partition_map[
        int(segment_id)
    ] = "TEST"


cell15_origin_table[
    "Partition"
] = (
    cell15_origin_table[
        "Segment_ID"
    ]
    .map(segment_partition_map)
)


if cell15_origin_table[
    "Partition"
].isna().any():

    missing_segments = sorted(
        cell15_origin_table.loc[
            cell15_origin_table[
                "Partition"
            ].isna(),
            "Segment_ID",
        ]
        .unique()
        .tolist()
    )

    raise RuntimeError(
        "Some origins were not assigned to a partition. "
        f"Segments: {missing_segments}"
    )


# ==================================================================================================
# 11. STRICT TEMPORAL ORDER VALIDATION
# ==================================================================================================

print("\n" + SEP)
print("STRICT TEMPORAL BLOCK VALIDATION")
print(SEP)


train_positions = [
    segment_order.index(x)
    for x in TRAIN_SEGMENTS
]

validation_positions = [
    segment_order.index(x)
    for x in VALIDATION_SEGMENTS
]

test_positions = [
    segment_order.index(x)
    for x in TEST_SEGMENTS
]


strict_block_order = (

    max(train_positions)
    < min(validation_positions)

    and

    max(validation_positions)
    < min(test_positions)
)


print(
    "TRAIN strictly precedes VALIDATION      : "
    f"{strict_block_order}"
)


if not strict_block_order:

    raise RuntimeError(
        "Selected segment blocks are not strictly chronological."
    )


print("✓ Strict block chronology validated.")


# ==================================================================================================
# 12. CROSS-PARTITION RAW-FOOTPRINT AUDIT
# ==================================================================================================
#
# Since complete capture segments are atomic and distinct, there should be no
# within-segment cross-partition overlap at all.
#
# We nevertheless audit explicitly.
#
# ==================================================================================================

print("\n" + SEP)
print("RAW TEMPORAL FOOTPRINT LEAKAGE AUDIT")
print(SEP)


partition_values = [
    "TRAIN",
    "VALIDATION",
    "TEST",
]


overlap_counts = {
    ("TRAIN", "VALIDATION"): 0,
    ("TRAIN", "TEST"): 0,
    ("VALIDATION", "TEST"): 0,
}


for segment_id, g in (
    cell15_origin_table
    .groupby("Segment_ID")
):

    partitions_here = (
        g["Partition"]
        .unique()
        .tolist()
    )

    if len(partitions_here) > 1:

        raise RuntimeError(
            f"Segment {segment_id} was split across partitions."
        )


# Generic defensive overlap audit.
for i, p1 in enumerate(
    partition_values
):

    for p2 in partition_values[
        i + 1:
    ]:

        left = (
            cell15_origin_table[
                cell15_origin_table[
                    "Partition"
                ]
                == p1
            ]
        )

        right = (
            cell15_origin_table[
                cell15_origin_table[
                    "Partition"
                ]
                == p2
            ]
        )

        common_segments = (
            set(
                left[
                    "Segment_ID"
                ].unique()
            )
            &
            set(
                right[
                    "Segment_ID"
                ].unique()
            )
        )


        count = 0


        for segment_id in common_segments:

            a = left[
                left["Segment_ID"]
                == segment_id
            ]

            b = right[
                right["Segment_ID"]
                == segment_id
            ]


            for row_a in a.itertuples():

                for row_b in b.itertuples():

                    if footprint_overlap(
                        row_a.Footprint_Start,
                        row_a.Footprint_End,
                        row_b.Footprint_Start,
                        row_b.Footprint_End,
                    ):

                        count += 1


        overlap_counts[
            (p1, p2)
        ] = count


for pair, count in overlap_counts.items():

    print(
        f"{pair[0]:>10} ↔ {pair[1]:<10} "
        f"footprint overlap pairs : {count:,}"
    )


if any(
    count != 0
    for count in overlap_counts.values()
):

    raise RuntimeError(
        "Cross-partition raw temporal footprint leakage detected."
    )


print(
    "✓ No retained raw-minute footprint is shared across partitions."
)


# ==================================================================================================
# 13. PROPAGATE PARTITIONS TO FIVE-HORIZON FORECAST TABLE
# ==================================================================================================

print("\n" + SEP)
print("PROPAGATING PARTITIONS TO FORECAST ROWS")
print(SEP)


cell15_partition_lookup = (
    cell15_origin_table[
        [
            "Segment_ID",
            "Origin_Time",
            "Partition",
        ]
    ]
    .copy()
)


cell15_forecast_metadata = (
    forecast
    .merge(
        cell15_partition_lookup,
        on=[
            "Segment_ID",
            "Origin_Time",
        ],
        how="left",
        validate="many_to_one",
    )
)


if cell15_forecast_metadata[
    "Partition"
].isna().any():

    raise RuntimeError(
        "Some forecast rows failed to inherit a partition."
    )


atomicity = (
    cell15_forecast_metadata
    .groupby(
        [
            "Segment_ID",
            "Origin_Time",
        ]
    )["Partition"]
    .nunique()
)


atomicity_errors = int(
    (
        atomicity != 1
    ).sum()
)


print(
    f"Five-horizon partition atomicity errors : "
    f"{atomicity_errors:,}"
)


if atomicity_errors != 0:

    raise RuntimeError(
        "Horizons belonging to the same origin crossed partitions."
    )


print(
    "✓ All five horizons belonging to an origin remain together."
)


# ==================================================================================================
# 14. ATTACK EPISODE ISOLATION AUDIT
# ==================================================================================================

print("\n" + SEP)
print("ATTACK EPISODE ISOLATION AUDIT")
print(SEP)


episode_leakage_count = 0
cell15_episode_partition_support = pd.DataFrame()


if episode_map_available:

    epmap = (
        cell14a_forecast_episode_map
        .copy()
    )


    required_episode_columns = {
        "Segment_ID",
        "Origin_Time",
        "Canonical_Type_Episode_ID",
        "Attack_Type",
    }


    if not required_episode_columns.issubset(
        epmap.columns
    ):

        print(
            "WARNING: Cell 14A episode map exists but does not contain "
            "the expected columns. Episode leakage audit skipped."
        )

    else:

        epmap["Segment_ID"] = (
            pd.to_numeric(
                epmap["Segment_ID"],
                errors="raise",
            )
            .astype(int)
        )

        epmap["Origin_Time"] = pd.to_datetime(
            epmap["Origin_Time"],
            errors="raise",
        )


        episode_partition = (
            epmap[
                [
                    "Segment_ID",
                    "Origin_Time",
                    "Attack_Type",
                    "Canonical_Type_Episode_ID",
                ]
            ]
            .drop_duplicates()
            .merge(
                cell15_partition_lookup,
                on=[
                    "Segment_ID",
                    "Origin_Time",
                ],
                how="left",
                validate="many_to_one",
            )
        )


        episode_partition_nunique = (
            episode_partition
            .groupby(
                "Canonical_Type_Episode_ID"
            )["Partition"]
            .nunique()
        )


        leaking_episode_ids = (
            episode_partition_nunique[
                episode_partition_nunique > 1
            ]
            .index
            .tolist()
        )


        episode_leakage_count = len(
            leaking_episode_ids
        )


        cell15_episode_partition_support = (
            episode_partition
            .groupby(
                [
                    "Attack_Type",
                    "Canonical_Type_Episode_ID",
                    "Partition",
                ]
            )
            .agg(
                Forecast_Origins=(
                    "Origin_Time",
                    "nunique",
                )
            )
            .reset_index()
        )


        print(
            "Episodes visible in >1 partition        : "
            f"{episode_leakage_count:,}"
        )


        if episode_leakage_count != 0:

            print(
                "\nLeaking episode IDs:"
            )

            for episode_id in leaking_episode_ids:
                print(
                    f"  • {episode_id}"
                )

            raise RuntimeError(
                "An attack episode crosses partitions."
            )


        print(
            "✓ No canonical attack episode crosses partitions."
        )


else:

    print(
        "Cell 14A episode map unavailable."
    )

    print(
        "Whole-segment splitting still prevents intentional "
        "within-segment episode splitting."
    )


# ==================================================================================================
# 15. BUILD TRAIN / VALIDATION / TEST OBJECTS
# ==================================================================================================

cell15_train_origins = (
    cell15_origin_table[
        cell15_origin_table[
            "Partition"
        ]
        == "TRAIN"
    ]
    .copy()
    .reset_index(drop=True)
)


cell15_validation_origins = (
    cell15_origin_table[
        cell15_origin_table[
            "Partition"
        ]
        == "VALIDATION"
    ]
    .copy()
    .reset_index(drop=True)
)


cell15_test_origins = (
    cell15_origin_table[
        cell15_origin_table[
            "Partition"
        ]
        == "TEST"
    ]
    .copy()
    .reset_index(drop=True)
)


cell15_train_forecast = (
    cell15_forecast_metadata[
        cell15_forecast_metadata[
            "Partition"
        ]
        == "TRAIN"
    ]
    .copy()
    .reset_index(drop=True)
)


cell15_validation_forecast = (
    cell15_forecast_metadata[
        cell15_forecast_metadata[
            "Partition"
        ]
        == "VALIDATION"
    ]
    .copy()
    .reset_index(drop=True)
)


cell15_test_forecast = (
    cell15_forecast_metadata[
        cell15_forecast_metadata[
            "Partition"
        ]
        == "TEST"
    ]
    .copy()
    .reset_index(drop=True)
)


# ==================================================================================================
# 16. TRAIN ATTACK-TYPE VOCABULARY
# ==================================================================================================

print("\n" + SEP)
print("TRAIN ATTACK-TYPE VOCABULARY")
print(SEP)


TRAIN_ATTACK_TYPES = sorted(
    attack_type_set(
        cell15_train_forecast
    )
)


VALIDATION_ATTACK_TYPES = sorted(
    attack_type_set(
        cell15_validation_forecast
    )
)


TEST_ATTACK_TYPES = sorted(
    attack_type_set(
        cell15_test_forecast
    )
)


TRAIN_ATTACK_TYPE_SET = set(
    TRAIN_ATTACK_TYPES
)


print(
    f"TRAIN attack types                     : "
    f"{len(TRAIN_ATTACK_TYPES)}"
)

for attack_type in TRAIN_ATTACK_TYPES:
    print(
        f"  • {attack_type}"
    )


# ==================================================================================================
# 17. KNOWN VS UNSEEN EVALUATION ATTACK TYPES
# ==================================================================================================

print("\n" + SEP)
print("KNOWN VS UNSEEN EVALUATION ATTACK TYPES")
print(SEP)


VALIDATION_KNOWN_TYPES = sorted(
    set(
        VALIDATION_ATTACK_TYPES
    )
    & TRAIN_ATTACK_TYPE_SET
)


VALIDATION_UNSEEN_TYPES = sorted(
    set(
        VALIDATION_ATTACK_TYPES
    )
    - TRAIN_ATTACK_TYPE_SET
)


TEST_KNOWN_TYPES = sorted(
    set(
        TEST_ATTACK_TYPES
    )
    & TRAIN_ATTACK_TYPE_SET
)


TEST_UNSEEN_TYPES = sorted(
    set(
        TEST_ATTACK_TYPES
    )
    - TRAIN_ATTACK_TYPE_SET
)


print("\nVALIDATION")
print(SUB)

print(
    f"All attack types                       : "
    f"{VALIDATION_ATTACK_TYPES}"
)

print(
    f"Known to TRAIN                         : "
    f"{VALIDATION_KNOWN_TYPES}"
)

print(
    f"Unseen relative to TRAIN               : "
    f"{VALIDATION_UNSEEN_TYPES}"
)


print("\nTEST")
print(SUB)

print(
    f"All attack types                       : "
    f"{TEST_ATTACK_TYPES}"
)

print(
    f"Known to TRAIN                         : "
    f"{TEST_KNOWN_TYPES}"
)

print(
    f"Unseen relative to TRAIN               : "
    f"{TEST_UNSEEN_TYPES}"
)


print(
    "\nNOTE: unseen evaluation attack types are NOT considered a split error."
)

print(
    "They remain valid for binary attack forecasting/generalization analysis."
)

print(
    "Closed-set attack-type classification metrics must be reported separately "
    "for targets whose attack type was represented in TRAIN."
)


# ==================================================================================================
# 18. ADD KNOWN/UNSEEN STATUS TO EVERY FORECAST ROW
# ==================================================================================================

def target_known_status(row):

    if int(
        row["Future_Attack_State"]
    ) == 0:

        return "BENIGN"

    types = [
        str(x)
        for x in normalize_collection(
            row[
                "Future_Attack_Types"
            ]
        )
    ]

    if len(types) == 0:

        return "ATTACK_TYPE_MISSING"

    known_flags = [
        x in TRAIN_ATTACK_TYPE_SET
        for x in types
    ]


    if all(known_flags):

        return "KNOWN_ATTACK_TYPE"


    if not any(known_flags):

        return "UNSEEN_ATTACK_TYPE"


    return "MIXED_KNOWN_UNSEEN_ATTACK_TYPES"


cell15_forecast_metadata[
    "Train_Type_Status"
] = (
    cell15_forecast_metadata
    .apply(
        target_known_status,
        axis=1,
    )
)


cell15_forecast_metadata[
    "Known_To_Train"
] = np.where(

    cell15_forecast_metadata[
        "Future_Attack_State"
    ]
    == 0,

    pd.NA,

    cell15_forecast_metadata[
        "Train_Type_Status"
    ]
    .isin(
        [
            "KNOWN_ATTACK_TYPE",
            "MIXED_KNOWN_UNSEEN_ATTACK_TYPES",
        ]
    )
    .astype(int),
)


cell15_forecast_metadata[
    "All_Attack_Types_Known_To_Train"
] = np.where(

    cell15_forecast_metadata[
        "Future_Attack_State"
    ]
    == 0,

    pd.NA,

    (
        cell15_forecast_metadata[
            "Train_Type_Status"
        ]
        == "KNOWN_ATTACK_TYPE"
    )
    .astype(int),
)


# Rebuild partition-specific forecast objects with final metadata.
cell15_train_forecast = (
    cell15_forecast_metadata[
        cell15_forecast_metadata[
            "Partition"
        ]
        == "TRAIN"
    ]
    .copy()
    .reset_index(drop=True)
)


cell15_validation_forecast = (
    cell15_forecast_metadata[
        cell15_forecast_metadata[
            "Partition"
        ]
        == "VALIDATION"
    ]
    .copy()
    .reset_index(drop=True)
)


cell15_test_forecast = (
    cell15_forecast_metadata[
        cell15_forecast_metadata[
            "Partition"
        ]
        == "TEST"
    ]
    .copy()
    .reset_index(drop=True)
)


# ==================================================================================================
# 19. PER-PARTITION / PER-HORIZON SUPPORT
# ==================================================================================================

print("\n" + SEP)
print("PARTITION × HORIZON FORECASTING SUPPORT")
print(SEP)


partition_horizon_rows = []


for partition in [
    "TRAIN",
    "VALIDATION",
    "TEST",
]:

    q = (
        cell15_forecast_metadata[
            cell15_forecast_metadata[
                "Partition"
            ]
            == partition
        ]
    )


    for horizon in EXPECTED_HORIZONS:

        h = q[
            q["Horizon_Minutes"]
            == horizon
        ]


        attack_count = int(
            h[
                "Future_Attack_State"
            ].sum()
        )


        benign_count = int(
            len(h)
            - attack_count
        )


        clean_attack_count = int(
            (
                (
                    h[
                        "Future_Attack_State"
                    ]
                    == 1
                )
                &
                (
                    h[
                        "Clean_History"
                    ]
                    == 1
                )
            ).sum()
        )


        known_attack_count = int(
            (
                (
                    h[
                        "Future_Attack_State"
                    ]
                    == 1
                )
                &
                (
                    h[
                        "Train_Type_Status"
                    ]
                    == "KNOWN_ATTACK_TYPE"
                )
            ).sum()
        )


        unseen_attack_count = int(
            (
                (
                    h[
                        "Future_Attack_State"
                    ]
                    == 1
                )
                &
                (
                    h[
                        "Train_Type_Status"
                    ]
                    == "UNSEEN_ATTACK_TYPE"
                )
            ).sum()
        )


        partition_horizon_rows.append({

            "Partition":
                partition,

            "Horizon_Minutes":
                horizon,

            "Forecast_Rows":
                len(h),

            "Attack_Targets":
                attack_count,

            "Benign_Targets":
                benign_count,

            "Attack_Percent":
                (
                    100
                    * attack_count
                    / len(h)
                    if len(h)
                    else np.nan
                ),

            "Clean_History_Attack_Targets":
                clean_attack_count,

            "Known_Attack_Targets":
                known_attack_count,

            "Unseen_Attack_Targets":
                unseen_attack_count,
        })


cell15_split_summary = pd.DataFrame(
    partition_horizon_rows
)


print(
    cell15_split_summary
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}",
    )
)


# ==================================================================================================
# 20. ATTACK-TYPE SUPPORT TABLE
# ==================================================================================================

print("\n" + SEP)
print("ATTACK-TYPE SUPPORT BY PARTITION")
print(SEP)


attack_type_rows = []


for partition in [
    "TRAIN",
    "VALIDATION",
    "TEST",
]:

    q = (
        cell15_forecast_metadata[
            cell15_forecast_metadata[
                "Partition"
            ]
            == partition
        ]
    )


    exploded = []


    for row in q.itertuples():

        if int(
            row.Future_Attack_State
        ) != 1:

            continue


        types = normalize_collection(
            row.Future_Attack_Types
        )


        for attack_type in types:

            exploded.append({

                "Partition":
                    partition,

                "Attack_Type":
                    str(attack_type),

                "Segment_ID":
                    int(row.Segment_ID),

                "Origin_Time":
                    row.Origin_Time,

                "Target_Time":
                    row.Target_Time,

                "Horizon_Minutes":
                    int(
                        row.Horizon_Minutes
                    ),

                "Clean_History":
                    int(
                        row.Clean_History
                    ),
            })


    if len(exploded) == 0:
        continue


    exploded_df = pd.DataFrame(
        exploded
    )


    for attack_type, g in (
        exploded_df
        .groupby("Attack_Type")
    ):

        attack_type_rows.append({

            "Partition":
                partition,

            "Attack_Type":
                attack_type,

            "Known_To_Train":
                int(
                    attack_type
                    in TRAIN_ATTACK_TYPE_SET
                ),

            "Forecast_Rows":
                len(g),

            "Unique_Origins":
                len(
                    g[
                        [
                            "Segment_ID",
                            "Origin_Time",
                        ]
                    ]
                    .drop_duplicates()
                ),

            "Unique_Target_Minutes":
                len(
                    g[
                        [
                            "Segment_ID",
                            "Target_Time",
                        ]
                    ]
                    .drop_duplicates()
                ),

            "Clean_History_Origins":
                len(
                    g.loc[
                        g[
                            "Clean_History"
                        ]
                        == 1,
                        [
                            "Segment_ID",
                            "Origin_Time",
                        ],
                    ]
                    .drop_duplicates()
                ),

            "Horizons":
                sorted(
                    g[
                        "Horizon_Minutes"
                    ]
                    .unique()
                    .tolist()
                ),
        })


cell15_attack_type_support = pd.DataFrame(
    attack_type_rows
)


if len(
    cell15_attack_type_support
):

    print(
        cell15_attack_type_support
        .sort_values(
            [
                "Partition",
                "Attack_Type",
            ]
        )
        .to_string(index=False)
    )

else:

    print(
        "No attack-type support rows found."
    )


# ==================================================================================================
# 21. BUILD EVALUATION ATTACK METADATA
# ==================================================================================================
#
# This table will be useful later for reporting:
#
#   overall binary forecasting
#   known-type forecasting
#   unseen-type binary forecasting
#   clean-history forecasting
#
# ==================================================================================================

evaluation_rows = []


for partition in [
    "VALIDATION",
    "TEST",
]:

    q = (
        cell15_forecast_metadata[
            (
                cell15_forecast_metadata[
                    "Partition"
                ]
                == partition
            )
            &
            (
                cell15_forecast_metadata[
                    "Future_Attack_State"
                ]
                == 1
            )
        ]
    )


    for row in q.itertuples():

        attack_types = [
            str(x)
            for x in normalize_collection(
                row.Future_Attack_Types
            )
        ]


        evaluation_rows.append({

            "Partition":
                partition,

            "Segment_ID":
                int(
                    row.Segment_ID
                ),

            "Origin_Time":
                row.Origin_Time,

            "Horizon_Minutes":
                int(
                    row.Horizon_Minutes
                ),

            "Target_Time":
                row.Target_Time,

            "Clean_History":
                int(
                    row.Clean_History
                ),

            "Future_Attack_Types":
                attack_types,

            "Train_Type_Status":
                row.Train_Type_Status,

            "Known_To_Train":
                row.Known_To_Train,

            "All_Attack_Types_Known_To_Train":
                row.All_Attack_Types_Known_To_Train,
        })


cell15_evaluation_attack_metadata = (
    pd.DataFrame(
        evaluation_rows
    )
)


# ==================================================================================================
# 22. FINAL ORIGIN-LEVEL PARTITION SUMMARY
# ==================================================================================================

print("\n" + SEP)
print("FINAL ORIGIN-LEVEL PARTITION SUMMARY")
print(SEP)


origin_summary_rows = []


for partition in [
    "TRAIN",
    "VALIDATION",
    "TEST",
]:

    o = (
        cell15_origin_table[
            cell15_origin_table[
                "Partition"
            ]
            == partition
        ]
    )


    f = (
        cell15_forecast_metadata[
            cell15_forecast_metadata[
                "Partition"
            ]
            == partition
        ]
    )


    origin_summary_rows.append({

        "Partition":
            partition,

        "Segments":
            sorted(
                o[
                    "Segment_ID"
                ]
                .unique()
                .tolist()
            ),

        "Origins":
            len(o),

        "Origin_Percent":
            (
                100
                * len(o)
                / len(
                    cell15_origin_table
                )
            ),

        "Forecast_Rows":
            len(f),

        "Attack_Target_Rows":
            int(
                f[
                    "Future_Attack_State"
                ].sum()
            ),

        "Clean_History_Attack_Target_Rows":
            int(
                (
                    (
                        f[
                            "Future_Attack_State"
                        ]
                        == 1
                    )
                    &
                    (
                        f[
                            "Clean_History"
                        ]
                        == 1
                    )
                ).sum()
            ),

        "Attack_Type_Count":
            len(
                attack_type_set(f)
            ),
    })


cell15_origin_partition_summary = pd.DataFrame(
    origin_summary_rows
)


print(
    cell15_origin_partition_summary
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.3f}",
    )
)


# ==================================================================================================
# 23. FINAL HARD CERTIFICATION
# ==================================================================================================

print("\n" + SEP)
print("FINAL SPLIT CERTIFICATION")
print(SEP)


hard_errors = []


# --------------------------------------------------------------------------------------------------
# A. Every origin assigned exactly once
# --------------------------------------------------------------------------------------------------

if cell15_origin_table[
    "Partition"
].isna().any():

    hard_errors.append(
        "At least one origin has no partition."
    )


if not set(
    cell15_origin_table[
        "Partition"
    ].unique()
).issubset(
    {
        "TRAIN",
        "VALIDATION",
        "TEST",
    }
):

    hard_errors.append(
        "Unexpected partition label found."
    )


# --------------------------------------------------------------------------------------------------
# B. Segment atomicity
# --------------------------------------------------------------------------------------------------

segment_partition_counts = (
    cell15_origin_table
    .groupby(
        "Segment_ID"
    )["Partition"]
    .nunique()
)


split_segments = (
    segment_partition_counts[
        segment_partition_counts > 1
    ]
)


if len(split_segments):

    hard_errors.append(
        "At least one capture segment crosses partitions."
    )


# --------------------------------------------------------------------------------------------------
# C. Five-horizon atomicity
# --------------------------------------------------------------------------------------------------

if atomicity_errors != 0:

    hard_errors.append(
        "At least one origin's five horizons cross partitions."
    )


# --------------------------------------------------------------------------------------------------
# D. Footprint overlap
# --------------------------------------------------------------------------------------------------

if any(
    count != 0
    for count in overlap_counts.values()
):

    hard_errors.append(
        "Cross-partition temporal footprint overlap exists."
    )


# --------------------------------------------------------------------------------------------------
# E. Episode leakage
# --------------------------------------------------------------------------------------------------

if episode_leakage_count != 0:

    hard_errors.append(
        "At least one canonical attack episode crosses partitions."
    )


# --------------------------------------------------------------------------------------------------
# F. Partition non-emptiness
# --------------------------------------------------------------------------------------------------

for partition in [
    "TRAIN",
    "VALIDATION",
    "TEST",
]:

    count = int(
        (
            cell15_origin_table[
                "Partition"
            ]
            == partition
        ).sum()
    )

    if count == 0:

        hard_errors.append(
            f"{partition} contains zero origins."
        )


# --------------------------------------------------------------------------------------------------
# G. Evaluation attack support
# --------------------------------------------------------------------------------------------------

validation_attack_targets = int(
    cell15_validation_forecast[
        "Future_Attack_State"
    ].sum()
)


test_attack_targets = int(
    cell15_test_forecast[
        "Future_Attack_State"
    ].sum()
)


if validation_attack_targets == 0:

    hard_errors.append(
        "VALIDATION contains zero attack targets."
    )


if test_attack_targets == 0:

    hard_errors.append(
        "TEST contains zero attack targets."
    )


# --------------------------------------------------------------------------------------------------
# H. Clean-history attack support
# --------------------------------------------------------------------------------------------------

validation_clean_attack = int(
    (
        (
            cell15_validation_forecast[
                "Future_Attack_State"
            ]
            == 1
        )
        &
        (
            cell15_validation_forecast[
                "Clean_History"
            ]
            == 1
        )
    ).sum()
)


test_clean_attack = int(
    (
        (
            cell15_test_forecast[
                "Future_Attack_State"
            ]
            == 1
        )
        &
        (
            cell15_test_forecast[
                "Clean_History"
            ]
            == 1
        )
    ).sum()
)


if validation_clean_attack == 0:

    hard_errors.append(
        "VALIDATION contains zero clean-history attack targets."
    )


if test_clean_attack == 0:

    hard_errors.append(
        "TEST contains zero clean-history attack targets."
    )


# --------------------------------------------------------------------------------------------------
# I. Forecast row accounting
# --------------------------------------------------------------------------------------------------

partition_row_total = (

    len(
        cell15_train_forecast
    )

    +

    len(
        cell15_validation_forecast
    )

    +

    len(
        cell15_test_forecast
    )
)


if partition_row_total != len(
    cell15_forecast_metadata
):

    hard_errors.append(
        "Forecast rows are missing or duplicated across partitions."
    )


# --------------------------------------------------------------------------------------------------
# FINAL RESULT
# --------------------------------------------------------------------------------------------------

if hard_errors:

    print("FINAL STATUS: FAILED")
    print()

    for error in hard_errors:
        print(
            f"  ✗ {error}"
        )

    raise RuntimeError(
        "Cell 15 split certification failed."
    )


print("✓ Every origin assigned exactly once.")
print("✓ Every capture segment belongs to exactly one partition.")
print("✓ All five horizons belonging to one origin remain atomic.")
print("✓ TRAIN strictly precedes VALIDATION in canonical capture order.")
print("✓ VALIDATION strictly precedes TEST in canonical capture order.")
print("✓ No cross-partition raw temporal footprint leakage.")
print("✓ No canonical attack episode leakage.")
print("✓ VALIDATION contains attack-positive targets.")
print("✓ TEST contains attack-positive targets.")
print("✓ VALIDATION contains genuine clean-history attack targets.")
print("✓ TEST contains genuine clean-history attack targets.")
print("✓ All forecast rows are accounted for exactly once.")

print()

print(
    "IMPORTANT: attack types absent from TRAIN are intentionally NOT "
    "treated as split failures."
)

print(
    "They are explicitly marked for unseen-type/generalization evaluation."
)


# ==================================================================================================
# 24. PERMANENT SPLIT METADATA
# ==================================================================================================

cell15_split_metadata = {

    "split_version":
        "CELL15_PERMANENT_STRICT_TEMPORAL_V1",

    "split_unit":
        "capture_segment",

    "history_length_minutes":
        HISTORY_LENGTH,

    "history_back_minutes":
        HISTORY_BACK_MINUTES,

    "forecast_horizons":
        EXPECTED_HORIZONS.copy(),

    "maximum_horizon_minutes":
        MAX_HORIZON,

    "train_segments":
        TRAIN_SEGMENTS.copy(),

    "validation_segments":
        VALIDATION_SEGMENTS.copy(),

    "test_segments":
        TEST_SEGMENTS.copy(),

    "train_origins":
        len(
            cell15_train_origins
        ),

    "validation_origins":
        len(
            cell15_validation_origins
        ),

    "test_origins":
        len(
            cell15_test_origins
        ),

    "train_attack_types":
        TRAIN_ATTACK_TYPES.copy(),

    "validation_attack_types":
        VALIDATION_ATTACK_TYPES.copy(),

    "test_attack_types":
        TEST_ATTACK_TYPES.copy(),

    "validation_known_types":
        VALIDATION_KNOWN_TYPES.copy(),

    "validation_unseen_types":
        VALIDATION_UNSEEN_TYPES.copy(),

    "test_known_types":
        TEST_KNOWN_TYPES.copy(),

    "test_unseen_types":
        TEST_UNSEEN_TYPES.copy(),

    "split_selection_uses_model_performance":
        False,

    "scaling_performed":
        False,

    "resampling_performed":
        False,

    "model_trained":
        False,
}


# ==================================================================================================
# 25. OBJECT INVENTORY
# ==================================================================================================

print("\n" + SEP)
print("PERMANENT OBJECTS CREATED")
print(SEP)


created_objects = [

    "cell15_segment_summary",

    "cell15_split_candidates",

    "cell15_origin_table",

    "cell15_partition_lookup",

    "cell15_forecast_metadata",

    "cell15_train_origins",

    "cell15_validation_origins",

    "cell15_test_origins",

    "cell15_train_forecast",

    "cell15_validation_forecast",

    "cell15_test_forecast",

    "cell15_split_summary",

    "cell15_origin_partition_summary",

    "cell15_attack_type_support",

    "cell15_evaluation_attack_metadata",

    "cell15_episode_partition_support",

    "cell15_split_metadata",
]


for name in created_objects:

    print(
        f"  {name}"
    )


# ==================================================================================================
# 26. FINAL METHODOLOGICAL CONTRACT
# ==================================================================================================

print("\n" + SEP)
print("CELL 15 — PERMANENT SPLIT COMPLETE")
print(SEP)


print(
f"""
FINAL EXPERIMENTAL CONTRACT
----------------------------------------------------------------------------------------------------

PRIMARY QUESTION

    Can a forecasting model trained on earlier capture blocks predict
    future malicious activity in later, completely held-out temporal blocks?


TRAIN
    Segments: {TRAIN_SEGMENTS}


VALIDATION
    Segments: {VALIDATION_SEGMENTS}


TEST
    Segments: {TEST_SEGMENTS}


BINARY ATTACK FORECASTING

    Evaluate ALL retained validation/test samples.

    This includes attacks whose exact attack type was not represented
    during training.


KNOWN-TYPE ATTACK CLASSIFICATION

    Evaluate attack-type classification separately for attack targets
    whose class was represented in TRAIN.


UNSEEN-TYPE GENERALIZATION

    Evaluation attacks absent from TRAIN are NOT discarded.

    They remain part of binary attack forecasting and are reported
    separately as unseen-type/generalization cases.


GENUINE PRE-ATTACK FORECASTING

    Clean_History == 1 is retained as a dedicated evaluation slice.

    This distinguishes advance warning from predictions made when the
    preceding 15-minute history already contains attack activity.


LEAKAGE CONTRACT

    Capture segments never cross partitions.

    Five horizons belonging to one origin never cross partitions.

    Complete history/future footprints never cross partitions.

    Canonical attack episodes never cross partitions.


PREPROCESSING CONTRACT

    No scaler has been fitted.

    No learned imputation has been performed.

    No resampling has been performed.

    No model has been trained.


NEXT — CELL 16

    Construct the actual 15-minute model input sequences for the
    frozen origins created here.

    Any learned preprocessing transformation MUST be fitted using
    TRAIN ONLY and then applied unchanged to VALIDATION and TEST.

    Cell 16 must NOT redesign or optimize this split using model results.
----------------------------------------------------------------------------------------------------
"""
)

CELL 15 — PERMANENT LEAKAGE-SAFE TEMPORAL SPLIT + EVALUATION CONTRACT

Configuration
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
History length                         : 15 minutes
History span before origin             : 14 minutes
Forecast horizons                      : [1, 5, 10, 15, 30]
Maximum forecast horizon               : 30 minutes

Primary evaluation                     : strict out-of-time forecasting
Split unit                             : complete capture segment
Known-type requirement                 : REPORT ONLY — NOT a split constraint
Unseen attack types in evaluation      : ALLOWED and explicitly marked

Target TRAIN fraction                  : 0.60
Target VALIDATION fraction             : 0.20
Target TEST fraction                   : 0.20

Scaling performed                      : NO
Resampling performed                   : NO
Fe

In [31]:
# ======================================================================
# DIAGNOSTIC — SHOW EVERYTHING CREATED BY CELL 15
# ======================================================================

print("=" * 100)
print("CELL 15 OBJECT DISCOVERY")
print("=" * 100)

for name in sorted(globals().keys()):
    if name.startswith("cell15") or name.startswith("final_") or name.startswith("split_"):
        obj = globals()[name]

        if isinstance(obj, pd.DataFrame):
            print(
                f"{name:<55} "
                f"DataFrame shape={obj.shape}"
            )
            print("  Columns:", list(obj.columns))

        elif isinstance(obj, dict):
            print(
                f"{name:<55} "
                f"dict keys={list(obj.keys())}"
            )

        elif isinstance(obj, (list, tuple, set)):
            print(
                f"{name:<55} "
                f"{type(obj).__name__} len={len(obj)}"
            )

        elif isinstance(obj, np.ndarray):
            print(
                f"{name:<55} "
                f"ndarray shape={obj.shape}"
            )

        else:
            print(
                f"{name:<55} "
                f"{type(obj).__name__}: {repr(obj)[:300]}"
            )

print("=" * 100)

CELL 15 OBJECT DISCOVERY
cell15_attack_type_support                              DataFrame shape=(14, 8)
  Columns: ['Partition', 'Attack_Type', 'Known_To_Train', 'Forecast_Rows', 'Unique_Origins', 'Unique_Target_Minutes', 'Clean_History_Origins', 'Horizons']
cell15_episode_partition_support                        DataFrame shape=(24, 4)
  Columns: ['Attack_Type', 'Canonical_Type_Episode_ID', 'Partition', 'Forecast_Origins']
cell15_evaluation_attack_metadata                       DataFrame shape=(1400, 10)
  Columns: ['Partition', 'Segment_ID', 'Origin_Time', 'Horizon_Minutes', 'Target_Time', 'Clean_History', 'Future_Attack_Types', 'Train_Type_Status', 'Known_To_Train', 'All_Attack_Types_Known_To_Train']
cell15_forecast_metadata                                DataFrame shape=(10070, 33)
  Columns: ['Segment_ID', 'History_Start', 'History_End', 'Origin_Time', 'Horizon_Minutes', 'Target_Time', 'Current_State', 'Attack_In_History', 'Clean_History', 'Attack_Minutes_In_History', 'Future_Att

In [35]:
# ==============================================================================================================
# CELL 16 — FINAL LEAKAGE-SAFE FEATURE CONTRACT + MULTI-HORIZON SEQUENCE CONSTRUCTION
# ==============================================================================================================
#
# PURPOSE
# -------
# Construct the final model-ready temporal sequences from the frozen Cell 15 split.
#
# Each forecasting sample contains:
#
#     X : 15 consecutive observed one-minute feature vectors
#         [t-14, ..., t]
#
#     y : binary attack state at:
#         t+1, t+5, t+10, t+15, t+30
#
# IMPORTANT
# ---------
# - Cell 15 partition membership is authoritative.
# - TRAIN / VALIDATION / TEST are NEVER re-created here.
# - No scaling is performed here.
# - No resampling is performed here.
# - No feature selection using labels is performed here.
# - No model is trained here.
# - Ground-truth attack annotation columns are NEVER used as predictors.
#
# ==============================================================================================================


import numpy as np
import pandas as pd


# ==============================================================================================================
# 1. CONFIGURATION
# ==============================================================================================================

HISTORY_LENGTH = 15

HORIZONS = [
    1,
    5,
    10,
    15,
    30,
]

EXPECTED_PARTITIONS = {
    "TRAIN",
    "VALIDATION",
    "TEST",
}


print("=" * 142)
print(
    "CELL 16 — FINAL LEAKAGE-SAFE FEATURE CONTRACT + "
    "MULTI-HORIZON SEQUENCE CONSTRUCTION"
)
print("=" * 142)

print()
print(f"History length : {HISTORY_LENGTH} minutes")
print(f"Horizons       : {HORIZONS}")
print(f"Targets/sample : {len(HORIZONS)}")
print()

print("Scaling performed          : NO")
print("Resampling performed       : NO")
print("Label-based feature select : NO")
print("Model trained              : NO")


# ==============================================================================================================
# 2. VERIFY EXACT PERMANENT UPSTREAM OBJECTS
# ==============================================================================================================

REQUIRED_OBJECTS = [
    "minute_df",
    "cell15_partition_lookup",
    "cell15_forecast_metadata",
    "cell15_split_metadata",
]

missing_objects = [
    name
    for name in REQUIRED_OBJECTS
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Required upstream objects are missing:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_objects
        )
        + "\n\nRun the FINAL Cells 9–15 first."
    )


if not isinstance(minute_df, pd.DataFrame):
    raise RuntimeError(
        "minute_df exists but is not a pandas DataFrame."
    )

if not isinstance(cell15_partition_lookup, pd.DataFrame):
    raise RuntimeError(
        "cell15_partition_lookup exists but is not a pandas DataFrame."
    )

if not isinstance(cell15_forecast_metadata, pd.DataFrame):
    raise RuntimeError(
        "cell15_forecast_metadata exists but is not a pandas DataFrame."
    )

if not isinstance(cell15_split_metadata, dict):
    raise RuntimeError(
        "cell15_split_metadata exists but is not a dictionary."
    )


print("\n" + "=" * 142)
print("UPSTREAM OBJECT VALIDATION")
print("=" * 142)

print(
    f"minute_df rows                 : "
    f"{len(minute_df):,}"
)

print(
    f"cell15_partition_lookup rows   : "
    f"{len(cell15_partition_lookup):,}"
)

print(
    f"cell15_forecast_metadata rows  : "
    f"{len(cell15_forecast_metadata):,}"
)

print("✓ Exact permanent Cell 15 objects found.")


# ==============================================================================================================
# 3. VERIFY FROZEN CELL 15 CONTRACT
# ==============================================================================================================

required_metadata_keys = [
    "split_version",
    "split_unit",
    "history_length_minutes",
    "history_back_minutes",
    "forecast_horizons",
    "maximum_horizon_minutes",
    "train_segments",
    "validation_segments",
    "test_segments",
    "train_origins",
    "validation_origins",
    "test_origins",
    "split_selection_uses_model_performance",
    "scaling_performed",
    "resampling_performed",
    "model_trained",
]

missing_metadata_keys = [
    key
    for key in required_metadata_keys
    if key not in cell15_split_metadata
]

if missing_metadata_keys:
    raise RuntimeError(
        "cell15_split_metadata is missing required keys:\n"
        + "\n".join(
            f"  - {key}"
            for key in missing_metadata_keys
        )
    )


if int(
    cell15_split_metadata["history_length_minutes"]
) != HISTORY_LENGTH:
    raise RuntimeError(
        "Cell 15 and Cell 16 history lengths disagree."
    )


cell15_horizons = [
    int(x)
    for x in cell15_split_metadata["forecast_horizons"]
]

if cell15_horizons != HORIZONS:
    raise RuntimeError(
        "Cell 15 and Cell 16 forecast horizons disagree.\n"
        f"Cell 15: {cell15_horizons}\n"
        f"Cell 16: {HORIZONS}"
    )


if int(
    cell15_split_metadata["maximum_horizon_minutes"]
) != max(HORIZONS):
    raise RuntimeError(
        "Cell 15 maximum horizon is inconsistent."
    )


if bool(
    cell15_split_metadata[
        "split_selection_uses_model_performance"
    ]
):
    raise RuntimeError(
        "Cell 15 split selection used model performance."
    )


if bool(cell15_split_metadata["scaling_performed"]):
    raise RuntimeError(
        "Cell 15 reports upstream scaling."
    )

if bool(cell15_split_metadata["resampling_performed"]):
    raise RuntimeError(
        "Cell 15 reports upstream resampling."
    )

if bool(cell15_split_metadata["model_trained"]):
    raise RuntimeError(
        "Cell 15 reports upstream model training."
    )


print("\n" + "=" * 142)
print("FROZEN CELL 15 CONTRACT")
print("=" * 142)

print(
    f"Split version      : "
    f"{cell15_split_metadata['split_version']}"
)

print(
    f"Split unit         : "
    f"{cell15_split_metadata['split_unit']}"
)

print(
    f"TRAIN segments     : "
    f"{cell15_split_metadata['train_segments']}"
)

print(
    f"VALIDATION segments: "
    f"{cell15_split_metadata['validation_segments']}"
)

print(
    f"TEST segments      : "
    f"{cell15_split_metadata['test_segments']}"
)

print("✓ Frozen split metadata validated.")


# ==============================================================================================================
# 4. NORMALIZE AUTHORITATIVE PARTITION LOOKUP
# ==============================================================================================================

required_lookup_columns = [
    "Segment_ID",
    "Origin_Time",
    "Partition",
]

missing_lookup_columns = [
    c
    for c in required_lookup_columns
    if c not in cell15_partition_lookup.columns
]

if missing_lookup_columns:
    raise RuntimeError(
        "cell15_partition_lookup is missing:\n"
        + "\n".join(
            f"  - {c}"
            for c in missing_lookup_columns
        )
    )


origin_assignments = (
    cell15_partition_lookup[
        required_lookup_columns
    ]
    .copy()
)


origin_assignments["Segment_ID"] = pd.to_numeric(
    origin_assignments["Segment_ID"],
    errors="raise",
).astype(int)


origin_assignments["Origin_Time"] = pd.to_datetime(
    origin_assignments["Origin_Time"],
    errors="raise",
)


origin_assignments["Partition"] = (
    origin_assignments["Partition"]
    .astype(str)
    .str.strip()
    .str.upper()
)


observed_partitions = set(
    origin_assignments["Partition"].unique()
)

if observed_partitions != EXPECTED_PARTITIONS:
    raise RuntimeError(
        "Unexpected partition labels.\n"
        f"Observed: {sorted(observed_partitions)}"
    )


duplicate_origins = int(
    origin_assignments.duplicated(
        subset=[
            "Segment_ID",
            "Origin_Time",
        ]
    ).sum()
)

if duplicate_origins != 0:
    raise RuntimeError(
        f"{duplicate_origins:,} duplicate origin assignments found."
    )


expected_origin_counts = {
    "TRAIN":
        int(cell15_split_metadata["train_origins"]),

    "VALIDATION":
        int(cell15_split_metadata["validation_origins"]),

    "TEST":
        int(cell15_split_metadata["test_origins"]),
}


actual_origin_counts = (
    origin_assignments["Partition"]
    .value_counts()
    .to_dict()
)


for partition in [
    "TRAIN",
    "VALIDATION",
    "TEST",
]:

    actual = int(
        actual_origin_counts.get(
            partition,
            0,
        )
    )

    expected = int(
        expected_origin_counts[partition]
    )

    if actual != expected:
        raise RuntimeError(
            f"{partition} origin count mismatch.\n"
            f"Expected: {expected:,}\n"
            f"Observed: {actual:,}"
        )


print("\nFrozen forecasting origins:")

for partition in [
    "TRAIN",
    "VALIDATION",
    "TEST",
]:
    print(
        f"  {partition:<10}: "
        f"{actual_origin_counts[partition]:,}"
    )

print(
    f"  {'TOTAL':<10}: "
    f"{len(origin_assignments):,}"
)

print("✓ Frozen Cell 15 origin assignments preserved.")


# ==============================================================================================================
# 5. VALIDATE FORECAST METADATA
# ==============================================================================================================

required_forecast_columns = [
    "Segment_ID",
    "Origin_Time",
    "Horizon_Minutes",
    "Target_Time",
    "Future_Attack_State",
    "Partition",
]

missing_forecast_columns = [
    c
    for c in required_forecast_columns
    if c not in cell15_forecast_metadata.columns
]

if missing_forecast_columns:
    raise RuntimeError(
        "cell15_forecast_metadata is missing:\n"
        + "\n".join(
            f"  - {c}"
            for c in missing_forecast_columns
        )
    )


forecast_df = (
    cell15_forecast_metadata
    .copy()
)


forecast_df["Segment_ID"] = pd.to_numeric(
    forecast_df["Segment_ID"],
    errors="raise",
).astype(int)


forecast_df["Origin_Time"] = pd.to_datetime(
    forecast_df["Origin_Time"],
    errors="raise",
)


forecast_df["Target_Time"] = pd.to_datetime(
    forecast_df["Target_Time"],
    errors="raise",
)


forecast_df["Horizon_Minutes"] = pd.to_numeric(
    forecast_df["Horizon_Minutes"],
    errors="raise",
).astype(int)


forecast_df["Future_Attack_State"] = pd.to_numeric(
    forecast_df["Future_Attack_State"],
    errors="raise",
).astype(int)


forecast_df["Partition"] = (
    forecast_df["Partition"]
    .astype(str)
    .str.strip()
    .str.upper()
)


invalid_binary_targets = int(
    (~forecast_df["Future_Attack_State"].isin([0, 1])).sum()
)

if invalid_binary_targets:
    raise RuntimeError(
        f"{invalid_binary_targets:,} invalid binary targets found."
    )


duplicate_forecast_keys = int(
    forecast_df.duplicated(
        subset=[
            "Segment_ID",
            "Origin_Time",
            "Horizon_Minutes",
        ]
    ).sum()
)

if duplicate_forecast_keys:
    raise RuntimeError(
        f"{duplicate_forecast_keys:,} duplicate forecast keys found."
    )


expected_horizon_tuple = tuple(HORIZONS)


origin_horizon_sets = (
    forecast_df
    .groupby(
        [
            "Segment_ID",
            "Origin_Time",
        ]
    )["Horizon_Minutes"]
    .apply(
        lambda s:
            tuple(
                sorted(
                    set(
                        int(x)
                        for x in s
                    )
                )
            )
    )
)


bad_horizon_sets = int(
    origin_horizon_sets.apply(
        lambda x:
            x != expected_horizon_tuple
    ).sum()
)

if bad_horizon_sets:
    raise RuntimeError(
        f"{bad_horizon_sets:,} origins have incorrect horizon sets."
    )


expected_target_times = (
    forecast_df["Origin_Time"]
    +
    pd.to_timedelta(
        forecast_df["Horizon_Minutes"],
        unit="m",
    )
)


target_alignment_errors = int(
    (
        forecast_df["Target_Time"]
        != expected_target_times
    ).sum()
)

if target_alignment_errors:
    raise RuntimeError(
        f"{target_alignment_errors:,} target-time alignment errors."
    )


partition_check = (
    origin_assignments
    .rename(
        columns={
            "Partition":
                "Expected_Partition"
        }
    )
)


forecast_df = forecast_df.merge(
    partition_check,
    on=[
        "Segment_ID",
        "Origin_Time",
    ],
    how="left",
    validate="many_to_one",
)


missing_assignments = int(
    forecast_df[
        "Expected_Partition"
    ].isna().sum()
)

if missing_assignments:
    raise RuntimeError(
        f"{missing_assignments:,} forecast rows lack partition assignments."
    )


partition_disagreements = int(
    (
        forecast_df["Partition"]
        != forecast_df["Expected_Partition"]
    ).sum()
)

if partition_disagreements:
    raise RuntimeError(
        f"{partition_disagreements:,} partition disagreements."
    )


expected_forecast_rows = (
    len(origin_assignments)
    * len(HORIZONS)
)

if len(forecast_df) != expected_forecast_rows:
    raise RuntimeError(
        "Forecast row count mismatch.\n"
        f"Expected: {expected_forecast_rows:,}\n"
        f"Observed: {len(forecast_df):,}"
    )


print("\n" + "=" * 142)
print("FORECASTING CONTRACT")
print("=" * 142)

print(
    f"Forecast origins             : "
    f"{len(origin_horizon_sets):,}"
)

print(
    f"Forecast rows                : "
    f"{len(forecast_df):,}"
)

print(
    f"Duplicate forecast keys      : "
    f"{duplicate_forecast_keys:,}"
)

print(
    f"Wrong horizon sets           : "
    f"{bad_horizon_sets:,}"
)

print(
    f"Target alignment errors      : "
    f"{target_alignment_errors:,}"
)

print(
    f"Partition disagreements      : "
    f"{partition_disagreements:,}"
)

print("✓ Forecasting contract validated.")


# ==============================================================================================================
# 6. NORMALIZE minute_df
# ==============================================================================================================

required_minute_columns = [
    "Segment_ID",
    "Minute",
]

missing_minute_columns = [
    c
    for c in required_minute_columns
    if c not in minute_df.columns
]

if missing_minute_columns:
    raise RuntimeError(
        "minute_df is missing required columns:\n"
        + "\n".join(
            f"  - {c}"
            for c in missing_minute_columns
        )
    )


minute_work = minute_df.copy()


minute_work["Segment_ID"] = pd.to_numeric(
    minute_work["Segment_ID"],
    errors="raise",
).astype(int)


minute_work["Minute"] = pd.to_datetime(
    minute_work["Minute"],
    errors="raise",
)


minute_work = (
    minute_work
    .sort_values(
        [
            "Segment_ID",
            "Minute",
        ]
    )
    .reset_index(drop=True)
)


duplicate_minute_keys = int(
    minute_work.duplicated(
        subset=[
            "Segment_ID",
            "Minute",
        ]
    ).sum()
)

if duplicate_minute_keys:
    raise RuntimeError(
        f"{duplicate_minute_keys:,} duplicate minute keys found."
    )


print("\n✓ minute_df key structure verified.")


# ==============================================================================================================
# 7. FINAL LEAKAGE-SAFE FEATURE CONTRACT
# ==============================================================================================================
#
# IMPORTANT:
#
# These columns are excluded even if numeric because they are derived from
# ground-truth attack labels / attack annotations.
#
# They would give the model privileged information unavailable to a genuine
# forecasting system unless a separate upstream detector produced them.
#
# ==============================================================================================================


IDENTITY_COLUMNS = {
    "Segment_ID",
    "Minute",
}


ATTACK_GROUND_TRUTH_COLUMNS = {
    "Attack_State",
    "Attack_Flow_Count",
    "Attack_Ratio",
    "Attack_Types",
    "Attack_Source_IPs",
    "Attack_Destination_IPs",
    "Attack_Edges",
    "Typed_Attack_Edges",
    "Attack_Type_Count",
    "Attack_Source_Count",
    "Attack_Destination_Count",
    "Attack_Edge_Count",
    "Typed_Attack_Edge_Count",
}


FORBIDDEN_PREDICTOR_COLUMNS = (
    IDENTITY_COLUMNS
    |
    ATTACK_GROUND_TRUTH_COLUMNS
)


feature_contract_rows = []

numeric_candidates = []


for column in minute_work.columns:

    dtype = minute_work[column].dtype

    if column in IDENTITY_COLUMNS:

        role = "IDENTITY — FORBIDDEN"

    elif column in ATTACK_GROUND_TRUTH_COLUMNS:

        role = "ATTACK GROUND TRUTH — FORBIDDEN"

    elif pd.api.types.is_numeric_dtype(
        minute_work[column]
    ):

        role = "NUMERIC CANDIDATE"

        numeric_candidates.append(
            column
        )

    else:

        role = "NON-NUMERIC METADATA"

    feature_contract_rows.append(
        {
            "Feature":
                column,

            "Role":
                role,

            "Dtype":
                str(dtype),
        }
    )


cell16_feature_contract = pd.DataFrame(
    feature_contract_rows
)


print("\n" + "=" * 142)
print("FINAL FEATURE CONTRACT")
print("=" * 142)

print(
    cell16_feature_contract
    .to_string(index=False)
)


print(
    f"\nInitial leakage-safe numeric candidates : "
    f"{len(numeric_candidates):,}"
)


# ==============================================================================================================
# 8. FEATURE QUALITY AUDIT
# ==============================================================================================================

quality_rows = []


for feature in numeric_candidates:

    values = pd.to_numeric(
        minute_work[feature],
        errors="coerce",
    ).to_numpy(
        dtype=np.float64
    )


    missing_count = int(
        np.isnan(values).sum()
    )

    positive_inf_count = int(
        np.isposinf(values).sum()
    )

    negative_inf_count = int(
        np.isneginf(values).sum()
    )


    finite_mask = np.isfinite(values)

    finite_values = values[
        finite_mask
    ]


    if len(finite_values) == 0:

        unique_count = 0

        dominant_fraction = np.nan

        minimum = np.nan
        maximum = np.nan
        mean = np.nan
        std = np.nan

        constant = True

    else:

        unique_values, counts = np.unique(
            finite_values,
            return_counts=True,
        )

        unique_count = len(
            unique_values
        )

        dominant_fraction = float(
            counts.max()
            /
            counts.sum()
        )

        minimum = float(
            finite_values.min()
        )

        maximum = float(
            finite_values.max()
        )

        mean = float(
            finite_values.mean()
        )

        std = float(
            finite_values.std()
        )

        constant = (
            unique_count <= 1
        )


    near_constant = bool(
        (not constant)
        and
        np.isfinite(dominant_fraction)
        and
        dominant_fraction >= 0.995
    )


    quality_rows.append(
        {
            "Feature":
                feature,

            "Missing":
                missing_count,

            "Positive_Inf":
                positive_inf_count,

            "Negative_Inf":
                negative_inf_count,

            "Unique_Finite_Values":
                unique_count,

            "Dominant_Fraction":
                dominant_fraction,

            "Constant":
                constant,

            "Near_Constant":
                near_constant,

            "Min":
                minimum,

            "Max":
                maximum,

            "Mean":
                mean,

            "Std":
                std,
        }
    )


cell16_feature_quality = pd.DataFrame(
    quality_rows
)


print("\n" + "=" * 142)
print("FEATURE QUALITY AUDIT")
print("=" * 142)

print(
    cell16_feature_quality
    .to_string(index=False)
)


# ==============================================================================================================
# 9. FREEZE FINAL PREDICTOR LIST
# ==============================================================================================================
#
# Only truly constant columns are removed here.
#
# Near-constant columns are NOT automatically removed.
#
# Missing / inf values are not silently repaired here.
# They must be handled TRAIN-only in the preprocessing cell.
#
# ==============================================================================================================


constant_features = (
    cell16_feature_quality.loc[
        cell16_feature_quality["Constant"],
        "Feature",
    ]
    .tolist()
)


near_constant_features = (
    cell16_feature_quality.loc[
        cell16_feature_quality["Near_Constant"],
        "Feature",
    ]
    .tolist()
)


cell16_predictor_columns = [
    feature
    for feature in numeric_candidates
    if feature not in constant_features
]


if not cell16_predictor_columns:
    raise RuntimeError(
        "No predictor columns remain after the feature audit."
    )


forbidden_leakage = sorted(
    set(cell16_predictor_columns)
    &
    ATTACK_GROUND_TRUTH_COLUMNS
)

if forbidden_leakage:
    raise RuntimeError(
        "Attack ground-truth columns entered the predictor set:\n"
        + "\n".join(
            f"  - {x}"
            for x in forbidden_leakage
        )
    )


print("\n" + "=" * 142)
print("FROZEN PREDICTOR CONTRACT")
print("=" * 142)

print(
    f"Leakage-safe numeric candidates : "
    f"{len(numeric_candidates):,}"
)

print(
    f"Constant removed                : "
    f"{len(constant_features):,}"
)

print(
    f"Near-constant retained          : "
    f"{len(near_constant_features):,}"
)

print(
    f"Final predictors                : "
    f"{len(cell16_predictor_columns):,}"
)


if constant_features:

    print("\nConstant features removed:")

    for feature in constant_features:
        print(
            f"  - {feature}"
        )


if near_constant_features:

    print(
        "\nNear-constant features retained "
        "for later TRAIN-only modelling decisions:"
    )

    for feature in near_constant_features:
        print(
            f"  - {feature}"
        )


print(
    "\n✓ No attack ground-truth annotation is present "
    "in the predictor set."
)


# ==============================================================================================================
# 10. BUILD FAST SEGMENT LOOKUPS
# ==============================================================================================================

segment_feature_lookup = {}

segment_minute_sets = {}


for segment_id, group in minute_work.groupby(
    "Segment_ID",
    sort=True,
):

    group = (
        group
        .sort_values("Minute")
        .copy()
    )


    indexed = (
        group
        .set_index("Minute")
    )


    segment_feature_lookup[
        int(segment_id)
    ] = indexed


    segment_minute_sets[
        int(segment_id)
    ] = set(
        indexed.index.tolist()
    )


print(
    f"\n✓ Prepared feature lookups for "
    f"{len(segment_feature_lookup):,} segments."
)


# ==============================================================================================================
# 11. BUILD TARGET LOOKUP
# ==============================================================================================================

target_pivot = (
    forecast_df
    .pivot(
        index=[
            "Segment_ID",
            "Origin_Time",
        ],
        columns="Horizon_Minutes",
        values="Future_Attack_State",
    )
)


target_pivot = (
    target_pivot[
        HORIZONS
    ]
    .sort_index()
)


if target_pivot.isna().any().any():
    raise RuntimeError(
        "Target pivot contains missing horizon targets."
    )


target_lookup = {
    (
        int(segment_id),
        pd.Timestamp(origin_time),
    ):
    row.to_numpy(
        dtype=np.float32
    )

    for (
        segment_id,
        origin_time
    ), row in target_pivot.iterrows()
}


if len(target_lookup) != len(origin_assignments):
    raise RuntimeError(
        "Target lookup does not contain exactly one target vector "
        "per frozen origin."
    )


print(
    f"✓ Prepared target vectors for "
    f"{len(target_lookup):,} origins."
)


# ==============================================================================================================
# 12. CONSTRUCT TEMPORAL SEQUENCES
# ==============================================================================================================

X_by_partition = {
    "TRAIN": [],
    "VALIDATION": [],
    "TEST": [],
}

y_by_partition = {
    "TRAIN": [],
    "VALIDATION": [],
    "TEST": [],
}

metadata_by_partition = {
    "TRAIN": [],
    "VALIDATION": [],
    "TEST": [],
}


history_errors = []

target_errors = []


ordered_origins = (
    origin_assignments
    .sort_values(
        [
            "Segment_ID",
            "Origin_Time",
        ]
    )
    .reset_index(drop=True)
)


for row in ordered_origins.itertuples(
    index=False
):

    segment_id = int(
        row.Segment_ID
    )

    origin = pd.Timestamp(
        row.Origin_Time
    )

    partition = str(
        row.Partition
    )


    if segment_id not in segment_feature_lookup:

        history_errors.append(
            {
                "Segment_ID":
                    segment_id,

                "Origin_Time":
                    origin,

                "Reason":
                    "Segment missing from minute_df",
            }
        )

        continue


    # ----------------------------------------------------------------------------------------------------------
    # CRITICAL PANDAS-SAFE TIMESTAMP ARITHMETIC
    #
    # Required sequence:
    #
    #     t-14, t-13, ..., t-1, t
    #
    # Never subtract bare integers from pd.Timestamp.
    # ----------------------------------------------------------------------------------------------------------

    history_minutes = [
        origin
        -
        pd.Timedelta(
            minutes=offset
        )

        for offset in range(
            HISTORY_LENGTH - 1,
            -1,
            -1,
        )
    ]


    if len(history_minutes) != HISTORY_LENGTH:
        raise RuntimeError(
            "Internal history-length construction error."
        )


    if history_minutes[-1] != origin:
        raise RuntimeError(
            "History sequence does not terminate at the forecast origin."
        )


    expected_history_start = (
        origin
        -
        pd.Timedelta(
            minutes=HISTORY_LENGTH - 1
        )
    )


    if history_minutes[0] != expected_history_start:
        raise RuntimeError(
            "History start is incorrectly aligned."
        )


    available_minutes = (
        segment_minute_sets[
            segment_id
        ]
    )


    missing_history_minutes = [
        minute
        for minute in history_minutes
        if minute not in available_minutes
    ]


    if missing_history_minutes:

        history_errors.append(
            {
                "Segment_ID":
                    segment_id,

                "Origin_Time":
                    origin,

                "Reason":
                    (
                        "Missing history minutes: "
                        +
                        str(
                            missing_history_minutes[:5]
                        )
                    ),
            }
        )

        continue


    segment_data = (
        segment_feature_lookup[
            segment_id
        ]
    )


    history_frame = (
        segment_data
        .loc[
            history_minutes,
            cell16_predictor_columns,
        ]
    )


    if history_frame.shape != (
        HISTORY_LENGTH,
        len(cell16_predictor_columns),
    ):

        history_errors.append(
            {
                "Segment_ID":
                    segment_id,

                "Origin_Time":
                    origin,

                "Reason":
                    (
                        "Unexpected history-frame shape "
                        f"{history_frame.shape}"
                    ),
            }
        )

        continue


    X_sequence = (
        history_frame
        .to_numpy(
            dtype=np.float32
        )
    )


    target_key = (
        segment_id,
        origin,
    )


    if target_key not in target_lookup:

        target_errors.append(
            {
                "Segment_ID":
                    segment_id,

                "Origin_Time":
                    origin,

                "Reason":
                    "Missing target vector",
            }
        )

        continue


    y_vector = (
        target_lookup[
            target_key
        ]
        .astype(
            np.float32,
            copy=True,
        )
    )


    if y_vector.shape != (
        len(HORIZONS),
    ):

        target_errors.append(
            {
                "Segment_ID":
                    segment_id,

                "Origin_Time":
                    origin,

                "Reason":
                    (
                        "Unexpected target shape "
                        f"{y_vector.shape}"
                    ),
            }
        )

        continue


    if not np.isin(
        y_vector,
        [0.0, 1.0],
    ).all():

        target_errors.append(
            {
                "Segment_ID":
                    segment_id,

                "Origin_Time":
                    origin,

                "Reason":
                    "Non-binary target value",
            }
        )

        continue


    X_by_partition[
        partition
    ].append(
        X_sequence
    )


    y_by_partition[
        partition
    ].append(
        y_vector
    )


    metadata_by_partition[
        partition
    ].append(
        {
            "Segment_ID":
                segment_id,

            "Origin_Time":
                origin,

            "History_Start":
                history_minutes[0],

            "History_End":
                history_minutes[-1],

            "Partition":
                partition,
        }
    )


# ==============================================================================================================
# 13. HARD-FAIL ON SEQUENCE CONSTRUCTION ERRORS
# ==============================================================================================================

if history_errors:

    history_error_df = pd.DataFrame(
        history_errors
    )

    raise RuntimeError(
        "History sequence construction failed for "
        f"{len(history_errors):,} origins.\n\n"
        + history_error_df.head(
            20
        ).to_string(
            index=False
        )
    )


if target_errors:

    target_error_df = pd.DataFrame(
        target_errors
    )

    raise RuntimeError(
        "Target construction failed for "
        f"{len(target_errors):,} origins.\n\n"
        + target_error_df.head(
            20
        ).to_string(
            index=False
        )
    )


print(
    "\n✓ Every frozen origin has a complete "
    "15-minute history and five-horizon target vector."
)


# ==============================================================================================================
# 14. CONVERT TO FINAL NUMPY ARRAYS
# ==============================================================================================================

N_FEATURES = len(
    cell16_predictor_columns
)


def finalize_partition_arrays(
    partition
):

    X_list = X_by_partition[
        partition
    ]

    y_list = y_by_partition[
        partition
    ]


    if len(X_list) == 0:

        X = np.empty(
            (
                0,
                HISTORY_LENGTH,
                N_FEATURES,
            ),
            dtype=np.float32,
        )

        y = np.empty(
            (
                0,
                len(HORIZONS),
            ),
            dtype=np.float32,
        )

    else:

        X = np.stack(
            X_list,
            axis=0,
        ).astype(
            np.float32,
            copy=False,
        )

        y = np.stack(
            y_list,
            axis=0,
        ).astype(
            np.float32,
            copy=False,
        )


    metadata = pd.DataFrame(
        metadata_by_partition[
            partition
        ]
    )


    return X, y, metadata


X_train_raw, y_train, cell16_train_metadata = (
    finalize_partition_arrays(
        "TRAIN"
    )
)


X_validation_raw, y_validation, cell16_validation_metadata = (
    finalize_partition_arrays(
        "VALIDATION"
    )
)


X_test_raw, y_test, cell16_test_metadata = (
    finalize_partition_arrays(
        "TEST"
    )
)


# Friendly aliases

X_val_raw = X_validation_raw
y_val = y_validation

cell16_val_metadata = (
    cell16_validation_metadata
)


# ==============================================================================================================
# 15. FINAL SHAPE VALIDATION
# ==============================================================================================================

expected_shapes = {
    "TRAIN":
        (
            expected_origin_counts["TRAIN"],
            HISTORY_LENGTH,
            N_FEATURES,
        ),

    "VALIDATION":
        (
            expected_origin_counts["VALIDATION"],
            HISTORY_LENGTH,
            N_FEATURES,
        ),

    "TEST":
        (
            expected_origin_counts["TEST"],
            HISTORY_LENGTH,
            N_FEATURES,
        ),
}


actual_X = {
    "TRAIN":
        X_train_raw,

    "VALIDATION":
        X_validation_raw,

    "TEST":
        X_test_raw,
}


actual_y = {
    "TRAIN":
        y_train,

    "VALIDATION":
        y_validation,

    "TEST":
        y_test,
}


for partition in [
    "TRAIN",
    "VALIDATION",
    "TEST",
]:

    X = actual_X[
        partition
    ]

    y = actual_y[
        partition
    ]


    if X.shape != expected_shapes[
        partition
    ]:

        raise RuntimeError(
            f"{partition} X shape mismatch.\n"
            f"Expected: {expected_shapes[partition]}\n"
            f"Observed: {X.shape}"
        )


    expected_y_shape = (
        expected_origin_counts[
            partition
        ],
        len(HORIZONS),
    )


    if y.shape != expected_y_shape:

        raise RuntimeError(
            f"{partition} y shape mismatch.\n"
            f"Expected: {expected_y_shape}\n"
            f"Observed: {y.shape}"
        )


print("\n" + "=" * 142)
print("FINAL RAW SEQUENCE SHAPES")
print("=" * 142)

print(
    f"X_train_raw      : "
    f"{X_train_raw.shape}"
)

print(
    f"y_train          : "
    f"{y_train.shape}"
)

print(
    f"X_validation_raw : "
    f"{X_validation_raw.shape}"
)

print(
    f"y_validation     : "
    f"{y_validation.shape}"
)

print(
    f"X_test_raw       : "
    f"{X_test_raw.shape}"
)

print(
    f"y_test           : "
    f"{y_test.shape}"
)


# ==============================================================================================================
# 16. NON-FINITE AUDIT
# ==============================================================================================================
#
# We do NOT repair anything here.
#
# If missing/inf values exist, preprocessing must learn any repair
# parameters using TRAIN only.
#
# ==============================================================================================================


def nonfinite_count(array):

    return int(
        (~np.isfinite(array)).sum()
    )


train_nonfinite = nonfinite_count(
    X_train_raw
)

validation_nonfinite = nonfinite_count(
    X_validation_raw
)

test_nonfinite = nonfinite_count(
    X_test_raw
)


print("\n" + "=" * 142)
print("RAW FEATURE NON-FINITE AUDIT")
print("=" * 142)

print(
    f"TRAIN non-finite values      : "
    f"{train_nonfinite:,}"
)

print(
    f"VALIDATION non-finite values : "
    f"{validation_nonfinite:,}"
)

print(
    f"TEST non-finite values       : "
    f"{test_nonfinite:,}"
)


# ==============================================================================================================
# 17. TARGET DISTRIBUTION AUDIT
# ==============================================================================================================


def target_distribution_table(
    y,
    partition,
):

    rows = []

    for j, horizon in enumerate(
        HORIZONS
    ):

        positives = int(
            y[:, j].sum()
        )

        total = int(
            y.shape[0]
        )

        negatives = (
            total
            -
            positives
        )

        positive_rate = (
            positives / total
            if total
            else np.nan
        )

        rows.append(
            {
                "Partition":
                    partition,

                "Horizon_Minutes":
                    horizon,

                "Samples":
                    total,

                "Positive_Attack_Targets":
                    positives,

                "Negative_Targets":
                    negatives,

                "Positive_Rate":
                    positive_rate,
            }
        )

    return rows


distribution_rows = []

distribution_rows.extend(
    target_distribution_table(
        y_train,
        "TRAIN",
    )
)

distribution_rows.extend(
    target_distribution_table(
        y_validation,
        "VALIDATION",
    )
)

distribution_rows.extend(
    target_distribution_table(
        y_test,
        "TEST",
    )
)


cell16_target_distribution = pd.DataFrame(
    distribution_rows
)


print("\n" + "=" * 142)
print("MULTI-HORIZON TARGET DISTRIBUTION")
print("=" * 142)

print(
    cell16_target_distribution
    .to_string(
        index=False
    )
)


# ==============================================================================================================
# 18. TEMPORAL / PARTITION INTEGRITY AUDIT
# ==============================================================================================================

metadata_frames = [
    cell16_train_metadata,
    cell16_validation_metadata,
    cell16_test_metadata,
]


all_sequence_metadata = pd.concat(
    metadata_frames,
    ignore_index=True,
)


duplicate_sequence_origins = int(
    all_sequence_metadata.duplicated(
        subset=[
            "Segment_ID",
            "Origin_Time",
        ]
    ).sum()
)


if duplicate_sequence_origins:
    raise RuntimeError(
        f"{duplicate_sequence_origins:,} sequence origins "
        "appear more than once across partitions."
    )


observed_sequence_keys = set(
    zip(
        all_sequence_metadata[
            "Segment_ID"
        ].astype(int),

        pd.to_datetime(
            all_sequence_metadata[
                "Origin_Time"
            ]
        ),
    )
)


expected_sequence_keys = set(
    zip(
        origin_assignments[
            "Segment_ID"
        ].astype(int),

        pd.to_datetime(
            origin_assignments[
                "Origin_Time"
            ]
        ),
    )
)


missing_sequence_keys = (
    expected_sequence_keys
    -
    observed_sequence_keys
)


extra_sequence_keys = (
    observed_sequence_keys
    -
    expected_sequence_keys
)


if missing_sequence_keys:
    raise RuntimeError(
        f"{len(missing_sequence_keys):,} frozen origins "
        "were lost during sequence construction."
    )


if extra_sequence_keys:
    raise RuntimeError(
        f"{len(extra_sequence_keys):,} unexpected origins "
        "appeared during sequence construction."
    )


print("\n" + "=" * 142)
print("SEQUENCE INTEGRITY AUDIT")
print("=" * 142)

print(
    f"Duplicate sequence origins : "
    f"{duplicate_sequence_origins:,}"
)

print(
    f"Missing frozen origins     : "
    f"{len(missing_sequence_keys):,}"
)

print(
    f"Unexpected origins         : "
    f"{len(extra_sequence_keys):,}"
)

print(
    "✓ Every frozen Cell 15 origin appears exactly once "
    "in Cell 16."
)


# ==============================================================================================================
# 19. CREATE PERMANENT CELL 16 CONTRACT
# ==============================================================================================================

cell16_sequence_metadata = (
    all_sequence_metadata
    .copy()
)


cell16_metadata = {

    "cell_version":
        "CELL16_FINAL_LEAKAGE_SAFE_SEQUENCE_V1",

    "source_split_version":
        cell15_split_metadata[
            "split_version"
        ],

    "history_length_minutes":
        HISTORY_LENGTH,

    "history_back_minutes":
        HISTORY_LENGTH - 1,

    "forecast_horizons":
        HORIZONS.copy(),

    "targets_per_sample":
        len(HORIZONS),

    "predictor_count":
        len(cell16_predictor_columns),

    "predictor_columns":
        cell16_predictor_columns.copy(),

    "forbidden_attack_ground_truth_columns":
        sorted(
            ATTACK_GROUND_TRUTH_COLUMNS
        ),

    "constant_features_removed":
        constant_features.copy(),

    "near_constant_features_retained":
        near_constant_features.copy(),

    "train_samples":
        int(
            X_train_raw.shape[0]
        ),

    "validation_samples":
        int(
            X_validation_raw.shape[0]
        ),

    "test_samples":
        int(
            X_test_raw.shape[0]
        ),

    "train_nonfinite_values":
        train_nonfinite,

    "validation_nonfinite_values":
        validation_nonfinite,

    "test_nonfinite_values":
        test_nonfinite,

    "scaling_performed":
        False,

    "imputation_performed":
        False,

    "resampling_performed":
        False,

    "model_trained":
        False,
}


# ==============================================================================================================
# 20. FINAL SUMMARY
# ==============================================================================================================

print("\n" + "=" * 142)
print("CELL 16 FINAL CONTRACT")
print("=" * 142)

print(
    f"Predictors                  : "
    f"{len(cell16_predictor_columns):,}"
)

print(
    f"History                     : "
    f"{HISTORY_LENGTH} minutes"
)

print(
    f"Horizons                    : "
    f"{HORIZONS}"
)

print()
print(
    f"TRAIN samples               : "
    f"{X_train_raw.shape[0]:,}"
)

print(
    f"VALIDATION samples          : "
    f"{X_validation_raw.shape[0]:,}"
)

print(
    f"TEST samples                : "
    f"{X_test_raw.shape[0]:,}"
)

print()
print(
    f"TRAIN tensor                : "
    f"{X_train_raw.shape}"
)

print(
    f"VALIDATION tensor           : "
    f"{X_validation_raw.shape}"
)

print(
    f"TEST tensor                 : "
    f"{X_test_raw.shape}"
)

print()
print(
    "Attack ground-truth predictors : EXCLUDED"
)

print(
    "Scaling                        : NOT PERFORMED"
)

print(
    "Imputation                     : NOT PERFORMED"
)

print(
    "Resampling                     : NOT PERFORMED"
)

print(
    "Model training                 : NOT PERFORMED"
)


print("\nPermanent Cell 16 objects created:")

print("  cell16_predictor_columns")
print("  cell16_feature_contract")
print("  cell16_feature_quality")
print("  cell16_target_distribution")
print("  cell16_sequence_metadata")
print("  cell16_metadata")

print("  X_train_raw")
print("  y_train")

print("  X_validation_raw")
print("  y_validation")

print("  X_test_raw")
print("  y_test")

print()
print("Aliases:")
print("  X_val_raw = X_validation_raw")
print("  y_val     = y_validation")


print("\n" + "=" * 142)
print("CELL 16 COMPLETE")
print("=" * 142)

print(
    "\n✓ Frozen Cell 15 partitions preserved."
)

print(
    "✓ Every sample contains exactly 15 causal history minutes."
)

print(
    "✓ Every sample contains exactly five future binary targets."
)

print(
    "✓ No future target information enters X."
)

print(
    "✓ Attack-label-derived ground-truth metadata is excluded from X."
)

print(
    "✓ No scaling/imputation/resampling/model fitting has occurred."
)

print(
    "\nNEXT: TRAIN-only preprocessing/scaling, followed by model construction."
)

CELL 16 — FINAL LEAKAGE-SAFE FEATURE CONTRACT + MULTI-HORIZON SEQUENCE CONSTRUCTION

History length : 15 minutes
Horizons       : [1, 5, 10, 15, 30]
Targets/sample : 5

Scaling performed          : NO
Resampling performed       : NO
Label-based feature select : NO
Model trained              : NO

UPSTREAM OBJECT VALIDATION
minute_df rows                 : 2,454
cell15_partition_lookup rows   : 2,014
cell15_forecast_metadata rows  : 10,070
✓ Exact permanent Cell 15 objects found.

FROZEN CELL 15 CONTRACT
Split version      : CELL15_PERMANENT_STRICT_TEMPORAL_V1
Split unit         : capture_segment
TRAIN segments     : [0, 1, 2, 3, 4, 5]
VALIDATION segments: [6, 7]
TEST segments      : [8, 9]
✓ Frozen split metadata validated.

Frozen forecasting origins:
  TRAIN     : 1,220
  VALIDATION: 398
  TEST      : 396
  TOTAL     : 2,014
✓ Frozen Cell 15 origin assignments preserved.

FORECASTING CONTRACT
Forecast origins             : 2,014
Forecast rows                : 10,070
Duplicate forecas

In [38]:
# ==============================================================================================================
# CELL 17 — FINAL TRAIN-ONLY PREPROCESSING + SCALING + MODEL-READY TENSORS
# ==============================================================================================================
#
# PURPOSE
# -------
# Convert the FINAL raw temporal tensors from Cell 16 into model-ready tensors.
#
# AUTHORITATIVE CELL 16 INPUTS
# ----------------------------
# X_train_raw
# X_validation_raw
# X_test_raw
#
# y_train
# y_validation
# y_test
#
# cell16_predictor_columns
# cell16_train_metadata
# cell16_validation_metadata
# cell16_test_metadata
# cell16_metadata
#
# IMPORTANT
# ---------
# Cell 17:
#
#   ✓ DOES NOT recreate the split
#   ✓ DOES NOT recreate sequences
#   ✓ DOES NOT change targets
#   ✓ DOES NOT perform feature selection
#   ✓ DOES NOT resample
#   ✓ DOES NOT choose thresholds
#   ✓ DOES NOT train a model
#
# It ONLY:
#
#   1. validates Cell 16,
#   2. checks non-finite values,
#   3. optionally fits TRAIN-only imputation IF actually required,
#   4. fits StandardScaler on TRAIN ONLY,
#   5. transforms VALIDATION and TEST using the TRAIN-fitted scaler,
#   6. restores the original 3-D temporal structure,
#   7. validates the final model-ready tensors.
#
# ==============================================================================================================


import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler


# ==============================================================================================================
# 1. CONFIGURATION
# ==============================================================================================================

HISTORY_LENGTH = 15

FORECAST_HORIZONS = [
    1,
    5,
    10,
    15,
    30,
]

N_HORIZONS = len(
    FORECAST_HORIZONS
)

SCALER_NAME = "StandardScaler"

IMPUTATION_STRATEGY = "median"


print("=" * 150)
print(
    "CELL 17 — FINAL TRAIN-ONLY PREPROCESSING + "
    "SCALING + MODEL-READY TENSORS"
)
print("=" * 150)

print("\nConfiguration")
print("-" * 110)

print(
    f"History length              : "
    f"{HISTORY_LENGTH} minutes"
)

print(
    f"Forecast horizons           : "
    f"{FORECAST_HORIZONS}"
)

print(
    f"Targets per sample          : "
    f"{N_HORIZONS}"
)

print(
    f"Scaler                      : "
    f"{SCALER_NAME}"
)

print(
    "Scaler fit source           : TRAIN ONLY"
)

print(
    "Imputation                  : ONLY IF REQUIRED"
)

print(
    f"Imputation strategy         : "
    f"{IMPUTATION_STRATEGY}"
)

print(
    "Validation used for fitting : NO"
)

print(
    "Test used for fitting       : NO"
)

print(
    "Resampling performed        : NO"
)

print(
    "Threshold selection         : NO"
)

print(
    "Model trained               : NO"
)


# ==============================================================================================================
# 2. EXACT CELL 16 OBJECT VALIDATION
# ==============================================================================================================

REQUIRED_CELL16_OBJECTS = [

    "X_train_raw",
    "X_validation_raw",
    "X_test_raw",

    "y_train",
    "y_validation",
    "y_test",

    "cell16_predictor_columns",

    "cell16_train_metadata",
    "cell16_validation_metadata",
    "cell16_test_metadata",

    "cell16_metadata",
]


missing_objects = [

    name

    for name in REQUIRED_CELL16_OBJECTS

    if name not in globals()
]


if missing_objects:

    raise RuntimeError(

        "Required FINAL Cell 16 objects are missing:\n"

        + "\n".join(

            f"  - {name}"

            for name in missing_objects

        )

        + "\n\nRun the FINAL Cell 16 first."
    )


print("\n" + "=" * 150)
print("EXACT CELL 16 OBJECT VALIDATION")
print("=" * 150)

print("✓ All required Cell 16 objects found.")


# ==============================================================================================================
# 3. COPY AUTHORITATIVE CELL 16 OBJECTS
# ==============================================================================================================

X_train_source = np.asarray(
    X_train_raw
).copy()


X_validation_source = np.asarray(
    X_validation_raw
).copy()


X_test_source = np.asarray(
    X_test_raw
).copy()


y_train_source = np.asarray(
    y_train
).copy()


y_validation_source = np.asarray(
    y_validation
).copy()


y_test_source = np.asarray(
    y_test
).copy()


MODEL_FEATURES = list(
    cell16_predictor_columns
)


N_MODEL_FEATURES = len(
    MODEL_FEATURES
)


train_metadata = (
    cell16_train_metadata
    .copy(deep=True)
)


validation_metadata = (
    cell16_validation_metadata
    .copy(deep=True)
)


test_metadata = (
    cell16_test_metadata
    .copy(deep=True)
)


print(
    f"\nFinal Cell 16 predictors : "
    f"{N_MODEL_FEATURES}"
)

print(
    f"X_train_raw              : "
    f"{X_train_source.shape}"
)

print(
    f"X_validation_raw         : "
    f"{X_validation_source.shape}"
)

print(
    f"X_test_raw               : "
    f"{X_test_source.shape}"
)

print(
    f"y_train                  : "
    f"{y_train_source.shape}"
)

print(
    f"y_validation             : "
    f"{y_validation_source.shape}"
)

print(
    f"y_test                   : "
    f"{y_test_source.shape}"
)


# ==============================================================================================================
# 4. CELL 16 METADATA CONTRACT VALIDATION
# ==============================================================================================================

required_metadata_keys = [

    "history_length_minutes",
    "forecast_horizons",
    "targets_per_sample",

    "predictor_count",
    "predictor_columns",

    "train_samples",
    "validation_samples",
    "test_samples",

    "scaling_performed",
    "imputation_performed",
    "resampling_performed",
]


missing_metadata_keys = [

    key

    for key in required_metadata_keys

    if key not in cell16_metadata
]


if missing_metadata_keys:

    raise RuntimeError(

        "cell16_metadata is missing required keys:\n"

        + "\n".join(

            f"  - {key}"

            for key in missing_metadata_keys

        )
    )


if int(
    cell16_metadata[
        "history_length_minutes"
    ]
) != HISTORY_LENGTH:

    raise RuntimeError(
        "Cell 16 history length does not match Cell 17."
    )


metadata_horizons = [

    int(x)

    for x in cell16_metadata[
        "forecast_horizons"
    ]
]


if metadata_horizons != FORECAST_HORIZONS:

    raise RuntimeError(

        "Cell 16 forecast horizons do not match Cell 17.\n"

        f"Cell 16: {metadata_horizons}\n"

        f"Cell 17: {FORECAST_HORIZONS}"
    )


if int(
    cell16_metadata[
        "targets_per_sample"
    ]
) != N_HORIZONS:

    raise RuntimeError(
        "Cell 16 target count is inconsistent."
    )


if int(
    cell16_metadata[
        "predictor_count"
    ]
) != N_MODEL_FEATURES:

    raise RuntimeError(

        "Cell 16 predictor count mismatch.\n"

        f"Metadata: "
        f"{cell16_metadata['predictor_count']}\n"

        f"Predictor list: "
        f"{N_MODEL_FEATURES}"
    )


metadata_predictors = list(
    cell16_metadata[
        "predictor_columns"
    ]
)


if metadata_predictors != MODEL_FEATURES:

    raise RuntimeError(
        "cell16_predictor_columns and "
        "cell16_metadata['predictor_columns'] disagree."
    )


if bool(
    cell16_metadata[
        "scaling_performed"
    ]
):

    raise RuntimeError(
        "Cell 16 reports that scaling was already performed."
    )


if bool(
    cell16_metadata[
        "imputation_performed"
    ]
):

    raise RuntimeError(
        "Cell 16 reports that imputation was already performed."
    )


if bool(
    cell16_metadata[
        "resampling_performed"
    ]
):

    raise RuntimeError(
        "Cell 16 reports that resampling was already performed."
    )


print(
    "\n✓ Cell 16 metadata contract validated."
)


# ==============================================================================================================
# 5. HARD SHAPE CONTRACT
# ==============================================================================================================

expected_shapes = {

    "TRAIN": (

        int(
            cell16_metadata[
                "train_samples"
            ]
        ),

        HISTORY_LENGTH,

        N_MODEL_FEATURES,
    ),

    "VALIDATION": (

        int(
            cell16_metadata[
                "validation_samples"
            ]
        ),

        HISTORY_LENGTH,

        N_MODEL_FEATURES,
    ),

    "TEST": (

        int(
            cell16_metadata[
                "test_samples"
            ]
        ),

        HISTORY_LENGTH,

        N_MODEL_FEATURES,
    ),
}


X_sources = {

    "TRAIN":
        X_train_source,

    "VALIDATION":
        X_validation_source,

    "TEST":
        X_test_source,
}


y_sources = {

    "TRAIN":
        y_train_source,

    "VALIDATION":
        y_validation_source,

    "TEST":
        y_test_source,
}


for partition in [

    "TRAIN",
    "VALIDATION",
    "TEST",

]:

    X = X_sources[
        partition
    ]

    y = y_sources[
        partition
    ]


    if X.ndim != 3:

        raise RuntimeError(

            f"{partition} X must be 3-D.\n"

            f"Observed shape: {X.shape}"
        )


    if X.shape != expected_shapes[
        partition
    ]:

        raise RuntimeError(

            f"{partition} X shape mismatch.\n"

            f"Expected: "
            f"{expected_shapes[partition]}\n"

            f"Observed: "
            f"{X.shape}"
        )


    expected_y_shape = (

        expected_shapes[
            partition
        ][0],

        N_HORIZONS,
    )


    if y.shape != expected_y_shape:

        raise RuntimeError(

            f"{partition} y shape mismatch.\n"

            f"Expected: "
            f"{expected_y_shape}\n"

            f"Observed: "
            f"{y.shape}"
        )


print("\n" + "=" * 150)
print("RAW TENSOR SHAPE CONTRACT")
print("=" * 150)

print(
    f"TRAIN      X={X_train_source.shape} "
    f"y={y_train_source.shape}"
)

print(
    f"VALIDATION X={X_validation_source.shape} "
    f"y={y_validation_source.shape}"
)

print(
    f"TEST       X={X_test_source.shape} "
    f"y={y_test_source.shape}"
)

print(
    "\n✓ Cell 16 raw tensor shapes validated."
)


# ==============================================================================================================
# 6. METADATA ALIGNMENT
# ==============================================================================================================

metadata_objects = {

    "TRAIN":
        train_metadata,

    "VALIDATION":
        validation_metadata,

    "TEST":
        test_metadata,
}


for partition in [

    "TRAIN",
    "VALIDATION",
    "TEST",

]:

    X = X_sources[
        partition
    ]

    metadata = metadata_objects[
        partition
    ]


    if len(metadata) != len(X):

        raise RuntimeError(

            f"{partition} metadata alignment failure.\n"

            f"Samples: {len(X):,}\n"

            f"Metadata rows: {len(metadata):,}"
        )


print(
    "✓ Cell 16 metadata remains aligned "
    "with every sequence."
)


# ==============================================================================================================
# 7. TARGET VALIDATION
# ==============================================================================================================

for partition in [

    "TRAIN",
    "VALIDATION",
    "TEST",

]:

    y = y_sources[
        partition
    ]


    if not np.isfinite(
        y
    ).all():

        raise RuntimeError(

            f"{partition} targets contain "
            "NaN or infinity."
        )


    if not np.isin(

        y,

        [
            0.0,
            1.0,
        ],

    ).all():

        raise RuntimeError(

            f"{partition} contains "
            "non-binary target values."
        )


print(
    "✓ All five future targets are valid binary labels."
)


# ==============================================================================================================
# 8. PRESERVE ORIGINAL TARGETS FOR IMMUTABILITY AUDIT
# ==============================================================================================================

y_train_original = (
    y_train_source
    .copy()
)


y_validation_original = (
    y_validation_source
    .copy()
)


y_test_original = (
    y_test_source
    .copy()
)


# ==============================================================================================================
# 9. RAW NON-FINITE AUDIT
# ==============================================================================================================

def raw_nonfinite_summary(
    X,
    partition,
):

    X64 = np.asarray(
        X,
        dtype=np.float64,
    )


    return {

        "Partition":
            partition,

        "Total_Values":
            int(
                X64.size
            ),

        "NaN":
            int(
                np.isnan(
                    X64
                ).sum()
            ),

        "Positive_Inf":
            int(
                np.isposinf(
                    X64
                ).sum()
            ),

        "Negative_Inf":
            int(
                np.isneginf(
                    X64
                ).sum()
            ),

        "Total_Nonfinite":
            int(
                (
                    ~np.isfinite(
                        X64
                    )
                ).sum()
            ),
    }


cell17_raw_nonfinite_audit = pd.DataFrame(

    [

        raw_nonfinite_summary(
            X_train_source,
            "TRAIN",
        ),

        raw_nonfinite_summary(
            X_validation_source,
            "VALIDATION",
        ),

        raw_nonfinite_summary(
            X_test_source,
            "TEST",
        ),
    ]
)


print("\n" + "=" * 150)
print("RAW NON-FINITE AUDIT")
print("=" * 150)

print(

    cell17_raw_nonfinite_audit
    .to_string(
        index=False
    )
)


# ==============================================================================================================
# 10. PREPARE WORKING ARRAYS
# ==============================================================================================================

X_train_work = np.asarray(

    X_train_source,

    dtype=np.float64,

).copy()


X_validation_work = np.asarray(

    X_validation_source,

    dtype=np.float64,

).copy()


X_test_work = np.asarray(

    X_test_source,

    dtype=np.float64,

).copy()


# Infinity is not a learned statistic.
# Convert +/-inf to NaN so that the optional imputer can handle it.

X_train_work[
    ~np.isfinite(
        X_train_work
    )
] = np.nan


X_validation_work[
    ~np.isfinite(
        X_validation_work
    )
] = np.nan


X_test_work[
    ~np.isfinite(
        X_test_work
    )
] = np.nan


# ==============================================================================================================
# 11. TEMPORARILY FLATTEN TIME DIMENSION
# ==============================================================================================================
#
# (samples, 15, features)
#
# becomes:
#
# (samples × 15, features)
#
# This lets preprocessing learn one statistic per network feature.
#
# Temporal structure is restored afterward.
#
# ==============================================================================================================

train_flat = (
    X_train_work
    .reshape(
        -1,
        N_MODEL_FEATURES,
    )
)


validation_flat = (
    X_validation_work
    .reshape(
        -1,
        N_MODEL_FEATURES,
    )
)


test_flat = (
    X_test_work
    .reshape(
        -1,
        N_MODEL_FEATURES,
    )
)


print("\n" + "=" * 150)
print("TEMPORARY FEATURE-WISE REPRESENTATION")
print("=" * 150)

print(
    f"TRAIN      : "
    f"{train_flat.shape}"
)

print(
    f"VALIDATION : "
    f"{validation_flat.shape}"
)

print(
    f"TEST       : "
    f"{test_flat.shape}"
)


# ==============================================================================================================
# 12. DETERMINE WHETHER IMPUTATION IS ACTUALLY REQUIRED
# ==============================================================================================================

train_missing = int(
    np.isnan(
        train_flat
    ).sum()
)


validation_missing = int(
    np.isnan(
        validation_flat
    ).sum()
)


test_missing = int(
    np.isnan(
        test_flat
    ).sum()
)


IMPUTATION_REQUIRED = bool(

    train_missing
    +
    validation_missing
    +
    test_missing
)


print("\n" + "=" * 150)
print("IMPUTATION REQUIREMENT AUDIT")
print("=" * 150)

print(
    f"TRAIN missing      : "
    f"{train_missing:,}"
)

print(
    f"VALIDATION missing : "
    f"{validation_missing:,}"
)

print(
    f"TEST missing       : "
    f"{test_missing:,}"
)

print(
    f"\nImputation required: "
    f"{IMPUTATION_REQUIRED}"
)


# ==============================================================================================================
# 13. OPTIONAL TRAIN-ONLY IMPUTATION
# ==============================================================================================================

feature_imputer = None


if IMPUTATION_REQUIRED:

    # ------------------------------------------------------------------------------------------
    # LEAKAGE RULE:
    #
    # fit() ONLY on TRAIN.
    #
    # VALIDATION and TEST use transform() only.
    # ------------------------------------------------------------------------------------------

    feature_imputer = SimpleImputer(

        strategy=
            IMPUTATION_STRATEGY
    )


    train_flat_prepared = (
        feature_imputer
        .fit_transform(
            train_flat
        )
    )


    validation_flat_prepared = (
        feature_imputer
        .transform(
            validation_flat
        )
    )


    test_flat_prepared = (
        feature_imputer
        .transform(
            test_flat
        )
    )


    print(
        "\n✓ Imputer fitted using TRAIN only."
    )


else:

    train_flat_prepared = (
        train_flat
        .copy()
    )


    validation_flat_prepared = (
        validation_flat
        .copy()
    )


    test_flat_prepared = (
        test_flat
        .copy()
    )


    print(
        "\n✓ Imputation not required."
    )


# ==============================================================================================================
# 14. POST-IMPUTATION FINITE CHECK
# ==============================================================================================================

for partition, array in [

    (
        "TRAIN",
        train_flat_prepared,
    ),

    (
        "VALIDATION",
        validation_flat_prepared,
    ),

    (
        "TEST",
        test_flat_prepared,
    ),

]:

    bad_values = int(

        (
            ~np.isfinite(
                array
            )
        ).sum()
    )


    if bad_values:

        raise RuntimeError(

            f"{partition} contains "
            f"{bad_values:,} non-finite values "
            "after optional imputation."
        )


print(
    "✓ All feature values finite before scaling."
)


# ==============================================================================================================
# 15. FIT STANDARD SCALER — TRAIN ONLY
# ==============================================================================================================

feature_scaler = StandardScaler()


train_flat_scaled = (
    feature_scaler
    .fit_transform(
        train_flat_prepared
    )
)


validation_flat_scaled = (
    feature_scaler
    .transform(
        validation_flat_prepared
    )
)


test_flat_scaled = (
    feature_scaler
    .transform(
        test_flat_prepared
    )
)


print("\n" + "=" * 150)
print("TRAIN-ONLY STANDARDIZATION")
print("=" * 150)

print(
    "✓ StandardScaler fitted on TRAIN only."
)

print(
    "✓ VALIDATION transformed with TRAIN scaler."
)

print(
    "✓ TEST transformed with TRAIN scaler."
)

print(
    "✓ No VALIDATION/TEST statistics were used "
    "during fitting."
)


# ==============================================================================================================
# 16. RESTORE 3-D TEMPORAL STRUCTURE
# ==============================================================================================================

X_train = (

    train_flat_scaled
    .reshape(
        X_train_source.shape
    )
    .astype(
        np.float32
    )
)


X_validation = (

    validation_flat_scaled
    .reshape(
        X_validation_source.shape
    )
    .astype(
        np.float32
    )
)


X_test = (

    test_flat_scaled
    .reshape(
        X_test_source.shape
    )
    .astype(
        np.float32
    )
)


# Convenient alias.

X_val = X_validation


# Targets remain unchanged.

y_train = (
    y_train_source
    .astype(
        np.float32,
        copy=True,
    )
)


y_validation = (
    y_validation_source
    .astype(
        np.float32,
        copy=True,
    )
)


y_test = (
    y_test_source
    .astype(
        np.float32,
        copy=True,
    )
)


y_val = y_validation


# ==============================================================================================================
# 17. FINAL MODEL-READY SHAPE CHECK
# ==============================================================================================================

final_arrays = {

    "TRAIN":
        (
            X_train,
            y_train,
        ),

    "VALIDATION":
        (
            X_validation,
            y_validation,
        ),

    "TEST":
        (
            X_test,
            y_test,
        ),
}


for partition, (
    X,
    y,
) in final_arrays.items():

    if X.shape != expected_shapes[
        partition
    ]:

        raise RuntimeError(

            f"{partition} final X shape changed unexpectedly.\n"

            f"Expected: "
            f"{expected_shapes[partition]}\n"

            f"Observed: "
            f"{X.shape}"
        )


    if y.shape != (

        X.shape[0],
        N_HORIZONS,

    ):

        raise RuntimeError(

            f"{partition} final y shape "
            "is inconsistent."
        )


print("\n" + "=" * 150)
print("FINAL MODEL-READY TENSORS")
print("=" * 150)

print(
    f"X_train      : {X_train.shape}"
)

print(
    f"y_train      : {y_train.shape}"
)

print()

print(
    f"X_validation : {X_validation.shape}"
)

print(
    f"y_validation : {y_validation.shape}"
)

print()

print(
    f"X_test       : {X_test.shape}"
)

print(
    f"y_test       : {y_test.shape}"
)


# ==============================================================================================================
# 18. FINAL NON-FINITE CHECK
# ==============================================================================================================

final_nonfinite_rows = []


for partition, X in [

    (
        "TRAIN",
        X_train,
    ),

    (
        "VALIDATION",
        X_validation,
    ),

    (
        "TEST",
        X_test,
    ),

]:

    bad_values = int(

        (
            ~np.isfinite(
                X
            )
        ).sum()
    )


    final_nonfinite_rows.append(

        {

            "Partition":
                partition,

            "Nonfinite_After_Preprocessing":
                bad_values,
        }
    )


cell17_postprocess_audit = pd.DataFrame(
    final_nonfinite_rows
)


print("\n" + "=" * 150)
print("FINAL NON-FINITE AUDIT")
print("=" * 150)

print(

    cell17_postprocess_audit
    .to_string(
        index=False
    )
)


if int(

    cell17_postprocess_audit[
        "Nonfinite_After_Preprocessing"
    ].sum()

) != 0:

    raise RuntimeError(
        "Final model-ready tensors contain non-finite values."
    )


print(
    "\n✓ All model-ready feature tensors are finite."
)


# ==============================================================================================================
# 19. TARGET IMMUTABILITY AUDIT
# ==============================================================================================================

target_checks = {

    "TRAIN":

        np.array_equal(
            y_train,
            y_train_original,
        ),

    "VALIDATION":

        np.array_equal(
            y_validation,
            y_validation_original,
        ),

    "TEST":

        np.array_equal(
            y_test,
            y_test_original,
        ),
}


print("\n" + "=" * 150)
print("TARGET IMMUTABILITY AUDIT")
print("=" * 150)


for partition, unchanged in (
    target_checks.items()
):

    print(

        f"{partition:<12}: "

        f"{'UNCHANGED' if unchanged else 'CHANGED'}"
    )


if not all(
    target_checks.values()
):

    raise RuntimeError(
        "Targets changed during preprocessing."
    )


print(
    "\n✓ Forecast targets are exactly unchanged."
)


# ==============================================================================================================
# 20. TRAIN-SCALER SANITY CHECK
# ==============================================================================================================

train_scaled_flat = (

    X_train
    .reshape(
        -1,
        N_MODEL_FEATURES,
    )
)


train_scaled_means = (
    train_scaled_flat
    .mean(
        axis=0
    )
)


train_scaled_stds = (
    train_scaled_flat
    .std(
        axis=0
    )
)


max_abs_train_mean = float(

    np.max(

        np.abs(
            train_scaled_means
        )
    )
)


min_train_std = float(

    np.min(
        train_scaled_stds
    )
)


max_train_std = float(

    np.max(
        train_scaled_stds
    )
)


print("\n" + "=" * 150)
print("TRAIN SCALER SANITY CHECK")
print("=" * 150)

print(

    f"Maximum |TRAIN feature mean| : "
    f"{max_abs_train_mean:.8f}"
)

print(

    f"Minimum TRAIN feature std    : "
    f"{min_train_std:.8f}"
)

print(

    f"Maximum TRAIN feature std    : "
    f"{max_train_std:.8f}"
)


# ==============================================================================================================
# 21. TARGET DISTRIBUTION — DESCRIPTIVE ONLY
# ==============================================================================================================

distribution_rows = []


for partition, y in [

    (
        "TRAIN",
        y_train,
    ),

    (
        "VALIDATION",
        y_validation,
    ),

    (
        "TEST",
        y_test,
    ),

]:

    for horizon_index, horizon in enumerate(
        FORECAST_HORIZONS
    ):

        horizon_targets = (
            y[
                :,
                horizon_index
            ]
        )


        total = int(
            len(
                horizon_targets
            )
        )


        attack = int(

            (
                horizon_targets
                == 1
            ).sum()
        )


        benign = (
            total
            -
            attack
        )


        attack_rate = (

            attack
            /
            total

            if total

            else np.nan
        )


        distribution_rows.append(

            {

                "Partition":
                    partition,

                "Horizon_Minutes":
                    horizon,

                "Samples":
                    total,

                "Benign":
                    benign,

                "Attack":
                    attack,

                "Attack_Rate":
                    attack_rate,
            }
        )


cell17_target_distribution = pd.DataFrame(
    distribution_rows
)


print("\n" + "=" * 150)
print("TARGET DISTRIBUTION — DESCRIPTIVE ONLY")
print("=" * 150)

print(

    cell17_target_distribution
    .to_string(
        index=False
    )
)


# ==============================================================================================================
# 22. FEATURE SCALER PARAMETER AUDIT
# ==============================================================================================================

cell17_scaler_parameters = pd.DataFrame(

    {

        "Feature":
            MODEL_FEATURES,

        "Train_Mean":
            feature_scaler.mean_,

        "Train_Scale":
            feature_scaler.scale_,

        "Train_Variance":
            feature_scaler.var_,
    }
)


if len(
    cell17_scaler_parameters
) != N_MODEL_FEATURES:

    raise RuntimeError(
        "Scaler parameter count does not match predictor count."
    )


print("\n" + "=" * 150)
print("SCALER PARAMETER CONTRACT")
print("=" * 150)

print(
    f"Scaler parameters stored for : "
    f"{len(cell17_scaler_parameters)} features"
)

print(
    "✓ Feature order exactly matches "
    "cell16_predictor_columns."
)


# ==============================================================================================================
# 23. CREATE PERMANENT CELL 17 CONTRACT
# ==============================================================================================================

cell17_metadata = {

    "cell_version":
        "CELL17_FINAL_TRAIN_ONLY_PREPROCESSING_V1",

    "source_cell16_version":
        cell16_metadata.get(
            "cell_version"
        ),

    "history_length_minutes":
        HISTORY_LENGTH,

    "forecast_horizons":
        FORECAST_HORIZONS.copy(),

    "targets_per_sample":
        N_HORIZONS,

    "predictor_count":
        N_MODEL_FEATURES,

    "predictor_columns":
        MODEL_FEATURES.copy(),

    "train_samples":
        int(
            X_train.shape[0]
        ),

    "validation_samples":
        int(
            X_validation.shape[0]
        ),

    "test_samples":
        int(
            X_test.shape[0]
        ),

    "imputation_required":
        IMPUTATION_REQUIRED,

    "imputation_strategy":
        (
            IMPUTATION_STRATEGY
            if IMPUTATION_REQUIRED
            else None
        ),

    "imputer_fit_source":
        (
            "TRAIN_ONLY"
            if IMPUTATION_REQUIRED
            else None
        ),

    "scaler":
        SCALER_NAME,

    "scaler_fit_source":
        "TRAIN_ONLY",

    "validation_used_for_preprocessing_fit":
        False,

    "test_used_for_preprocessing_fit":
        False,

    "temporal_structure_preserved":
        True,

    "targets_modified":
        False,

    "feature_selection_performed":
        False,

    "resampling_performed":
        False,

    "threshold_selection_performed":
        False,

    "model_trained":
        False,
}


# ==============================================================================================================
# 24. FINAL PERMANENT OBJECT ALIASES
# ==============================================================================================================

# These are intentionally explicit so later cells have ONE canonical contract.

MODEL_FEATURE_COLUMNS = (
    MODEL_FEATURES.copy()
)


cell17_train_metadata = (
    train_metadata.copy(
        deep=True
    )
)


cell17_validation_metadata = (
    validation_metadata.copy(
        deep=True
    )
)


cell17_test_metadata = (
    test_metadata.copy(
        deep=True
    )
)


# ==============================================================================================================
# 25. FINAL SUMMARY
# ==============================================================================================================

print("\n" + "=" * 150)
print("CELL 17 FINAL CONTRACT")
print("=" * 150)

print(
    f"Predictors                  : "
    f"{N_MODEL_FEATURES}"
)

print(
    f"History length              : "
    f"{HISTORY_LENGTH}"
)

print(
    f"Forecast horizons           : "
    f"{FORECAST_HORIZONS}"
)

print()

print(
    f"TRAIN samples               : "
    f"{X_train.shape[0]:,}"
)

print(
    f"VALIDATION samples          : "
    f"{X_validation.shape[0]:,}"
)

print(
    f"TEST samples                : "
    f"{X_test.shape[0]:,}"
)

print()

print(
    f"TRAIN tensor                : "
    f"{X_train.shape}"
)

print(
    f"VALIDATION tensor           : "
    f"{X_validation.shape}"
)

print(
    f"TEST tensor                 : "
    f"{X_test.shape}"
)

print()

print(
    f"Imputation required         : "
    f"{IMPUTATION_REQUIRED}"
)

print(
    "Scaler fit source           : TRAIN ONLY"
)

print(
    "Validation fit contribution : NONE"
)

print(
    "Test fit contribution       : NONE"
)

print(
    "Resampling                  : NONE"
)

print(
    "Threshold selection         : NONE"
)

print(
    "Model training              : NONE"
)


print("\nPermanent Cell 17 objects:")

print(
    "  X_train"
)

print(
    "  X_validation"
)

print(
    "  X_val"
)

print(
    "  X_test"
)

print()

print(
    "  y_train"
)

print(
    "  y_validation"
)

print(
    "  y_val"
)

print(
    "  y_test"
)

print()

print(
    "  MODEL_FEATURES"
)

print(
    "  MODEL_FEATURE_COLUMNS"
)

print(
    "  feature_imputer"
)

print(
    "  feature_scaler"
)

print()

print(
    "  cell17_train_metadata"
)

print(
    "  cell17_validation_metadata"
)

print(
    "  cell17_test_metadata"
)

print()

print(
    "  cell17_raw_nonfinite_audit"
)

print(
    "  cell17_postprocess_audit"
)

print(
    "  cell17_target_distribution"
)

print(
    "  cell17_scaler_parameters"
)

print(
    "  cell17_metadata"
)


print("\n" + "=" * 150)
print("CELL 17 COMPLETE")
print("=" * 150)

print(
    "\n✓ Exact final Cell 16 feature contract used."
)

print(
    "✓ 80 final predictors preserved."
)

print(
    "✓ 15-minute temporal sequences preserved."
)

print(
    "✓ Five-horizon targets preserved."
)

print(
    "✓ TRAIN-only preprocessing enforced."
)

print(
    "✓ VALIDATION and TEST never used to fit preprocessing."
)

print(
    "✓ Raw Cell 16 tensors were not modified."
)

print(
    "✓ No resampling, threshold selection, or model training occurred."
)

print(
    "\nNEXT: Cell 18 — final pre-model audit, "
    "baselines, imbalance strategy, and evaluation contract."
)

CELL 17 — FINAL TRAIN-ONLY PREPROCESSING + SCALING + MODEL-READY TENSORS

Configuration
--------------------------------------------------------------------------------------------------------------
History length              : 15 minutes
Forecast horizons           : [1, 5, 10, 15, 30]
Targets per sample          : 5
Scaler                      : StandardScaler
Scaler fit source           : TRAIN ONLY
Imputation                  : ONLY IF REQUIRED
Imputation strategy         : median
Validation used for fitting : NO
Test used for fitting       : NO
Resampling performed        : NO
Threshold selection         : NO
Model trained               : NO

EXACT CELL 16 OBJECT VALIDATION
✓ All required Cell 16 objects found.

Final Cell 16 predictors : 80
X_train_raw              : (1220, 15, 80)
X_validation_raw         : (398, 15, 80)
X_test_raw               : (396, 15, 80)
y_train                  : (1220, 5)
y_validation             : (398, 5)
y_test                   : (396, 5)

✓ Cell 1

In [39]:
# ==============================================================================================================
# CELL 18 — FINAL PRE-MODEL AUDIT + BASELINES + LOSS/EVALUATION CONTRACT
# ==============================================================================================================
#
# PURPOSE
# -------
# Freeze the FINAL model-development and evaluation contract BEFORE any learned model is trained.
#
# UPSTREAM CONTRACT
# -----------------
# Cell 15 : frozen leakage-safe temporal split
# Cell 16 : frozen 15-minute raw sequence construction
# Cell 17 : TRAIN-only preprocessing/scaling + model-ready tensors
#
# PRIMARY MODELING TASK
# ---------------------
# Input:
#       15 consecutive observed minutes
#
# Output:
#       five binary point forecasts:
#
#           P(attack at t+1)
#           P(attack at t+5)
#           P(attack at t+10)
#           P(attack at t+15)
#           P(attack at t+30)
#
# IMPORTANT
# ---------
# This cell DOES NOT:
#
#   ✗ modify the frozen split
#   ✗ modify features
#   ✗ remove features
#   ✗ refit preprocessing
#   ✗ resample data
#   ✗ perform SMOTE
#   ✗ train a learned model
#   ✗ inspect TEST predictions
#   ✗ select a learned-model threshold
#
# This cell DOES:
#
#   ✓ validate the final Cell 17 tensors
#   ✓ audit frozen feature behavior
#   ✓ quantify class imbalance
#   ✓ derive TRAIN-only candidate BCE pos_weight values
#   ✓ evaluate non-learned reference baselines
#   ✓ freeze the primary/secondary metric contract
#   ✓ freeze validation-only threshold-selection rules
#   ✓ freeze untouched-test rules
#   ✓ freeze onset/continuation and lead-time evaluation principles
#
# ==============================================================================================================


import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)


# ==============================================================================================================
# 1. CONFIGURATION
# ==============================================================================================================

HISTORY_LENGTH = 15

FORECAST_HORIZONS = [
    1,
    5,
    10,
    15,
    30,
]

N_HORIZONS = len(
    FORECAST_HORIZONS
)


# Threshold GRID for FUTURE learned-model validation.
#
# IMPORTANT:
# No learned-model threshold is selected in Cell 18.
# We only freeze the candidate search space.

THRESHOLD_GRID = np.round(
    np.arange(
        0.05,
        0.951,
        0.01,
    ),
    2,
)


DEFAULT_THRESHOLD = 0.50


print("=" * 160)
print(
    "CELL 18 — FINAL PRE-MODEL AUDIT + "
    "BASELINES + LOSS/EVALUATION CONTRACT"
)
print("=" * 160)

print("\nConfiguration")
print("-" * 120)

print(
    f"History length                    : "
    f"{HISTORY_LENGTH} minutes"
)

print(
    f"Forecast horizons                 : "
    f"{FORECAST_HORIZONS}"
)

print(
    f"Targets per sample                : "
    f"{N_HORIZONS}"
)

print(
    f"Default probability threshold     : "
    f"{DEFAULT_THRESHOLD:.2f}"
)

print(
    f"Future threshold search grid      : "
    f"{THRESHOLD_GRID[0]:.2f} → "
    f"{THRESHOLD_GRID[-1]:.2f}"
)

print(
    "Threshold selection source        : VALIDATION ONLY"
)

print(
    "TEST used for model selection     : NO"
)

print(
    "Resampling                        : NO"
)

print(
    "Model trained                     : NO"
)


# ==============================================================================================================
# 2. REQUIRED CELL 17 OBJECTS
# ==============================================================================================================

REQUIRED_CELL17_OBJECTS = [

    "X_train",
    "X_validation",
    "X_test",

    "y_train",
    "y_validation",
    "y_test",

    "MODEL_FEATURES",

    "feature_scaler",

    "cell17_train_metadata",
    "cell17_validation_metadata",
    "cell17_test_metadata",

    "cell17_metadata",
]


missing_objects = [

    name

    for name in REQUIRED_CELL17_OBJECTS

    if name not in globals()
]


if missing_objects:

    raise RuntimeError(

        "Cell 18 is missing required FINAL Cell 17 objects:\n"

        + "\n".join(
            f"  - {name}"
            for name in missing_objects
        )

        + "\n\nRun the FINAL Cell 17 first."
    )


print("\n" + "=" * 160)
print("UPSTREAM CELL 17 CONTRACT VALIDATION")
print("=" * 160)

print(
    "✓ Required Cell 17 objects found."
)


# ==============================================================================================================
# 3. DEFENSIVE REFERENCES / COPIES
# ==============================================================================================================

X_train_18 = np.asarray(
    X_train
)

X_validation_18 = np.asarray(
    X_validation
)

X_test_18 = np.asarray(
    X_test
)


y_train_18 = np.asarray(
    y_train
)

y_validation_18 = np.asarray(
    y_validation
)

y_test_18 = np.asarray(
    y_test
)


MODEL_FEATURES_18 = list(
    MODEL_FEATURES
)


N_FEATURES = len(
    MODEL_FEATURES_18
)


train_metadata_18 = (
    cell17_train_metadata
    .copy(deep=True)
)

validation_metadata_18 = (
    cell17_validation_metadata
    .copy(deep=True)
)

test_metadata_18 = (
    cell17_test_metadata
    .copy(deep=True)
)


# ==============================================================================================================
# 4. HARD CONTRACT CHECK
# ==============================================================================================================

expected_feature_count = int(
    cell17_metadata[
        "predictor_count"
    ]
)


if N_FEATURES != expected_feature_count:

    raise RuntimeError(
        "Cell 17 predictor-count contract is inconsistent."
    )


expected_shapes = {

    "TRAIN": (
        int(
            cell17_metadata[
                "train_samples"
            ]
        ),
        HISTORY_LENGTH,
        N_FEATURES,
    ),

    "VALIDATION": (
        int(
            cell17_metadata[
                "validation_samples"
            ]
        ),
        HISTORY_LENGTH,
        N_FEATURES,
    ),

    "TEST": (
        int(
            cell17_metadata[
                "test_samples"
            ]
        ),
        HISTORY_LENGTH,
        N_FEATURES,
    ),
}


for partition, X, y, metadata in [

    (
        "TRAIN",
        X_train_18,
        y_train_18,
        train_metadata_18,
    ),

    (
        "VALIDATION",
        X_validation_18,
        y_validation_18,
        validation_metadata_18,
    ),

    (
        "TEST",
        X_test_18,
        y_test_18,
        test_metadata_18,
    ),

]:

    if X.shape != expected_shapes[
        partition
    ]:

        raise RuntimeError(

            f"{partition} feature shape changed.\n"

            f"Expected: "
            f"{expected_shapes[partition]}\n"

            f"Observed: "
            f"{X.shape}"
        )


    expected_y_shape = (
        X.shape[0],
        N_HORIZONS,
    )


    if y.shape != expected_y_shape:

        raise RuntimeError(

            f"{partition} target shape mismatch.\n"

            f"Expected: "
            f"{expected_y_shape}\n"

            f"Observed: "
            f"{y.shape}"
        )


    if len(
        metadata
    ) != len(
        X
    ):

        raise RuntimeError(

            f"{partition} metadata alignment failure."
        )


    if not np.isfinite(
        X
    ).all():

        raise RuntimeError(

            f"{partition} feature tensor contains "
            "non-finite values."
        )


    if not np.isfinite(
        y
    ).all():

        raise RuntimeError(

            f"{partition} target tensor contains "
            "non-finite values."
        )


    if not np.isin(
        y,
        [0, 1],
    ).all():

        raise RuntimeError(

            f"{partition} target tensor is not binary."
        )


print(
    f"Predictors                         : "
    f"{N_FEATURES}"
)

print(
    f"TRAIN tensor                       : "
    f"{X_train_18.shape}"
)

print(
    f"VALIDATION tensor                  : "
    f"{X_validation_18.shape}"
)

print(
    f"TEST tensor                        : "
    f"{X_test_18.shape}"
)

print(
    "\n✓ Frozen Cell 17 tensors are intact."
)


# ==============================================================================================================
# 5. TRAIN FEATURE-VARIANCE AUDIT
# ==============================================================================================================
#
# We inspect zero-variance features AFTER TRAIN-only scaling.
#
# IMPORTANT:
# We DO NOT remove them here.
#
# The 80-feature representation is already frozen.
# Constant features are simply documented.
#
# ==============================================================================================================

train_flat_18 = (
    X_train_18
    .reshape(
        -1,
        N_FEATURES,
    )
)


train_feature_variance = np.var(
    train_flat_18,
    axis=0,
)


ZERO_VARIANCE_TOLERANCE = 1e-12


zero_variance_mask = (
    train_feature_variance
    <= ZERO_VARIANCE_TOLERANCE
)


ZERO_VARIANCE_FEATURES = [

    feature

    for feature, is_zero in zip(
        MODEL_FEATURES_18,
        zero_variance_mask,
    )

    if is_zero
]


cell18_feature_variance_audit = pd.DataFrame({

    "Feature":
        MODEL_FEATURES_18,

    "Train_Variance_After_Scaling":
        train_feature_variance,

    "Zero_Variance":
        zero_variance_mask,
})


print("\n" + "=" * 160)
print("TRAIN FEATURE-VARIANCE AUDIT")
print("=" * 160)

print(
    f"Total predictors                   : "
    f"{N_FEATURES}"
)

print(
    f"Zero-variance TRAIN predictors     : "
    f"{len(ZERO_VARIANCE_FEATURES)}"
)


if ZERO_VARIANCE_FEATURES:

    print(
        "\nZero-variance features:"
    )

    for feature in ZERO_VARIANCE_FEATURES:

        print(
            f"  - {feature}"
        )

else:

    print(
        "\n✓ No zero-variance TRAIN features."
    )


print(
    "\nNOTE: No feature is removed in Cell 18."
)


# ==============================================================================================================
# 6. CLASS DISTRIBUTION + TRAIN-ONLY POS_WEIGHT
# ==============================================================================================================
#
# Candidate BCEWithLogitsLoss positive-class weighting:
#
#       pos_weight_H = N_negative_H / N_positive_H
#
# calculated from TRAIN ONLY.
#
# VALIDATION and TEST are reported descriptively,
# but NEVER contribute to loss-weight estimation.
#
# ==============================================================================================================

imbalance_rows = []


for partition, y in [

    (
        "TRAIN",
        y_train_18,
    ),

    (
        "VALIDATION",
        y_validation_18,
    ),

    (
        "TEST",
        y_test_18,
    ),

]:

    for h_idx, horizon in enumerate(
        FORECAST_HORIZONS
    ):

        target = (
            y[
                :,
                h_idx
            ]
            .astype(
                int
            )
        )


        positive = int(
            np.sum(
                target == 1
            )
        )


        negative = int(
            np.sum(
                target == 0
            )
        )


        total = int(
            len(
                target
            )
        )


        prevalence = (

            positive
            /
            total

            if total > 0

            else np.nan
        )


        pos_weight = (

            negative
            /
            positive

            if (
                partition == "TRAIN"
                and positive > 0
            )

            else np.nan
        )


        imbalance_rows.append({

            "Partition":
                partition,

            "Horizon_Minutes":
                horizon,

            "Samples":
                total,

            "Negative":
                negative,

            "Positive":
                positive,

            "Positive_Rate":
                prevalence,

            "TRAIN_Pos_Weight":
                pos_weight,
        })


cell18_class_balance = pd.DataFrame(
    imbalance_rows
)


train_balance = (
    cell18_class_balance[
        cell18_class_balance[
            "Partition"
        ] == "TRAIN"
    ]
    .copy()
)


if (
    train_balance[
        "Positive"
    ] <= 0
).any():

    raise RuntimeError(
        "At least one TRAIN horizon contains no positive attacks."
    )


if (
    train_balance[
        "Negative"
    ] <= 0
).any():

    raise RuntimeError(
        "At least one TRAIN horizon contains no benign examples."
    )


TRAIN_POS_WEIGHT = np.asarray(

    train_balance[
        "TRAIN_Pos_Weight"
    ].to_numpy(),

    dtype=np.float32,
)


print("\n" + "=" * 160)
print("CLASS DISTRIBUTION + TRAIN-ONLY LOSS-WEIGHT AUDIT")
print("=" * 160)

print(

    cell18_class_balance
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)


print(
    "\nCandidate TRAIN-only BCE pos_weight vector:"
)

print(
    dict(
        zip(
            FORECAST_HORIZONS,
            [
                float(x)
                for x in TRAIN_POS_WEIGHT
            ],
        )
    )
)


print(
    "\nIMPORTANT:"
)

print(
    "  These are candidate TRAIN-derived loss weights."
)

print(
    "  Validation/test prevalence was NOT used to calculate them."
)

print(
    "  No resampling is performed."
)


# ==============================================================================================================
# 7. PREVALENCE SHIFT AUDIT
# ==============================================================================================================

prevalence_pivot = (

    cell18_class_balance
    .pivot(
        index="Horizon_Minutes",
        columns="Partition",
        values="Positive_Rate",
    )
    .reset_index()
)


for column in [
    "TRAIN",
    "VALIDATION",
    "TEST",
]:

    if column not in prevalence_pivot.columns:

        prevalence_pivot[
            column
        ] = np.nan


prevalence_pivot[
    "Validation_minus_Train"
] = (

    prevalence_pivot[
        "VALIDATION"
    ]

    -

    prevalence_pivot[
        "TRAIN"
    ]
)


prevalence_pivot[
    "Test_minus_Train"
] = (

    prevalence_pivot[
        "TEST"
    ]

    -

    prevalence_pivot[
        "TRAIN"
    ]
)


cell18_prevalence_shift = (
    prevalence_pivot
    .copy()
)


print("\n" + "=" * 160)
print("OUT-OF-TIME PREVALENCE SHIFT")
print("=" * 160)

print(

    cell18_prevalence_shift
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)


print(
    "\nNOTE:"
)

print(
    "  Distribution shift is retained as part of the "
    "out-of-time evaluation."
)

print(
    "  Validation/test will NOT be resampled to imitate TRAIN."
)


# ==============================================================================================================
# 8. SAFE METRIC HELPERS
# ==============================================================================================================

def safe_roc_auc(
    y_true,
    y_score,
):

    y_true = np.asarray(
        y_true
    )

    y_score = np.asarray(
        y_score
    )


    if np.unique(
        y_true
    ).size < 2:

        return np.nan


    return float(
        roc_auc_score(
            y_true,
            y_score,
        )
    )


def safe_pr_auc(
    y_true,
    y_score,
):

    y_true = np.asarray(
        y_true
    )

    y_score = np.asarray(
        y_score
    )


    if np.unique(
        y_true
    ).size < 2:

        return np.nan


    return float(
        average_precision_score(
            y_true,
            y_score,
        )
    )


def binary_metric_record(
    y_true,
    probabilities,
    predictions,
):

    y_true = np.asarray(
        y_true
    ).astype(
        int
    )

    probabilities = np.asarray(
        probabilities
    ).astype(
        float
    )

    predictions = np.asarray(
        predictions
    ).astype(
        int
    )


    tn, fp, fn, tp = (
        confusion_matrix(
            y_true,
            predictions,
            labels=[
                0,
                1,
            ],
        )
        .ravel()
    )


    return {

        "Accuracy":
            float(
                accuracy_score(
                    y_true,
                    predictions,
                )
            ),

        "Balanced_Accuracy":
            float(
                balanced_accuracy_score(
                    y_true,
                    predictions,
                )
            ),

        "Precision":
            float(
                precision_score(
                    y_true,
                    predictions,
                    zero_division=0,
                )
            ),

        "Recall":
            float(
                recall_score(
                    y_true,
                    predictions,
                    zero_division=0,
                )
            ),

        "F1":
            float(
                f1_score(
                    y_true,
                    predictions,
                    zero_division=0,
                )
            ),

        "MCC":
            float(
                matthews_corrcoef(
                    y_true,
                    predictions,
                )
            ),

        "ROC_AUC":
            safe_roc_auc(
                y_true,
                probabilities,
            ),

        "PR_AUC":
            safe_pr_auc(
                y_true,
                probabilities,
            ),

        "TN":
            int(
                tn
            ),

        "FP":
            int(
                fp
            ),

        "FN":
            int(
                fn
            ),

        "TP":
            int(
                tp
            ),
    }


# ==============================================================================================================
# 9. NON-LEARNED REFERENCE BASELINES
# ==============================================================================================================
#
# Baselines are intentionally simple.
#
# BASELINE A — ALWAYS BENIGN
#       probability = 0
#
# BASELINE B — TRAIN PREVALENCE
#       probability for horizon H =
#       TRAIN positive prevalence at horizon H
#
# These baselines:
#
#   - train no model
#   - use no TEST information
#   - provide minimum reference performance
#
# ==============================================================================================================

TRAIN_PREVALENCE = (

    train_balance[
        "Positive_Rate"
    ]
    .to_numpy(
        dtype=float
    )
)


baseline_rows = []


for partition, y in [

    (
        "TRAIN",
        y_train_18,
    ),

    (
        "VALIDATION",
        y_validation_18,
    ),

    (
        "TEST",
        y_test_18,
    ),

]:

    for h_idx, horizon in enumerate(
        FORECAST_HORIZONS
    ):

        y_true = (
            y[
                :,
                h_idx
            ]
            .astype(
                int
            )
        )


        # --------------------------------------------------------------------------------------
        # Baseline A: Always benign
        # --------------------------------------------------------------------------------------

        benign_probability = np.zeros(
            len(
                y_true
            ),
            dtype=float,
        )


        benign_prediction = np.zeros(
            len(
                y_true
            ),
            dtype=int,
        )


        benign_metrics = binary_metric_record(

            y_true,
            benign_probability,
            benign_prediction,
        )


        baseline_rows.append({

            "Partition":
                partition,

            "Horizon_Minutes":
                horizon,

            "Baseline":
                "Always_Benign",

            **benign_metrics,
        })


        # --------------------------------------------------------------------------------------
        # Baseline B: TRAIN prevalence
        # --------------------------------------------------------------------------------------

        prevalence_probability = np.full(

            len(
                y_true
            ),

            TRAIN_PREVALENCE[
                h_idx
            ],

            dtype=float,
        )


        prevalence_prediction = (

            prevalence_probability
            >= DEFAULT_THRESHOLD
        ).astype(
            int
        )


        prevalence_metrics = binary_metric_record(

            y_true,
            prevalence_probability,
            prevalence_prediction,
        )


        baseline_rows.append({

            "Partition":
                partition,

            "Horizon_Minutes":
                horizon,

            "Baseline":
                "Train_Prevalence",

            **prevalence_metrics,
        })


cell18_reference_baselines = pd.DataFrame(
    baseline_rows
)


print("\n" + "=" * 160)
print("NON-LEARNED REFERENCE BASELINES")
print("=" * 160)

display_columns = [

    "Partition",
    "Horizon_Minutes",
    "Baseline",

    "Accuracy",
    "Balanced_Accuracy",

    "Precision",
    "Recall",
    "F1",

    "MCC",

    "ROC_AUC",
    "PR_AUC",
]


print(

    cell18_reference_baselines[
        display_columns
    ]
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)


# ==============================================================================================================
# 10. OPTIONAL PERSISTENCE BASELINE SUPPORT AUDIT
# ==============================================================================================================
#
# A persistence baseline would predict:
#
#       future attack state = attack state at origin t
#
# This is a useful temporal baseline ONLY if Cell 17 metadata contains
# a canonical current-state field.
#
# We do NOT invent that field if it is absent.
#
# ==============================================================================================================

CURRENT_STATE_CANDIDATES = [

    "Current_State",
    "Attack_State",
    "Origin_Attack_State",
]


current_state_column = next(

    (
        column

        for column in CURRENT_STATE_CANDIDATES

        if (
            column
            in train_metadata_18.columns

            and column
            in validation_metadata_18.columns

            and column
            in test_metadata_18.columns
        )
    ),

    None,
)


PERSISTENCE_BASELINE_AVAILABLE = (
    current_state_column
    is not None
)


print("\n" + "=" * 160)
print("TEMPORAL PERSISTENCE BASELINE SUPPORT")
print("=" * 160)

print(
    f"Current-state metadata column      : "
    f"{current_state_column}"
)

print(
    f"Persistence baseline available     : "
    f"{PERSISTENCE_BASELINE_AVAILABLE}"
)


if PERSISTENCE_BASELINE_AVAILABLE:

    persistence_rows = []


    for partition, y, metadata in [

        (
            "TRAIN",
            y_train_18,
            train_metadata_18,
        ),

        (
            "VALIDATION",
            y_validation_18,
            validation_metadata_18,
        ),

        (
            "TEST",
            y_test_18,
            test_metadata_18,
        ),

    ]:

        current_state = pd.to_numeric(

            metadata[
                current_state_column
            ],

            errors="coerce",

        ).to_numpy()


        if np.isnan(
            current_state
        ).any():

            raise RuntimeError(
                "Persistence-baseline current state contains NaN."
            )


        if not np.isin(
            current_state,
            [0, 1],
        ).all():

            raise RuntimeError(
                "Persistence-baseline current state is not binary."
            )


        current_state = (
            current_state
            .astype(
                int
            )
        )


        for h_idx, horizon in enumerate(
            FORECAST_HORIZONS
        ):

            y_true = (
                y[
                    :,
                    h_idx
                ]
                .astype(
                    int
                )
            )


            probability = (
                current_state
                .astype(
                    float
                )
            )


            prediction = (
                current_state
                .copy()
            )


            metrics = binary_metric_record(

                y_true,
                probability,
                prediction,
            )


            persistence_rows.append({

                "Partition":
                    partition,

                "Horizon_Minutes":
                    horizon,

                "Baseline":
                    "Current_State_Persistence",

                **metrics,
            })


    cell18_persistence_baseline = pd.DataFrame(
        persistence_rows
    )


    cell18_reference_baselines = pd.concat(

        [
            cell18_reference_baselines,
            cell18_persistence_baseline,
        ],

        ignore_index=True,
    )


    print(
        "\n✓ Current-state persistence baseline calculated."
    )


else:

    cell18_persistence_baseline = pd.DataFrame()


    print(
        "\nNOTE: Persistence baseline not calculated because "
        "Cell 17 metadata does not expose a canonical "
        "current-state field."
    )


# ==============================================================================================================
# 11. METRIC CONTRACT
# ==============================================================================================================
#
# WHY PR-AUC IS PRIMARY
# ---------------------
#
# Attack forecasting is imbalanced in TRAIN and prevalence shifts over time.
#
# Accuracy can look good merely by predicting benign.
#
# ROC-AUC remains useful but may appear optimistic when negatives dominate.
#
# Therefore:
#
# PRIMARY probability-ranking metric:
#       PR-AUC / Average Precision
#
# SECONDARY threshold-free metric:
#       ROC-AUC
#
# THRESHOLD-DEPENDENT metrics:
#       Precision
#       Recall
#       F1
#       MCC
#       Balanced Accuracy
#
# Operational forecasting metrics:
#       onset recall
#       false-alert behavior
#       lead time
#
# Those require model predictions and/or episode alignment,
# and are therefore evaluated later.
#
# ==============================================================================================================

PRIMARY_SELECTION_METRIC = (
    "Macro_Validation_PR_AUC"
)


PRIMARY_PER_HORIZON_METRIC = (
    "PR_AUC"
)


SECONDARY_PROBABILITY_METRIC = (
    "ROC_AUC"
)


THRESHOLD_METRICS = [

    "Precision",
    "Recall",
    "F1",
    "MCC",
    "Balanced_Accuracy",
]


REPORT_METRICS = [

    "PR_AUC",
    "ROC_AUC",

    "Precision",
    "Recall",
    "F1",

    "MCC",
    "Balanced_Accuracy",

    "Accuracy",

    "TN",
    "FP",
    "FN",
    "TP",
]


print("\n" + "=" * 160)
print("FINAL METRIC CONTRACT")
print("=" * 160)

print(
    f"Primary model-selection metric     : "
    f"{PRIMARY_SELECTION_METRIC}"
)

print(
    f"Primary per-horizon metric         : "
    f"{PRIMARY_PER_HORIZON_METRIC}"
)

print(
    f"Secondary probability metric       : "
    f"{SECONDARY_PROBABILITY_METRIC}"
)

print(
    "Threshold-dependent metrics       : "
    + ", ".join(
        THRESHOLD_METRICS
    )
)

print(
    "\nAccuracy will be reported, "
    "but it is NOT a primary selection metric."
)


# ==============================================================================================================
# 12. MODEL-SELECTION AGGREGATION CONTRACT
# ==============================================================================================================
#
# A multi-horizon model produces five outputs.
#
# We need ONE validation quantity for checkpoint/model comparison.
#
# We therefore define:
#
#       Macro Validation PR-AUC
#
# = arithmetic mean of PR-AUC over the five horizons.
#
# Every horizon receives equal weight.
#
# This prevents the shortest horizon or the largest class from
# silently dominating model selection.
#
# ==============================================================================================================

HORIZON_SELECTION_WEIGHTS = np.full(

    N_HORIZONS,

    1.0 / N_HORIZONS,

    dtype=np.float32,
)


cell18_horizon_selection_weights = pd.DataFrame({

    "Horizon_Minutes":
        FORECAST_HORIZONS,

    "Selection_Weight":
        HORIZON_SELECTION_WEIGHTS,
})


print("\n" + "=" * 160)
print("MULTI-HORIZON MODEL-SELECTION CONTRACT")
print("=" * 160)

print(
    cell18_horizon_selection_weights
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)

print(
    "\n✓ All five horizons receive equal weight "
    "during model selection."
)


# ==============================================================================================================
# 13. LOSS CONTRACT
# ==============================================================================================================
#
# For the initial neural model:
#
#       BCEWithLogitsLoss
#
# with one TRAIN-derived positive-class weight per horizon.
#
# We are NOT applying the loss here.
#
# Cell 19 will convert TRAIN_POS_WEIGHT to the framework tensor.
#
# IMPORTANT:
# pos_weight is NOT a decision threshold.
#
# ==============================================================================================================

LOSS_NAME = (
    "BCEWithLogitsLoss"
)


LOSS_WEIGHTING = (
    "TRAIN_ONLY_PER_HORIZON_POS_WEIGHT"
)


LOSS_REDUCTION = (
    "mean"
)


cell18_loss_contract = pd.DataFrame({

    "Horizon_Minutes":
        FORECAST_HORIZONS,

    "Train_Pos_Weight":
        TRAIN_POS_WEIGHT,

    "Loss":
        LOSS_NAME,
})


print("\n" + "=" * 160)
print("LOSS CONTRACT")
print("=" * 160)

print(
    cell18_loss_contract
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

print(
    "\n✓ Loss weighting uses TRAIN labels only."
)

print(
    "✓ No oversampling/undersampling is required."
)


# ==============================================================================================================
# 14. VALIDATION-ONLY THRESHOLD-SELECTION CONTRACT
# ==============================================================================================================
#
# Cell 18 does NOT select thresholds because no learned model exists yet.
#
# AFTER a model is trained:
#
#   1. produce VALIDATION probabilities,
#   2. search THRESHOLD_GRID independently for each horizon,
#   3. choose threshold maximizing validation F1,
#   4. deterministic tie-break:
#          higher recall,
#          then lower threshold,
#   5. freeze those five thresholds,
#   6. apply them unchanged to TEST exactly once.
#
# Why per-horizon thresholds?
#
# The five horizons have different prevalence/calibration behavior.
#
# ==============================================================================================================

THRESHOLD_SELECTION_METRIC = (
    "F1"
)


THRESHOLD_TIE_BREAK = [

    "Higher_Recall",
    "Lower_Threshold",
]


PER_HORIZON_THRESHOLDS = True


TEST_THRESHOLD_TUNING_ALLOWED = False


cell18_threshold_contract = {

    "default_threshold":
        DEFAULT_THRESHOLD,

    "candidate_threshold_grid":
        THRESHOLD_GRID.copy(),

    "selection_partition":
        "VALIDATION_ONLY",

    "selection_metric":
        THRESHOLD_SELECTION_METRIC,

    "tie_break":
        THRESHOLD_TIE_BREAK.copy(),

    "per_horizon_thresholds":
        PER_HORIZON_THRESHOLDS,

    "test_threshold_tuning_allowed":
        TEST_THRESHOLD_TUNING_ALLOWED,

    "thresholds_selected_in_cell18":
        False,
}


print("\n" + "=" * 160)
print("VALIDATION-ONLY THRESHOLD CONTRACT")
print("=" * 160)

print(
    f"Selection partition                : "
    f"{cell18_threshold_contract['selection_partition']}"
)

print(
    f"Optimization metric                : "
    f"{THRESHOLD_SELECTION_METRIC}"
)

print(
    "Per-horizon thresholds            : YES"
)

print(
    "TEST threshold tuning             : FORBIDDEN"
)

print(
    "Thresholds selected now           : NO"
)


# ==============================================================================================================
# 15. TEST-SET FIREWALL
# ==============================================================================================================
#
# TEST labels may be used here only for:
#
#   - static descriptive auditing
#   - non-learned baseline reporting
#
# Once model development begins:
#
# TEST MUST NOT influence:
#
#   - architecture
#   - hyperparameters
#   - epochs
#   - early stopping
#   - class weighting
#   - feature selection
#   - threshold selection
#   - model checkpoint choice
#
# ==============================================================================================================

TEST_FIREWALL_RULES = {

    "architecture_selection":
        False,

    "hyperparameter_selection":
        False,

    "epoch_selection":
        False,

    "early_stopping":
        False,

    "loss_weight_selection":
        False,

    "feature_selection":
        False,

    "threshold_selection":
        False,

    "checkpoint_selection":
        False,

    "final_evaluation":
        True,
}


print("\n" + "=" * 160)
print("TEST-SET FIREWALL")
print("=" * 160)

for rule, allowed in (
    TEST_FIREWALL_RULES.items()
):

    print(
        f"{rule:<32}: "
        f"{'ALLOWED' if allowed else 'FORBIDDEN'}"
    )


# ==============================================================================================================
# 16. ATTACK-TYPE / ENDPOINT GENERALIZATION CONTRACT
# ==============================================================================================================
#
# IMPORTANT ARCHITECTURAL SEPARATION
# ----------------------------------
#
# The primary model is a BINARY multi-horizon forecaster.
#
# Attack type, source, destination/victim-candidate, edge, and typed-edge
# metadata are NOT ordinary predictors.
#
# They are retained for later stratified/generalization analysis.
#
# Cell 15 already froze an out-of-time split in which some attack types
# may be unseen in TRAIN.
#
# Therefore:
#
#   - unseen attack types are NOT removed from binary TEST evaluation
#   - binary forecasting remains evaluated on all valid samples
#   - known/unseen attack-type performance is reported separately later
#
# ==============================================================================================================

GENERALIZATION_CONTRACT = {

    "primary_task":
        "binary_multi_horizon_attack_forecasting",

    "attack_type_used_as_predictor":
        False,

    "source_endpoint_used_as_predictor_target_metadata":
        False,

    "destination_endpoint_used_as_predictor_target_metadata":
        False,

    "unseen_attack_types_removed_from_test":
        False,

    "known_vs_unseen_type_analysis":
        True,

    "endpoint_generalization_analysis":
        True,

    "graph_generalization_analysis":
        True,
}


print("\n" + "=" * 160)
print("GENERALIZATION CONTRACT")
print("=" * 160)

print(
    "Primary task                        : "
    "binary multi-horizon attack forecasting"
)

print(
    "Unseen attack types in TEST         : RETAINED"
)

print(
    "Known/unseen type analysis          : SEPARATE LATER ANALYSIS"
)

print(
    "Endpoint/graph metadata             : NOT ordinary predictors"
)


# ==============================================================================================================
# 17. ONSET / CONTINUATION CONTRACT
# ==============================================================================================================
#
# We previously discovered that naive origin-state definitions can
# conflict with longer-horizon semantics.
#
# Therefore we do NOT recreate New_Onset / Continuation labels here.
#
# Later evaluation must use the already aligned canonical temporal
# metadata / episode structure.
#
# Cell 18 only freezes the rule.
#
# ==============================================================================================================

ONSET_EVALUATION_CONTRACT = {

    "reconstruct_onset_from_origin_state_only":
        False,

    "use_canonical_temporal_alignment":
        True,

    "report_clean_history_forecasting":
        True,

    "report_continuation_forecasting":
        True,

    "report_onset_forecasting":
        True,

    "modify_primary_binary_target":
        False,
}


print("\n" + "=" * 160)
print("ONSET / CONTINUATION EVALUATION CONTRACT")
print("=" * 160)

print(
    "Primary binary targets              : UNCHANGED"
)

print(
    "Naive onset reconstruction          : FORBIDDEN"
)

print(
    "Canonical temporal alignment        : REQUIRED"
)

print(
    "Clean-history/onset analysis        : REPORT SEPARATELY"
)


# ==============================================================================================================
# 18. LEAD-TIME EVALUATION CONTRACT
# ==============================================================================================================
#
# IMPORTANT:
#
# The current labels are POINT forecasts at:
#
#       t+1, t+5, t+10, t+15, t+30
#
# Lead time is NOT identical to ordinary per-horizon classification.
#
# Later, after predictions exist:
#
#   attack episode start = T_attack
#
#   warning origin       = T_warning
#
#   lead time            = T_attack - T_warning
#
# A valid warning must occur BEFORE the episode start and satisfy the
# frozen validation-selected alert threshold/rule.
#
# We do not compute lead time in Cell 18 because no model predictions exist.
#
# ==============================================================================================================

LEAD_TIME_CONTRACT = {

    "primary_kpi":
        "lead_time_minutes",

    "definition":
        "attack_episode_start_minus_first_valid_pre_attack_warning",

    "warning_must_precede_attack_start":
        True,

    "threshold_source":
        "VALIDATION_ONLY",

    "episode_source":
        "canonical_episode_alignment",

    "compute_during_training":
        False,

    "compute_after_predictions":
        True,

    "test_rule_changed_after_test":
        False,
}


print("\n" + "=" * 160)
print("LEAD-TIME EVALUATION CONTRACT")
print("=" * 160)

print(
    "Lead-time KPI                       : "
    "attack start − first valid pre-attack warning"
)

print(
    "Warning must precede attack         : YES"
)

print(
    "Threshold source                    : VALIDATION ONLY"
)

print(
    "Computed in Cell 18                 : NO"
)

print(
    "Computed after model predictions    : YES"
)


# ==============================================================================================================
# 19. FINAL MODEL-DEVELOPMENT CONTRACT
# ==============================================================================================================

cell18_evaluation_contract = {

    "cell_version":
        "CELL18_FINAL_PREMODEL_CONTRACT_V1",

    "history_length_minutes":
        HISTORY_LENGTH,

    "forecast_horizons":
        FORECAST_HORIZONS.copy(),

    "predictor_count":
        N_FEATURES,

    "predictor_columns":
        MODEL_FEATURES_18.copy(),

    "primary_task":
        "binary_multi_horizon_point_forecasting",

    "primary_model_selection_metric":
        PRIMARY_SELECTION_METRIC,

    "primary_per_horizon_metric":
        PRIMARY_PER_HORIZON_METRIC,

    "secondary_probability_metric":
        SECONDARY_PROBABILITY_METRIC,

    "report_metrics":
        REPORT_METRICS.copy(),

    "horizon_selection_weights":
        HORIZON_SELECTION_WEIGHTS.copy(),

    "loss_name":
        LOSS_NAME,

    "loss_weighting":
        LOSS_WEIGHTING,

    "train_pos_weight":
        TRAIN_POS_WEIGHT.copy(),

    "resampling":
        False,

    "threshold_selection_partition":
        "VALIDATION_ONLY",

    "threshold_selection_metric":
        THRESHOLD_SELECTION_METRIC,

    "threshold_grid":
        THRESHOLD_GRID.copy(),

    "per_horizon_thresholds":
        True,

    "test_used_for_model_selection":
        False,

    "test_threshold_tuning":
        False,

    "zero_variance_features":
        ZERO_VARIANCE_FEATURES.copy(),

    "zero_variance_features_removed":
        False,

    "persistence_baseline_available":
        PERSISTENCE_BASELINE_AVAILABLE,

    "known_unseen_attack_analysis":
        True,

    "lead_time_analysis":
        True,

    "model_trained":
        False,
}


# ==============================================================================================================
# 20. HARD SAFETY ASSERTIONS
# ==============================================================================================================

if cell18_evaluation_contract[
    "test_used_for_model_selection"
]:

    raise RuntimeError(
        "TEST leakage detected in evaluation contract."
    )


if cell18_evaluation_contract[
    "test_threshold_tuning"
]:

    raise RuntimeError(
        "TEST threshold tuning must remain forbidden."
    )


if len(
    TRAIN_POS_WEIGHT
) != N_HORIZONS:

    raise RuntimeError(
        "TRAIN pos_weight vector does not match horizon count."
    )


if not np.isfinite(
    TRAIN_POS_WEIGHT
).all():

    raise RuntimeError(
        "TRAIN pos_weight contains non-finite values."
    )


if np.any(
    TRAIN_POS_WEIGHT <= 0
):

    raise RuntimeError(
        "TRAIN pos_weight must be strictly positive."
    )


if not np.isclose(
    HORIZON_SELECTION_WEIGHTS.sum(),
    1.0,
):

    raise RuntimeError(
        "Horizon model-selection weights must sum to 1."
    )


print("\n" + "=" * 160)
print("HARD SAFETY ASSERTIONS")
print("=" * 160)

print(
    "✓ TEST is excluded from model selection."
)

print(
    "✓ TEST is excluded from threshold selection."
)

print(
    "✓ Loss weights are finite and TRAIN-derived."
)

print(
    "✓ Horizon-selection weights sum to 1."
)

print(
    "✓ No frozen feature tensor was modified."
)


# ==============================================================================================================
# 21. PERMANENT CELL 18 OBJECTS
# ==============================================================================================================

CELL18_COMPLETE = True

CELL18_MODEL_TRAINING_ALLOWED_NEXT = True


# ==============================================================================================================
# 22. FINAL SUMMARY
# ==============================================================================================================

print("\n" + "=" * 160)
print("CELL 18 COMPLETE — FINAL PRE-MODEL CONTRACT FROZEN")
print("=" * 160)

print(
    f"""
DATA CONTRACT
-------------
Predictors                         : {N_FEATURES}
History                            : {HISTORY_LENGTH} minutes
Forecast horizons                  : {FORECAST_HORIZONS}

TRAIN samples                      : {len(X_train_18):,}
VALIDATION samples                 : {len(X_validation_18):,}
TEST samples                       : {len(X_test_18):,}

CLASS IMBALANCE
---------------
Loss                               : {LOSS_NAME}
Loss weighting                     : TRAIN-only per-horizon pos_weight
Resampling                         : NONE

MODEL SELECTION
---------------
Primary metric                     : Macro validation PR-AUC
Per-horizon primary metric         : PR-AUC
Secondary metric                   : ROC-AUC
All five horizons weighted equally : YES

THRESHOLDING
------------
Threshold fitting                  : NOT performed yet
Future threshold source            : VALIDATION ONLY
Per-horizon thresholds             : YES
TEST threshold tuning              : FORBIDDEN

TEST
----
Used for architecture selection    : NO
Used for hyperparameter selection  : NO
Used for early stopping            : NO
Used for threshold selection       : NO
Used for final evaluation          : YES

GENERALIZATION
--------------
Unseen attack types retained       : YES
Known/unseen analysis later        : YES
Endpoint/graph analysis later      : YES

OPERATIONAL EVALUATION
----------------------
Onset analysis                     : later, using canonical alignment
Continuation analysis              : later
Lead-time analysis                 : later
Primary lead-time definition       :
    attack episode start
    minus first valid pre-attack warning

MODEL TRAINED IN CELL 18            : NO
"""
)


print("Permanent Cell 18 objects")
print("-" * 120)

print(
    "cell18_feature_variance_audit"
)

print(
    "ZERO_VARIANCE_FEATURES"
)

print(
    "cell18_class_balance"
)

print(
    "TRAIN_POS_WEIGHT"
)

print(
    "cell18_prevalence_shift"
)

print(
    "cell18_reference_baselines"
)

print(
    "cell18_persistence_baseline"
)

print(
    "cell18_horizon_selection_weights"
)

print(
    "cell18_loss_contract"
)

print(
    "cell18_threshold_contract"
)

print(
    "TEST_FIREWALL_RULES"
)

print(
    "GENERALIZATION_CONTRACT"
)

print(
    "ONSET_EVALUATION_CONTRACT"
)

print(
    "LEAD_TIME_CONTRACT"
)

print(
    "cell18_evaluation_contract"
)


print("\nNEXT")
print("-" * 120)

print(
    """
CELL 19 — FIRST LEARNED TEMPORAL BASELINE MODEL

Cell 19 should:
    1. define a deliberately simple learned temporal baseline,
    2. consume X_train / y_train,
    3. use BCEWithLogitsLoss with TRAIN_POS_WEIGHT,
    4. use VALIDATION for early stopping/model selection,
    5. select the best checkpoint by macro validation PR-AUC,
    6. NOT touch TEST for model selection,
    7. save validation probabilities for later threshold selection.

Only after the baseline is established should we compare:
    - LSTM
    - GRU
    - attention-based recurrent models
    - more advanced temporal architectures.
"""
)

print("=" * 160)

CELL 18 — FINAL PRE-MODEL AUDIT + BASELINES + LOSS/EVALUATION CONTRACT

Configuration
------------------------------------------------------------------------------------------------------------------------
History length                    : 15 minutes
Forecast horizons                 : [1, 5, 10, 15, 30]
Targets per sample                : 5
Default probability threshold     : 0.50
Future threshold search grid      : 0.05 → 0.95
Threshold selection source        : VALIDATION ONLY
TEST used for model selection     : NO
Resampling                        : NO
Model trained                     : NO

UPSTREAM CELL 17 CONTRACT VALIDATION
✓ Required Cell 17 objects found.
Predictors                         : 80
TRAIN tensor                       : (1220, 15, 80)
VALIDATION tensor                  : (398, 15, 80)
TEST tensor                        : (396, 15, 80)

✓ Frozen Cell 17 tensors are intact.

TRAIN FEATURE-VARIANCE AUDIT
Total predictors                   : 80
Zero-variance TRAIN p

In [40]:
# ==============================================================================================================
# CELL 19 — FREEZE + EXPORT FINAL FORECASTING DATASET PACKAGE
# ==============================================================================================================
#
# PURPOSE
# -------
# This is the FINAL cell of the dataset-preparation notebook.
#
# It exports the complete frozen forecasting dataset produced by Cells 1–18
# into a versioned Google Drive package that can be loaded by completely
# separate model-training notebooks.
#
# THIS CELL DOES NOT:
#
#   ✗ change the split
#   ✗ change predictors
#   ✗ change targets
#   ✗ refit the scaler
#   ✗ resample
#   ✗ train a model
#   ✗ tune thresholds
#
# THIS CELL DOES:
#
#   ✓ save model-ready TRAIN / VALIDATION / TEST tensors
#   ✓ save aligned sample metadata
#   ✓ save feature order
#   ✓ save the TRAIN-fitted scaler
#   ✓ save split/evaluation contracts
#   ✓ save rich forecast metadata for later evaluation
#   ✓ create SHA-256 checksums
#   ✓ reload exported artifacts
#   ✓ verify exported artifacts against RAM
#   ✓ create a machine-readable manifest
#
# ==============================================================================================================


import os
import json
import hashlib
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import joblib


# ==============================================================================================================
# 1. DATASET VERSION + GOOGLE DRIVE LOCATION
# ==============================================================================================================

DATASET_VERSION = "forecasting_v1"

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Network_Attack_Forecasting"
)

DATASET_ROOT = (
    PROJECT_ROOT
    / "processed"
    / DATASET_VERSION
)


TENSOR_DIR = DATASET_ROOT / "tensors"
METADATA_DIR = DATASET_ROOT / "metadata"
PREPROCESSING_DIR = DATASET_ROOT / "preprocessing"
CONTRACT_DIR = DATASET_ROOT / "contracts"
MANIFEST_DIR = DATASET_ROOT / "manifest"


print("=" * 170)
print(
    "CELL 19 — FREEZE + EXPORT FINAL FORECASTING DATASET PACKAGE"
)
print("=" * 170)

print("\nConfiguration")
print("-" * 120)

print(
    f"Dataset version                   : {DATASET_VERSION}"
)

print(
    f"Project root                      : {PROJECT_ROOT}"
)

print(
    f"Export directory                  : {DATASET_ROOT}"
)

print(
    "Split modification                 : NO"
)

print(
    "Scaling/refitting                  : NO"
)

print(
    "Resampling                         : NO"
)

print(
    "Model training                     : NO"
)


# ==============================================================================================================
# 2. VERIFY GOOGLE DRIVE
# ==============================================================================================================

DRIVE_ROOT = Path(
    "/content/drive/MyDrive"
)

if not DRIVE_ROOT.exists():

    raise RuntimeError(
        "Google Drive does not appear to be mounted.\n\n"
        "Run:\n"
        "from google.colab import drive\n"
        "drive.mount('/content/drive')\n\n"
        "Then rerun Cell 19."
    )


print("\n✓ Google Drive is mounted.")


# ==============================================================================================================
# 3. REQUIRED FROZEN OBJECTS
# ==============================================================================================================

REQUIRED_OBJECTS = [

    # Final model-ready tensors
    "X_train",
    "X_validation",
    "X_test",

    "y_train",
    "y_validation",
    "y_test",

    # Feature contract
    "MODEL_FEATURES",

    # TRAIN-fitted preprocessing
    "feature_scaler",

    # Sample metadata
    "cell17_train_metadata",
    "cell17_validation_metadata",
    "cell17_test_metadata",

    # Split contract
    "cell15_split_metadata",

    # Cell 17 preprocessing contract
    "cell17_metadata",

    # Cell 18 evaluation contract
    "cell18_evaluation_contract",

    # Loss/evaluation objects
    "TRAIN_POS_WEIGHT",
    "cell18_threshold_contract",
]


missing_required_objects = [

    name

    for name in REQUIRED_OBJECTS

    if name not in globals()
]


print("\n" + "=" * 170)
print("UPSTREAM FROZEN-PIPELINE VALIDATION")
print("=" * 170)


if missing_required_objects:

    raise RuntimeError(

        "Cell 19 cannot export the dataset because the following "
        "required frozen objects are missing:\n\n"

        + "\n".join(
            f"  - {name}"
            for name in missing_required_objects
        )

        + "\n\nRun the FINAL Cells 15–18 first."
    )


print(
    "✓ All required frozen objects found."
)


# ==============================================================================================================
# 4. NORMALIZE FINAL REFERENCES
# ==============================================================================================================

X_train_export = np.asarray(
    X_train,
    dtype=np.float32,
)

X_validation_export = np.asarray(
    X_validation,
    dtype=np.float32,
)

X_test_export = np.asarray(
    X_test,
    dtype=np.float32,
)


y_train_export = np.asarray(
    y_train,
    dtype=np.float32,
)

y_validation_export = np.asarray(
    y_validation,
    dtype=np.float32,
)

y_test_export = np.asarray(
    y_test,
    dtype=np.float32,
)


MODEL_FEATURES_EXPORT = list(
    MODEL_FEATURES
)


FORECAST_HORIZONS_EXPORT = list(
    cell18_evaluation_contract[
        "forecast_horizons"
    ]
)


N_FEATURES = len(
    MODEL_FEATURES_EXPORT
)

N_HORIZONS = len(
    FORECAST_HORIZONS_EXPORT
)


print(
    f"Predictors                        : {N_FEATURES}"
)

print(
    f"Forecast horizons                 : {FORECAST_HORIZONS_EXPORT}"
)

print(
    f"TRAIN                             : {X_train_export.shape}"
)

print(
    f"VALIDATION                        : {X_validation_export.shape}"
)

print(
    f"TEST                              : {X_test_export.shape}"
)


# ==============================================================================================================
# 5. HARD PRE-EXPORT INTEGRITY CHECK
# ==============================================================================================================

EXPECTED_HISTORY_LENGTH = int(
    cell18_evaluation_contract[
        "history_length_minutes"
    ]
)


expected_X_shapes = {

    "TRAIN": (
        len(y_train_export),
        EXPECTED_HISTORY_LENGTH,
        N_FEATURES,
    ),

    "VALIDATION": (
        len(y_validation_export),
        EXPECTED_HISTORY_LENGTH,
        N_FEATURES,
    ),

    "TEST": (
        len(y_test_export),
        EXPECTED_HISTORY_LENGTH,
        N_FEATURES,
    ),
}


expected_y_shapes = {

    "TRAIN": (
        len(X_train_export),
        N_HORIZONS,
    ),

    "VALIDATION": (
        len(X_validation_export),
        N_HORIZONS,
    ),

    "TEST": (
        len(X_test_export),
        N_HORIZONS,
    ),
}


for partition, X_arr, y_arr in [

    (
        "TRAIN",
        X_train_export,
        y_train_export,
    ),

    (
        "VALIDATION",
        X_validation_export,
        y_validation_export,
    ),

    (
        "TEST",
        X_test_export,
        y_test_export,
    ),

]:

    if X_arr.shape != expected_X_shapes[
        partition
    ]:

        raise RuntimeError(

            f"{partition} X shape violates frozen contract.\n"
            f"Observed: {X_arr.shape}\n"
            f"Expected: {expected_X_shapes[partition]}"
        )


    if y_arr.shape != expected_y_shapes[
        partition
    ]:

        raise RuntimeError(

            f"{partition} y shape violates frozen contract.\n"
            f"Observed: {y_arr.shape}\n"
            f"Expected: {expected_y_shapes[partition]}"
        )


    if not np.isfinite(
        X_arr
    ).all():

        raise RuntimeError(
            f"{partition} X contains NaN/Inf."
        )


    if not np.isfinite(
        y_arr
    ).all():

        raise RuntimeError(
            f"{partition} y contains NaN/Inf."
        )


    if not np.isin(
        y_arr,
        [0.0, 1.0],
    ).all():

        raise RuntimeError(
            f"{partition} targets are not binary."
        )


metadata_length_checks = {

    "TRAIN":
        len(cell17_train_metadata),

    "VALIDATION":
        len(cell17_validation_metadata),

    "TEST":
        len(cell17_test_metadata),
}


expected_metadata_lengths = {

    "TRAIN":
        len(X_train_export),

    "VALIDATION":
        len(X_validation_export),

    "TEST":
        len(X_test_export),
}


if (
    metadata_length_checks
    != expected_metadata_lengths
):

    raise RuntimeError(

        "Cell 17 sample metadata is not aligned "
        "with final model tensors.\n\n"

        f"Observed : {metadata_length_checks}\n"
        f"Expected : {expected_metadata_lengths}"
    )


if len(
    TRAIN_POS_WEIGHT
) != N_HORIZONS:

    raise RuntimeError(
        "TRAIN_POS_WEIGHT length does not match horizon count."
    )


if not np.isfinite(
    np.asarray(
        TRAIN_POS_WEIGHT,
        dtype=float,
    )
).all():

    raise RuntimeError(
        "TRAIN_POS_WEIGHT contains non-finite values."
    )


print("\n✓ Tensor shapes validated.")
print("✓ Binary targets validated.")
print("✓ Metadata alignment validated.")
print("✓ TRAIN loss weights validated.")
print("✓ No NaN/Inf values detected.")


# ==============================================================================================================
# 6. PREVENT ACCIDENTAL SILENT OVERWRITE
# ==============================================================================================================
#
# If forecasting_v1 already exists, we DO NOT silently merge files into it.
#
# We permit overwrite only when the existing directory is clearly an earlier
# Cell-19 export of the SAME dataset version.
#
# A backup is created before replacement.
#
# ==============================================================================================================

if DATASET_ROOT.exists():

    existing_manifest = (
        DATASET_ROOT
        / "manifest"
        / "dataset_manifest.json"
    )


    if existing_manifest.exists():

        with open(
            existing_manifest,
            "r",
            encoding="utf-8",
        ) as f:

            old_manifest = json.load(
                f
            )


        old_version = old_manifest.get(
            "dataset_version"
        )


        if old_version != DATASET_VERSION:

            raise RuntimeError(

                "Existing export directory contains a different "
                "dataset version.\n\n"

                f"Directory : {DATASET_ROOT}\n"
                f"Expected  : {DATASET_VERSION}\n"
                f"Found     : {old_version}"
            )


        backup_root = (
            PROJECT_ROOT
            / "processed"
            / f"{DATASET_VERSION}_previous_export"
        )


        if backup_root.exists():

            shutil.rmtree(
                backup_root
            )


        shutil.copytree(
            DATASET_ROOT,
            backup_root,
        )


        print(
            "\nExisting forecasting_v1 export found."
        )

        print(
            f"✓ Previous export backed up to:\n"
            f"  {backup_root}"
        )


    else:

        raise RuntimeError(

            f"Directory already exists but has no valid manifest:\n"
            f"{DATASET_ROOT}\n\n"

            "Cell 19 will not overwrite an unidentified directory."
        )


# ==============================================================================================================
# 7. CREATE CLEAN VERSIONED EXPORT DIRECTORY
# ==============================================================================================================

if DATASET_ROOT.exists():

    shutil.rmtree(
        DATASET_ROOT
    )


for directory in [

    TENSOR_DIR,
    METADATA_DIR,
    PREPROCESSING_DIR,
    CONTRACT_DIR,
    MANIFEST_DIR,

]:

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


print("\n" + "=" * 170)
print("EXPORT DIRECTORY CREATED")
print("=" * 170)

print(
    DATASET_ROOT
)


# ==============================================================================================================
# 8. EXPORT MODEL-READY TENSORS
# ==============================================================================================================

tensor_exports = {

    "X_train.npy":
        X_train_export,

    "X_validation.npy":
        X_validation_export,

    "X_test.npy":
        X_test_export,

    "y_train.npy":
        y_train_export,

    "y_validation.npy":
        y_validation_export,

    "y_test.npy":
        y_test_export,
}


for filename, array in (
    tensor_exports.items()
):

    np.save(
        TENSOR_DIR / filename,
        array,
        allow_pickle=False,
    )


print("\n" + "=" * 170)
print("MODEL-READY TENSORS EXPORTED")
print("=" * 170)

for filename, array in (
    tensor_exports.items()
):

    print(
        f"✓ {filename:<28} {str(array.shape):<20} {array.dtype}"
    )


# ==============================================================================================================
# 9. EXPORT SAMPLE METADATA
# ==============================================================================================================

metadata_exports = {

    "train_sample_metadata.csv":
        cell17_train_metadata,

    "validation_sample_metadata.csv":
        cell17_validation_metadata,

    "test_sample_metadata.csv":
        cell17_test_metadata,
}


for filename, frame in (
    metadata_exports.items()
):

    frame.to_csv(
        METADATA_DIR / filename,
        index=False,
    )


print("\n" + "=" * 170)
print("ALIGNED SAMPLE METADATA EXPORTED")
print("=" * 170)

for filename, frame in (
    metadata_exports.items()
):

    print(
        f"✓ {filename:<40} rows={len(frame):,}"
    )


# ==============================================================================================================
# 10. EXPORT RICH CANONICAL FORECAST METADATA
# ==============================================================================================================
#
# These objects are NOT ordinary model predictors.
#
# They are retained because later evaluation needs:
#
#   - attack-type analysis
#   - known/unseen attack analysis
#   - clean-history/onset analysis
#   - source/destination analysis
#   - graph/edge analysis
#   - episode/lead-time evaluation
#
# We export them separately from model tensors to prevent accidental leakage.
#
# ==============================================================================================================

OPTIONAL_RICH_METADATA = {

    "cell15_forecast_metadata.csv":
        "cell15_forecast_metadata",

    "cell12_forecast_metadata.csv":
        "cell12_forecast_metadata",

    "cell16_sequence_metadata.csv":
        "cell16_sequence_metadata",

    "cell15_evaluation_attack_metadata.csv":
        "cell15_evaluation_attack_metadata",

    "cell14a_forecast_episode_map.csv":
        "cell14a_forecast_episode_map",
}


rich_metadata_exported = []


for filename, object_name in (
    OPTIONAL_RICH_METADATA.items()
):

    if object_name not in globals():

        continue


    obj = globals()[
        object_name
    ]


    if not isinstance(
        obj,
        pd.DataFrame,
    ):

        continue


    obj.to_csv(
        METADATA_DIR / filename,
        index=False,
    )


    rich_metadata_exported.append(
        filename
    )


print("\n" + "=" * 170)
print("RICH EVALUATION METADATA")
print("=" * 170)


if rich_metadata_exported:

    for filename in (
        rich_metadata_exported
    ):

        print(
            f"✓ {filename}"
        )

else:

    print(
        "NOTE: No optional rich metadata objects were available."
    )


# ==============================================================================================================
# 11. EXPORT FEATURE ORDER
# ==============================================================================================================

feature_payload = {

    "dataset_version":
        DATASET_VERSION,

    "feature_count":
        N_FEATURES,

    "history_length_minutes":
        EXPECTED_HISTORY_LENGTH,

    "feature_order":
        MODEL_FEATURES_EXPORT,
}


with open(
    CONTRACT_DIR / "feature_contract.json",
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        feature_payload,
        f,
        indent=2,
        ensure_ascii=False,
    )


print("\n✓ Exact feature order exported.")


# ==============================================================================================================
# 12. EXPORT TRAIN-FITTED PREPROCESSING OBJECTS
# ==============================================================================================================

joblib.dump(
    feature_scaler,
    PREPROCESSING_DIR / "feature_scaler.joblib",
)


# Export imputer only if Cell 17 created one.
#
# Cell 17 may have determined that imputation was unnecessary.
# We preserve the object if present rather than inventing one.

IMPUTER_EXPORTED = False


if (
    "feature_imputer" in globals()
    and feature_imputer is not None
):

    joblib.dump(
        feature_imputer,
        PREPROCESSING_DIR / "feature_imputer.joblib",
    )

    IMPUTER_EXPORTED = True


print("\n" + "=" * 170)
print("PREPROCESSING ARTIFACTS EXPORTED")
print("=" * 170)

print(
    "✓ feature_scaler.joblib"
)

print(
    f"{'✓' if IMPUTER_EXPORTED else '—'} "
    f"feature_imputer.joblib "
    f"{'(saved)' if IMPUTER_EXPORTED else '(not required / not available)'}"
)


# ==============================================================================================================
# 13. JSON SERIALIZATION HELPER
# ==============================================================================================================

def make_json_safe(value):

    if isinstance(
        value,
        dict,
    ):

        return {
            str(k):
                make_json_safe(v)

            for k, v in value.items()
        }


    if isinstance(
        value,
        (list, tuple, set),
    ):

        return [
            make_json_safe(v)
            for v in value
        ]


    if isinstance(
        value,
        np.ndarray,
    ):

        return (
            value
            .tolist()
        )


    if isinstance(
        value,
        np.integer,
    ):

        return int(
            value
        )


    if isinstance(
        value,
        np.floating,
    ):

        value = float(
            value
        )

        if np.isnan(value):

            return None

        if np.isposinf(value):

            return "Infinity"

        if np.isneginf(value):

            return "-Infinity"

        return value


    if isinstance(
        value,
        np.bool_,
    ):

        return bool(
            value
        )


    if isinstance(
        value,
        pd.Timestamp,
    ):

        return value.isoformat()


    if isinstance(
        value,
        Path,
    ):

        return str(
            value
        )


    if isinstance(
        value,
        float,
    ):

        if np.isnan(value):

            return None

        if np.isposinf(value):

            return "Infinity"

        if np.isneginf(value):

            return "-Infinity"


    return value


# ==============================================================================================================
# 14. EXPORT SPLIT / PREPROCESSING / EVALUATION CONTRACTS
# ==============================================================================================================

contract_exports = {

    "split_contract.json":
        cell15_split_metadata,

    "preprocessing_contract.json":
        cell17_metadata,

    "evaluation_contract.json":
        cell18_evaluation_contract,

    "threshold_contract.json":
        cell18_threshold_contract,
}


OPTIONAL_CONTRACTS = {

    "loss_contract":
        "cell18_loss_contract",

    "test_firewall_rules":
        "TEST_FIREWALL_RULES",

    "generalization_contract":
        "GENERALIZATION_CONTRACT",

    "onset_evaluation_contract":
        "ONSET_EVALUATION_CONTRACT",

    "lead_time_contract":
        "LEAD_TIME_CONTRACT",
}


for key, object_name in (
    OPTIONAL_CONTRACTS.items()
):

    if object_name in globals():

        contract_exports[
            f"{key}.json"
        ] = globals()[
            object_name
        ]


for filename, contract in (
    contract_exports.items()
):

    # DataFrames are exported as records.
    if isinstance(
        contract,
        pd.DataFrame,
    ):

        payload = (
            contract
            .to_dict(
                orient="records"
            )
        )

    else:

        payload = contract


    with open(
        CONTRACT_DIR / filename,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            make_json_safe(
                payload
            ),
            f,
            indent=2,
            ensure_ascii=False,
        )


print("\n" + "=" * 170)
print("PIPELINE CONTRACTS EXPORTED")
print("=" * 170)

for filename in contract_exports:

    print(
        f"✓ {filename}"
    )


# ==============================================================================================================
# 15. EXPORT TRAIN POSITIVE-CLASS WEIGHTS
# ==============================================================================================================

loss_weight_payload = {

    "forecast_horizons":
        FORECAST_HORIZONS_EXPORT,

    "train_pos_weight":
        np.asarray(
            TRAIN_POS_WEIGHT,
            dtype=float,
        ).tolist(),

    "source":
        "TRAIN_ONLY",

    "resampling":
        False,
}


with open(
    CONTRACT_DIR / "train_loss_weights.json",
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        loss_weight_payload,
        f,
        indent=2,
    )


print(
    "\n✓ TRAIN-only loss weights exported."
)


# ==============================================================================================================
# 16. OPTIONAL SUPPORT TABLE EXPORTS
# ==============================================================================================================

OPTIONAL_TABLE_EXPORTS = {

    "cell18_class_balance.csv":
        "cell18_class_balance",

    "cell18_prevalence_shift.csv":
        "cell18_prevalence_shift",

    "cell18_reference_baselines.csv":
        "cell18_reference_baselines",

    "cell18_feature_variance_audit.csv":
        "cell18_feature_variance_audit",

    "cell18_horizon_selection_weights.csv":
        "cell18_horizon_selection_weights",

    "cell16_feature_contract.csv":
        "cell16_feature_contract",

    "cell16_feature_quality.csv":
        "cell16_feature_quality",

    "cell16_target_distribution.csv":
        "cell16_target_distribution",
}


support_tables_exported = []


for filename, object_name in (
    OPTIONAL_TABLE_EXPORTS.items()
):

    if object_name not in globals():

        continue


    obj = globals()[
        object_name
    ]


    if not isinstance(
        obj,
        pd.DataFrame,
    ):

        continue


    obj.to_csv(
        METADATA_DIR / filename,
        index=False,
    )


    support_tables_exported.append(
        filename
    )


print("\n" + "=" * 170)
print("DIAGNOSTIC / SUPPORT TABLES")
print("=" * 170)

for filename in support_tables_exported:

    print(
        f"✓ {filename}"
    )


# ==============================================================================================================
# 17. SHA-256 HELPER
# ==============================================================================================================

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):

    digest = hashlib.sha256()


    with open(
        path,
        "rb",
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )


            if not chunk:

                break


            digest.update(
                chunk
            )


    return digest.hexdigest()


# ==============================================================================================================
# 18. BUILD PRELIMINARY MANIFEST
# ==============================================================================================================

manifest = {

    "dataset_version":
        DATASET_VERSION,

    "dataset_status":
        "FROZEN",

    "task":
        "binary_multi_horizon_network_attack_forecasting",

    "temporal_resolution":
        "1_minute",

    "history_length_minutes":
        EXPECTED_HISTORY_LENGTH,

    "forecast_semantics":
        "POINT_TARGET_AT_EXACTLY_T_PLUS_H",

    "forecast_horizons_minutes":
        FORECAST_HORIZONS_EXPORT,

    "feature_count":
        N_FEATURES,

    "feature_dtype":
        str(
            X_train_export.dtype
        ),

    "target_dtype":
        str(
            y_train_export.dtype
        ),

    "splits": {

        "TRAIN": {

            "samples":
                int(
                    len(
                        X_train_export
                    )
                ),

            "X_shape":
                list(
                    X_train_export.shape
                ),

            "y_shape":
                list(
                    y_train_export.shape
                ),
        },

        "VALIDATION": {

            "samples":
                int(
                    len(
                        X_validation_export
                    )
                ),

            "X_shape":
                list(
                    X_validation_export.shape
                ),

            "y_shape":
                list(
                    y_validation_export.shape
                ),
        },

        "TEST": {

            "samples":
                int(
                    len(
                        X_test_export
                    )
                ),

            "X_shape":
                list(
                    X_test_export.shape
                ),

            "y_shape":
                list(
                    y_test_export.shape
                ),
        },
    },

    "preprocessing": {

        "scaler":
            "StandardScaler",

        "scaler_fit_partition":
            "TRAIN_ONLY",

        "imputer_exported":
            IMPUTER_EXPORTED,

        "resampling":
            False,
    },

    "evaluation": {

        "primary_model_selection_metric":
            cell18_evaluation_contract.get(
                "primary_model_selection_metric"
            ),

        "primary_per_horizon_metric":
            cell18_evaluation_contract.get(
                "primary_per_horizon_metric"
            ),

        "threshold_selection_partition":
            cell18_evaluation_contract.get(
                "threshold_selection_partition"
            ),

        "test_used_for_model_selection":
            False,

        "test_threshold_tuning":
            False,
    },

    "artifact_structure": {

        "tensors":
            "tensors/",

        "metadata":
            "metadata/",

        "preprocessing":
            "preprocessing/",

        "contracts":
            "contracts/",

        "manifest":
            "manifest/",
    },

    "rich_metadata_exports":
        rich_metadata_exported,

    "support_table_exports":
        support_tables_exported,
}


# ==============================================================================================================
# 19. SAVE PRELIMINARY MANIFEST
# ==============================================================================================================

manifest_path = (
    MANIFEST_DIR
    / "dataset_manifest.json"
)


with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        make_json_safe(
            manifest
        ),
        f,
        indent=2,
        ensure_ascii=False,
    )


# ==============================================================================================================
# 20. GENERATE CHECKSUMS FOR ALL EXPORTED FILES
# ==============================================================================================================
#
# Exclude checksums.json itself because otherwise its own checksum would
# recursively change the file.
#
# ==============================================================================================================

checksum_records = {}


for path in sorted(
    DATASET_ROOT.rglob("*")
):

    if not path.is_file():

        continue


    if path.name == "checksums.json":

        continue


    relative_path = str(
        path.relative_to(
            DATASET_ROOT
        )
    )


    checksum_records[
        relative_path
    ] = {

        "sha256":
            sha256_file(
                path
            ),

        "size_bytes":
            int(
                path.stat().st_size
            ),
    }


checksum_path = (
    MANIFEST_DIR
    / "checksums.json"
)


with open(
    checksum_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        checksum_records,
        f,
        indent=2,
    )


print("\n" + "=" * 170)
print("SHA-256 INTEGRITY MANIFEST CREATED")
print("=" * 170)

print(
    f"Files checksummed                  : "
    f"{len(checksum_records)}"
)


# ==============================================================================================================
# 21. RELOAD SAVED TENSORS FROM GOOGLE DRIVE
# ==============================================================================================================
#
# This verifies that what exists on disk is exactly what existed in RAM.
#
# ==============================================================================================================

reloaded_arrays = {

    "X_train":
        np.load(
            TENSOR_DIR / "X_train.npy",
            allow_pickle=False,
        ),

    "X_validation":
        np.load(
            TENSOR_DIR / "X_validation.npy",
            allow_pickle=False,
        ),

    "X_test":
        np.load(
            TENSOR_DIR / "X_test.npy",
            allow_pickle=False,
        ),

    "y_train":
        np.load(
            TENSOR_DIR / "y_train.npy",
            allow_pickle=False,
        ),

    "y_validation":
        np.load(
            TENSOR_DIR / "y_validation.npy",
            allow_pickle=False,
        ),

    "y_test":
        np.load(
            TENSOR_DIR / "y_test.npy",
            allow_pickle=False,
        ),
}


original_arrays = {

    "X_train":
        X_train_export,

    "X_validation":
        X_validation_export,

    "X_test":
        X_test_export,

    "y_train":
        y_train_export,

    "y_validation":
        y_validation_export,

    "y_test":
        y_test_export,
}


reload_validation_rows = []


for name in original_arrays:

    original = original_arrays[
        name
    ]

    reloaded = reloaded_arrays[
        name
    ]


    shape_match = (
        original.shape
        == reloaded.shape
    )


    dtype_match = (
        original.dtype
        == reloaded.dtype
    )


    exact_match = np.array_equal(
        original,
        reloaded,
        equal_nan=True,
    )


    reload_validation_rows.append({

        "Artifact":
            name,

        "Shape":
            str(
                reloaded.shape
            ),

        "DType":
            str(
                reloaded.dtype
            ),

        "Shape_Match":
            shape_match,

        "DType_Match":
            dtype_match,

        "Exact_Value_Match":
            exact_match,
    })


cell19_reload_validation = pd.DataFrame(
    reload_validation_rows
)


print("\n" + "=" * 170)
print("DISK RELOAD VALIDATION")
print("=" * 170)

print(
    cell19_reload_validation
    .to_string(
        index=False
    )
)


if not (
    cell19_reload_validation[
        [
            "Shape_Match",
            "DType_Match",
            "Exact_Value_Match",
        ]
    ]
    .all()
    .all()
):

    raise RuntimeError(

        "At least one exported tensor differs from "
        "the frozen in-memory tensor."
    )


print(
    "\n✓ All six tensors reload exactly."
)


# ==============================================================================================================
# 22. RELOAD + VERIFY FEATURE CONTRACT
# ==============================================================================================================

with open(
    CONTRACT_DIR / "feature_contract.json",
    "r",
    encoding="utf-8",
) as f:

    reloaded_feature_contract = json.load(
        f
    )


if (
    reloaded_feature_contract[
        "feature_order"
    ]
    != MODEL_FEATURES_EXPORT
):

    raise RuntimeError(
        "Reloaded feature order does not match frozen feature order."
    )


if int(
    reloaded_feature_contract[
        "feature_count"
    ]
) != N_FEATURES:

    raise RuntimeError(
        "Reloaded feature count does not match frozen feature count."
    )


print(
    "✓ Feature order reload verified."
)


# ==============================================================================================================
# 23. RELOAD + VERIFY SCALER
# ==============================================================================================================

reloaded_scaler = joblib.load(
    PREPROCESSING_DIR / "feature_scaler.joblib"
)


for attribute in [
    "mean_",
    "scale_",
]:

    if not hasattr(
        feature_scaler,
        attribute,
    ):

        continue


    if not hasattr(
        reloaded_scaler,
        attribute,
    ):

        raise RuntimeError(

            f"Reloaded scaler is missing {attribute}."
        )


    original_values = np.asarray(
        getattr(
            feature_scaler,
            attribute,
        )
    )


    reloaded_values = np.asarray(
        getattr(
            reloaded_scaler,
            attribute,
        )
    )


    if not np.array_equal(
        original_values,
        reloaded_values,
        equal_nan=True,
    ):

        raise RuntimeError(

            f"Reloaded scaler {attribute} differs "
            "from frozen scaler."
        )


print(
    "✓ TRAIN-fitted scaler reload verified."
)


# ==============================================================================================================
# 24. VERIFY CHECKSUM FILE AGAINST CURRENT ARTIFACTS
# ==============================================================================================================

with open(
    checksum_path,
    "r",
    encoding="utf-8",
) as f:

    stored_checksums = json.load(
        f
    )


checksum_errors = []


for relative_path, record in (
    stored_checksums.items()
):

    absolute_path = (
        DATASET_ROOT
        / relative_path
    )


    if not absolute_path.exists():

        checksum_errors.append(

            (
                relative_path,
                "MISSING",
            )
        )

        continue


    current_hash = sha256_file(
        absolute_path
    )


    if current_hash != record[
        "sha256"
    ]:

        checksum_errors.append(

            (
                relative_path,
                "HASH_MISMATCH",
            )
        )


if checksum_errors:

    raise RuntimeError(

        "Checksum verification failed:\n"

        + "\n".join(
            str(x)
            for x in checksum_errors
        )
    )


print(
    "✓ SHA-256 checksum verification passed."
)


# ==============================================================================================================
# 25. EXPORT HUMAN-READABLE README
# ==============================================================================================================

dataset_readme = f"""
FORECASTING DATASET PACKAGE
===========================

Version
-------
{DATASET_VERSION}

Task
----
Binary multi-horizon network attack forecasting.

Input
-----
15 consecutive 1-minute observations.

Each observation contains:
{N_FEATURES} frozen numeric predictors.

Input tensor shape
------------------
(samples, 15, {N_FEATURES})

Targets
-------
Binary point forecasts at:

{FORECAST_HORIZONS_EXPORT}

Meaning:

y[:, 0] = attack state exactly t + 1 minute
y[:, 1] = attack state exactly t + 5 minutes
y[:, 2] = attack state exactly t + 10 minutes
y[:, 3] = attack state exactly t + 15 minutes
y[:, 4] = attack state exactly t + 30 minutes

Final partitions
----------------
TRAIN      : {X_train_export.shape}
VALIDATION : {X_validation_export.shape}
TEST       : {X_test_export.shape}

Important leakage rules
-----------------------
1. The scaler was fitted on TRAIN only.
2. VALIDATION and TEST were never used to fit preprocessing.
3. No resampling was performed.
4. TEST must not be used for:
   - architecture selection
   - hyperparameter selection
   - epoch selection
   - early stopping
   - threshold selection
   - checkpoint selection

5. Validation is used for model selection.
6. Thresholds are selected from validation only.
7. TEST is reserved for final evaluation.

Model-development notebooks
---------------------------
Ordinary model-development notebooks should load:

    X_train.npy
    y_train.npy
    X_validation.npy
    y_validation.npy

They should NOT load TEST during architecture development.

Final-evaluation notebook
-------------------------
The final evaluation notebook may load:

    X_test.npy
    y_test.npy

only after the architecture, checkpoint strategy, and
validation-derived thresholds have been frozen.

Rich metadata
-------------
Files under metadata/ are evaluation metadata.

They must NOT be silently inserted into the model predictor tensor.

They are preserved for later analyses including:

    - attack-type generalization
    - known vs unseen attack types
    - onset / continuation
    - source/destination analysis
    - graph analysis
    - episode-level warning analysis
    - lead-time evaluation

Integrity
---------
manifest/checksums.json contains SHA-256 checksums for the exported artifacts.

Do not manually edit files inside this frozen dataset package.

If the dataset pipeline changes, create a NEW dataset version
rather than silently changing forecasting_v1.
""".strip()


with open(
    DATASET_ROOT / "README_DATASET.txt",
    "w",
    encoding="utf-8",
) as f:

    f.write(
        dataset_readme
    )


# README was created after the earlier checksum pass.
# Add it to the checksum manifest.

readme_relative_path = str(
    (
        DATASET_ROOT
        / "README_DATASET.txt"
    )
    .relative_to(
        DATASET_ROOT
    )
)


stored_checksums[
    readme_relative_path
] = {

    "sha256":
        sha256_file(
            DATASET_ROOT
            / "README_DATASET.txt"
        ),

    "size_bytes":
        int(
            (
                DATASET_ROOT
                / "README_DATASET.txt"
            )
            .stat()
            .st_size
        ),
}


with open(
    checksum_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        stored_checksums,
        f,
        indent=2,
    )


print(
    "✓ Human-readable dataset README created."
)


# ==============================================================================================================
# 26. FINAL DIRECTORY INVENTORY
# ==============================================================================================================

inventory_rows = []


for path in sorted(
    DATASET_ROOT.rglob("*")
):

    if not path.is_file():

        continue


    inventory_rows.append({

        "File":
            str(
                path.relative_to(
                    DATASET_ROOT
                )
            ),

        "Size_MB":
            path.stat().st_size
            / (1024 ** 2),
    })


cell19_export_inventory = pd.DataFrame(
    inventory_rows
)


print("\n" + "=" * 170)
print("FINAL EXPORTED DATASET INVENTORY")
print("=" * 170)

print(

    cell19_export_inventory
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)


total_size_mb = (

    cell19_export_inventory[
        "Size_MB"
    ].sum()
)


print(
    f"\nTotal frozen package size         : "
    f"{total_size_mb:.2f} MB"
)


# ==============================================================================================================
# 27. FINAL DATASET FREEZE FLAGS
# ==============================================================================================================

CELL19_EXPORT_COMPLETE = True

FORECASTING_DATASET_VERSION = (
    DATASET_VERSION
)

FORECASTING_DATASET_PATH = str(
    DATASET_ROOT
)


# ==============================================================================================================
# 28. FINAL SUMMARY
# ==============================================================================================================

print("\n" + "=" * 170)
print("FORECASTING DATASET v1 SUCCESSFULLY FROZEN")
print("=" * 170)

print(
    f"""
DATASET
-------
Version                            : {DATASET_VERSION}
Location                           : {DATASET_ROOT}

TEMPORAL CONTRACT
-----------------
Resolution                         : 1 minute
History                            : {EXPECTED_HISTORY_LENGTH} minutes
Forecast horizons                  : {FORECAST_HORIZONS_EXPORT}
Forecast semantics                 : POINT target exactly at t + H

FEATURES
--------
Final predictors                   : {N_FEATURES}
Feature order exported             : YES
Scaler exported                    : YES
Scaler fit source                  : TRAIN ONLY

MODEL-READY DATA
----------------
TRAIN                              : {X_train_export.shape}
VALIDATION                         : {X_validation_export.shape}
TEST                               : {X_test_export.shape}

TARGETS
-------
TRAIN                              : {y_train_export.shape}
VALIDATION                         : {y_validation_export.shape}
TEST                               : {y_test_export.shape}

SAFETY
------
Split changed                      : NO
Features changed                   : NO
Targets changed                    : NO
Scaler refitted                    : NO
Resampling                         : NO
Model trained                      : NO

REPRODUCIBILITY
---------------
Dataset manifest                   : YES
Feature contract                   : YES
Split contract                     : YES
Preprocessing contract             : YES
Evaluation contract                : YES
Threshold contract                 : YES
TRAIN loss weights                 : YES
SHA-256 checksums                  : YES
Disk reload verification           : PASSED

STATUS
------
DATASET PREPARATION                : COMPLETE
DATASET VERSION                    : FROZEN
MODEL DEVELOPMENT                  : READY
"""
)


print("NEXT NOTEBOOK")
print("-" * 120)

print(
    """
02_mlp_baseline.ipynb

That notebook should load ONLY:

    TRAIN
    VALIDATION
    feature contract
    evaluation contract
    TRAIN loss weights

during model development.

TEST remains locked until final evaluation.
"""
)

print("=" * 170)

CELL 19 — FREEZE + EXPORT FINAL FORECASTING DATASET PACKAGE

Configuration
------------------------------------------------------------------------------------------------------------------------
Dataset version                   : forecasting_v1
Project root                      : /content/drive/MyDrive/Network_Attack_Forecasting
Export directory                  : /content/drive/MyDrive/Network_Attack_Forecasting/processed/forecasting_v1
Split modification                 : NO
Scaling/refitting                  : NO
Resampling                         : NO
Model training                     : NO

✓ Google Drive is mounted.

UPSTREAM FROZEN-PIPELINE VALIDATION
✓ All required frozen objects found.
Predictors                        : 80
Forecast horizons                 : [1, 5, 10, 15, 30]
TRAIN                             : (1220, 15, 80)
VALIDATION                        : (398, 15, 80)
TEST                              : (396, 15, 80)

✓ Tensor shapes validated.
✓ Binary targets vali